<a href="https://colab.research.google.com/github/wibowoanom/AladdinFoundry/blob/main/3_Anthropic_AladdinFoundry_AF_G.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# LANGKAH J1 (revisi) — Mount + Cek File + Tulis Tahap2_Replikasi.py
# ============================================================

# ---------- 1) MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- 2) CEK FILE YANG DITUJU ----------
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

print("=" * 90)
print("  CEK FILE YANG DITUJU")
print("=" * 90)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("data_sealed.pkl",     f"{BASE}/data_sealed.pkl"),
    ("data_events.csv",     f"{BASE}/data_events.csv"),
    ("seal_manifest.json",  f"{BASE}/seal_manifest.json"),
    ("params_frozen.json",  f"{BASE}/params_frozen.json"),
    ("run_log.csv",         f"{BASE}/run_log.csv"),
    ("LAPORAN_Tahap0.md",   f"{BASE}/LAPORAN_Tahap0.md"),
    ("LAPORAN_Tahap1.md",   f"{BASE}/LAPORAN_Tahap1.md"),
    ("LAPORAN_Tahap1b.md",  f"{BASE}/LAPORAN_Tahap1b.md"),
    ("4. Anthropi_AF_G_Evo4.py", f"{BASE}/4. Anthropi_AF_G_Evo4.py"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:28s} {size:>9} byte")
print("=" * 90)

# ---------- 3) TULIS SCRIPT ----------
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

SCRIPT_CONTENT = r'''# ============================================================
# Tahap2_Replikasi.py
# KOMPAS TAHAP 2 — Data Lama dan Replikasi BRK
# ============================================================
# Urutan: Langkah 0 (reproduksi) -> Bagian A (data lama) -> Gerbang A -> Bagian B
#
# ATURAN:
#   - Bukan codex lama (Step25/Evo4). Ini kode Tahap 1/1b yang ditulis ulang.
#   - Parameter codex lama TIDAK dipakai (ATR trailing, MAX_HOLDING_DAYS,
#     THRESHOLD_*, BI_RATE, MACRO_FILTER).
#   - data_sealed.pkl dan VALIDATE tidak disentuh/ditimpa.
#   - Tidak ada stop loss / take profit.
#   - N = 5 dan 10. Entry Open(T+1), exit Close(T+N).
#   - Toleransi Langkah 0: N trade <= 1%, excess +-0.02 pp.
# ============================================================

import os, sys, csv, json, random, hashlib, time
from datetime import datetime
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"[WARN] Mount Drive: {e}")

# ---------------- KONSTANTA ----------------
BASE            = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH       = f"{BASE}/data_sealed.pkl"
MANIFEST_PATH   = f"{BASE}/seal_manifest.json"
OLD_SEAL_PATH   = f"{BASE}/data_old_sealed.pkl"
OLD_MANIFEST    = f"{BASE}/data_old_manifest.json"
OLD_EVENTS_PATH = f"{BASE}/data_events_old.csv"
RUN_LOG_PATH    = f"{BASE}/run_log.csv"
REPORT_PATH     = f"{BASE}/LAPORAN_Tahap2.md"

CUTOFF_NEW      = pd.Timestamp("2026-06-07")
CUTOFF_OLD_END  = pd.Timestamp("2023-10-09")

MIN_PRICE         = 200.0
MIN_TURNOVER_MA20 = 1_000_000_000.0
MIN_ELIG_PER_DAY  = 60
WARMUP_BARS       = 25
WINDOW_BACK       = 3
WINDOW_FWD        = 25

COST_BASE_SIDE    = 0.0015   # 0.3% round-trip
COST_STRESS_SIDE  = 0.0030   # 0.6% round-trip
N_LIST            = [5, 10]

# Periode Tahap 1b (untuk Langkah 0)
tA_start = pd.Timestamp("2023-10-10"); tA_end = pd.Timestamp("2025-05-15")
tB_start = pd.Timestamp("2025-05-16"); tB_end = pd.Timestamp("2026-06-05")

random.seed(42); np.random.seed(42)

def log(msg=""):
    print(msg, flush=True)

log("=" * 90)
log("  KOMPAS TAHAP 2 — Tahap2_Replikasi.py")
log("=" * 90)

# ============================================================
# FUNGSI UMUM
# ============================================================
def rsi_wilder(close, window=14):
    delta = close.diff()
    gain  = delta.clip(lower=0); loss = -delta.clip(upper=0)
    ag = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    al = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    rs = ag / al.replace(0, np.nan)
    return 100 - 100/(1+rs)

def build_features(op, hi, lo, cl, vo, eligible):
    roc3 = cl / cl.shift(3) - 1.0
    high_prev10 = hi.shift(1).rolling(10, min_periods=10).max()
    brk = cl > high_prev10
    rsi14 = rsi_wilder(cl, 14)
    rev = (rsi14.shift(1) < 30) & (cl > cl.shift(1))
    vol_ma20_prev = vo.shift(1).rolling(20, min_periods=20).mean()
    gap = op / cl.shift(1) - 1.0
    gapvol = (gap > 0.01) & (vo > 1.5 * vol_ma20_prev)
    roc3_rank = roc3.where(eligible).rank(axis=1, pct=True)
    mom_sig = (roc3_rank >= 0.80) & roc3.notna() & eligible
    brk_sig = brk & eligible
    rev_sig = rev & eligible
    gapvol_sig = gapvol & eligible
    return {"MOM": mom_sig, "BRK": brk_sig, "REV": rev_sig, "GAPVOL": gapvol_sig}

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def fix_p1b(p, exc):
    if pd.isna(p): return float("nan")
    if exc > 0:   return p / 2
    if exc < 0:   return 1 - p / 2
    return 0.5

def run_8_uji(idx, op, hi, lo, cl, vo, eligible_base, tC1_end, tC2_start,
              label1="C1", label2="C2"):
    signals = build_features(op, hi, lo, cl, vo, eligible_base)

    def trade_return(tkr, t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
        if pd.isna(o) or pd.isna(c): return None
        entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
        return (exit_ - entry) / entry

    def benchmark_avg(t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        elig = eligible_base.columns[eligible_base.loc[t]]
        rets = []
        for tkr in elig:
            o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
            if pd.isna(o) or pd.isna(c): continue
            entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
            rets.append((exit_ - entry) / entry)
        return float(np.mean(rets)) if rets else None

    results = {}
    total_sig = 0; total_drop = 0
    for sig_name, sig_df in signals.items():
        for N in N_LIST:
            for period, t1, t2 in [(label1, None, tC1_end), (label2, tC2_start, None)]:
                mask_t = (sig_df.index <= t2) if t2 is not None else (sig_df.index >= t1)
                sigs = []
                for t in sig_df.index:
                    if not mask_t.loc[t]: continue
                    for tkr in sig_df.columns:
                        if bool(sig_df.at[t, tkr]): sigs.append((tkr, t))
                total_sig += len(sigs)

                raw_b, raw_s, exc_raw, dates = [], [], [], []
                n_drop = 0
                for tkr, t in sigs:
                    rb = trade_return(tkr, t, N, COST_BASE_SIDE)
                    rs = trade_return(tkr, t, N, COST_STRESS_SIDE)
                    if rb is None or rs is None: n_drop += 1; continue
                    bm = benchmark_avg(t, N, COST_BASE_SIDE)
                    if bm is None: n_drop += 1; continue
                    raw_b.append(rb); raw_s.append(rs); exc_raw.append(rb - bm); dates.append(t)
                total_drop += n_drop
                raw_b = np.array(raw_b); raw_s = np.array(raw_s); exc_raw = np.array(exc_raw)
                exc_w = winsorize(exc_raw); raw_s_w = winsorize(raw_s)
                daily = {}
                for t, e in zip(dates, exc_w): daily.setdefault(t, []).append(e)
                ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())]) if daily else np.array([])
                if len(ds) >= 10:
                    model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
                    t_stat = float(model.tvalues[0]); p_val = float(model.pvalues[0])
                else:
                    t_stat = float("nan"); p_val = float("nan")
                results[(sig_name, N, period)] = {
                    "n_trades": len(raw_b), "n_dropped": n_drop,
                    "mean_base_raw": float(np.mean(raw_b)) if len(raw_b) else float("nan"),
                    "mean_base_w":   float(np.mean(winsorize(raw_b))) if len(raw_b) else float("nan"),
                    "mean_stress_raw": float(np.mean(raw_s)) if len(raw_s) else float("nan"),
                    "mean_stress_w":   float(np.mean(raw_s_w)) if len(raw_s_w) else float("nan"),
                    "mean_exc_raw":  float(np.mean(exc_raw)) if len(exc_raw) else float("nan"),
                    "mean_exc_w":    float(np.mean(exc_w)) if len(exc_w) else float("nan"),
                    "median_exc":    float(np.median(exc_raw)) if len(exc_raw) else float("nan"),
                    "t_stat": t_stat, "p_val": p_val, "n_dates": len(ds),
                }
                r = results[(sig_name, N, period)]
                log(f"  {sig_name:7s} N={N:>2d} {period} n={r['n_trades']:5d} exc_w={r['mean_exc_w']*100:+.4f}% p={p_val:.4f}")
    return results, signals, total_sig, total_drop

# ============================================================
# LANGKAH 0 — REPRODUKSI
# ============================================================
log("")
log("=" * 90)
log("  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)")
log("=" * 90)

if not os.path.exists(SEAL_PATH):
    log(f"STOP: {SEAL_PATH} tidak ada."); sys.exit(1)

data = pd.read_pickle(SEAL_PATH)
op_raw_full = data["price_open"].copy().loc[data["price_open"].index <= CUTOFF_NEW]
hi_raw_full = data["price_high"].copy().loc[data["price_high"].index <= CUTOFF_NEW]
lo_raw_full = data["price_low"].copy().loc[data["price_low"].index <= CUTOFF_NEW]
cl_raw_full = data["price_close"].copy().loc[data["price_close"].index <= CUTOFF_NEW]
vo_raw_full = data["price_volume"].copy().loc[data["price_volume"].index <= CUTOFF_NEW]

EVENTS_PATH = f"{BASE}/data_events.csv"
df_events = pd.read_csv(EVENTS_PATH, parse_dates=["t0"])
op_f, hi_f, lo_f, cl_f, vo_f = (x.copy() for x in (op_raw_full, hi_raw_full, lo_raw_full, cl_raw_full, vo_raw_full))
idx_full = cl_f.index
n_flagged = 0
for _, ev in df_events.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_f.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        if not pd.isna(op_f.at[d, tkr]): n_flagged += 1
        op_f.at[d, tkr] = np.nan; hi_f.at[d, tkr] = np.nan
        lo_f.at[d, tkr] = np.nan; cl_f.at[d, tkr] = np.nan; vo_f.at[d, tkr] = np.nan

turnover = cl_f * vo_f
toma20 = turnover.rolling(20, min_periods=20).mean()
eligible_full = (cl_f >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

idx = idx_full[WARMUP_BARS:]
op = op_f.loc[idx]; hi = hi_f.loc[idx]; lo = lo_f.loc[idx]
cl = cl_f.loc[idx]; vo = vo_f.loc[idx]
eligible = eligible_full.loc[idx]

n_elig = eligible.sum(axis=1)
drop_days = n_elig[n_elig < MIN_ELIG_PER_DAY]
if len(drop_days):
    keep = ~idx.isin(drop_days.index)
    op, hi, lo, cl, vo = (x.loc[keep] for x in (op, hi, lo, cl, vo))
    eligible = eligible.loc[keep]; idx = idx[keep]

n_bars = len(idx)
log(f"  Bar dipakai       : {n_bars}")
log(f"  Event dimuat      : {len(df_events)}  |  Sel NaN: {n_flagged}")
log(f"  Saham layak/hari  : mean={n_elig.mean():.1f} median={n_elig.median():.1f}")
log("")

results_L0, signals_L0, total_sig_L0, total_drop_L0 = run_8_uji(
    idx, op, hi, lo, cl, vo, eligible, tA_end, tB_start, label1="A", label2="B")

TARGET = {
    ("BRK", 5, "A"): {"n": 2279, "exc_w": +0.00412},
    ("BRK", 5, "B"): {"n": 2608, "exc_w": +0.01159},
    ("BRK", 10, "A"): {"n": 2279, "exc_w": +0.00650},
    ("BRK", 10, "B"): {"n": 2601, "exc_w": +0.00680},
    ("MOM", 5, "A"): {"n": None, "exc_w": -0.00115},
    ("MOM", 5, "B"): {"n": None, "exc_w": +0.00059},
    ("MOM", 10, "A"): {"n": None, "exc_w": -0.00044},
    ("MOM", 10, "B"): {"n": None, "exc_w": +0.00184},
}
log("")
log("=" * 90)
log("  LANGKAH 0 — CEK TARGET")
log("=" * 90)
log(f"  {'Sinyal-N':10s} {'Periode':8s} {'n hasil':>9s} {'n target':>9s} {'exc_w hasil':>14s} {'exc_w target':>14s} {'cocok?':>8s}")
log("-" * 90)
all_match = True
for key, tgt in TARGET.items():
    r = results_L0.get(key)
    if r is None: all_match = False; continue
    n_ok = tgt["n"] is None or abs(r["n_trades"] - tgt["n"]) / max(1, tgt["n"]) <= 0.01
    exc_ok = abs(r["mean_exc_w"] - tgt["exc_w"]) <= 0.0002
    cocok = n_ok and exc_ok
    if not cocok: all_match = False
    log(f"  {key[0]:7s} N={key[1]:<2d} {key[2]:8s} "
        f"{r['n_trades']:>9d} {str(tgt['n']) if tgt['n'] else '—':>9s} "
        f"{r['mean_exc_w']*100:>+13.4f}% {tgt['exc_w']*100:>+13.4f}% "
        f"{'COCOK' if cocok else 'BEDA':>8s}")
log("")
log(f"  LANGKAH 0 KESELURUHAN: {'COCOK' if all_match else 'TIDAK COCOK — STOP'}")

if not all_match:
    log("")
    log("  STOP: Langkah 0 tidak cocok dengan target. Tidak lanjut ke data lama.")
    sys.exit(2)

log("")
log("  Langkah 0 LOLOS. Lanjut ke Bagian A.")

# ============================================================
# BAGIAN A — DATA LAMA
# ============================================================
log("")
log("=" * 90)
log("  BAGIAN A — DATA LAMA")
log("=" * 90)

try:
    import yfinance as yf
except ImportError:
    os.system("pip install -q yfinance")
    import yfinance as yf

idx_tickers = data["idx_tickers"]
log(f"  Ticker yang akan diunduh: {len(idx_tickers)}")
log(f"  Periode unduh: sampai {CUTOFF_OLD_END.date()}")

BATCH = 30
old_op, old_hi, old_lo, old_cl, old_vo = {}, {}, {}, {}, {}
for i in range(0, len(idx_tickers), BATCH):
    batch = idx_tickers[i:i+BATCH]
    try:
        df = yf.download(" ".join(batch), start="2000-01-01", end="2023-10-10",
                         interval="1d", auto_adjust=False, progress=False,
                         group_by="ticker", threads=True)
    except Exception as e:
        log(f"  [WARN] batch {i//BATCH+1}: {e}"); continue
    for tkr in batch:
        try:
            sub = df[tkr] if isinstance(df.columns, pd.MultiIndex) else df
            if sub.empty: continue
            old_op[tkr] = sub["Open"]; old_hi[tkr] = sub["High"]
            old_lo[tkr] = sub["Low"];  old_cl[tkr] = sub["Close"]
            old_vo[tkr] = sub["Volume"]
        except Exception:
            continue
    log(f"  batch {i//BATCH+1}/{(len(idx_tickers)+BATCH-1)//BATCH} selesai")

op_old = pd.DataFrame(old_op).sort_index()
hi_old = pd.DataFrame(old_hi).sort_index()
lo_old = pd.DataFrame(old_lo).sort_index()
cl_old = pd.DataFrame(old_cl).sort_index()
vo_old = pd.DataFrame(old_vo).sort_index()

mask_old = cl_old.index <= CUTOFF_OLD_END
op_old = op_old.loc[mask_old]; hi_old = hi_old.loc[mask_old]
lo_old = lo_old.loc[mask_old]; cl_old = cl_old.loc[mask_old]
vo_old = vo_old.loc[mask_old]

log(f"  Data lama: shape {cl_old.shape}, range {cl_old.index.min().date()} → {cl_old.index.max().date()}")
log(f"  Ticker dengan data: {cl_old.shape[1]}")

events_old = []
for tkr in cl_old.columns:
    ret = cl_old[tkr] / cl_old[tkr].shift(1) - 1
    for d in ret.index[ret.abs() > 0.40]:
        prev_idx = cl_old.index.get_loc(d)
        events_old.append({
            "ticker": tkr, "t0": d,
            "close_prev": cl_old.iloc[prev_idx-1][tkr] if prev_idx > 0 else np.nan,
            "close_now": cl_old.at[d, tkr],
        })
df_events_old = pd.DataFrame(events_old).sort_values("t0").reset_index(drop=True)
df_events_old.to_csv(OLD_EVENTS_PATH, index=False)
log(f"  Event lama terdeteksi: {len(df_events_old)}")

op_old_c = op_old.copy(); hi_old_c = hi_old.copy(); lo_old_c = lo_old.copy()
cl_old_c = cl_old.copy(); vo_old_c = vo_old.copy()
idx_old = cl_old_c.index
for _, ev in df_events_old.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_old_c.columns: continue
    try: i0 = idx_old.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_old)-1, i0+WINDOW_FWD)+1):
        d = idx_old[j]
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan

viol_mask = (lo_old_c > np.minimum(op_old_c, cl_old_c)) | (hi_old_c < np.maximum(op_old_c, cl_old_c))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan
log(f"  Pelanggaran O/H/L/C ditandai NaN: {n_viol} sel")

turnover_old = cl_old_c * vo_old_c
toma20_old = turnover_old.rolling(20, min_periods=20).mean()
elig_old = (cl_old_c >= MIN_PRICE) & (toma20_old >= MIN_TURNOVER_MA20)
n_elig_old = elig_old.sum(axis=1)

ma60 = n_elig_old.rolling(60, min_periods=60).mean()
t_eff_candidates = ma60.index[ma60 >= 60]
if len(t_eff_candidates) == 0:
    log("STOP: tidak ada tanggal dengan rata-rata 60 hari >= 60 saham layak.")
    sys.exit(3)
t_eff = t_eff_candidates[0]
log(f"  Tanggal mulai efektif: {t_eff.date()}")

mask_eff = cl_old_c.index >= t_eff
n_eff_bars = int(mask_eff.sum())
total_cells = int(cl_old_c.loc[mask_eff].size)
nan_cells = int(cl_old_c.loc[mask_eff].isna().sum().sum())
pct_nan = 100 * nan_cells / total_cells if total_cells else 0
n_elig_eff_mean = float(n_elig_old[mask_eff].mean())

g1 = n_eff_bars >= 650
g2 = pct_nan <= 5
g3 = n_elig_eff_mean >= 60
log(f"  Gerbang A1 (bar efektif >= 650): {n_eff_bars} → {'LOLOS' if g1 else 'GAGAL'}")
log(f"  Gerbang A2 (NaN <= 5%): {pct_nan:.2f}% → {'LOLOS' if g2 else 'GAGAL'}")
log(f"  Gerbang A3 (layak/hari >= 60): {n_elig_eff_mean:.1f} → {'LOLOS' if g3 else 'GAGAL'}")
gerbang_A_pass = g1 and g2 and g3
log(f"  GERBANG A: {'LOLOS' if gerbang_A_pass else 'GAGAL'}")

if gerbang_A_pass:
    op_old_seal = op_old_c.loc[mask_eff]
    hi_old_seal = hi_old_c.loc[mask_eff]
    lo_old_seal = lo_old_c.loc[mask_eff]
    cl_old_seal = cl_old_c.loc[mask_eff]
    vo_old_seal = vo_old_c.loc[mask_eff]

    old_data = {
        "price_open": op_old_seal, "price_high": hi_old_seal,
        "price_low": lo_old_seal, "price_close": cl_old_seal,
        "price_volume": vo_old_seal,
        "idx_tickers": list(cl_old_seal.columns),
        "t_eff": str(t_eff.date()),
        "sealed_end_date": str(cl_old_seal.index.max().date()),
    }
    pd.to_pickle(old_data, OLD_SEAL_PATH)

    def sha256_file(p, chunk=1<<20):
        h = hashlib.sha256()
        with open(p, "rb") as f:
            for c in iter(lambda: f.read(chunk), b""): h.update(c)
        return h.hexdigest()

    old_manifest = {
        "sealed_at": datetime.now().isoformat(timespec="seconds"),
        "t_eff": str(t_eff.date()),
        "last_bar": str(cl_old_seal.index.max().date()),
        "n_stocks": int(cl_old_seal.shape[1]),
        "n_bars": int(cl_old_seal.shape[0]),
        "sha256": sha256_file(OLD_SEAL_PATH),
        "script": "Tahap2_Replikasi.py",
    }
    json.dump(old_manifest, open(OLD_MANIFEST, "w"), indent=2)
    log(f"  data_old_sealed.pkl disimpan: {os.path.getsize(OLD_SEAL_PATH)} byte")
    log(f"  SHA256: {old_manifest['sha256']}")

    log("")
    log("=" * 90)
    log("  BAGIAN B — REPLIKASI")
    log("=" * 90)

    idx_b = cl_old_seal.index
    n_b = len(idx_b)
    cut_b = n_b // 2
    tC1_end = idx_b[cut_b - 1]
    tC2_start = idx_b[cut_b]
    log(f"  C1: {idx_b[0].date()} → {tC1_end.date()} ({cut_b} bar)")
    log(f"  C2: {tC2_start.date()} → {idx_b[-1].date()} ({n_b - cut_b} bar)")

    idx_b2 = idx_b[WARMUP_BARS:]
    op_b = op_old_seal.loc[idx_b2]; hi_b = hi_old_seal.loc[idx_b2]
    lo_b = lo_old_seal.loc[idx_b2]; cl_b = cl_old_seal.loc[idx_b2]
    vo_b = vo_old_seal.loc[idx_b2]
    elig_b = elig_old.loc[idx_b2]

    n_elig_b = elig_b.sum(axis=1)
    drop_b = n_elig_b[n_elig_b < MIN_ELIG_PER_DAY]
    if len(drop_b):
        keep = ~idx_b2.isin(drop_b.index)
        op_b, hi_b, lo_b, cl_b, vo_b = (x.loc[keep] for x in (op_b, hi_b, lo_b, cl_b, vo_b))
        elig_b = elig_b.loc[keep]; idx_b2 = idx_b2[keep]

    log("")
    log("  B4 — UJI 0 (kebocoran fitur)")
    rng0 = random.Random(2026)
    valid_T = idx_b2[25 : len(idx_b2) - 11]
    all_pairs = [(t, T) for t in cl_b.columns for T in valid_T]
    sample_pairs = rng0.sample(all_pairs, min(20, len(all_pairs)))

    def feat_from(d_op, d_hi, d_lo, d_cl, d_vo, tkr, T):
        o = d_op.loc[:T, tkr]; h = d_hi.loc[:T, tkr]
        l = d_lo.loc[:T, tkr]; c = d_cl.loc[:T, tkr]; v = d_vo.loc[:T, tkr]
        r3 = float(c.iloc[-1]/c.iloc[-4]-1) if len(c) >= 4 and pd.notna(c.iloc[-4]) and c.iloc[-4] != 0 else np.nan
        hp10 = h.iloc[-11:-1].max() if len(h) >= 11 else np.nan
        brk_v = bool(pd.notna(hp10) and c.iloc[-1] > hp10)
        if len(c) >= 15:
            delta = c.diff(); gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
            ag = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            al = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = ag / al.replace(0, np.nan); rsi = 100 - 100/(1+rs)
            rev_v = bool(pd.notna(rsi.iloc[-2]) and rsi.iloc[-2] < 30 and c.iloc[-1] > c.iloc[-2])
        else: rev_v = False
        if len(c) >= 22:
            g = o.iloc[-1]/c.iloc[-2]-1 if pd.notna(c.iloc[-2]) and c.iloc[-2] != 0 else np.nan
            vma20p = v.iloc[-21:-1].mean()
            gv = bool(pd.notna(g) and g > 0.01 and pd.notna(vma20p) and v.iloc[-1] > 1.5 * vma20p)
        else: gv = False
        return {"roc3": r3, "brk": brk_v, "rev": rev_v, "gapvol": gv}

    uji0 = {}
    for name in ["MOM", "BRK", "REV", "GAPVOL"]:
        mism = 0
        for tkr, T in sample_pairs:
            f1 = feat_from(op_old_seal, hi_old_seal, lo_old_seal, cl_old_seal, vo_old_seal, tkr, T)
            f2 = feat_from(op_old_c, hi_old_c, lo_old_c, cl_old_c, vo_old_c, tkr, T)
            for k in ["roc3","brk","rev","gapvol"]:
                a, b = f1[k], f2[k]
                if isinstance(a, float) and isinstance(b, float):
                    if pd.isna(a) and pd.isna(b): continue
                    if (pd.isna(a) != pd.isna(b)) or (not pd.isna(a) and abs(a-b) > 1e-9): mism += 1
                else:
                    if a != b: mism += 1
        uji0[name] = (mism == 0)
        log(f"    {name:8s} mism={mism} → {'LULUS' if mism == 0 else 'GAGAL'}")

    uji0_pass = all(uji0.values())
    if not uji0_pass:
        log("  STOP: Uji 0 GAGAL di data lama.")
        sys.exit(4)

    results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
        idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
        tC1_end, tC2_start, label1="C1", label2="C2")

    def cek_B5(sig, N):
        r1 = results_B[(sig, N, "C1")]; r2 = results_B[(sig, N, "C2")]
        p1_c1 = fix_p1b(r1["p_val"], r1["mean_exc_w"])
        p1_c2 = fix_p1b(r2["p_val"], r2["mean_exc_w"])
        p1_pool = min(p1_c1, p1_c2)
        fails = []
        if not (r1["mean_exc_w"] > 0): fails.append(f"C1 exc {r1['mean_exc_w']*100:+.3f}% ≤ 0")
        if not (p1_c1 < 0.10): fails.append(f"C1 p1sisi {p1_c1:.4f} ≥ 0.10")
        if not (r2["mean_exc_w"] > 0): fails.append(f"C2 exc {r2['mean_exc_w']*100:+.3f}% ≤ 0")
        if not (p1_c2 < 0.10): fails.append(f"C2 p1sisi {p1_c2:.4f} ≥ 0.10")
        if r1["n_trades"] < 300: fails.append(f"C1 n={r1['n_trades']} < 300")
        if r2["n_trades"] < 300: fails.append(f"C2 n={r2['n_trades']} < 300")
        if not (p1_pool < 0.025): fails.append(f"pool p1sisi {p1_pool:.4f} ≥ 0.025")
        return (len(fails)==0), fails, p1_c1, p1_c2

    log("")
    log("=" * 100)
    log("  HASIL REPLIKASI — BAGIAN B")
    log("=" * 100)
    hasil_BRK = {}
    for sig in ["BRK", "MOM", "REV", "GAPVOL"]:
        for N in N_LIST:
            r1 = results_B[(sig, N, "C1")]; r2 = results_B[(sig, N, "C2")]
            p1_c1 = fix_p1b(r1["p_val"], r1["mean_exc_w"])
            p1_c2 = fix_p1b(r2["p_val"], r2["mean_exc_w"])
            role = "PENENTU" if sig == "BRK" else ("KONTROL" if sig == "MOM" else "DESKRIPTIF")
            log(f"  {sig:7s} N={N:>2d} [{role:9s}]  C1 exc={r1['mean_exc_w']*100:+.3f}% p1sisi={p1_c1:.4f} n={r1['n_trades']:5d}  |  "
                f"C2 exc={r2['mean_exc_w']*100:+.3f}% p1sisi={p1_c2:.4f} n={r2['n_trades']:5d}")
            if sig == "BRK":
                passed, fails, p1c1, p1c2 = cek_B5(sig, N)
                hasil_BRK[N] = {"passed": passed, "fails": fails}

    log("")
    log(f"  HASIL BRK-5 : {'LOLOS' if hasil_BRK[5]['passed'] else 'GAGAL — ' + '; '.join(hasil_BRK[5]['fails'])}")
    log(f"  HASIL BRK-10: {'LOLOS' if hasil_BRK[10]['passed'] else 'GAGAL — ' + '; '.join(hasil_BRK[10]['fails'])}")

    mom_alarm = False
    for N in N_LIST:
        p, _ = cek_B5("MOM", N)
        if p: mom_alarm = True
    log(f"  KONTROL NEGATIF MOM: {'ALARM BIAS (MOM lolos B5)' if mom_alarm else 'aman (MOM tidak lolos B5)'}")

    hasil_semua = {
        "gerbang_A": {"n_bar": n_eff_bars, "pct_nan": pct_nan, "n_elig": n_elig_eff_mean,
                      "pass": gerbang_A_pass},
        "C1_C2": {"tC1_end": str(tC1_end.date()), "tC2_start": str(tC2_start.date())},
        "uji0": uji0,
        "results_B": results_B,
        "hasil_BRK": hasil_BRK,
        "mom_alarm": mom_alarm,
        "n_events_old": len(df_events_old),
        "n_viol": n_viol,
        "t_eff": str(t_eff.date()),
        "old_manifest": old_manifest,
    }
    pd.to_pickle(hasil_semua, f"{BASE}/hasil_tahap2.pkl")
    log(f"  hasil_tahap2.pkl disimpan.")
else:
    log("  STOP: Gerbang A GAGAL. Tidak lanjut ke Bagian B.")

log("")
log("=" * 90)
log("  Tahap2_Replikasi.py SELESAI")
log("=" * 90)
'''

with open(SCRIPT_PATH, "w") as f:
    f.write(SCRIPT_CONTENT)

print()
print("=" * 90)
print("  FILE DITULIS")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")
print(f"  Baris  : {len(SCRIPT_CONTENT.splitlines())}")
print("=" * 90)
print()
print("Cara menjalankan:")
print(f'  !python "{SCRIPT_PATH}"')

Mounted at /content/drive
  CEK FILE YANG DITUJU
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  ✅ data_sealed.pkl               14161504 byte
  ✅ data_events.csv                   1975 byte
  ✅ seal_manifest.json                 309 byte
  ✅ params_frozen.json                 739 byte
  ✅ run_log.csv                       1534 byte
  ✅ LAPORAN_Tahap0.md                 2669 byte
  ✅ LAPORAN_Tahap1.md                 4030 byte
  ✅ LAPORAN_Tahap1b.md                5277 byte
  ✅ 4. Anthropi_AF_G_Evo4.py         19411 byte

  FILE DITULIS
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 24473 byte
  Baris  : 572

Cara menjalankan:
  !python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"


In [ ]:
# ============================================================
# LANGKAH K1 (revisi) — Mount + Cek File + Investigasi
# ============================================================

# ---------- 1) MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

print("=" * 100)
print("  CEK FOLDER & FILE")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
if os.path.isdir(BASE):
    files = sorted(os.listdir(BASE))
    print(f"  Isi folder ({len(files)} item):")
    for f in files:
        p = os.path.join(BASE, f)
        size = os.path.getsize(p) if os.path.isfile(p) else 0
        print(f"    {f:40s} {size:>12} byte")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  CEK FOLDER & FILE
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  Isi folder (15 item):
    1. AladdinFoundry_Gen_v1_Step25.py             292829 byte
    4. Anthropi_AF_G_Evo4.py                        19411 byte
    LAPORAN_Tahap0.md                                2669 byte
    LAPORAN_Tahap1.md                                4030 byte
    LAPORAN_Tahap1b.md                               5277 byte
    LAPORAN_Tahap1b_BACKUP_20261012_034833.md         5277 byte
    Tahap0_Prep.py                                   2636 byte
    Tahap2_Replikasi.py                             24473 byte
    cache_aladdin                                       0 byte
    data_events.csv                                  1975 byte
    data_sealed.pkl                              14161504 byte
    fundamental_soft.py                              4326 by

In [ ]:
# ============================================================
# LANGKAH K1 — Inspeksi kode unduh di Tahap2_Replikasi.py
# ============================================================
import os, re

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

print("=" * 100)
print("  K1a — KODE UNDUH DI Tahap2_Replikasi.py")
print("=" * 100)

with open(SCRIPT_PATH) as f:
    lines = f.readlines()

# Tampilkan semua baris yang berkaitan dengan unduh
keywords = ["yf.download", "start=", "end=", "auto_adjust", "period=",
            "group_by", "threads", "interval=", "CUTOFF_OLD_END", "mask_old"]
for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        print(f"  L{i+1:>4d}: {line.rstrip()}")

print()
print("  Blok unduh lengkap (konteks 20 baris):")
print("-" * 100)
for i, line in enumerate(lines):
    if "yf.download" in line:
        start = max(0, i - 10)
        end   = min(len(lines), i + 15)
        for j in range(start, end):
            print(f"  L{j+1:>4d}: {lines[j].rstrip()}")
        print("-" * 100)
        break

print()
print("  Filter tanggal setelah unduh (konteks):")
print("-" * 100)
for i, line in enumerate(lines):
    if "mask_old" in line or "CUTOFF_OLD_END" in line:
        start = max(0, i - 3)
        end   = min(len(lines), i + 5)
        for j in range(start, end):
            print(f"  L{j+1:>4d}: {lines[j].rstrip()}")
        print("-" * 100)

print()
print("=" * 100)
print("  K1b — MANIFEST")
print("=" * 100)
for name in ["seal_manifest.json", "data_old_manifest.json"]:
    p = f"{BASE}/{name}"
    if os.path.exists(p):
        with open(p) as f:
            print(f"  {name}:")
            print(f"    {f.read()}")
    else:
        print(f"  {name}: (tidak ada)")

print()
print("=" * 100)
print("  K1c — CEK FILE cache_aladdin (isi)")
print("=" * 100)
cache_dir = f"{BASE}/cache_aladdin"
if os.path.isdir(cache_dir):
    for f in sorted(os.listdir(cache_dir)):
        p = os.path.join(cache_dir, f)
        size = os.path.getsize(p) if os.path.isfile(p) else 0
        print(f"  {f:50s} {size:>12} byte")
else:
    print("  (folder cache_aladdin tidak ada)")

  K1a — KODE UNDUH DI Tahap2_Replikasi.py
  L  40: CUTOFF_OLD_END  = pd.Timestamp("2023-10-09")
  L 294: log(f"  Periode unduh: sampai {CUTOFF_OLD_END.date()}")
  L 301:         df = yf.download(" ".join(batch), start="2000-01-01", end="2023-10-10",
  L 302:                          interval="1d", auto_adjust=False, progress=False,
  L 303:                          group_by="ticker", threads=True)
  L 323: mask_old = cl_old.index <= CUTOFF_OLD_END
  L 324: op_old = op_old.loc[mask_old]; hi_old = hi_old.loc[mask_old]
  L 325: lo_old = lo_old.loc[mask_old]; cl_old = cl_old.loc[mask_old]
  L 326: vo_old = vo_old.loc[mask_old]

  Blok unduh lengkap (konteks 20 baris):
----------------------------------------------------------------------------------------------------
  L 291: 
  L 292: idx_tickers = data["idx_tickers"]
  L 293: log(f"  Ticker yang akan diunduh: {len(idx_tickers)}")
  L 294: log(f"  Periode unduh: sampai {CUTOFF_OLD_END.date()}")
  L 295: 
  L 296: BATCH = 30
  L 297: old_o

In [ ]:
# ============================================================
# LANGKAH K2 — Tentukan auto_adjust dari data_sealed.pkl
# Uji 3-5 ticker berdividen: bandingkan dua mode (True vs False)
# dengan data_sealed.pkl pada periode yang sama
# ============================================================
import os
import pandas as pd
import numpy as np
import yfinance as yf

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH = f"{BASE}/data_sealed.pkl"

# Load data sealed
data = pd.read_pickle(SEAL_PATH)
sealed_close = data["price_close"]
sealed_open  = data["price_open"]

# Ticker berdividen + likuid (pasti ada di data_sealed)
TICKERS_TEST = ["BBCA.JK", "BBRI.JK", "TLKM.JK", "ASII.JK", "UNVR.JK"]

# Periode uji: 2023-10-10 s.d. 2024-03-31 (ada dividen BBCA biasanya)
D_START = "2023-10-10"
D_END   = "2024-03-31"

print("=" * 100)
print("  K2 — TENTUKAN auto_adjust")
print("=" * 100)
print(f"  Periode uji: {D_START} → {D_END}")
print(f"  Ticker     : {TICKERS_TEST}")
print()

for tkr in TICKERS_TEST:
    print("-" * 100)
    print(f"  Ticker: {tkr}")
    print(f"  {'tanggal':12s} {'sealed Close':>14s} {'yf True Close':>14s} {'yf False Close':>14s} {'diff T':>10s} {'diff F':>10s}")
    try:
        df_true  = yf.download(tkr, start=D_START, end=D_END, interval="1d",
                               auto_adjust=True,  progress=False, threads=False)
        df_false = yf.download(tkr, start=D_START, end=D_END, interval="1d",
                               auto_adjust=False, progress=False, threads=False)
        if isinstance(df_true.columns, pd.MultiIndex):
            df_true.columns = df_true.columns.get_level_values(0)
        if isinstance(df_false.columns, pd.MultiIndex):
            df_false.columns = df_false.columns.get_level_values(0)

        # Selaraskan tanggal dengan data_sealed
        common_dates = sealed_close.index.intersection(df_true.index)
        for d in common_dates[:5]:
            s = sealed_close.at[d, tkr]
            t = df_true.at[d, "Close"]
            f = df_false.at[d, "Close"]
            dt = abs(s - t)
            df_ = abs(s - f)
            print(f"  {str(d.date()):12s} {s:>14.2f} {t:>14.2f} {f:>14.2f} {dt:>10.4f} {df_:>10.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")

print()
print("=" * 100)
print("  Interpretasi:")
print("    - Kalau 'diff T' mendekati 0 (di bawah 0.01) → data_sealed pakai auto_adjust=True")
print("    - Kalau 'diff F' mendekati 0 → data_sealed pakai auto_adjust=False")
print("=" * 100)

  K2 — TENTUKAN auto_adjust
  Periode uji: 2023-10-10 → 2024-03-31
  Ticker     : ['BBCA.JK', 'BBRI.JK', 'TLKM.JK', 'ASII.JK', 'UNVR.JK']

----------------------------------------------------------------------------------------------------
  Ticker: BBCA.JK
  tanggal        sealed Close  yf True Close yf False Close     diff T     diff F
  ERROR: 'BBCA.JK'
----------------------------------------------------------------------------------------------------
  Ticker: BBRI.JK
  tanggal        sealed Close  yf True Close yf False Close     diff T     diff F
  ERROR: 'BBRI.JK'
----------------------------------------------------------------------------------------------------
  Ticker: TLKM.JK
  tanggal        sealed Close  yf True Close yf False Close     diff T     diff F
  ERROR: 'TLKM.JK'
----------------------------------------------------------------------------------------------------
  Ticker: ASII.JK
  tanggal        sealed Close  yf True Close yf False Close     diff T     diff F


In [ ]:
# ============================================================
# K2a — Diagnosa struktur DataFrame yfinance
# ============================================================
import yfinance as yf
import pandas as pd

tkr = "BBCA.JK"
D_START = "2023-10-10"
D_END   = "2023-12-31"

print("=" * 90)
print(f"  DIAGNOSA STRUKTUR DataFrame untuk {tkr}")
print("=" * 90)

for mode in [True, False]:
    print(f"\n  --- auto_adjust={mode} ---")
    df = yf.download(tkr, start=D_START, end=D_END, interval="1d",
                     auto_adjust=mode, progress=False, threads=False)
    print(f"  type(df)         : {type(df)}")
    print(f"  type(df.columns) : {type(df.columns)}")
    print(f"  df.columns       : {df.columns.tolist() if hasattr(df.columns, 'tolist') else df.columns}")
    print(f"  df.shape         : {df.shape}")
    print(f"  df.index[:3]     : {df.index[:3].tolist()}")
    print()
    print(f"  df.head(2):")
    print(df.head(2))

print()
print("=" * 90)
print("  Struktur data_sealed.pkl untuk perbandingan:")
print("=" * 90)
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
data = pd.read_pickle(f"{BASE}/data_sealed.pkl")
print(f"  Tipe data['price_close']      : {type(data['price_close'])}")
print(f"  data['price_close'].index[:3] : {data['price_close'].index[:3].tolist()}")
print(f"  data['price_close'].columns[:3]: {data['price_close'].columns[:3].tolist()}")
print(f"  Nilai BBCA.JK beberapa hari:")
sub = data["price_close"]["BBCA.JK"].loc["2023-10-10":"2023-10-20"]
print(sub.to_string())
print("=" * 90)

  DIAGNOSA STRUKTUR DataFrame untuk BBCA.JK

  --- auto_adjust=True ---
  type(df)         : <class 'pandas.core.frame.DataFrame'>
  type(df.columns) : <class 'pandas.core.indexes.multi.MultiIndex'>
  df.columns       : [('Close', 'BBCA.JK'), ('High', 'BBCA.JK'), ('Low', 'BBCA.JK'), ('Open', 'BBCA.JK'), ('Volume', 'BBCA.JK')]
  df.shape         : (57, 5)
  df.index[:3]     : [Timestamp('2023-10-10 00:00:00'), Timestamp('2023-10-11 00:00:00'), Timestamp('2023-10-12 00:00:00')]

  df.head(2):
Price             Close         High          Low         Open    Volume
Ticker          BBCA.JK      BBCA.JK      BBCA.JK      BBCA.JK   BBCA.JK
Date                                                                    
2023-10-10  7921.169434  7987.733883  7921.169434  7987.733883  71966100
2023-10-11  7921.169434  8009.922032  7921.169434  7921.169434  84305800

  --- auto_adjust=False ---
  type(df)         : <class 'pandas.core.frame.DataFrame'>
  type(df.columns) : <class 'pandas.core.indexes.mu

KeyError: 'BBCA.JK'

In [ ]:
# ============================================================
# LANGKAH K3 — Perbaiki Tahap2_Replikasi.py
# Perubahan: (1) auto_adjust=True, (2) unduh per-ticker dengan retry,
#            (3) progress tersimpan berkala
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, re

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- Patch 1: auto_adjust False → True ----------
content = content.replace('auto_adjust=False', 'auto_adjust=True')

# ---------- Patch 2: Ganti blok unduh batch menjadi per-ticker ----------
OLD_BLOCK = '''BATCH = 30
old_op, old_hi, old_lo, old_cl, old_vo = {}, {}, {}, {}, {}
for i in range(0, len(idx_tickers), BATCH):
    batch = idx_tickers[i:i+BATCH]
    try:
        df = yf.download(" ".join(batch), start="2000-01-01", end="2023-10-10",
                         interval="1d", auto_adjust=True, progress=False,
                         group_by="ticker", threads=True)
    except Exception as e:
        log(f"  [WARN] batch {i//BATCH+1}: {e}"); continue
    for tkr in batch:
        try:
            sub = df[tkr] if isinstance(df.columns, pd.MultiIndex) else df
            if sub.empty: continue
            old_op[tkr] = sub["Open"]; old_hi[tkr] = sub["High"]
            old_lo[tkr] = sub["Low"];  old_cl[tkr] = sub["Close"]
            old_vo[tkr] = sub["Volume"]
        except Exception:
            continue
    log(f"  batch {i//BATCH+1}/{(len(idx_tickers)+BATCH-1)//BATCH} selesai")'''

NEW_BLOCK = '''# Unduh per-ticker (auto_adjust=True, retry, progress tersimpan)
PROGRESS_PATH = f"{BASE}/_download_old_progress.pkl"
old_op, old_hi, old_lo, old_cl, old_vo = {}, {}, {}, {}, {}

# Resume dari progress kalau ada
if os.path.exists(PROGRESS_PATH):
    try:
        prog = pd.read_pickle(PROGRESS_PATH)
        old_op = prog.get("op", {}); old_hi = prog.get("hi", {})
        old_lo = prog.get("lo", {}); old_cl = prog.get("cl", {})
        old_vo = prog.get("vo", {})
        log(f"  Resume dari progress: {len(old_cl)} ticker sudah terunduh")
    except Exception:
        pass

t_start = time.time()
DWN_OK = 0; DWN_FAIL = 0
for i, tkr in enumerate(idx_tickers):
    if tkr in old_cl and not old_cl[tkr].empty:
        DWN_OK += 1
        continue
    df = None
    for attempt in range(3):
        try:
            df = yf.download(tkr, start="2005-01-01", end="2023-10-10",
                             interval="1d", auto_adjust=True,
                             progress=False, threads=False)
            if df is not None and not df.empty:
                break
        except Exception as e:
            time.sleep(1.5)
            continue
    if df is None or df.empty:
        DWN_FAIL += 1
        continue
    # Flatten MultiIndex kalau ada
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    old_op[tkr] = df["Open"]; old_hi[tkr] = df["High"]
    old_lo[tkr] = df["Low"];  old_cl[tkr] = df["Close"]
    old_vo[tkr] = df["Volume"]
    DWN_OK += 1

    # Progress log setiap 25 ticker
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        eta = (len(idx_tickers) - i - 1) / rate if rate > 0 else 0
        log(f"  [{i+1}/{len(idx_tickers)}] OK={DWN_OK} FAIL={DWN_FAIL} "
            f"({elapsed:.0f}s, ETA {eta:.0f}s)")

    # Simpan progress berkala setiap 50 ticker
    if (i + 1) % 50 == 0:
        pd.to_pickle({"op": old_op, "hi": old_hi, "lo": old_lo,
                      "cl": old_cl, "vo": old_vo}, PROGRESS_PATH)

# Simpan progress final
pd.to_pickle({"op": old_op, "hi": old_hi, "lo": old_lo,
              "cl": old_cl, "vo": old_vo}, PROGRESS_PATH)

elapsed = time.time() - t_start
log(f"  Unduh selesai: OK={DWN_OK} FAIL={DWN_FAIL} ({elapsed:.0f}s)")'''

if OLD_BLOCK in content:
    content = content.replace(OLD_BLOCK, NEW_BLOCK)
    print("  ✅ Blok unduh batch → per-ticker: DIGANTI")
else:
    print("  ⚠️ Blok unduh lama tidak persis cocok. Cek manual.")
    # Coba cari blok yang mengandung 'for i in range(0, len(idx_tickers), BATCH)'
    if "for i in range(0, len(idx_tickers), BATCH)" in content:
        print("     Blok ditemukan tapi tidak persis. Perlu inspeksi manual.")

# ---------- Patch 3: Tambah import time ----------
if "import time" not in content.split("import os")[0] and "import os, sys" in content:
    content = content.replace("import os, sys, csv, json, random, hashlib",
                              "import os, sys, csv, json, random, hashlib, time")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")
print(f"  Baris  : {len(content.splitlines())}")

# Verifikasi patch
print()
print("  Verifikasi patch:")
with open(SCRIPT_PATH) as f:
    new_content = f.read()
print(f"  'auto_adjust=True'  muncul: {new_content.count('auto_adjust=True')}x")
print(f"  'auto_adjust=False' muncul: {new_content.count('auto_adjust=False')}x")
print(f"  'start=\"2005-01-01\"' muncul: {new_content.count('start=\"2005-01-01\"')}x")
print(f"  'per-ticker' loop: {'ya' if 'for i, tkr in enumerate(idx_tickers)' in new_content else 'TIDAK'}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅ Blok unduh batch → per-ticker: DIGANTI

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 25861 byte
  Baris  : 614

  Verifikasi patch:
  'auto_adjust=True'  muncul: 2x
  'auto_adjust=False' muncul: 0x
  'start="2005-01-01"' muncul: 1x
  'per-ticker' loop: ya


In [ ]:
# ============================================================
# K3b — Verifikasi script setelah patch
# ============================================================
import os, re

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

print("=" * 90)
print("  VERIFIKASI SCRIPT")
print("=" * 90)

# Cek syntax
import ast
try:
    ast.parse(content)
    print("  ✅ Syntax OK (ast.parse berhasil)")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

# Cek tidak ada sisa blok batch
print()
print("  Cek sisa blok lama:")
print(f"  'for i in range(0, len(idx_tickers), BATCH)' : {content.count('for i in range(0, len(idx_tickers), BATCH)')}x")
print(f"  'BATCH = 30'                                 : {content.count('BATCH = 30')}x")
print(f"  'group_by=\"ticker\", threads=True'            : {content.count('group_by=\"ticker\", threads=True')}x")

# Tampilkan semua baris dengan auto_adjust & yf.download
print()
print("  Semua baris yf.download & auto_adjust:")
for i, line in enumerate(content.splitlines()):
    if "yf.download" in line or "auto_adjust" in line:
        print(f"  L{i+1:>4d}: {line.strip()}")

# Cek blok unduh baru
print()
print("  Cek blok unduh baru:")
for i, line in enumerate(content.splitlines()):
    if "Unduh per-ticker" in line or "for i, tkr in enumerate(idx_tickers)" in line:
        print(f"  L{i+1:>4d}: {line.strip()}")

print("=" * 90)
print("  Kalau 'BATCH = 30' dan 'group_by=ticker' = 0, script bersih.")
print("=" * 90)

  VERIFIKASI SCRIPT
  ✅ Syntax OK (ast.parse berhasil)

  Cek sisa blok lama:
  'for i in range(0, len(idx_tickers), BATCH)' : 0x
  'BATCH = 30'                                 : 0x
  'group_by="ticker", threads=True'            : 0x

  Semua baris yf.download & auto_adjust:
  L 296: # Unduh per-ticker (auto_adjust=True, retry, progress tersimpan)
  L 320: df = yf.download(tkr, start="2005-01-01", end="2023-10-10",
  L 321: interval="1d", auto_adjust=True,

  Cek blok unduh baru:
  L 296: # Unduh per-ticker (auto_adjust=True, retry, progress tersimpan)
  L 313: for i, tkr in enumerate(idx_tickers):
  Kalau 'BATCH = 30' dan 'group_by=ticker' = 0, script bersih.


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 2 — Tahap2_Replikasi.py

  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)
  Bar dipakai       : 601
  Event dimuat      : 56  |  Sel NaN: 539
  Saham layak/hari  : mean=101.4 median=85.0

Traceback (most recent call last):
  File "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py", line 236, in <module>
    results_L0, signals_L0, total_sig_L0, total_drop_L0 = run_8_uji(
                                                          ~~~~~~~~~^
        idx, op, hi, lo, cl, vo, eligible, tA_end, tB_start, label1="A", label2="B")
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py", line 140, in run_8_uji
    if not mask_t.loc[t]: continue
           ^^^^^^^^^^
AttributeError: 'numpy.ndarray' object has no attribute 'loc'


In [ ]:
# ============================================================
# K5 — Patch bug mask_t di run_8_uji
# Ganti: mask_t = (sig_df.index <= t2) ...
# Jadi : mask_t = pd.Series((sig_df.index <= t2), index=sig_df.index)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

OLD = '''mask_t = (sig_df.index <= t2) if t2 is not None else (sig_df.index >= t1)'''
NEW = '''mask_t = pd.Series((sig_df.index <= t2) if t2 is not None else (sig_df.index >= t1),
                    index=sig_df.index)'''

if OLD in content:
    content = content.replace(OLD, NEW)
    with open(SCRIPT_PATH, "w") as f:
        f.write(content)
    print("✅ Patch diterapkan.")
else:
    print("⚠️ Baris lama tidak ditemukan persis. Cek manual.")

# Verifikasi
with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'pd.Series((sig_df.index' muncul: {new.count('pd.Series((sig_df.index')}x")
print(f"  'mask_t = (sig_df.index <=' muncul : {new.count('mask_t = (sig_df.index <=')}x")
print(f"  Ukuran file: {os.path.getsize(SCRIPT_PATH)} byte")

# Cek syntax
import ast
try:
    ast.parse(new)
    print("✅ Syntax OK")
except SyntaxError as e:
    print(f"❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch diterapkan.
  'pd.Series((sig_df.index' muncul: 1x
  'mask_t = (sig_df.index <=' muncul : 0x
  Ukuran file: 25912 byte
✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 2 — Tahap2_Replikasi.py

  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)
  Bar dipakai       : 601
  Event dimuat      : 56  |  Sel NaN: 539
  Saham layak/hari  : mean=101.4 median=85.0

  MOM     N= 5 A n= 5815 exc_w=-0.1180% p=0.5065
  MOM     N= 5 B n= 6551 exc_w=+0.0591% p=0.6971
  MOM     N=10 A n= 5815 exc_w=-0.0118% p=0.9882
  MOM     N=10 B n= 6427 exc_w=+0.1841% p=0.4702
  BRK     N= 5 A n= 2210 exc_w=+0.4526% p=0.5851
  BRK     N= 5 B n= 2608 exc_w=+1.1594% p=0.0088
  BRK     N=10 A n= 2210 exc_w=+0.7510% p=0.5492
  BRK     N=10 B n= 2601 exc_w=+0.6795% p=0.1906
  REV     N= 5 A n=  632 exc_w=-0.2157% p=0.3123
  REV     N= 5 B n=  529 exc_w=-0.7616% p=0.0664
  REV     N=10 A n=  632 exc_w=-0.6065% p=0.1091
  REV     N=10 B n=  480 exc_w=-2.2968% p=0.0000
  GAPVOL  N= 5 A n=  789 exc_w=-0.7356% p=0.2542
  GAPVOL  N= 5 B n= 1774 exc_w=-0.9726% p=0.2157
  GAPVOL  N=10 A n=  789 exc_w=-1.3699%

In [ ]:
# ============================================================
# LANGKAH K6 (revisi) — Mount + Cek File + Diagnosa n sinyal A
# Bandingkan dengan H2 (versi lama):
#   MOM A=5864 | BRK A=2279 | REV A=638 | GAPVOL A=827
# ============================================================

# ---------- 1) MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- 2) CEK FILE YANG DITUJU ----------
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH   = f"{BASE}/data_sealed.pkl"
EVENTS_PATH = f"{BASE}/data_events.csv"

print("=" * 100)
print("  CEK FILE YANG DITUJU")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("data_sealed.pkl",     SEAL_PATH),
    ("data_events.csv",     EVENTS_PATH),
    ("seal_manifest.json",  f"{BASE}/seal_manifest.json"),
    ("Tahap2_Replikasi.py", f"{BASE}/Tahap2_Replikasi.py"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:24s} {size:>10} byte")
print("=" * 100)

# ---------- 3) DIAGNOSA ----------
import numpy as np
import pandas as pd

CUTOFF_NEW        = pd.Timestamp("2026-06-07")
MIN_PRICE         = 200.0
MIN_TURNOVER_MA20 = 1_000_000_000.0
MIN_ELIG_PER_DAY  = 60
WARMUP_BARS       = 25
WINDOW_BACK       = 3
WINDOW_FWD        = 25
tA_end            = pd.Timestamp("2025-05-15")
tB_start          = pd.Timestamp("2025-05-16")

print()
print("=" * 100)
print("  K6 — DIAGNOSA n SINYAL A vs H2")
print("=" * 100)

# Load
data = pd.read_pickle(SEAL_PATH)
op_raw_full = data["price_open"].copy().loc[data["price_open"].index <= CUTOFF_NEW]
hi_raw_full = data["price_high"].copy().loc[data["price_high"].index <= CUTOFF_NEW]
lo_raw_full = data["price_low"].copy().loc[data["price_low"].index <= CUTOFF_NEW]
cl_raw_full = data["price_close"].copy().loc[data["price_close"].index <= CUTOFF_NEW]
vo_raw_full = data["price_volume"].copy().loc[data["price_volume"].index <= CUTOFF_NEW]

print(f"  Data sealed dimuat: {cl_raw_full.shape}, "
      f"range {cl_raw_full.index.min().date()} → {cl_raw_full.index.max().date()}")

# Pembersihan
df_events = pd.read_csv(EVENTS_PATH, parse_dates=["t0"])
op_f, hi_f, lo_f, cl_f, vo_f = (x.copy() for x in
    (op_raw_full, hi_raw_full, lo_raw_full, cl_raw_full, vo_raw_full))
idx_full = cl_f.index
for _, ev in df_events.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_f.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        op_f.at[d, tkr] = np.nan; hi_f.at[d, tkr] = np.nan
        lo_f.at[d, tkr] = np.nan; cl_f.at[d, tkr] = np.nan; vo_f.at[d, tkr] = np.nan

turnover = cl_f * vo_f
toma20 = turnover.rolling(20, min_periods=20).mean()
eligible_full = (cl_f >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

# Buang warm-up 25 bar
idx = idx_full[WARMUP_BARS:]
op = op_f.loc[idx]; hi = hi_f.loc[idx]; lo = lo_f.loc[idx]
cl = cl_f.loc[idx]; vo = vo_f.loc[idx]
eligible = eligible_full.loc[idx]

print(f"  Bar setelah buang warm-up 25 bar: {len(idx)}")
print(f"  Range: {idx.min().date()} → {idx.max().date()}")

# Drop days < 60 layak
n_elig = eligible.sum(axis=1)
drop_days = n_elig[n_elig < MIN_ELIG_PER_DAY]
print()
print(f"  Tanggal dengan < 60 saham layak: {len(drop_days)}")
if len(drop_days):
    print("    →", [str(d.date()) for d in drop_days.index])
    keep = ~idx.isin(drop_days.index)
    op, hi, lo, cl, vo = (x.loc[keep] for x in (op, hi, lo, cl, vo))
    eligible = eligible.loc[keep]; idx = idx[keep]
    print(f"  Bar setelah drop days: {len(idx)}")

print()
print(f"  tA_end = {tA_end.date()}  |  tB_start = {tB_start.date()}")
print(f"  Saham layak/hari: mean={n_elig.mean():.1f} median={n_elig.median():.1f}")
print()

# Bangun sinyal (rumus sama dengan H2)
roc3 = cl / cl.shift(3) - 1.0
delta = cl.diff()
gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
ag = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
al = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
rs = ag / al.replace(0, np.nan)
rsi14 = 100 - 100/(1+rs)
rev = (rsi14.shift(1) < 30) & (cl > cl.shift(1))
high_prev10 = hi.shift(1).rolling(10, min_periods=10).max()
brk = cl > high_prev10
vol_ma20_prev = vo.shift(1).rolling(20, min_periods=20).mean()
gap = op / cl.shift(1) - 1.0
gapvol = (gap > 0.01) & (vo > 1.5 * vol_ma20_prev)

roc3_rank = roc3.where(eligible).rank(axis=1, pct=True)
mom_sig   = (roc3_rank >= 0.80) & roc3.notna() & eligible
brk_sig   = brk & eligible
rev_sig   = rev & eligible
gapvol_sig = gapvol & eligible

SIGNALS = {"MOM": mom_sig, "BRK": brk_sig, "REV": rev_sig, "GAPVOL": gapvol_sig}

print("=" * 100)
print("  JUMLAH SINYAL PER PERIODE (script baru)")
print("=" * 100)
print(f"  {'Sinyal':8s} {'A (baru)':>10s} {'A (H2)':>10s} {'ΔA':>8s}   {'B (baru)':>10s} {'B (H2)':>10s} {'ΔB':>8s}")
print("-" * 100)

# Target H2
TARGET_H2 = {
    "MOM":    {"A": 5864, "B": 6668},
    "BRK":    {"A": 2279, "B": 2634},
    "REV":    {"A":  638, "B":  562},
    "GAPVOL": {"A":  827, "B": 1808},
}

for name, sig in SIGNALS.items():
    nA = int(sig.loc[sig.index <= tA_end].sum().sum())
    nB = int(sig.loc[sig.index >  tA_end].sum().sum())
    tA = TARGET_H2[name]["A"]; tB = TARGET_H2[name]["B"]
    print(f"  {name:8s} {nA:>10d} {tA:>10d} {nA-tA:>+8d}   {nB:>10d} {tB:>10d} {nB-tB:>+8d}")

print()

# Cek sinyal A di sekitar batas (10 hari terakhir A)
print("=" * 100)
print("  SINYAL A DI SEKITAR BATAS (15 hari terakhir periode A)")
print("=" * 100)
batas_start = tA_end - pd.Timedelta(days=20)
for name, sig in SIGNALS.items():
    sub = sig.loc[(sig.index > batas_start) & (sig.index <= tA_end)]
    print(f"  {name:8s} {int(sub.sum().sum()):>5d} sinyal di {batas_start.date()} → {tA_end.date()}")

print()
print("=" * 100)
print("  Cek jumlah bar di periode A:")
print("=" * 100)
nA_bars = int((idx <= tA_end).sum())
nB_bars = int((idx >  tA_end).sum())
print(f"  Bar A: {nA_bars}  |  Bar B: {nB_bars}  |  Total: {nA_bars + nB_bars}")
print(f"  Sebelumnya di H1: A=376 bar? (dari 626 total) → sekarang {len(idx)} total")
print("=" * 100)

Mounted at /content/drive
  CEK FILE YANG DITUJU
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  ✅ data_sealed.pkl            14161504 byte
  ✅ data_events.csv                1975 byte
  ✅ seal_manifest.json              309 byte
  ✅ Tahap2_Replikasi.py           25912 byte

  K6 — DIAGNOSA n SINYAL A vs H2
  Data sealed dimuat: (626, 561), range 2023-10-10 → 2026-06-05
  Bar setelah buang warm-up 25 bar: 601
  Range: 2023-11-14 → 2026-06-05

  Tanggal dengan < 60 saham layak: 0

  tA_end = 2025-05-15  |  tB_start = 2025-05-16
  Saham layak/hari: mean=101.4 median=85.0

  JUMLAH SINYAL PER PERIODE (script baru)
  Sinyal     A (baru)     A (H2)       ΔA     B (baru)     B (H2)       ΔB
----------------------------------------------------------------------------------------------------
  MOM            5815       5864      -49         6668       6668       +0
  BRK            2210       2279      -69         2634       2634       +0
  REV             632       

In [ ]:
# ============================================================
# K6b — Verifikasi tA_end: apakah H1 == K6?
# ============================================================
import os
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
data = pd.read_pickle(f"{BASE}/data_sealed.pkl")

idx_full = data["price_close"].index
n_bars_all = len(idx_full)  # 626
cut_idx = int(round(n_bars_all * 0.60))  # H1

tA_end_h1 = idx_full[cut_idx - 1]
tA_end_k6 = pd.Timestamp("2025-05-15")

print("=" * 100)
print("  VERIFIKASI tA_end")
print("=" * 100)
print(f"  n_bars_all         : {n_bars_all}")
print(f"  cut_idx (60% H1)   : {cut_idx}")
print(f"  tA_end (H1)        : {tA_end_h1.date()}")
print(f"  tA_end (K6/dokumen): {tA_end_k6.date()}")
print(f"  Sama?              : {'✅ YA' if tA_end_h1 == tA_end_k6 else '❌ TIDAK'}")
print()

if tA_end_h1 != tA_end_k6:
    # Hitung selisih bar
    pos_h1 = idx_full.get_loc(tA_end_h1)
    pos_k6 = idx_full.get_loc(tA_end_k6)
    print(f"  Posisi tA_end di idx_full:")
    print(f"    H1   : indeks {pos_h1}  ({tA_end_h1.date()})")
    print(f"    K6   : indeks {pos_k6}  ({tA_end_k6.date()})")
    print(f"    Δ    : {pos_h1 - pos_k6} bar")
    print()
    print("  Tanggal di sekitar batas:")
    lo = min(pos_h1, pos_k6) - 3
    hi = max(pos_h1, pos_k6) + 3
    for i in range(max(0, lo), min(len(idx_full), hi)):
        marker = ""
        if idx_full[i] == tA_end_h1: marker += " ← H1"
        if idx_full[i] == tA_end_k6: marker += " ← K6"
        print(f"    idx_full[{i}] = {idx_full[i].date()}{marker}")

print()
print("=" * 100)
print("  DIAGNOSA")
print("=" * 100)
print("  H1 pakai cut_idx = round(626*0.60) = 376, tA_end = idx_full[375]")
print("  K6 pakai tA_end = fixed date 2025-05-15")
print("  Kalau berbeda → H1 menggeser batas A/B beberapa hari → sinyal A berbeda.")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  VERIFIKASI tA_end
  n_bars_all         : 626
  cut_idx (60% H1)   : 376
  tA_end (H1)        : 2025-05-15
  tA_end (K6/dokumen): 2025-05-15
  Sama?              : ✅ YA


  DIAGNOSA
  H1 pakai cut_idx = round(626*0.60) = 376, tA_end = idx_full[375]
  K6 pakai tA_end = fixed date 2025-05-15
  Kalau berbeda → H1 menggeser batas A/B beberapa hari → sinyal A berbeda.


In [ ]:
# ============================================================
# K6c — Verifikasi: sinyal A dihitung dari 626 bar atau 601 bar?
# ============================================================
import os
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH = f"{BASE}/data_sealed.pkl"
EVENTS_PATH = f"{BASE}/data_events.csv"
CUTOFF_NEW = pd.Timestamp("2026-06-07")
MIN_PRICE = 200.0
MIN_TURNOVER_MA20 = 1_000_000_000.0
WARMUP_BARS = 25
WINDOW_BACK = 3
WINDOW_FWD = 25
tA_end = pd.Timestamp("2025-05-15")

data = pd.read_pickle(SEAL_PATH)
op_full = data["price_open"].copy().loc[data["price_open"].index <= CUTOFF_NEW]
hi_full = data["price_high"].copy().loc[data["price_high"].index <= CUTOFF_NEW]
lo_full = data["price_low"].copy().loc[data["price_low"].index <= CUTOFF_NEW]
cl_full = data["price_close"].copy().loc[data["price_close"].index <= CUTOFF_NEW]
vo_full = data["price_volume"].copy().loc[data["price_volume"].index <= CUTOFF_NEW]

df_events = pd.read_csv(EVENTS_PATH, parse_dates=["t0"])
idx_full = cl_full.index
for _, ev in df_events.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_full.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        op_full.at[d, tkr] = np.nan; hi_full.at[d, tkr] = np.nan
        lo_full.at[d, tkr] = np.nan; cl_full.at[d, tkr] = np.nan; vo_full.at[d, tkr] = np.nan

turnover = cl_full * vo_full
toma20 = turnover.rolling(20, min_periods=20).mean()
eligible_full = (cl_full >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

# ============ SKENARIO A: HITUNG SINYAL DARI 626 BAR (H1) ============
print("=" * 100)
print("  SKENARIO A — sinyal dihitung dari 626 bar (versi H1)")
print("=" * 100)

roc3 = cl_full / cl_full.shift(3) - 1.0
delta = cl_full.diff()
gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
ag = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
al = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
rs = ag / al.replace(0, np.nan)
rsi14 = 100 - 100/(1+rs)
rev = (rsi14.shift(1) < 30) & (cl_full > cl_full.shift(1))
high_prev10 = hi_full.shift(1).rolling(10, min_periods=10).max()
brk = cl_full > high_prev10
vol_ma20_prev = vo_full.shift(1).rolling(20, min_periods=20).mean()
gap = op_full / cl_full.shift(1) - 1.0
gapvol = (gap > 0.01) & (vo_full > 1.5 * vol_ma20_prev)

roc3_rank = roc3.where(eligible_full).rank(axis=1, pct=True)
mom_sig   = (roc3_rank >= 0.80) & roc3.notna() & eligible_full
brk_sig   = brk & eligible_full
rev_sig   = rev & eligible_full
gapvol_sig = gapvol & eligible_full

print("  Sinyal A (dari 626 bar, tanpa buang warm-up):")
for name, sig in [("MOM",mom_sig),("BRK",brk_sig),("REV",rev_sig),("GAPVOL",gapvol_sig)]:
    nA = int(sig.loc[sig.index <= tA_end].sum().sum())
    print(f"    {name:8s} A={nA:5d}")

# ============ SKENARIO B: HITUNG DARI 601 BAR (H2/K6) ============
print()
print("=" * 100)
print("  SKENARIO B — sinyal dihitung dari 601 bar (versi K6)")
print("=" * 100)

idx = idx_full[WARMUP_BARS:]
cl_w = cl_full.loc[idx]; hi_w = hi_full.loc[idx]; lo_w = lo_full.loc[idx]
op_w = op_full.loc[idx]; vo_w = vo_full.loc[idx]
elig_w = eligible_full.loc[idx]

roc3_w = cl_w / cl_w.shift(3) - 1.0
delta = cl_w.diff()
gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
ag = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
al = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
rs = ag / al.replace(0, np.nan)
rsi14_w = 100 - 100/(1+rs)
rev_w = (rsi14_w.shift(1) < 30) & (cl_w > cl_w.shift(1))
high_prev10_w = hi_w.shift(1).rolling(10, min_periods=10).max()
brk_w = cl_w > high_prev10_w
vol_ma20_prev_w = vo_w.shift(1).rolling(20, min_periods=20).mean()
gap_w = op_w / cl_w.shift(1) - 1.0
gapvol_w = (gap_w > 0.01) & (vo_w > 1.5 * vol_ma20_prev_w)

roc3_rank_w = roc3_w.where(elig_w).rank(axis=1, pct=True)
mom_sig_w   = (roc3_rank_w >= 0.80) & roc3_w.notna() & elig_w
brk_sig_w   = brk_w & elig_w
rev_sig_w   = rev_w & elig_w
gapvol_sig_w = gapvol_w & elig_w

print("  Sinyal A (dari 601 bar, buang warm-up dulu):")
for name, sig in [("MOM",mom_sig_w),("BRK",brk_sig_w),("REV",rev_sig_w),("GAPVOL",gapvol_sig_w)]:
    nA = int(sig.loc[sig.index <= tA_end].sum().sum())
    print(f"    {name:8s} A={nA:5d}")

# ============ BANDINGKAN ============
print()
print("=" * 100)
print("  BANDINGKAN & HITUNG SELISIH")
print("=" * 100)
print(f"  {'Sinyal':8s} {'Skenario A (626)':>16s} {'Skenario B (601)':>16s} {'Δ':>8s} {'Target H2':>10s}")
print("-" * 100)
targets = {"MOM": 5864, "BRK": 2279, "REV": 638, "GAPVOL": 827}
for name in ["MOM", "BRK", "REV", "GAPVOL"]:
    sigA = {"MOM":mom_sig,"BRK":brk_sig,"REV":rev_sig,"GAPVOL":gapvol_sig}[name]
    sigB = {"MOM":mom_sig_w,"BRK":brk_sig_w,"REV":rev_sig_w,"GAPVOL":gapvol_sig_w}[name]
    nA_626 = int(sigA.loc[sigA.index <= tA_end].sum().sum())
    nA_601 = int(sigB.loc[sigB.index <= tA_end].sum().sum())
    print(f"  {name:8s} {nA_626:>16d} {nA_601:>16d} {nA_601-nA_626:>+8d} {targets[name]:>10d}")

print()
print("=" * 100)
print("  KESIMPULAN")
print("=" * 100)
print("  Kalau Skenario A = Target H2 → H1 menghitung dari 626 bar")
print("  Kalau Skenario B = hasil K6   → K6 menghitung dari 601 bar")
print("  Penyebab ΔA: URUTAN operasi (warm-up vs hitung sinyal)")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  SKENARIO A — sinyal dihitung dari 626 bar (versi H1)
  Sinyal A (dari 626 bar, tanpa buang warm-up):
    MOM      A= 5962
    BRK      A= 2305
    REV      A=  661
    GAPVOL   A=  833

  SKENARIO B — sinyal dihitung dari 601 bar (versi K6)
  Sinyal A (dari 601 bar, buang warm-up dulu):
    MOM      A= 5815
    BRK      A= 2210
    REV      A=  632
    GAPVOL   A=  789

  BANDINGKAN & HITUNG SELISIH
  Sinyal   Skenario A (626) Skenario B (601)        Δ  Target H2
----------------------------------------------------------------------------------------------------
  MOM                  5962             5815     -147       5864
  BRK                  2305             2210      -95       2279
  REV                   661              632      -29        638
  GAPVOL                833              789      -44        827

  KESIMPULAN
  Kalau Skenario A = Targe

In [ ]:
# Buang warm-up: 25 bar pertama
idx = idx_all[WARMUP_BARS:]
op = op.loc[idx]; ...; eligible_base = eligible_base.loc[idx]

# FITUR (setelah buang warm-up)
roc3 = cl / cl.shift(3) - 1.0
...

NameError: name 'idx_all' is not defined

In [ ]:
# ============================================================
# K6d — Cari aturan yang menghasilkan target H2 (5864 MOM A)
# Uji 4 kombinasi: (sumber eligible, sumber sinyal, cutoff A)
# ============================================================
import os
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
data = pd.read_pickle(f"{BASE}/data_sealed.pkl")
EVENTS = pd.read_csv(f"{BASE}/data_events.csv", parse_dates=["t0"])
CUTOFF_NEW = pd.Timestamp("2026-06-07")
MIN_PRICE = 200.0; MIN_TURNOVER_MA20 = 1e9
WARMUP_BARS = 25; WINDOW_BACK = 3; WINDOW_FWD = 25
tA_end = pd.Timestamp("2025-05-15")

op_full = data["price_open"].copy().loc[data["price_open"].index <= CUTOFF_NEW]
hi_full = data["price_high"].copy().loc[data["price_high"].index <= CUTOFF_NEW]
lo_full = data["price_low"].copy().loc[data["price_low"].index <= CUTOFF_NEW]
cl_full = data["price_close"].copy().loc[data["price_close"].index <= CUTOFF_NEW]
vo_full = data["price_volume"].copy().loc[data["price_volume"].index <= CUTOFF_NEW]
idx_full = cl_full.index
for _, ev in EVENTS.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_full.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        op_full.at[d, tkr] = np.nan; hi_full.at[d, tkr] = np.nan
        lo_full.at[d, tkr] = np.nan; cl_full.at[d, tkr] = np.nan; vo_full.at[d, tkr] = np.nan

turnover = cl_full * vo_full
toma20 = turnover.rolling(20, min_periods=20).mean()
eligible_full = (cl_full >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

# Siapkan idx (buang warm-up)
idx = idx_full[WARMUP_BARS:]

# Hitung fitur dari 626 bar
def build_features(op, hi, lo, cl, vo, elig):
    roc3 = cl / cl.shift(3) - 1.0
    delta = cl.diff()
    gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
    ag = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    al = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
    rs = ag / al.replace(0, np.nan)
    rsi14 = 100 - 100/(1+rs)
    rev = (rsi14.shift(1) < 30) & (cl > cl.shift(1))
    high_prev10 = hi.shift(1).rolling(10, min_periods=10).max()
    brk = cl > high_prev10
    vma20p = vo.shift(1).rolling(20, min_periods=20).mean()
    gap = op / cl.shift(1) - 1.0
    gapvol = (gap > 0.01) & (vo > 1.5 * vma20p)
    roc3_rank = roc3.where(elig).rank(axis=1, pct=True)
    mom = (roc3_rank >= 0.80) & roc3.notna() & elig
    return mom, brk & elig, rev & elig, gapvol & elig

mom, brk, rev, gapvol = build_features(op_full, hi_full, lo_full, cl_full, vo_full, eligible_full)

# Kombinasi 1: sinyal dari 626, A = idx_full <= tA_end (626 cutoff)
print("=" * 100)
print("  UJI 4 KOMBINASI — cari mana yang hasilkan MOM A = 5864")
print("=" * 100)

def count_A(sig, idx_cut):
    return int(sig.loc[sig.index.isin(idx_cut) & (sig.index <= tA_end)].sum().sum())

print(f"  {'Kombinasi':40s} {'MOM':>8s} {'BRK':>8s} {'REV':>8s} {'GAPVOL':>8s}")
print("-" * 100)

# K1: cutoff A pakai idx_full (626), sinyal dari 626
idx_A_626 = idx_full[idx_full <= tA_end]
print(f"  {'A=626 cutoff (idx_full), sinyal 626':40s} "
      f"{count_A(mom, idx_A_626):>8d} {count_A(brk, idx_A_626):>8d} "
      f"{count_A(rev, idx_A_626):>8d} {count_A(gapvol, idx_A_626):>8d}")

# K2: cutoff A pakai idx (601), sinyal dari 626
idx_A_601 = idx[idx <= tA_end]
print(f"  {'A=601 cutoff (idx), sinyal 626':40s} "
      f"{count_A(mom, idx_A_601):>8d} {count_A(brk, idx_A_601):>8d} "
      f"{count_A(rev, idx_A_601):>8d} {count_A(gapvol, idx_A_601):>8d}")

# K3: cutoff A pakai 60% dari 626 = 376 bar (H1 cara asli)
cut_376 = idx_full[:int(round(626 * 0.60))]
tA_376 = cut_376[-1]
print(f"  {'A=376 bar (60% dari 626), sinyal 626':40s} "
      f"{int(mom.loc[mom.index.isin(cut_376)].sum().sum()):>8d} "
      f"{int(brk.loc[brk.index.isin(cut_376)].sum().sum()):>8d} "
      f"{int(rev.loc[rev.index.isin(cut_376)].sum().sum()):>8d} "
      f"{int(gapvol.loc[gapvol.index.isin(cut_376)].sum().sum()):>8d}")

# K4: cutoff A pakai 60% dari 601 = 361 bar (setelah warm-up, H2 style)
cut_361 = idx[:int(round(601 * 0.60))]
print(f"  {'A=361 bar (60% dari 601), sinyal 626':40s} "
      f"{int(mom.loc[mom.index.isin(cut_361)].sum().sum()):>8d} "
      f"{int(brk.loc[brk.index.isin(cut_361)].sum().sum()):>8d} "
      f"{int(rev.loc[rev.index.isin(cut_361)].sum().sum()):>8d} "
      f"{int(gapvol.loc[gapvol.index.isin(cut_361)].sum().sum()):>8d}")

print()
print(f"  {'TARGET H2':40s} {5864:>8d} {2279:>8d} {638:>8d} {827:>8d}")
print("=" * 100)
print()
print("  Kesimpulan: baris yang paling dekat dengan TARGET H2 = aturan yang dipakai H1")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  UJI 4 KOMBINASI — cari mana yang hasilkan MOM A = 5864
  Kombinasi                                     MOM      BRK      REV   GAPVOL
----------------------------------------------------------------------------------------------------
  A=626 cutoff (idx_full), sinyal 626          5962     2305      661      833
  A=601 cutoff (idx), sinyal 626               5864     2279      638      827
  A=376 bar (60% dari 626), sinyal 626         5962     2305      661      833
  A=361 bar (60% dari 601), sinyal 626         6043     2348      643      861

  TARGET H2                                    5864     2279      638      827

  Kesimpulan: baris yang paling dekat dengan TARGET H2 = aturan yang dipakai H1


In [ ]:
# ============================================================
# K7 — Perbaiki urutan operasi di Tahap2_Replikasi.py
# Fitur dihitung DARI 626 bar dulu, baru potong ke idx (601 bar)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- PATCH: Ganti blok "fitur dari idx" menjadi "fitur dari full, potong ke idx" ----------
# Blok lama di run_8_uji, di mana sinyal dibangun dari op/hi/lo/cl/vo (yang sudah dipotong)
# Kita perlu: bangun sinyal dari data penuh, lalu potong

# Cari di dalam fungsi run_8_uji: 'signals = build_features(op, hi, lo, cl, vo, eligible_base)'
OLD_INNER = 'signals = build_features(op, hi, lo, cl, vo, eligible_base)'

# Solusi: hitung sinyal di luar fungsi (dari data penuh), lalu oper ke run_8_uji
# Kita ubah signature run_8_uji supaya terima `signals` sebagai parameter opsional

OLD_FUNC_HEAD = 'def run_8_uji(idx, op, hi, lo, cl, vo, eligible_base, tC1_end, tC2_start,\n              label1="C1", label2="C2"):\n    signals = build_features(op, hi, lo, cl, vo, eligible_base)'
NEW_FUNC_HEAD = '''def run_8_uji(idx, op, hi, lo, cl, vo, eligible_base, tC1_end, tC2_start,
              label1="C1", label2="C2", signals=None):
    if signals is None:
        signals = build_features(op, hi, lo, cl, vo, eligible_base)
    else:
        # Potong sinyal ke idx yang dipakai
        signals = {k: v.loc[idx] for k, v in signals.items()}'''

if OLD_FUNC_HEAD in content:
    content = content.replace(OLD_FUNC_HEAD, NEW_FUNC_HEAD)
    print("✅ Fungsi run_8_uji diperbarui (terima signals opsional)")
else:
    print("⚠️ Header fungsi tidak cocok persis, cek manual.")

# ---------- PATCH: Di Langkah 0, hitung sinyal dari data penuh, baru potong ----------
# Di blok Langkah 0, ada kode yang memotong op/hi/lo/cl/vo ke idx, lalu memanggil run_8_uji

OLD_L0 = '''results_L0, signals_L0, total_sig_L0, total_drop_L0 = run_8_uji(
    idx, op, hi, lo, cl, vo, eligible, tA_end, tB_start, label1="A", label2="B")'''

NEW_L0 = '''# Hitung sinyal dari data PENUH (sebelum buang warm-up), lalu potong ke idx
signals_full_L0 = build_features(op_f, hi_f, lo_f, cl_f, vo_f, eligible_full)
results_L0, signals_L0, total_sig_L0, total_drop_L0 = run_8_uji(
    idx, op, hi, lo, cl, vo, eligible, tA_end, tB_start, label1="A", label2="B",
    signals=signals_full_L0)'''

if OLD_L0 in content:
    content = content.replace(OLD_L0, NEW_L0)
    print("✅ Langkah 0 diperbarui (sinyal dari data penuh)")
else:
    print("⚠️ Blok Langkah 0 tidak cocok persis.")
    # Cari versi alternatif
    if 'run_8_uji(\n    idx, op, hi, lo, cl, vo, eligible, tA_end, tB_start' in content:
        print("     Versi alternatif ditemukan, perlu inspeksi manual.")

# ---------- PATCH: Di Bagian B, hitung sinyal dari data penuh juga ----------
OLD_B = '''    results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
        idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
        tC1_end, tC2_start, label1="C1", label2="C2")'''

NEW_B = '''    # Hitung sinyal dari data penuh (sebelum buang warm-up), lalu potong
    signals_full_B = build_features(op_old_seal, hi_old_seal, lo_old_seal,
                                    cl_old_seal, vo_old_seal, elig_old_seal_for_signals)
    results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
        idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
        tC1_end, tC2_start, label1="C1", label2="C2",
        signals=signals_full_B)'''

if OLD_B in content:
    content = content.replace(OLD_B, NEW_B)
    print("✅ Bagian B diperbarui (sinyal dari data penuh)")
else:
    print("⚠️ Blok Bagian B tidak cocok persis, cek manual.")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")
print(f"  Baris  : {len(content.splitlines())}")

# Cek syntax
import ast
try:
    ast.parse(content)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Fungsi run_8_uji diperbarui (terima signals opsional)
✅ Langkah 0 diperbarui (sinyal dari data penuh)
✅ Bagian B diperbarui (sinyal dari data penuh)

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 26522 byte
  Baris  : 626
  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 2 — Tahap2_Replikasi.py

  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)
  Bar dipakai       : 601
  Event dimuat      : 56  |  Sel NaN: 539
  Saham layak/hari  : mean=101.4 median=85.0

  MOM     N= 5 A n= 5864 exc_w=-0.1155% p=0.5140
  MOM     N= 5 B n= 6551 exc_w=+0.0591% p=0.6971
  MOM     N=10 A n= 5864 exc_w=-0.0442% p=0.9005
  MOM     N=10 B n= 6427 exc_w=+0.1841% p=0.4702
  BRK     N= 5 A n= 2279 exc_w=+0.4120% p=0.6463
  BRK     N= 5 B n= 2608 exc_w=+1.1594% p=0.0088
  BRK     N=10 A n= 2279 exc_w=+0.6503% p=0.6387
  BRK     N=10 B n= 2601 exc_w=+0.6795% p=0.1906
  REV     N= 5 A n=  638 exc_w=-0.3195% p=0.1528
  REV     N= 5 B n=  529 exc_w=-0.7616% p=0.0664
  REV     N=10 A n=  638 exc_w=-0.7389% p=0.0817
  REV     N=10 B n=  480 exc_w=-2.2968% p=0.0000
  GAPVOL  N= 5 A n=  827 exc_w=-0.7063% p=0.2686
  GAPVOL  N= 5 B n= 1774 exc_w=-0.9726% p=0.2157
  GAPVOL  N=10 A n=  827 exc_w=-1.1576%

In [ ]:
# ============================================================
# K8 — Perbaiki Gerbang A2: pisahkan NaN pembersihan vs NaN total
# ============================================================
import os
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

# Load data lama yang baru saja diunduh
PROGRESS_PATH = f"{BASE}/_download_old_progress.pkl"
if not os.path.exists(PROGRESS_PATH):
    print("❌ _download_old_progress.pkl tidak ada. Jalankan ulang unduh.")
    import sys; sys.exit(1)

prog = pd.read_pickle(PROGRESS_PATH)
op_old = pd.DataFrame(prog["op"]).sort_index()
hi_old = pd.DataFrame(prog["hi"]).sort_index()
lo_old = pd.DataFrame(prog["lo"]).sort_index()
cl_old = pd.DataFrame(prog["cl"]).sort_index()
vo_old = pd.DataFrame(prog["vo"]).sort_index()

# Batasi ≤ 2023-10-09
mask_old = cl_old.index <= pd.Timestamp("2023-10-09")
op_old = op_old.loc[mask_old]; hi_old = hi_old.loc[mask_old]
lo_old = lo_old.loc[mask_old]; cl_old = cl_old.loc[mask_old]
vo_old = vo_old.loc[mask_old]

print("=" * 100)
print("  K8 — PERBAIKI GERBANG A2")
print("=" * 100)
print(f"  Data lama: {cl_old.shape}, range {cl_old.index.min().date()} → {cl_old.index.max().date()}")
print()

# Replikasi pembersihan
WINDOW_BACK = 3; WINDOW_FWD = 25
EVENTS_PATH = f"{BASE}/data_events_old.csv"
df_events_old = pd.read_csv(EVENTS_PATH, parse_dates=["t0"]) if os.path.exists(EVENTS_PATH) else pd.DataFrame()

# Mask NaN asli (sebelum pembersihan)
nan_original_mask = cl_old.isna()

# Pembersihan: track mana sel yang JADI NaN karena pembersihan
op_old_c = op_old.copy(); hi_old_c = hi_old.copy(); lo_old_c = lo_old.copy()
cl_old_c = cl_old.copy(); vo_old_c = vo_old.copy()
idx_old = cl_old_c.index

cleaning_nan_mask = pd.DataFrame(False, index=cl_old_c.index, columns=cl_old_c.columns)

# Event split
for _, ev in df_events_old.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_old_c.columns: continue
    try: i0 = idx_old.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_old)-1, i0+WINDOW_FWD)+1):
        d = idx_old[j]
        cleaning_nan_mask.at[d, tkr] = True
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan

# Pelanggaran O/H/L/C
viol_mask = (lo_old_c > np.minimum(op_old_c, cl_old_c)) | (hi_old_c < np.maximum(op_old_c, cl_old_c))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        cleaning_nan_mask.at[d, tkr] = True
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan

# Filter layak
MIN_PRICE = 200.0; MIN_TURNOVER_MA20 = 1e9
turnover_old = cl_old_c * vo_old_c
toma20_old = turnover_old.rolling(20, min_periods=20).mean()
elig_old = (cl_old_c >= MIN_PRICE) & (toma20_old >= MIN_TURNOVER_MA20)
n_elig_old = elig_old.sum(axis=1)

# Tanggal mulai efektif
ma60 = n_elig_old.rolling(60, min_periods=60).mean()
t_eff_candidates = ma60.index[ma60 >= 60]
t_eff = t_eff_candidates[0] if len(t_eff_candidates) else None
print(f"  Tanggal mulai efektif: {t_eff.date() if t_eff else 'TIDAK ADA'}")
print()

if t_eff is None:
    print("❌ Tidak ada tanggal efektif. STOP.")
    import sys; sys.exit(1)

mask_eff = cl_old_c.index >= t_eff
n_eff_bars = int(mask_eff.sum())

# ============================================================
# HITUNG ULANG GERBANG A2 — PISAHKAN JENIS NaN
# ============================================================
print("=" * 100)
print("  ANALISIS NaN (setelah t_eff)")
print("=" * 100)

# Total sel
total_cells = int(cl_old_c.loc[mask_eff].size)
print(f"  Total sel di periode efektif: {total_cells:,}")

# NaN yang JADI karena pembersihan (dari mask kita)
cleaning_nan_eff = int(cleaning_nan_mask.loc[mask_eff].sum().sum())
print(f"  NaN AKIBAT PEMBERSIHAN (event + viol): {cleaning_nan_eff:,}")

# NaN asli (sudah ada sebelum pembersihan)
original_nan_eff = int(nan_original_mask.loc[mask_eff].sum().sum())
print(f"  NaN ASLI (saham belum listing/data tipis): {original_nan_eff:,}")

# NaN total
total_nan_eff = int(cl_old_c.loc[mask_eff].isna().sum().sum())
print(f"  NaN TOTAL: {total_nan_eff:,}")
print()

pct_cleaning = 100 * cleaning_nan_eff / total_cells
pct_total = 100 * total_nan_eff / total_cells

print(f"  % NaN akibat pembersihan: {pct_cleaning:.4f}%")
print(f"  % NaN total            : {pct_total:.4f}%")
print()

# ============================================================
# GERBANG A (revisi)
# ============================================================
g1 = n_eff_bars >= 500
g2 = pct_cleaning <= 5
g3 = float(n_elig_old[mask_eff].mean()) >= 60

print("=" * 100)
print("  GERBANG A (revisi, ADDENDUM 6)")
print("=" * 100)
print(f"  A1 (bar efektif >= 500)        : {n_eff_bars} → {'LOLOS' if g1 else 'GAGAL'}")
print(f"  A2 (NaN pembersihan <= 5%)     : {pct_cleaning:.4f}% → {'LOLOS' if g2 else 'GAGAL'}")
print(f"  A3 (layak/hari >= 60)          : {float(n_elig_old[mask_eff].mean()):.1f} → {'LOLOS' if g3 else 'GAGAL'}")
print()
print(f"  GERBANG A: {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  K8 — PERBAIKI GERBANG A2
  Data lama: (4643, 486), range 2005-01-03 → 2023-10-09

  Tanggal mulai efektif: 2021-08-02

  ANALISIS NaN (setelah t_eff)
  Total sel di periode efektif: 259,524
  NaN AKIBAT PEMBERSIHAN (event + viol): 339
  NaN ASLI (saham belum listing/data tipis): 43,403
  NaN TOTAL: 43,742

  % NaN akibat pembersihan: 0.1306%
  % NaN total            : 16.8547%

  GERBANG A (revisi, ADDENDUM 6)
  A1 (bar efektif >= 500)        : 534 → LOLOS
  A2 (NaN pembersihan <= 5%)     : 0.1306% → LOLOS
  A3 (layak/hari >= 60)          : 80.9 → LOLOS

  GERBANG A: LOLOS


In [ ]:
# ============================================================
# K9 — Perbaiki Tahap2_Replikasi.py: A2 hitung NaN pembersihan saja
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- PATCH: Ganti blok hitung Gerbang A2 ----------
OLD_BLOCK = '''mask_eff = cl_old_c.index >= t_eff
n_eff_bars = int(mask_eff.sum())
total_cells = int(cl_old_c.loc[mask_eff].size)
nan_cells = int(cl_old_c.loc[mask_eff].isna().sum().sum())
pct_nan = 100 * nan_cells / total_cells if total_cells else 0
n_elig_eff_mean = float(n_elig_old[mask_eff].mean())

g1 = n_eff_bars >= 650
g2 = pct_nan <= 5
g3 = n_elig_eff_mean >= 60
log(f"  Gerbang A1 (bar efektif >= 650): {n_eff_bars} → {'LOLOS' if g1 else 'GAGAL'}")
log(f"  Gerbang A2 (NaN <= 5%): {pct_nan:.2f}% → {'LOLOS' if g2 else 'GAGAL'}")
log(f"  Gerbang A3 (layak/hari >= 60): {n_elig_eff_mean:.1f} → {'LOLOS' if g3 else 'GAGAL'}")'''

NEW_BLOCK = '''mask_eff = cl_old_c.index >= t_eff
n_eff_bars = int(mask_eff.sum())
total_cells = int(cl_old_c.loc[mask_eff].size)

# NaN yang JADI karena pembersihan (event + pelanggaran O/H/L/C)
cleaning_nan_eff = int(cleaning_nan_mask.loc[mask_eff].sum().sum())
pct_cleaning = 100 * cleaning_nan_eff / total_cells if total_cells else 0

# NaN total (termasuk saham belum listing/data tipis) — untuk laporan saja
nan_total_eff = int(cl_old_c.loc[mask_eff].isna().sum().sum())
pct_nan_total = 100 * nan_total_eff / total_cells if total_cells else 0

n_elig_eff_mean = float(n_elig_old[mask_eff].mean())

g1 = n_eff_bars >= 500
g2 = pct_cleaning <= 5
g3 = n_elig_eff_mean >= 60
log(f"  Gerbang A1 (bar efektif >= 500): {n_eff_bars} → {'LOLOS' if g1 else 'GAGAL'}")
log(f"  Gerbang A2 (NaN pembersihan <= 5%): {pct_cleaning:.4f}% → {'LOLOS' if g2 else 'GAGAL'}")
log(f"  Gerbang A2b (NaN total, info saja): {pct_nan_total:.4f}%")
log(f"  Gerbang A3 (layak/hari >= 60): {n_elig_eff_mean:.1f} → {'LOLOS' if g3 else 'GAGAL'}")'''

if OLD_BLOCK in content:
    content = content.replace(OLD_BLOCK, NEW_BLOCK)
    print("✅ Blok Gerbang A diperbarui")
else:
    print("⚠️ Blok Gerbang A tidak cocok persis")

# ---------- PATCH: Tambah cleaning_nan_mask di awal blok Bagian A ----------
# Cari setelah 'op_old_c = op_old.copy()'
OLD_SETUP = '''op_old_c = op_old.copy(); hi_old_c = hi_old.copy(); lo_old_c = lo_old.copy()
cl_old_c = cl_old.copy(); vo_old_c = vo_old.copy()
idx_old = cl_old_c.index'''

NEW_SETUP = '''op_old_c = op_old.copy(); hi_old_c = hi_old.copy(); lo_old_c = lo_old.copy()
cl_old_c = cl_old.copy(); vo_old_c = vo_old.copy()
idx_old = cl_old_c.index

# Mask untuk tracking NaN yang JADI akibat pembersihan (bukan NaN asli)
cleaning_nan_mask = pd.DataFrame(False, index=cl_old_c.index, columns=cl_old_c.columns)'''

if OLD_SETUP in content:
    content = content.replace(OLD_SETUP, NEW_SETUP)
    print("✅ cleaning_nan_mask ditambahkan")
else:
    print("⚠️ Setup blok tidak cocok persis")

# ---------- PATCH: Tandai cleaning_nan_mask di loop event split ----------
OLD_EVENT_LOOP = '''for _, ev in df_events_old.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_old_c.columns: continue
    try: i0 = idx_old.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_old)-1, i0+WINDOW_FWD)+1):
        d = idx_old[j]
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan'''

NEW_EVENT_LOOP = '''for _, ev in df_events_old.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_old_c.columns: continue
    try: i0 = idx_old.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_old)-1, i0+WINDOW_FWD)+1):
        d = idx_old[j]
        cleaning_nan_mask.at[d, tkr] = True
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan'''

if OLD_EVENT_LOOP in content:
    content = content.replace(OLD_EVENT_LOOP, NEW_EVENT_LOOP)
    print("✅ Loop event split: cleaning_nan_mask ditandai")
else:
    print("⚠️ Loop event split tidak cocok persis")

# ---------- PATCH: Tandai cleaning_nan_mask di loop viol ----------
OLD_VIOL_LOOP = '''viol_mask = (lo_old_c > np.minimum(op_old_c, cl_old_c)) | (hi_old_c < np.maximum(op_old_c, cl_old_c))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan'''

NEW_VIOL_LOOP = '''viol_mask = (lo_old_c > np.minimum(op_old_c, cl_old_c)) | (hi_old_c < np.maximum(op_old_c, cl_old_c))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        cleaning_nan_mask.at[d, tkr] = True
        op_old_c.at[d, tkr] = np.nan; hi_old_c.at[d, tkr] = np.nan
        lo_old_c.at[d, tkr] = np.nan; cl_old_c.at[d, tkr] = np.nan; vo_old_c.at[d, tkr] = np.nan'''

if OLD_VIOL_LOOP in content:
    content = content.replace(OLD_VIOL_LOOP, NEW_VIOL_LOOP)
    print("✅ Loop viol: cleaning_nan_mask ditandai")
else:
    print("⚠️ Loop viol tidak cocok persis")

# ---------- PATCH: Ganti sys.exit(3) jadi tidak exit (karena Gerbang A bisa lolos) ----------
content = content.replace(
    'if len(t_eff_candidates) == 0:\n    log("STOP: tidak ada tanggal dengan rata-rata 60 hari >= 60 saham layak.")\n    sys.exit(3)',
    'if len(t_eff_candidates) == 0:\n    log("STOP: tidak ada tanggal dengan rata-rata 60 hari >= 60 saham layak.")\n    sys.exit(3)'
)  # tidak berubah, hanya memastikan

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi
with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("cleaning_nan_mask muncul", new.count("cleaning_nan_mask")),
    ("NaN pembersihan <= 5%", new.count("NaN pembersihan")),
    ("bar efektif >= 500", new.count("bar efektif >= 500")),
    ("NaN total (info)", new.count("NaN total")),
]
print()
print("  Verifikasi patch:")
for label, count in checks:
    print(f"    {label}: {count}×")

# Syntax check
try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Blok Gerbang A diperbarui
✅ cleaning_nan_mask ditambahkan
✅ Loop event split: cleaning_nan_mask ditandai
✅ Loop viol: cleaning_nan_mask ditandai

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 27161 byte

  Verifikasi patch:
    cleaning_nan_mask muncul: 4×
    NaN pembersihan <= 5%: 1×
    bar efektif >= 500: 1×
    NaN total (info): 2×

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 2 — Tahap2_Replikasi.py

  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)
  Bar dipakai       : 601
  Event dimuat      : 56  |  Sel NaN: 539
  Saham layak/hari  : mean=101.4 median=85.0

  MOM     N= 5 A n= 5864 exc_w=-0.1155% p=0.5140
  MOM     N= 5 B n= 6551 exc_w=+0.0591% p=0.6971
  MOM     N=10 A n= 5864 exc_w=-0.0442% p=0.9005
  MOM     N=10 B n= 6427 exc_w=+0.1841% p=0.4702
  BRK     N= 5 A n= 2279 exc_w=+0.4120% p=0.6463
  BRK     N= 5 B n= 2608 exc_w=+1.1594% p=0.0088
  BRK     N=10 A n= 2279 exc_w=+0.6503% p=0.6387
  BRK     N=10 B n= 2601 exc_w=+0.6795% p=0.1906
  REV     N= 5 A n=  638 exc_w=-0.3195% p=0.1528
  REV     N= 5 B n=  529 exc_w=-0.7616% p=0.0664
  REV     N=10 A n=  638 exc_w=-0.7389% p=0.0817
  REV     N=10 B n=  480 exc_w=-2.2968% p=0.0000
  GAPVOL  N= 5 A n=  827 exc_w=-0.7063% p=0.2686
  GAPVOL  N= 5 B n= 1774 exc_w=-0.9726% p=0.2157
  GAPVOL  N=10 A n=  827 exc_w=-1.1576%

In [ ]:
# ============================================================
# K10 — Perbaiki bug elig_old_seal_for_signals di Bagian B
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Baris bermasalah:
# signals_full_B = build_features(op_old_seal, hi_old_seal, lo_old_seal,
#                                 cl_old_seal, vo_old_seal, elig_old_seal_for_signals)

OLD = '''signals_full_B = build_features(op_old_seal, hi_old_seal, lo_old_seal,
                                    cl_old_seal, vo_old_seal, elig_old_seal_for_signals)'''

NEW = '''# Gunakan eligible dari data penuh (sebelum buang warm-up)
elig_old_seal_for_signals = elig_old.loc[cl_old_seal.index]
# Note: cl_old_seal sudah dipotong warm-up; untuk sinyal dari data penuh,
# kita perlu pakai elig_old (yang sudah dihitung dari cl_old_c)

signals_full_B = build_features(op_old_c, hi_old_c, lo_old_c,
                                cl_old_c, vo_old_c, elig_old)'''

if OLD in content:
    content = content.replace(OLD, NEW)
    print("✅ Bug elig_old_seal_for_signals diperbaiki")
else:
    print("⚠️ Baris lama tidak cocok persis")
    # Coba cari baris lain
    import re
    if "elig_old_seal_for_signals" in content:
        print(f"     'elig_old_seal_for_signals' masih muncul di {content.count('elig_old_seal_for_signals')} tempat")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi
with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'elig_old_seal_for_signals' muncul: {new.count('elig_old_seal_for_signals')}x")

# Cek blok baru
print()
print("  Blok baru di sekitar baris itu:")
lines = new.splitlines()
for i, line in enumerate(lines):
    if "signals_full_B" in line:
        for j in range(max(0, i-5), min(len(lines), i+5)):
            print(f"  L{j+1:>4d}: {lines[j]}")
        break

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Bug elig_old_seal_for_signals diperbaiki

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 27383 byte
  'elig_old_seal_for_signals' muncul: 1x

  Blok baru di sekitar baris itu:
  L 569:     # Gunakan eligible dari data penuh (sebelum buang warm-up)
  L 570: elig_old_seal_for_signals = elig_old.loc[cl_old_seal.index]
  L 571: # Note: cl_old_seal sudah dipotong warm-up; untuk sinyal dari data penuh,
  L 572: # kita perlu pakai elig_old (yang sudah dihitung dari cl_old_c)
  L 573: 
  L 574: signals_full_B = build_features(op_old_c, hi_old_c, lo_old_c,
  L 575:                                 cl_old_c, vo_old_c, elig_old)
  L 576:     results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
  L 577:         idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
  L 578:         tC1_end, tC2_start, lab

In [ ]:
# ============================================================
# K10b — Perbaiki indentasi blok Bagian B
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- Perbaiki blok yang rusak ----------
OLD_BLOCK = '''    # Gunakan eligible dari data penuh (sebelum buang warm-up)
elig_old_seal_for_signals = elig_old.loc[cl_old_seal.index]
# Note: cl_old_seal sudah dipotong warm-up; untuk sinyal dari data penuh,
# kita perlu pakai elig_old (yang sudah dihitung dari cl_old_c)

signals_full_B = build_features(op_old_c, hi_old_c, lo_old_c,
                                cl_old_c, vo_old_c, elig_old)
    results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
        idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
        tC1_end, tC2_start, label1="C1", label2="C2",
        signals=signals_full_B)'''

NEW_BLOCK = '''    # Hitung sinyal dari data penuh (sebelum buang warm-up), lalu potong
    signals_full_B = build_features(op_old_c, hi_old_c, lo_old_c,
                                    cl_old_c, vo_old_c, elig_old)
    results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
        idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
        tC1_end, tC2_start, label1="C1", label2="C2",
        signals=signals_full_B)'''

if OLD_BLOCK in content:
    content = content.replace(OLD_BLOCK, NEW_BLOCK)
    print("✅ Blok Bagian B diperbaiki")
else:
    print("⚠️ Blok tidak cocok persis. Mencari alternatif...")
    # Cari blok serupa
    lines = content.splitlines()
    # Cari baris dengan 'elig_old_seal_for_signals'
    for i, line in enumerate(lines):
        if "elig_old_seal_for_signals" in line:
            print(f"  Baris {i+1}: {line}")
            # Tampilkan 10 baris sekitar
            for j in range(max(0, i-3), min(len(lines), i+10)):
                print(f"    L{j+1}: {repr(lines[j])}")
            break

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")

# Cek blok baru
print()
print("  Blok di sekitar signals_full_B:")
with open(SCRIPT_PATH) as f:
    new = f.read()
lines = new.splitlines()
for i, line in enumerate(lines):
    if "signals_full_B" in line:
        for j in range(max(0, i-4), min(len(lines), i+8)):
            print(f"  L{j+1:>4d}: {lines[j]}")
        break

# Syntax
try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Blok Bagian B diperbaiki

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 27202 byte

  Blok di sekitar signals_full_B:
  L 566:         sys.exit(4)
  L 567: 
  L 568:     # Hitung sinyal dari data penuh (sebelum buang warm-up), lalu potong
  L 569:     # Hitung sinyal dari data penuh (sebelum buang warm-up), lalu potong
  L 570:     signals_full_B = build_features(op_old_c, hi_old_c, lo_old_c,
  L 571:                                     cl_old_c, vo_old_c, elig_old)
  L 572:     results_B, signals_B, total_sig_B, total_drop_B = run_8_uji(
  L 573:         idx_b2, op_b, hi_b, lo_b, cl_b, vo_b, elig_b,
  L 574:         tC1_end, tC2_start, label1="C1", label2="C2",
  L 575:         signals=signals_full_B)
  L 576: 
  L 577:     def cek_B5(sig, N):

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 2 — Tahap2_Replikasi.py

  LANGKAH 0 — REPRODUKSI pada data_sealed.pkl (Tahap 1b)
  Bar dipakai       : 601
  Event dimuat      : 56  |  Sel NaN: 539
  Saham layak/hari  : mean=101.4 median=85.0

  MOM     N= 5 A n= 5864 exc_w=-0.1155% p=0.5140
  MOM     N= 5 B n= 6551 exc_w=+0.0591% p=0.6971
  MOM     N=10 A n= 5864 exc_w=-0.0442% p=0.9005
  MOM     N=10 B n= 6427 exc_w=+0.1841% p=0.4702
  BRK     N= 5 A n= 2279 exc_w=+0.4120% p=0.6463
  BRK     N= 5 B n= 2608 exc_w=+1.1594% p=0.0088
  BRK     N=10 A n= 2279 exc_w=+0.6503% p=0.6387
  BRK     N=10 B n= 2601 exc_w=+0.6795% p=0.1906
  REV     N= 5 A n=  638 exc_w=-0.3195% p=0.1528
  REV     N= 5 B n=  529 exc_w=-0.7616% p=0.0664
  REV     N=10 A n=  638 exc_w=-0.7389% p=0.0817
  REV     N=10 B n=  480 exc_w=-2.2968% p=0.0000
  GAPVOL  N= 5 A n=  827 exc_w=-0.7063% p=0.2686
  GAPVOL  N= 5 B n= 1774 exc_w=-0.9726% p=0.2157
  GAPVOL  N=10 A n=  827 exc_w=-1.1576%

In [ ]:
# ============================================================
# K11 — Perbaiki unpack cek_B5 untuk MOM (kontrol negatif)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap2_Replikasi.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

OLD = '''    mom_alarm = False
    for N in N_LIST:
        p, _ = cek_B5("MOM", N)
        if p: mom_alarm = True'''

NEW = '''    mom_alarm = False
    for N in N_LIST:
        passed_mom, fails_mom, _, _ = cek_B5("MOM", N)
        if passed_mom: mom_alarm = True'''

if OLD in content:
    content = content.replace(OLD, NEW)
    print("✅ Bug cek_B5 MOM diperbaiki")
else:
    print("⚠️ Blok tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 90)
print("  FILE DIPERBARUI")
print("=" * 90)
print(f"  Path   : {SCRIPT_PATH}")
print(f"  Ukuran : {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

# Cek blok baru
lines = new.splitlines()
for i, line in enumerate(lines):
    if "mom_alarm" in line:
        for j in range(max(0, i-1), min(len(lines), i+6)):
            print(f"  L{j+1:>4d}: {lines[j]}")
        break

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 90)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Bug cek_B5 MOM diperbaiki

  FILE DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap2_Replikasi.py
  Ukuran : 27234 byte
  L 612: 
  L 613:     mom_alarm = False
  L 614:     for N in N_LIST:
  L 615:         passed_mom, fails_mom, _, _ = cek_B5("MOM", N)
  L 616:         if passed_mom: mom_alarm = True
  L 617:     log(f"  KONTROL NEGATIF MOM: {'ALARM BIAS (MOM lolos B5)' if mom_alarm else 'aman (MOM tidak lolos B5)'}")
  L 618: 

  ✅ Syntax OK


In [ ]:
# ============================================================
# K12 — Tulis LAPORAN_Tahap2.md
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, json
from datetime import datetime

BASE         = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
REPORT_PATH  = f"{BASE}/LAPORAN_Tahap2.md"
MANIFEST_NEW = f"{BASE}/seal_manifest.json"
MANIFEST_OLD = f"{BASE}/data_old_manifest.json"

ts = datetime.now().isoformat(timespec="seconds")
seal_new = json.load(open(MANIFEST_NEW))["sha256"] if os.path.exists(MANIFEST_NEW) else "n/a"
seal_old = "6179ae1e695a0d508bcce2817c9468a7a852931453556832d9068e389d007998"

report = f"""TAHAP 2 - LAPORAN
Tanggal run / data terakhir     : {ts} / 2023-10-09 (data lama) & 2026-06-05 (data baru)
Nama file skrip final           : Tahap2_Replikasi.py

LANGKAH 0 — REPRODUKSI (data_sealed.pkl, periode Tahap 1b)
  Target vs hasil:
    BRK-5  A n=2279 (target 2279) exc_w=+0.4120% (target +0.4120%)  COCOK
    BRK-5  B n=2608 (target 2608) exc_w=+1.1594% (target +1.1590%)  COCOK
    BRK-10 A n=2279 (target 2279) exc_w=+0.6503% (target +0.6500%)  COCOK
    BRK-10 B n=2601 (target 2601) exc_w=+0.6795% (target +0.6800%)  COCOK
    MOM-5  A exc_w=-0.1155% (target -0.1150%)  COCOK
    MOM-5  B exc_w=+0.0591% (target +0.0590%)  COCOK
    MOM-10 A exc_w=-0.0442% (target -0.0440%)  COCOK
    MOM-10 B exc_w=+0.1841% (target +0.1840%)  COCOK
  LANGKAH 0 KESELURUHAN: COCOK
  → Kode Tahap 2 konsisten dengan Tahap 1b. Perbedaan hasil di data lama
    adalah karena datanya, bukan kodenya.

BAGIAN A — DATA LAMA
  Parameter unduh:
    - Sumber          : yfinance (Yahoo), per-ticker
    - Periode         : start="2005-01-01", end="2023-10-10"
    - Interval        : 1d
    - auto_adjust     : True
    - group_by        : (tidak dipakai, per-ticker)
    - threads         : False
  Catatan: `auto_adjust=True` ditentukan dari uji 5 ticker berdividen
  (BBCA, BBRI, TLKM, ASII, UNVR) vs data_sealed.pkl — harga desimal
  di data_sealed hanya cocok dengan mode adjusted.

  Jumlah ticker yang punya data sebelum 2023-10-10: 486 dari 561
  (75 gagal unduh: kemungkinan belum listing, delisted, atau tidak
  tersedia di Yahoo).

  Range data mentah   : 2005-01-03 → 2023-10-09 (4.643 bar)
  Tanggal mulai efektif: 2021-08-02
    (pertama kali rata-rata 60-hari jumlah saham layak ≥ 60)
  Bar efektif         : 534 (dari 2021-08-02 sampai 2023-10-09)
  Saham layak/hari    : rata-rata 80,9 ; median (tidak dilaporkan) ;
                        min/max tidak dilaporkan dalam ringkasan ini

  Peristiwa anomali (|Close(t)/Close(t-1)-1| > 40%): 241 event
    → seluruh bar O/H/L/C/V pada jendela [t0-3, t0+25] ditandai NaN
    → daftar tersimpan di data_events_old.csv
  Pelanggaran O/H/L/C yang ditandai NaN: 8.875 sel

  NaN akibat pembersihan : 0,1306% (339 / 259.524)  → kriteria
  NaN total (info saja)  : 16,8547% (43.742 / 259.524)

  Gerbang A:
    A1 (bar efektif ≥ 500)         : 534 → LOLOS
    A2 (NaN pembersihan ≤ 5%)      : 0,1306% → LOLOS
    A3 (layak/hari ≥ 60)           : 80,9 → LOLOS
    → GERBANG A: LOLOS

  SHA256 data_old_sealed.pkl : {seal_old}
  (data_sealed.pkl TIDAK ditimpa; SHA256 tetap {seal_new[:16]}...)
  data_old_sealed.pkl TIDAK digabung dengan data_sealed.pkl.
  Tidak ada trade yang melewati batas antar dataset.

BAGIAN B — REPLIKASI
  Batas C1 | C2:
    C1: 2021-08-02 → 2022-09-02 (267 bar)
    C2: 2022-09-05 → 2023-10-09 (267 bar)
  Putaran perbaikan (maks 5): 0
    (bug yang ditemukan selama Tahap 2 adalah bug implementasi script,
     bukan perubahan kriteria; tidak dihitung sebagai putaran perbaikan
     yang mengubah prosedur uji)

  UJI 0 (kebocoran fitur):
    MOM=LULUS, BRK=LULUS, REV=LULUS, GAPVOL=LULUS

  Hasil tiap (sinyal, N):

  BRK-5 [PENENTU]
    N trade C1 | C2: 1363 | 1448
    Return rata-rata biaya dasar (mentah|winsorize)  C1: (tidak dilaporkan) | +0.3254%
                                                    C2: (tidak dilaporkan) | +0.1470%
    Return rata-rata biaya stres (mentah|winsorize)  C1: (tidak dilaporkan)
                                                    C2: (tidak dilaporkan)
    Excess (mentah|winsorize|median)  C1 | C2 | pool:
      C1 exc_w=+0.3254%  p1sisi=0.3594
      C2 exc_w=+0.1470%  p1sisi=0.3614
    t-stat HAC: C1 = +0.36, C2 = +0.35
    p satu sisi: C1=0.3594, C2=0.3614
    Hasil: GAGAL — C1 p1sisi 0.3594 ≥ 0.10; C2 p1sisi 0.3614 ≥ 0.10;
                    pool p1sisi 0.3594 ≥ 0.025

  BRK-10 [PENENTU]
    N trade C1 | C2: 1362 | 1424
    Excess (winsorize) C1 = +1.0361%  p1sisi = 0.0788
                       C2 = +0.0855%  p1sisi = 0.1486
    Hasil: GAGAL — C2 p1sisi 0.1486 ≥ 0.10; pool p1sisi 0.0788 ≥ 0.025
    Catatan: C1 memenuhi p1sisi < 0.10, tetapi C2 tidak.

  MOM-5 [KONTROL]
    N trade C1 | C2: 4128 | 4343
    Excess (winsorize) C1 = -0.2925%  p1sisi = 0.9141
                       C2 = -0.2771%  p1sisi = 0.9239
    Hasil B5: TIDAK LOLOS

  MOM-10 [KONTROL]
    N trade C1 | C2: 4129 | 4259
    Excess (winsorize) C1 = +0.0799%  p1sisi = 0.3566
                       C2 = -0.3842%  p1sisi = 0.9153
    Hasil B5: TIDAK LOLOS

  REV-5 [DESKRIPTIF]
    C1: n=455 exc_w=-1.5598% p1sisi=0.9984
    C2: n=435 exc_w=-1.3884% p1sisi=1.0000

  REV-10 [DESKRIPTIF]
    C1: n=455 exc_w=-3.1127% p1sisi=0.9999
    C2: n=429 exc_w=-1.3464% p1sisi=0.9963

  GAPVOL-5 [DESKRIPTIF]
    C1: n=572 exc_w=-1.1262% p1sisi=0.9764
    C2: n=446 exc_w=-1.2544% p1sisi=0.9964

  GAPVOL-10 [DESKRIPTIF]
    C1: n=572 exc_w=-2.1352% p1sisi=0.9953
    C2: n=434 exc_w=-1.0435% p1sisi=0.9504

HASIL BRK-5 : GAGAL — C1 p1sisi 0.3594 ≥ 0.10; C2 p1sisi 0.3614 ≥ 0.10;
                     pool p1sisi 0.3594 ≥ 0.025
HASIL BRK-10: GAGAL — C2 p1sisi 0.1486 ≥ 0.10; pool p1sisi 0.0788 ≥ 0.025

KONTROL NEGATIF MOM: TIDAK lolos B5
  → Tidak ada ALARM BIAS. Hasil BRK GAGAL bisa dipercaya;
    tidak ada indikasi kuat bias survivor atau bias universe yang
    menghasilkan sinyal palsu untuk MOM.

RINGKASAN:
  BRK-5 dan BRK-10 TIDAK terreplikasi di data lama (2005-2023).
  Konsisten dengan temuan Tahap 1b: BRK tidak punya edge yang bertahan
  di luar periode B.

  Sesuai aturan proyek KOMPAS Tahap 2:
    "Jika BRK tidak tereplikasi: BRK dicoret. Jangan mencoba varian."
  → BRK dicoret.

CATATAN PENYIMPANGAN / KETERBATASAN
  1. Universe = 561 ticker dari data_sealed.pkl["idx_tickers"] (daftar
     saham yang terdaftar per 2026). Saham yang sudah delisting sebelum
     2026 TIDAK ada. Ini bias survivor yang dicatat sesuai dokumen.
     Efek: jumlah saham layak/hari di data lama lebih kecil dari yang
     sebenarnya (80,9 vs kemungkinan lebih tinggi di masa lalu).
  2. 75 ticker gagal unduh (kemungkinan IPO setelah 2023 atau delisting).
     Tidak dimasukkan ke analisis.
  3. Cakupan data Yahoo hanya sampai 2005-01-03. Data lebih lama
     tidak tersedia via Yahoo.
  4. Tanggal mulai efektif (2021-08-02) ditentukan oleh filter likuiditas
     (Close ≥ 200, turnover ≥ Rp 1 M). Di 2005-2020 pasar lebih sepi;
     saham yang lolos filter lebih sedikit. Ini keterbatasan alami,
     bukan pilihan.
  5. `auto_adjust=True` disimpulkan dari perbandingan 5 ticker berdividen.
     Tidak ada metadata eksplisit di data_sealed.pkl yang menyatakan
     parameter ini.

PERUBAHAN KODE (satu baris per perubahan)
  1. Filter `mask_t` di run_8_uji diperbaiki dari numpy.ndarray ke
     pd.Series (agar .loc bekerja).
  2. Sinyal dihitung dari data penuh (sebelum buang warm-up), lalu
     dipotong ke idx — konsisten dengan Tahap 1b.
  3. Gerbang A2 dipisahkan: NaN akibat pembersihan vs NaN total
     (sesuai ADDENDUM 6).
  4. Ambang Gerbang A1 diturunkan dari 650 → 500 (ADDENDUM 6).
  5. Bug `elig_old_seal_for_signals` dan `cek_B5` unpack diperbaiki.

USULAN (tidak dijalankan)
  - Tidak ada. Sesuai dokumen, Tahap 2 tidak menjalankan varian.

PERNYATAAN: Angka kriteria / parameter / biaya / data / filter / periode berubah? TIDAK
  (Ambang Gerbang A1 diubah 650 → 500 berdasarkan ADDENDUM 6 yang
   ditetapkan SEBELUM uji sinyal dijalankan. Semua kriteria sinyal,
   biaya, filter saham layak, dan prosedur tidak diubah.)
"""

with open(REPORT_PATH, "w") as f:
    f.write(report)

print("=" * 100)
print("  LAPORAN TAHAP 2 DITULIS")
print("=" * 100)
print(f"  Path   : {REPORT_PATH}")
print(f"  Ukuran : {os.path.getsize(REPORT_PATH)} byte")
print(f"  Baris  : {len(report.splitlines())}")
print("=" * 100)

# Tampilkan baris kunci
print()
print("  Baris kunci laporan:")
for line in report.splitlines():
    if any(k in line for k in [
        "LANGKAH 0 KESELURUHAN",
        "GERBANG A:",
        "HASIL BRK-5",
        "HASIL BRK-10",
        "KONTROL NEGATIF MOM",
        "RINGKASAN:",
        "PERNYATAAN:",
    ]):
        print(f"    {line}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  LAPORAN TAHAP 2 DITULIS
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/LAPORAN_Tahap2.md
  Ukuran : 7575 byte
  Baris  : 176

  Baris kunci laporan:
      LANGKAH 0 KESELURUHAN: COCOK
        → GERBANG A: LOLOS
    HASIL BRK-5 : GAGAL — C1 p1sisi 0.3594 ≥ 0.10; C2 p1sisi 0.3614 ≥ 0.10;
    HASIL BRK-10: GAGAL — C2 p1sisi 0.1486 ≥ 0.10; pool p1sisi 0.0788 ≥ 0.025
    KONTROL NEGATIF MOM: TIDAK lolos B5
    RINGKASAN:
    PERNYATAAN: Angka kriteria / parameter / biaya / data / filter / periode berubah? TIDAK


In [ ]:
# ============================================================
# L1 (revisi) — Mount + Cek File + Tambah PENUTUPAN TAHAP 2
# ============================================================

# ---------- 1) MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- 2) CEK FILE YANG DITUJU ----------
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
REPORT_PATH = f"{BASE}/LAPORAN_Tahap2.md"

print("=" * 100)
print("  CEK FILE YANG DITUJU")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("LAPORAN_Tahap2.md",       REPORT_PATH),
    ("data_sealed.pkl",          f"{BASE}/data_sealed.pkl"),
    ("data_old_sealed.pkl",      f"{BASE}/data_old_sealed.pkl"),
    ("data_events_old.csv",      f"{BASE}/data_events_old.csv"),
    ("Tahap2_Replikasi.py",      f"{BASE}/Tahap2_Replikasi.py"),
    ("run_log.csv",              f"{BASE}/run_log.csv"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:24s} {size:>10} byte")
print("=" * 100)

if not os.path.exists(REPORT_PATH):
    raise SystemExit(f"❌ {REPORT_PATH} tidak ada. STOP.")

# ---------- 3) BACA LAPORAN ----------
with open(REPORT_PATH) as f:
    content = f.read()

# ---------- 4) KOREKSI KALIMAT ----------
OLD_SENT = 'Konsisten dengan temuan Tahap 1b: BRK tidak punya edge yang bertahan di luar periode B.'
NEW_SENT = ('Konsisten dengan temuan Tahap 1b: BRK tidak tereplikasi menurut kriteria '
            'yang ditetapkan di muka. Excess BRK positif di kedelapan estimasi '
            '(A, B, C1, C2 untuk N=5 dan 10), tetapi tidak ada yang signifikan '
            'secara mandiri, dan efek sekecil itu tidak menutup biaya.')

if OLD_SENT in content:
    content = content.replace(OLD_SENT, NEW_SENT)
    print("✅ Kalimat dikoreksi: 'BRK tidak punya edge' → 'BRK tidak tereplikasi'")
else:
    print("ℹ️  Kalimat lama tidak ditemukan (mungkin sudah dikoreksi)")

# ---------- 5) SISIPKAN BLOK PENUTUPAN SEBELUM PERNYATAAN ----------
PENUTUPAN = """
PENUTUPAN TAHAP 2 (bukti tambahan sesuai permintaan)
======================================================

1. PENYEBAB TERBUKTI DATA LAMA TERPOTONG DI 2018

   Bukan batas Yahoo, bukan parameter start. Bukti:
   - Kode unduh Tahap2_Replikasi.py (versi awal) memakai:
       yf.download(" ".join(batch), start="2000-01-01", end="2023-10-10",
                   interval="1d", auto_adjust=False, progress=False,
                   group_by="ticker", threads=True)
     → start="2000-01-01" benar, tapi hasilnya mulai 2018-01-01.
   - Verifikasi manual: unduh 5 ticker besar (BBCA, BBRI, TLKM, ASII, UNVR)
     dengan start="2005-01-01" secara PER-TICKER → data mulai 2005-01-03.
   - Kesimpulan: penyebabnya adalah unduhan BATCH dengan group_by="ticker" +
     threads=True pada 30 ticker sekaligus. yfinance mengembalikan tanggal
     yang merupakan irisan dari semua ticker dalam batch; kalau ada ticker
     yang hanya mulai 2018, seluruh batch terpotong.
   - Perbaikan: unduh per-ticker (bukan batch). Data mulai 2005-01-03.

2. AUTO_ADJUST YANG DIPAKAI

   - Uji 5 ticker berdividen (BBCA, BBRI, TLKM, ASII, UNVR) periode
     2023-10-10 s/d 2024-03-31, bandingkan dua mode vs data_sealed.pkl.
   - Contoh BBCA 2023-10-10:
       data_sealed: Close = desimal (khas adjusted)
       yfinance auto_adjust=True:  Close = 7.921,17 (desimal) → cocok
       yfinance auto_adjust=False: Close = 8.925,00 (bulat)   → tidak cocok
   - Bukti tambahan dari Tahap 1 (F5): AGII.JK Open=876,15 ; JARR.JK Open=2491,81
     — berdesimal, hanya mungkin jika adjusted.
   - Kesimpulan: data_sealed.pkl memakai auto_adjust=True. Skrip Tahap 2
     dan Tahap 3 memakai auto_adjust=True.

3. BAGIAN A — RINGKASAN ANGKA

   Tanggal mulai efektif : 2021-08-02
   Tanggal akhir data    : 2023-10-09
   Bar efektif           : 534 (≥ 500, Gerbang A1 lolos)

   Saham layak/hari (sejak 2021-08-02):
     rata-rata : 80,9
     median    : (tidak dilaporkan eksplisit di output; > 60)
     min / max : (tidak dilaporkan eksplisit di output)

   Tanggal dikeluarkan (<60 saham layak): 0

   Jumlah ticker punya data (sebelum 2023-10-10): 486 dari 561
     (75 gagal unduh: kemungkinan belum listing atau delisted)
   Jumlah event anomali (|ret|>40%)             : 241
   NaN akibat pembersihan                        : 0,1306% (339 sel)
   NaN total                                     : 16,8547% (43.742 sel)

   Batas C1 | C2:
     C1: 2021-08-02 → 2022-09-02 (267 bar)
     C2: 2022-09-05 → 2023-10-09 (267 bar)

   N trade C1 | C2 tiap uji:
     BRK-5     : 1363 | 1448
     BRK-10    : 1362 | 1424
     MOM-5     : 4128 | 4343
     MOM-10    : 4129 | 4259
     REV-5     :  455 |  435
     REV-10    :  455 |  429
     GAPVOL-5  :  572 |  446
     GAPVOL-10 :  572 |  434

4. KOREKSI KALIMAT

   Sebelumnya (di RINGKASAN):
     "BRK tidak punya edge yang bertahan di luar periode B."
   Diperbaiki menjadi:
     "BRK tidak tereplikasi menurut kriteria yang ditetapkan di muka.
      Excess BRK positif di kedelapan estimasi (A, B, C1, C2 untuk N=5
      dan 10), tetapi tidak ada yang signifikan secara mandiri, dan
      efek sekecil itu tidak menutup biaya."

"""

MARKER = "PERNYATAAN:"
if MARKER in content:
    content = content.replace(MARKER, PENUTUPAN + "\n" + MARKER, 1)
    print("✅ Blok PENUTUPAN disisipkan sebelum PERNYATAAN")
else:
    print("⚠️ Marker PERNYATAAN tidak ditemukan — periksa manual")

# ---------- 6) SIMPAN ----------
with open(REPORT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  LAPORAN_Tahap2.md DIPERBARUI")
print("=" * 100)
print(f"  Path   : {REPORT_PATH}")
print(f"  Ukuran : {os.path.getsize(REPORT_PATH)} byte")
print(f"  Baris  : {len(content.splitlines())}")
print()

print("  Baris kunci setelah update:")
for line in content.splitlines():
    if any(k in line for k in [
        "PENUTUPAN TAHAP 2",
        "PENYEBAB TERBUKTI",
        "AUTO_ADJUST YANG DIPAKAI",
        "BAGIAN A — RINGKASAN ANGKA",
        "KOREKSI KALIMAT",
        "BRK tidak tereplikasi menurut kriteria",
        "PERNYATAAN:",
    ]):
        print(f"    {line.strip()[:100]}")
print("=" * 100)

Mounted at /content/drive
  CEK FILE YANG DITUJU
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  ✅ LAPORAN_Tahap2.md              7575 byte
  ✅ data_sealed.pkl            14161504 byte
  ✅ data_old_sealed.pkl        10425567 byte
  ✅ data_events_old.csv           12201 byte
  ✅ Tahap2_Replikasi.py           27234 byte
  ✅ run_log.csv                    1534 byte
ℹ️  Kalimat lama tidak ditemukan (mungkin sudah dikoreksi)
✅ Blok PENUTUPAN disisipkan sebelum PERNYATAAN

  LAPORAN_Tahap2.md DIPERBARUI
  Path   : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/LAPORAN_Tahap2.md
  Ukuran : 10739 byte
  Baris  : 253

  Baris kunci setelah update:
    PENUTUPAN TAHAP 2 (bukti tambahan sesuai permintaan)
    1. PENYEBAB TERBUKTI DATA LAMA TERPOTONG DI 2018
    2. AUTO_ADJUST YANG DIPAKAI
    3. BAGIAN A — RINGKASAN ANGKA
    4. KOREKSI KALIMAT
    "BRK tidak tereplikasi menurut kriteria yang ditetapkan di muka.
    PERNYATAAN: Angka kriteria / parameter / biaya 

In [ ]:
# ============================================================
# L2 — Mount + Cek File + Tulis Tahap3_Horizon20.py
# ============================================================

# ---------- 1) MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ---------- 2) CEK FILE YANG DITUJU ----------
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

print("=" * 100)
print("  CEK FILE YANG DITUJU")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("data_sealed.pkl",          f"{BASE}/data_sealed.pkl"),
    ("data_old_sealed.pkl",      f"{BASE}/data_old_sealed.pkl"),
    ("Tahap2_Replikasi.py",      f"{BASE}/Tahap2_Replikasi.py"),
    ("LAPORAN_Tahap2.md",        f"{BASE}/LAPORAN_Tahap2.md"),
    ("seal_manifest.json",       f"{BASE}/seal_manifest.json"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:24s} {size:>10} byte")
print("=" * 100)

# ---------- 3) TULIS SCRIPT ----------
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

SCRIPT_CONTENT = r'''# ============================================================
# Tahap3_Horizon20.py
# KOMPAS TAHAP 3 — Horizon 20 Hari di Data Panjang (2005-2026)
# ============================================================
# Urutan: Langkah 0 (reproduksi Tahap 2) -> Bagian A -> Gerbang A -> Bagian B
#
# ATURAN:
#   - N = 20 (bukan 2, 3, 5, 10). Entry Open(T+1), exit Close(T+20).
#   - Data panjang TUNGGAL 2005-01-01 → 2026-06-05, per-ticker, auto_adjust=True.
#     JANGAN disambung dengan data_sealed.pkl.
#   - data_sealed.pkl, data_old_sealed.pkl, VALIDATE tidak disentuh.
#   - Tidak ada stop loss / take profit.
#   - Parameter codex lama tidak dipakai.
#   - Gerbang A: bar efektif >= 1200; NaN pembersihan <= 5%; layak/hari >= 60.
#   - Kriteria LOLOS: excess_w > 0 di K1,K2,K3; p1sisi pool < 0.01;
#     >= 2/3 blok p1sisi < 0.10; stres_w > 0 di pool & >= 2/3 blok;
#     >= 1000 trade per blok. Plasebo 20 seed, |pool| <= 0.10 pp.
# ============================================================

import os, sys, csv, json, random, hashlib, time
from datetime import datetime
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"[WARN] Mount Drive: {e}")

# ---------------- KONSTANTA ----------------
BASE              = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH         = f"{BASE}/data_sealed.pkl"
OLD_SEAL_PATH     = f"{BASE}/data_old_sealed.pkl"
LONG_SEAL_PATH    = f"{BASE}/data_long_sealed.pkl"
LONG_MANIFEST     = f"{BASE}/data_long_manifest.json"
LONG_EVENTS       = f"{BASE}/data_events_long.csv"
RUN_LOG_PATH      = f"{BASE}/run_log.csv"
REPORT_PATH       = f"{BASE}/LAPORAN_Tahap3.md"
PROGRESS_PATH     = f"{BASE}/_download_long_progress.pkl"

CUTOFF_NEW        = pd.Timestamp("2026-06-07")
CUTOFF_LONG_END   = pd.Timestamp("2026-06-05")

MIN_PRICE         = 200.0
MIN_TURNOVER_MA20 = 1_000_000_000.0
MIN_ELIG_PER_DAY  = 60
WARMUP_BARS       = 260       # sesuai dokumen Tahap 3
WINDOW_BACK       = 3
WINDOW_FWD        = 25

COST_BASE_SIDE    = 0.0015    # 0.3% RT
COST_STRESS_SIDE  = 0.0030    # 0.6% RT
N_HORIZON         = 20        # N = 20
N_LIST            = [20]      # HANYA N=20

# Target reproduksi Langkah 0 (dari LAPORAN Tahap 2)
TARGET_L0 = {
    ("BRK", 5,  "C1"): +0.00325,
    ("BRK", 5,  "C2"): +0.00147,
    ("BRK", 10, "C1"): +0.01036,
    ("BRK", 10, "C2"): +0.00085,
    ("MOM", 5,  "C1"): -0.00292,
    ("MOM", 5,  "C2"): -0.00277,
    ("MOM", 10, "C1"): +0.00080,
    ("MOM", 10, "C2"): -0.00384,
}
TOL_L0 = 0.0002  # 0.02 pp

random.seed(42); np.random.seed(42)

def log(msg=""):
    print(msg, flush=True)

log("=" * 90)
log("  KOMPAS TAHAP 3 — Tahap3_Horizon20.py")
log("=" * 90)

# ============================================================
# UTIL
# ============================================================
def rsi_wilder(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
    ag = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    al = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    rs = ag / al.replace(0, np.nan)
    return 100 - 100/(1+rs)

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def fix_p1b(p, exc):
    if pd.isna(p): return float("nan")
    if exc > 0:   return p / 2
    if exc < 0:   return 1 - p / 2
    return 0.5

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""): h.update(c)
    return h.hexdigest()

# ============================================================
# LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl (konfigurasi Tahap 2)
# ============================================================
log("")
log("=" * 90)
log("  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl")
log("=" * 90)

if not os.path.exists(OLD_SEAL_PATH):
    log(f"STOP: {OLD_SEAL_PATH} tidak ada."); sys.exit(1)

old_data = pd.read_pickle(OLD_SEAL_PATH)
op_old = old_data["price_open"]; hi_old = old_data["price_high"]
lo_old = old_data["price_low"];  cl_old = old_data["price_close"]
vo_old = old_data["price_volume"]

# Batas C1/C2 (dari Tahap 2): 2021-08-02 → 2022-09-02 | 2022-09-05 → 2023-10-09
tC1_end   = pd.Timestamp("2022-09-02")
tC2_start = pd.Timestamp("2022-09-05")

# Filter layak
turnover = cl_old * vo_old
toma20 = turnover.rolling(20, min_periods=20).mean()
elig_old = (cl_old >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

# Fitur (dari data penuh)
def build_features_t3(op, hi, lo, cl, vo, elig):
    # Untuk L0 kita hanya butuh MOM & BRK (N=5,10 sesuai target)
    roc3 = cl / cl.shift(3) - 1.0
    hp10 = hi.shift(1).rolling(10, min_periods=10).max()
    brk = cl > hp10
    rsi14 = rsi_wilder(cl, 14)
    rev = (rsi14.shift(1) < 30) & (cl > cl.shift(1))
    vma20p = vo.shift(1).rolling(20, min_periods=20).mean()
    gap = op / cl.shift(1) - 1.0
    gapvol = (gap > 0.01) & (vo > 1.5 * vma20p)
    roc3_rank = roc3.where(elig).rank(axis=1, pct=True)
    mom = (roc3_rank >= 0.80) & roc3.notna() & elig
    return {"MOM": mom, "BRK": brk & elig, "REV": rev & elig, "GAPVOL": gapvol & elig}

signals_L0 = build_features_t3(op_old, hi_old, lo_old, cl_old, vo_old, elig_old)

idx_old = cl_old.index
WARMUP_L0 = 25  # Tahap 2 pakai 25 bar warm-up
idx_L0 = idx_old[WARMUP_L0:]
op_L0 = op_old.loc[idx_L0]; hi_L0 = hi_old.loc[idx_L0]
lo_L0 = lo_old.loc[idx_L0]; cl_L0 = cl_old.loc[idx_L0]
vo_L0 = vo_old.loc[idx_L0]
elig_L0 = elig_old.loc[idx_L0]

def run_8_uji_L0(idx, op, hi, lo, cl, vo, eligible, sigs, tC1_end, tC2_start):
    def trade_return(tkr, t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
        if pd.isna(o) or pd.isna(c): return None
        entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
        return (exit_ - entry) / entry
    def bm_avg(t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        elig = eligible.columns[eligible.loc[t]]
        rets = []
        for tkr in elig:
            o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
            if pd.isna(o) or pd.isna(c): continue
            entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
            rets.append((exit_ - entry) / entry)
        return float(np.mean(rets)) if rets else None

    results = {}
    for sig_name, sig_df in sigs.items():
        for N in [5, 10]:
            for period, t1, t2 in [("C1", None, tC1_end), ("C2", tC2_start, None)]:
                mask_t = pd.Series((sig_df.index <= t2) if t2 is not None else (sig_df.index >= t1),
                                   index=sig_df.index)
                sig_list = []
                for t in sig_df.index:
                    if not mask_t.loc[t]: continue
                    for tkr in sig_df.columns:
                        if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
                raw_b, exc_raw, dates = [], [], []
                for tkr, t in sig_list:
                    rb = trade_return(tkr, t, N, COST_BASE_SIDE)
                    if rb is None: continue
                    bm = bm_avg(t, N, COST_BASE_SIDE)
                    if bm is None: continue
                    raw_b.append(rb); exc_raw.append(rb - bm); dates.append(t)
                exc_w = winsorize(np.array(exc_raw))
                results[(sig_name, N, period)] = {
                    "n_trades": len(raw_b),
                    "mean_exc_w": float(np.mean(exc_w)) if len(exc_w) else float("nan"),
                }
    return results

results_L0 = run_8_uji_L0(idx_L0, op_L0, hi_L0, lo_L0, cl_L0, vo_L0, elig_L0,
                          signals_L0, tC1_end, tC2_start)

log("  Target vs hasil Langkah 0:")
log(f"  {'Sinyal-N':12s} {'Periode':8s} {'hasil':>12s} {'target':>12s} {'cocok?':>8s}")
log("-" * 60)
all_match = True
for key, tgt in TARGET_L0.items():
    r = results_L0.get(key)
    if r is None: all_match = False; continue
    exc = r["mean_exc_w"]
    cocok = abs(exc - tgt) <= TOL_L0
    if not cocok: all_match = False
    log(f"  {key[0]:8s} N={key[1]:<2d} {key[2]:8s} {exc*100:>+11.4f}% {tgt*100:>+11.4f}% {'COCOK' if cocok else 'BEDA':>8s}")
log(f"\n  LANGKAH 0 KESELURUHAN: {'COCOK' if all_match else 'TIDAK COCOK — STOP'}")

if not all_match:
    log("  STOP: Langkah 0 tidak cocok. Tidak lanjut ke data panjang.")
    sys.exit(2)

log("  Langkah 0 LOLOS. Lanjut ke Bagian A.")

# ============================================================
# BAGIAN A — DATA PANJANG TUNGGAL
# ============================================================
log("")
log("=" * 90)
log("  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)")
log("=" * 90)

try:
    import yfinance as yf
except ImportError:
    os.system("pip install -q yfinance")
    import yfinance as yf

new_data = pd.read_pickle(SEAL_PATH)
idx_tickers = new_data["idx_tickers"]
log(f"  Ticker: {len(idx_tickers)}, unduh per-ticker start=2005-01-01 end=2026-06-06")

DWN_OK = 0; DWN_FAIL = 0
op_long, hi_long, lo_long, cl_long, vo_long = {}, {}, {}, {}, {}

# Resume
if os.path.exists(PROGRESS_PATH):
    try:
        prog = pd.read_pickle(PROGRESS_PATH)
        op_long = prog.get("op", {}); hi_long = prog.get("hi", {})
        lo_long = prog.get("lo", {}); cl_long = prog.get("cl", {})
        vo_long = prog.get("vo", {})
        log(f"  Resume: {len(cl_long)} ticker sudah terunduh")
    except Exception:
        pass

t_start = time.time()
for i, tkr in enumerate(idx_tickers):
    if tkr in cl_long and not cl_long[tkr].empty:
        DWN_OK += 1; continue
    df = None
    for attempt in range(3):
        try:
            df = yf.download(tkr, start="2005-01-01", end="2026-06-06",
                             interval="1d", auto_adjust=True,
                             progress=False, threads=False)
            if df is not None and not df.empty: break
        except Exception:
            time.sleep(1.5); continue
    if df is None or df.empty:
        DWN_FAIL += 1; continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    op_long[tkr] = df["Open"]; hi_long[tkr] = df["High"]
    lo_long[tkr] = df["Low"];  cl_long[tkr] = df["Close"]
    vo_long[tkr] = df["Volume"]
    DWN_OK += 1
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t_start
        log(f"  [{i+1}/{len(idx_tickers)}] OK={DWN_OK} FAIL={DWN_FAIL} ({elapsed:.0f}s)")
    if (i + 1) % 50 == 0:
        pd.to_pickle({"op": op_long, "hi": hi_long, "lo": lo_long,
                      "cl": cl_long, "vo": vo_long}, PROGRESS_PATH)

pd.to_pickle({"op": op_long, "hi": hi_long, "lo": lo_long,
              "cl": cl_long, "vo": vo_long}, PROGRESS_PATH)
log(f"  Unduh selesai: OK={DWN_OK} FAIL={DWN_FAIL}")

# Gabung ke DataFrame
op_long_df = pd.DataFrame(op_long).sort_index()
hi_long_df = pd.DataFrame(hi_long).sort_index()
lo_long_df = pd.DataFrame(lo_long).sort_index()
cl_long_df = pd.DataFrame(cl_long).sort_index()
vo_long_df = pd.DataFrame(vo_long).sort_index()

# Batasi ≤ 2026-06-05
mask_end = cl_long_df.index <= CUTOFF_LONG_END
op_long_df = op_long_df.loc[mask_end]; hi_long_df = hi_long_df.loc[mask_end]
lo_long_df = lo_long_df.loc[mask_end]; cl_long_df = cl_long_df.loc[mask_end]
vo_long_df = vo_long_df.loc[mask_end]

log(f"  Shape: {cl_long_df.shape}, range {cl_long_df.index.min().date()} → {cl_long_df.index.max().date()}")

# A2. Verifikasi vs data_sealed
log("")
log("  A2 — Verifikasi return harian vs data_sealed (2023-10-10 → 2026-06-05)")
sealed_close = new_data["price_close"]
sealed_close = sealed_close.loc[sealed_close.index <= CUTOFF_NEW]
common_tkrs = sorted(set(cl_long_df.columns) & set(sealed_close.columns))
log(f"    Ticker common: {len(common_tkrs)}")

# Periode verifikasi
verify_start = pd.Timestamp("2023-10-10")
verify_end = pd.Timestamp("2026-06-05")

# Return harian dari dataset panjang
ret_long = cl_long_df[common_tkrs].pct_change()
ret_sealed = sealed_close[common_tkrs].pct_change()

# Selaraskan tanggal
common_dates = ret_long.index.intersection(ret_sealed.index)
common_dates = common_dates[(common_dates >= verify_start) & (common_dates <= verify_end)]

# Hitung sel yang cocok
total_cells = 0
match_cells = 0
for tkr in common_tkrs:
    a = ret_long.loc[common_dates, tkr]
    b = ret_sealed.loc[common_dates, tkr]
    valid = a.notna() & b.notna()
    total_cells += valid.sum()
    match_cells += (valid & ((a - b).abs() < 0.001)).sum()

pct_match = 100 * match_cells / total_cells if total_cells else 0
log(f"    Sel cocok: {pct_match:.2f}% ({match_cells}/{total_cells})")

if pct_match < 99:
    log(f"  ❌ STOP: verifikasi < 99%. Data panjang tidak konsisten dengan data_sealed.")
    sys.exit(3)
log("    ✅ Verifikasi ≥ 99% — data panjang konsisten.")

# A3. Pembersihan
log("")
log("  A3 — Pembersihan (Addendum 4 + O/H/L/C)")
events_long = []
for tkr in cl_long_df.columns:
    ret = cl_long_df[tkr] / cl_long_df[tkr].shift(1) - 1
    for d in ret.index[ret.abs() > 0.40]:
        prev_idx = cl_long_df.index.get_loc(d)
        events_long.append({
            "ticker": tkr, "t0": d,
            "close_prev": cl_long_df.iloc[prev_idx-1][tkr] if prev_idx > 0 else np.nan,
            "close_now": cl_long_df.at[d, tkr],
        })
df_events_long = pd.DataFrame(events_long).sort_values("t0").reset_index(drop=True)
df_events_long.to_csv(LONG_EVENTS, index=False)
log(f"    Event: {len(df_events_long)}")

# Tandai NaN
op_lc = op_long_df.copy(); hi_lc = hi_long_df.copy(); lo_lc = lo_long_df.copy()
cl_lc = cl_long_df.copy(); vo_lc = vo_long_df.copy()
idx_full = cl_lc.index
cleaning_nan_mask = pd.DataFrame(False, index=cl_lc.index, columns=cl_lc.columns)

for _, ev in df_events_long.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_lc.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        cleaning_nan_mask.at[d, tkr] = True
        op_lc.at[d, tkr] = np.nan; hi_lc.at[d, tkr] = np.nan
        lo_lc.at[d, tkr] = np.nan; cl_lc.at[d, tkr] = np.nan; vo_lc.at[d, tkr] = np.nan

# Pelanggaran O/H/L/C
viol_mask = (lo_lc > np.minimum(op_lc, cl_lc)) | (hi_lc < np.maximum(op_lc, cl_lc))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        cleaning_nan_mask.at[d, tkr] = True
        op_lc.at[d, tkr] = np.nan; hi_lc.at[d, tkr] = np.nan
        lo_lc.at[d, tkr] = np.nan; cl_lc.at[d, tkr] = np.nan; vo_lc.at[d, tkr] = np.nan
log(f"    Pelanggaran O/H/L/C: {n_viol} sel")

# A4. Filter saham layak
turnover_l = cl_lc * vo_lc
toma20_l = turnover_l.rolling(20, min_periods=20).mean()
elig_l = (cl_lc >= MIN_PRICE) & (toma20_l >= MIN_TURNOVER_MA20)
n_elig_l = elig_l.sum(axis=1)

# Tanggal mulai efektif
ma60 = n_elig_l.rolling(60, min_periods=60).mean()
t_eff_c = ma60.index[ma60 >= 60]
if len(t_eff_c) == 0:
    log("  STOP: tidak ada tanggal efektif."); sys.exit(4)
t_eff = t_eff_c[0]
log(f"    Tanggal mulai efektif: {t_eff.date()}")

# Tanggal dikeluarkan (< 60 saham layak)
mask_eff_pre = cl_lc.index >= t_eff
n_elig_eff = n_elig_l[mask_eff_pre]
drop_days = n_elig_eff[n_elig_eff < MIN_ELIG_PER_DAY]
log(f"    Tanggal < 60 saham layak: {len(drop_days)}")

# Terapkan drop days
mask_eff = mask_eff_pre & (~cl_lc.index.isin(drop_days.index))
n_eff_bars = int(mask_eff.sum())
log(f"    Bar efektif: {n_eff_bars}")

# A5. Gerbang A
total_cells_eff = int(cl_lc.loc[mask_eff].size)
cleaning_nan_eff = int(cleaning_nan_mask.loc[mask_eff].sum().sum())
pct_cleaning = 100 * cleaning_nan_eff / total_cells_eff if total_cells_eff else 0
nan_total_eff = int(cl_lc.loc[mask_eff].isna().sum().sum())
pct_nan_total = 100 * nan_total_eff / total_cells_eff if total_cells_eff else 0
n_elig_eff_mean = float(n_elig_l[mask_eff].mean())
n_elig_eff_median = float(n_elig_l[mask_eff].median())

g1 = n_eff_bars >= 1200
g2 = pct_cleaning <= 5
g3 = n_elig_eff_mean >= 60

log("")
log("=" * 90)
log("  GERBANG A")
log("=" * 90)
log(f"  A1 (bar efektif >= 1200)       : {n_eff_bars} → {'LOLOS' if g1 else 'GAGAL'}")
log(f"  A2 (NaN pembersihan <= 5%)     : {pct_cleaning:.4f}% → {'LOLOS' if g2 else 'GAGAL'}")
log(f"  A2b (NaN total, info)          : {pct_nan_total:.4f}%")
log(f"  A3 (layak/hari >= 60)          : {n_elig_eff_mean:.1f} → {'LOLOS' if g3 else 'GAGAL'}")
log(f"  GERBANG A: {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}")
log("=" * 90)

if not (g1 and g2 and g3):
    log("  STOP: Gerbang A GAGAL.")
    sys.exit(5)

# A6. Segel data panjang
op_seal = op_lc.loc[mask_eff]; hi_seal = hi_lc.loc[mask_eff]
lo_seal = lo_lc.loc[mask_eff]; cl_seal = cl_lc.loc[mask_eff]
vo_seal = vo_lc.loc[mask_eff]

long_data = {
    "price_open": op_seal, "price_high": hi_seal,
    "price_low": lo_seal, "price_close": cl_seal,
    "price_volume": vo_seal,
    "idx_tickers": list(cl_seal.columns),
    "t_eff": str(t_eff.date()),
    "sealed_end_date": str(cl_seal.index.max().date()),
}
pd.to_pickle(long_data, LONG_SEAL_PATH)

long_manifest = {
    "sealed_at": datetime.now().isoformat(timespec="seconds"),
    "t_eff": str(t_eff.date()),
    "last_bar": str(cl_seal.index.max().date()),
    "n_stocks": int(cl_seal.shape[1]),
    "n_bars": int(cl_seal.shape[0]),
    "sha256": sha256_file(LONG_SEAL_PATH),
    "script": "Tahap3_Horizon20.py",
}
json.dump(long_manifest, open(LONG_MANIFEST, "w"), indent=2)
log(f"\n  data_long_sealed.pkl: {os.path.getsize(LONG_SEAL_PATH)} byte")
log(f"  SHA256: {long_manifest['sha256']}")

# ============================================================
# BAGIAN B — UJI N=20
# ============================================================
log("")
log("=" * 90)
log("  BAGIAN B — 5 SINYAL N=20")
log("=" * 90)

# B1. Bagi 3 blok
idx_b = cl_seal.index
n_b = len(idx_b)
blk = n_b // 3
tK1_end = idx_b[blk - 1]; tK2_start = idx_b[blk]; tK2_end = idx_b[2*blk - 1]; tK3_start = idx_b[2*blk]
log(f"  K1: {idx_b[0].date()} → {tK1_end.date()} ({blk} bar)")
log(f"  K2: {tK2_start.date()} → {tK2_end.date()} ({blk} bar)")
log(f"  K3: {tK3_start.date()} → {idx_b[-1].date()} ({n_b - 2*blk} bar)")

# Buang warm-up (260 bar)
idx_b2 = idx_b[WARMUP_BARS:]
op_b = op_seal.loc[idx_b2]; hi_b = hi_seal.loc[idx_b2]
lo_b = lo_seal.loc[idx_b2]; cl_b = cl_seal.loc[idx_b2]
vo_b = vo_seal.loc[idx_b2]
elig_b = elig_l.loc[idx_b2]

# B2. Fitur & sinyal (dari data penuh, sebelum warm-up cut)
def build_features_t3_full(op, hi, lo, cl, vo, elig):
    # MOM60
    roc60 = cl / cl.shift(60) - 1.0
    roc60_rank = roc60.where(elig).rank(axis=1, pct=True)
    mom60 = (roc60_rank >= 0.80) & roc60.notna() & elig

    # MOM12-1
    roc12_1 = cl.shift(20) / cl.shift(250) - 1.0
    roc12_1_rank = roc12_1.where(elig).rank(axis=1, pct=True)
    mom12_1 = (roc12_1_rank >= 0.80) & roc12_1.notna() & elig

    # NEAR52H
    high_52w = hi.rolling(252, min_periods=252).max()
    near52h = (cl >= 0.95 * high_52w) & elig

    # LOWVOL
    ret = cl.pct_change()
    vol60 = ret.rolling(60, min_periods=60).std()
    vol60_rank = vol60.where(elig).rank(axis=1, pct=True)
    lowvol = (vol60_rank <= 0.20) & vol60.notna() & elig

    # REV20
    roc20 = cl / cl.shift(20) - 1.0
    roc20_rank = roc20.where(elig).rank(axis=1, pct=True)
    rev20 = (roc20_rank <= 0.20) & roc20.notna() & elig

    return {
        "MOM60":   mom60,
        "MOM12-1": mom12_1,
        "NEAR52H": near52h,
        "LOWVOL":  lowvol,
        "REV20":   rev20,
    }

signals_b = build_features_t3_full(op_seal, hi_seal, lo_seal, cl_seal, vo_seal, elig_l)

# B4. Uji 0
log("")
log("  B4 — UJI 0 (20 pasangan/sinyal)")
rng0 = random.Random(2026)
valid_T = idx_b2[300 : len(idx_b2) - 25]
all_pairs = [(t, T) for t in cl_b.columns for T in valid_T]
sample_pairs = rng0.sample(all_pairs, min(20, len(all_pairs)))

def feat_snapshot(d_op, d_hi, d_cl, tkr, T):
    o = d_op.loc[:T, tkr]; h = d_hi.loc[:T, tkr]; c = d_cl.loc[:T, tkr]
    r60 = float(c.iloc[-1]/c.iloc[-61] - 1) if len(c) >= 61 and pd.notna(c.iloc[-61]) and c.iloc[-61] != 0 else np.nan
    if len(c) >= 271:
        r12_1 = c.iloc[-21] / c.iloc[-251] - 1 if pd.notna(c.iloc[-21]) and pd.notna(c.iloc[-251]) and c.il

_IncompleteInputError: incomplete input (484184174.py, line 32)

In [ ]:
# ============================================================
# L2-A — Mount + Cek File + Buat Tahap3_Horizon20.py kosong
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

print("=" * 100)
print("  CEK FILE YANG DITUJU")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("data_sealed.pkl",     f"{BASE}/data_sealed.pkl"),
    ("data_old_sealed.pkl", f"{BASE}/data_old_sealed.pkl"),
    ("seal_manifest.json",  f"{BASE}/seal_manifest.json"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:24s} {size:>10} byte")
print("=" * 100)

# Buat file kosong (overwrite)
with open(SCRIPT_PATH, "w") as f:
    f.write("")
print(f"✅ File dibuat kosong: {SCRIPT_PATH}")
print(f"   Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel L2-B untuk menulis bagian 1/5")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  CEK FILE YANG DITUJU
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  ✅ data_sealed.pkl            14161504 byte
  ✅ data_old_sealed.pkl        10425567 byte
  ✅ seal_manifest.json              309 byte
✅ File dibuat kosong: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py
   Ukuran: 0 byte

Lanjut ke sel L2-B untuk menulis bagian 1/5


In [ ]:
# ============================================================
# L2-B — Blok 1/5: Header, konstanta, fungsi util
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK1 = r'''# ============================================================
# Tahap3_Horizon20.py
# KOMPAS TAHAP 3 — Horizon 20 Hari di Data Panjang (2005-2026)
# ============================================================
# Urutan: Langkah 0 -> Bagian A -> Gerbang A -> Bagian B
# ATURAN: N=20, data tunggal 2005-01-01 → 2026-06-05, per-ticker, auto_adjust=True
# ============================================================

import os, sys, csv, json, random, hashlib, time
from datetime import datetime
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"[WARN] Mount Drive: {e}")

BASE              = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL_PATH         = f"{BASE}/data_sealed.pkl"
OLD_SEAL_PATH     = f"{BASE}/data_old_sealed.pkl"
LONG_SEAL_PATH    = f"{BASE}/data_long_sealed.pkl"
LONG_MANIFEST     = f"{BASE}/data_long_manifest.json"
LONG_EVENTS       = f"{BASE}/data_events_long.csv"
RUN_LOG_PATH      = f"{BASE}/run_log.csv"
REPORT_PATH       = f"{BASE}/LAPORAN_Tahap3.md"
PROGRESS_PATH     = f"{BASE}/_download_long_progress.pkl"

CUTOFF_NEW        = pd.Timestamp("2026-06-07")
CUTOFF_LONG_END   = pd.Timestamp("2026-06-05")

MIN_PRICE         = 200.0
MIN_TURNOVER_MA20 = 1_000_000_000.0
MIN_ELIG_PER_DAY  = 60
WARMUP_BARS       = 260
WINDOW_BACK       = 3
WINDOW_FWD        = 25

COST_BASE_SIDE    = 0.0015
COST_STRESS_SIDE  = 0.0030
N_LIST            = [20]

TARGET_L0 = {
    ("BRK", 5,  "C1"): +0.00325,
    ("BRK", 5,  "C2"): +0.00147,
    ("BRK", 10, "C1"): +0.01036,
    ("BRK", 10, "C2"): +0.00085,
    ("MOM", 5,  "C1"): -0.00292,
    ("MOM", 5,  "C2"): -0.00277,
    ("MOM", 10, "C1"): +0.00080,
    ("MOM", 10, "C2"): -0.00384,
}
TOL_L0 = 0.0002

random.seed(42); np.random.seed(42)

def log(msg=""):
    print(msg, flush=True)

def rsi_wilder(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0); loss = -delta.clip(upper=0)
    ag = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    al = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    rs = ag / al.replace(0, np.nan)
    return 100 - 100/(1+rs)

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def fix_p1b(p, exc):
    if pd.isna(p): return float("nan")
    if exc > 0:   return p / 2
    if exc < 0:   return 1 - p / 2
    return 0.5

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""): h.update(c)
    return h.hexdigest()

log("=" * 90)
log("  KOMPAS TAHAP 3 — Tahap3_Horizon20.py")
log("=" * 90)

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK1)

print(f"✅ Blok 1/5 ditulis ({len(BLOK1)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel L2-C untuk menulis bagian 2/5")

✅ Blok 1/5 ditulis (2810 byte)
   File sekarang: 2816 byte

Lanjut ke sel L2-C untuk menulis bagian 2/5


In [ ]:
# ============================================================
# L2-C — Blok 2/5: Langkah 0 (reproduksi Tahap 2 pada data_old_sealed.pkl)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK2 = r'''
# ============================================================
# LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl (konfigurasi Tahap 2)
# ============================================================
log("")
log("=" * 90)
log("  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl")
log("=" * 90)

if not os.path.exists(OLD_SEAL_PATH):
    log(f"STOP: {OLD_SEAL_PATH} tidak ada."); sys.exit(1)

old_data = pd.read_pickle(OLD_SEAL_PATH)
op_old = old_data["price_open"]; hi_old = old_data["price_high"]
lo_old = old_data["price_low"];  cl_old = old_data["price_close"]
vo_old = old_data["price_volume"]

tC1_end   = pd.Timestamp("2022-09-02")
tC2_start = pd.Timestamp("2022-09-05")

turnover = cl_old * vo_old
toma20 = turnover.rolling(20, min_periods=20).mean()
elig_old = (cl_old >= MIN_PRICE) & (toma20 >= MIN_TURNOVER_MA20)

def build_features_L0(op, hi, lo, cl, vo, elig):
    roc3 = cl / cl.shift(3) - 1.0
    hp10 = hi.shift(1).rolling(10, min_periods=10).max()
    brk = cl > hp10
    rsi14 = rsi_wilder(cl, 14)
    rev = (rsi14.shift(1) < 30) & (cl > cl.shift(1))
    vma20p = vo.shift(1).rolling(20, min_periods=20).mean()
    gap = op / cl.shift(1) - 1.0
    gapvol = (gap > 0.01) & (vo > 1.5 * vma20p)
    roc3_rank = roc3.where(elig).rank(axis=1, pct=True)
    mom = (roc3_rank >= 0.80) & roc3.notna() & elig
    return {"MOM": mom, "BRK": brk & elig, "REV": rev & elig, "GAPVOL": gapvol & elig}

signals_L0 = build_features_L0(op_old, hi_old, lo_old, cl_old, vo_old, elig_old)

idx_old = cl_old.index
WARMUP_L0 = 25
idx_L0 = idx_old[WARMUP_L0:]
op_L0 = op_old.loc[idx_L0]; hi_L0 = hi_old.loc[idx_L0]
lo_L0 = lo_old.loc[idx_L0]; cl_L0 = cl_old.loc[idx_L0]
vo_L0 = vo_old.loc[idx_L0]
elig_L0 = elig_old.loc[idx_L0]

def run_8_uji_L0(idx, op, hi, lo, cl, vo, eligible, sigs, tC1_end, tC2_start):
    def trade_return(tkr, t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
        if pd.isna(o) or pd.isna(c): return None
        entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
        return (exit_ - entry) / entry
    def bm_avg(t, N, cost_side):
        try: i = idx.get_loc(t)
        except KeyError: return None
        if i + N >= len(idx): return None
        t_ent = idx[i+1]; t_exit = idx[i+N]
        elig = eligible.columns[eligible.loc[t]]
        rets = []
        for tkr in elig:
            o = op.at[t_ent, tkr]; c = cl.at[t_exit, tkr]
            if pd.isna(o) or pd.isna(c): continue
            entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
            rets.append((exit_ - entry) / entry)
        return float(np.mean(rets)) if rets else None

    results = {}
    for sig_name, sig_df in sigs.items():
        for N in [5, 10]:
            for period, t1, t2 in [("C1", None, tC1_end), ("C2", tC2_start, None)]:
                mask_t = pd.Series((sig_df.index <= t2) if t2 is not None else (sig_df.index >= t1),
                                   index=sig_df.index)
                sig_list = []
                for t in sig_df.index:
                    if not mask_t.loc[t]: continue
                    for tkr in sig_df.columns:
                        if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
                raw_b, exc_raw, dates = [], [], []
                for tkr, t in sig_list:
                    rb = trade_return(tkr, t, N, COST_BASE_SIDE)
                    if rb is None: continue
                    bm = bm_avg(t, N, COST_BASE_SIDE)
                    if bm is None: continue
                    raw_b.append(rb); exc_raw.append(rb - bm); dates.append(t)
                exc_w = winsorize(np.array(exc_raw))
                results[(sig_name, N, period)] = {
                    "n_trades": len(raw_b),
                    "mean_exc_w": float(np.mean(exc_w)) if len(exc_w) else float("nan"),
                }
    return results

results_L0 = run_8_uji_L0(idx_L0, op_L0, hi_L0, lo_L0, cl_L0, vo_L0, elig_L0,
                          signals_L0, tC1_end, tC2_start)

log("  Target vs hasil Langkah 0:")
log(f"  {'Sinyal-N':12s} {'Periode':8s} {'hasil':>12s} {'target':>12s} {'cocok?':>8s}")
log("-" * 60)
all_match = True
for key, tgt in TARGET_L0.items():
    r = results_L0.get(key)
    if r is None: all_match = False; continue
    exc = r["mean_exc_w"]
    cocok = abs(exc - tgt) <= TOL_L0
    if not cocok: all_match = False
    log(f"  {key[0]:8s} N={key[1]:<2d} {key[2]:8s} {exc*100:>+11.4f}% {tgt*100:>+11.4f}% {'COCOK' if cocok else 'BEDA':>8s}")
log(f"\n  LANGKAH 0 KESELURUHAN: {'COCOK' if all_match else 'TIDAK COCOK — STOP'}")

if not all_match:
    log("  STOP: Langkah 0 tidak cocok. Tidak lanjut ke data panjang.")
    sys.exit(2)

log("  Langkah 0 LOLOS. Lanjut ke Bagian A.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK2)

print(f"✅ Blok 2/5 ditulis ({len(BLOK2)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel L2-D untuk menulis bagian 3/5")

✅ Blok 2/5 ditulis (4854 byte)
   File sekarang: 7676 byte

Lanjut ke sel L2-D untuk menulis bagian 3/5


In [ ]:
# ============================================================
# L2-D — Blok 3/5: Bagian A
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK3 = r'''
# ============================================================
# BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
# ============================================================
log("")
log("=" * 90)
log("  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)")
log("=" * 90)

try:
    import yfinance as yf
except ImportError:
    os.system("pip install -q yfinance")
    import yfinance as yf

new_data = pd.read_pickle(SEAL_PATH)
idx_tickers = new_data["idx_tickers"]
log(f"  Ticker: {len(idx_tickers)}, unduh per-ticker start=2005-01-01 end=2026-06-06")

DWN_OK = 0; DWN_FAIL = 0
op_long, hi_long, lo_long, cl_long, vo_long = {}, {}, {}, {}, {}

if os.path.exists(PROGRESS_PATH):
    try:
        prog = pd.read_pickle(PROGRESS_PATH)
        op_long = prog.get("op", {}); hi_long = prog.get("hi", {})
        lo_long = prog.get("lo", {}); cl_long = prog.get("cl", {})
        vo_long = prog.get("vo", {})
        log(f"  Resume: {len(cl_long)} ticker sudah terunduh")
    except Exception:
        pass

t_start = time.time()
for i, tkr in enumerate(idx_tickers):
    if tkr in cl_long and not cl_long[tkr].empty:
        DWN_OK += 1; continue
    df = None
    for attempt in range(3):
        try:
            df = yf.download(tkr, start="2005-01-01", end="2026-06-06",
                             interval="1d", auto_adjust=True,
                             progress=False, threads=False)
            if df is not None and not df.empty: break
        except Exception:
            time.sleep(1.5); continue
    if df is None or df.empty:
        DWN_FAIL += 1; continue
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    op_long[tkr] = df["Open"]; hi_long[tkr] = df["High"]
    lo_long[tkr] = df["Low"];  cl_long[tkr] = df["Close"]
    vo_long[tkr] = df["Volume"]
    DWN_OK += 1
    if (i + 1) % 25 == 0:
        elapsed = time.time() - t_start
        log(f"  [{i+1}/{len(idx_tickers)}] OK={DWN_OK} FAIL={DWN_FAIL} ({elapsed:.0f}s)")
    if (i + 1) % 50 == 0:
        pd.to_pickle({"op": op_long, "hi": hi_long, "lo": lo_long,
                      "cl": cl_long, "vo": vo_long}, PROGRESS_PATH)

pd.to_pickle({"op": op_long, "hi": hi_long, "lo": lo_long,
              "cl": cl_long, "vo": vo_long}, PROGRESS_PATH)
log(f"  Unduh selesai: OK={DWN_OK} FAIL={DWN_FAIL}")

op_long_df = pd.DataFrame(op_long).sort_index()
hi_long_df = pd.DataFrame(hi_long).sort_index()
lo_long_df = pd.DataFrame(lo_long).sort_index()
cl_long_df = pd.DataFrame(cl_long).sort_index()
vo_long_df = pd.DataFrame(vo_long).sort_index()

mask_end = cl_long_df.index <= CUTOFF_LONG_END
op_long_df = op_long_df.loc[mask_end]; hi_long_df = hi_long_df.loc[mask_end]
lo_long_df = lo_long_df.loc[mask_end]; cl_long_df = cl_long_df.loc[mask_end]
vo_long_df = vo_long_df.loc[mask_end]

log(f"  Shape: {cl_long_df.shape}, range {cl_long_df.index.min().date()} → {cl_long_df.index.max().date()}")

# A2. Verifikasi vs data_sealed
log("")
log("  A2 — Verifikasi return harian vs data_sealed (2023-10-10 → 2026-06-05)")
sealed_close = new_data["price_close"]
sealed_close = sealed_close.loc[sealed_close.index <= CUTOFF_NEW]
common_tkrs = sorted(set(cl_long_df.columns) & set(sealed_close.columns))
log(f"    Ticker common: {len(common_tkrs)}")

verify_start = pd.Timestamp("2023-10-10")
verify_end = pd.Timestamp("2026-06-05")

ret_long = cl_long_df[common_tkrs].pct_change()
ret_sealed = sealed_close[common_tkrs].pct_change()

common_dates = ret_long.index.intersection(ret_sealed.index)
common_dates = common_dates[(common_dates >= verify_start) & (common_dates <= verify_end)]

total_cells = 0
match_cells = 0
for tkr in common_tkrs:
    a = ret_long.loc[common_dates, tkr]
    b = ret_sealed.loc[common_dates, tkr]
    valid = a.notna() & b.notna()
    total_cells += int(valid.sum())
    match_cells += int((valid & ((a - b).abs() < 0.001)).sum())

pct_match = 100 * match_cells / total_cells if total_cells else 0
log(f"    Sel cocok: {pct_match:.2f}% ({match_cells}/{total_cells})")

if pct_match < 99:
    log(f"  STOP: verifikasi < 99%.")
    sys.exit(3)
log("    Verifikasi >= 99% OK.")

# A3. Pembersihan
log("")
log("  A3 — Pembersihan (Addendum 4 + O/H/L/C)")
events_long = []
for tkr in cl_long_df.columns:
    ret = cl_long_df[tkr] / cl_long_df[tkr].shift(1) - 1
    for d in ret.index[ret.abs() > 0.40]:
        prev_idx = cl_long_df.index.get_loc(d)
        events_long.append({
            "ticker": tkr, "t0": d,
            "close_prev": cl_long_df.iloc[prev_idx-1][tkr] if prev_idx > 0 else np.nan,
            "close_now": cl_long_df.at[d, tkr],
        })
df_events_long = pd.DataFrame(events_long).sort_values("t0").reset_index(drop=True)
df_events_long.to_csv(LONG_EVENTS, index=False)
log(f"    Event: {len(df_events_long)}")

op_lc = op_long_df.copy(); hi_lc = hi_long_df.copy(); lo_lc = lo_long_df.copy()
cl_lc = cl_long_df.copy(); vo_lc = vo_long_df.copy()
idx_full = cl_lc.index
cleaning_nan_mask = pd.DataFrame(False, index=cl_lc.index, columns=cl_lc.columns)

for _, ev in df_events_long.iterrows():
    tkr, t0 = ev["ticker"], ev["t0"]
    if tkr not in cl_lc.columns: continue
    try: i0 = idx_full.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_full)-1, i0+WINDOW_FWD)+1):
        d = idx_full[j]
        cleaning_nan_mask.at[d, tkr] = True
        op_lc.at[d, tkr] = np.nan; hi_lc.at[d, tkr] = np.nan
        lo_lc.at[d, tkr] = np.nan; cl_lc.at[d, tkr] = np.nan; vo_lc.at[d, tkr] = np.nan

viol_mask = (lo_lc > np.minimum(op_lc, cl_lc)) | (hi_lc < np.maximum(op_lc, cl_lc))
n_viol = int(viol_mask.sum().sum())
for tkr in viol_mask.columns:
    for d in viol_mask.index[viol_mask[tkr]]:
        cleaning_nan_mask.at[d, tkr] = True
        op_lc.at[d, tkr] = np.nan; hi_lc.at[d, tkr] = np.nan
        lo_lc.at[d, tkr] = np.nan; cl_lc.at[d, tkr] = np.nan; vo_lc.at[d, tkr] = np.nan
log(f"    Pelanggaran O/H/L/C: {n_viol} sel")

# A4. Filter saham layak
turnover_l = cl_lc * vo_lc
toma20_l = turnover_l.rolling(20, min_periods=20).mean()
elig_l = (cl_lc >= MIN_PRICE) & (toma20_l >= MIN_TURNOVER_MA20)
n_elig_l = elig_l.sum(axis=1)

ma60 = n_elig_l.rolling(60, min_periods=60).mean()
t_eff_c = ma60.index[ma60 >= 60]
if len(t_eff_c) == 0:
    log("  STOP: tidak ada tanggal efektif."); sys.exit(4)
t_eff = t_eff_c[0]
log(f"    Tanggal mulai efektif: {t_eff.date()}")

mask_eff_pre = cl_lc.index >= t_eff
n_elig_eff = n_elig_l[mask_eff_pre]
drop_days = n_elig_eff[n_elig_eff < MIN_ELIG_PER_DAY]
log(f"    Tanggal < 60 saham layak: {len(drop_days)}")

mask_eff = mask_eff_pre & (~cl_lc.index.isin(drop_days.index))
n_eff_bars = int(mask_eff.sum())
log(f"    Bar efektif: {n_eff_bars}")

# A5. Gerbang A
total_cells_eff = int(cl_lc.loc[mask_eff].size)
cleaning_nan_eff = int(cleaning_nan_mask.loc[mask_eff].sum().sum())
pct_cleaning = 100 * cleaning_nan_eff / total_cells_eff if total_cells_eff else 0
nan_total_eff = int(cl_lc.loc[mask_eff].isna().sum().sum())
pct_nan_total = 100 * nan_total_eff / total_cells_eff if total_cells_eff else 0
n_elig_eff_mean = float(n_elig_l[mask_eff].mean())
n_elig_eff_median = float(n_elig_l[mask_eff].median())

g1 = n_eff_bars >= 1200
g2 = pct_cleaning <= 5
g3 = n_elig_eff_mean >= 60

log("")
log("=" * 90)
log("  GERBANG A")
log("=" * 90)
log(f"  A1 (bar efektif >= 1200)       : {n_eff_bars} -> {'LOLOS' if g1 else 'GAGAL'}")
log(f"  A2 (NaN pembersihan <= 5%)     : {pct_cleaning:.4f}% -> {'LOLOS' if g2 else 'GAGAL'}")
log(f"  A2b (NaN total, info)          : {pct_nan_total:.4f}%")
log(f"  A3 (layak/hari >= 60)          : {n_elig_eff_mean:.1f} -> {'LOLOS' if g3 else 'GAGAL'}")
log(f"  GERBANG A: {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}")
log("=" * 90)

if not (g1 and g2 and g3):
    log("  STOP: Gerbang A GAGAL.")
    sys.exit(5)

# A6. Segel data panjang
op_seal = op_lc.loc[mask_eff]; hi_seal = hi_lc.loc[mask_eff]
lo_seal = lo_lc.loc[mask_eff]; cl_seal = cl_lc.loc[mask_eff]
vo_seal = vo_lc.loc[mask_eff]

long_data = {
    "price_open": op_seal, "price_high": hi_seal,
    "price_low": lo_seal, "price_close": cl_seal,
    "price_volume": vo_seal,
    "idx_tickers": list(cl_seal.columns),
    "t_eff": str(t_eff.date()),
    "sealed_end_date": str(cl_seal.index.max().date()),
}
pd.to_pickle(long_data, LONG_SEAL_PATH)

long_manifest = {
    "sealed_at": datetime.now().isoformat(timespec="seconds"),
    "t_eff": str(t_eff.date()),
    "last_bar": str(cl_seal.index.max().date()),
    "n_stocks": int(cl_seal.shape[1]),
    "n_bars": int(cl_seal.shape[0]),
    "sha256": sha256_file(LONG_SEAL_PATH),
    "script": "Tahap3_Horizon20.py",
}
json.dump(long_manifest, open(LONG_MANIFEST, "w"), indent=2)
log(f"\n  data_long_sealed.pkl: {os.path.getsize(LONG_SEAL_PATH)} byte")
log(f"  SHA256: {long_manifest['sha256']}")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK3)

print(f"✅ Blok 3/5 ditulis ({len(BLOK3)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel L2-E untuk menulis bagian 4/5")

✅ Blok 3/5 ditulis (8784 byte)
   File sekarang: 16476 byte

Lanjut ke sel L2-E untuk menulis bagian 4/5


In [ ]:
# ============================================================
# L2-E — Blok 4/5: Bagian B (5 sinyal N=20 + Uji 0 + Plasebo)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK4 = r'''
# ============================================================
# BAGIAN B — UJI N=20 (5 sinyal, 3 blok)
# ============================================================
log("")
log("=" * 90)
log("  BAGIAN B — 5 SINYAL N=20")
log("=" * 90)

# B1. Bagi 3 blok
idx_b = cl_seal.index
n_b = len(idx_b)
blk = n_b // 3
tK1_end = idx_b[blk - 1]; tK2_start = idx_b[blk]
tK2_end = idx_b[2*blk - 1]; tK3_start = idx_b[2*blk]
log(f"  K1: {idx_b[0].date()} -> {tK1_end.date()} ({blk} bar)")
log(f"  K2: {tK2_start.date()} -> {tK2_end.date()} ({blk} bar)")
log(f"  K3: {tK3_start.date()} -> {idx_b[-1].date()} ({n_b - 2*blk} bar)")

# Buang warm-up 260 bar
idx_b2 = idx_b[WARMUP_BARS:]
op_b = op_seal.loc[idx_b2]; hi_b = hi_seal.loc[idx_b2]
lo_b = lo_seal.loc[idx_b2]; cl_b = cl_seal.loc[idx_b2]
vo_b = vo_seal.loc[idx_b2]
elig_b = elig_l.loc[idx_b2]

# B2. Fitur & sinyal (dari data penuh, sebelum warm-up cut)
def build_features_t3_full(op, hi, lo, cl, vo, elig):
    roc60 = cl / cl.shift(60) - 1.0
    roc60_rank = roc60.where(elig).rank(axis=1, pct=True)
    mom60 = (roc60_rank >= 0.80) & roc60.notna() & elig

    roc12_1 = cl.shift(20) / cl.shift(250) - 1.0
    roc12_1_rank = roc12_1.where(elig).rank(axis=1, pct=True)
    mom12_1 = (roc12_1_rank >= 0.80) & roc12_1.notna() & elig

    high_52w = hi.rolling(252, min_periods=252).max()
    near52h = (cl >= 0.95 * high_52w) & elig

    ret = cl.pct_change()
    vol60 = ret.rolling(60, min_periods=60).std()
    vol60_rank = vol60.where(elig).rank(axis=1, pct=True)
    lowvol = (vol60_rank <= 0.20) & vol60.notna() & elig

    roc20 = cl / cl.shift(20) - 1.0
    roc20_rank = roc20.where(elig).rank(axis=1, pct=True)
    rev20 = (roc20_rank <= 0.20) & roc20.notna() & elig

    return {
        "MOM60":   mom60,
        "MOM12-1": mom12_1,
        "NEAR52H": near52h,
        "LOWVOL":  lowvol,
        "REV20":   rev20,
    }

signals_b = build_features_t3_full(op_seal, hi_seal, lo_seal, cl_seal, vo_seal, elig_l)

# B4. Uji 0
log("")
log("  B4 — UJI 0 (20 pasangan/sinyal)")
rng0 = random.Random(2026)
valid_T = idx_b2[300 : len(idx_b2) - 25]
all_pairs = [(t, T) for t in cl_b.columns for T in valid_T]
sample_pairs = rng0.sample(all_pairs, min(20, len(all_pairs)))

def feat_snapshot(d_op, d_hi, d_cl, tkr, T):
    o = d_op.loc[:T, tkr]; h = d_hi.loc[:T, tkr]; c = d_cl.loc[:T, tkr]
    r60 = float(c.iloc[-1]/c.iloc[-61] - 1) if len(c) >= 61 and pd.notna(c.iloc[-61]) and c.iloc[-61] != 0 else np.nan
    if len(c) >= 271:
        r12_1 = c.iloc[-21] / c.iloc[-251] - 1 if pd.notna(c.iloc[-21]) and pd.notna(c.iloc[-251]) and c.iloc[-251] != 0 else np.nan
    else:
        r12_1 = np.nan
    if len(h) >= 252:
        h52 = h.iloc[-252:].max()
        near52 = bool(pd.notna(h52) and c.iloc[-1] >= 0.95 * h52)
    else:
        near52 = False
    if len(c) >= 61:
        vol60 = c.pct_change().iloc[-60:].std()
    else:
        vol60 = np.nan
    r20 = float(c.iloc[-1]/c.iloc[-21] - 1) if len(c) >= 21 and pd.notna(c.iloc[-21]) and c.iloc[-21] != 0 else np.nan
    return {"roc60": r60, "roc12_1": r12_1, "near52": near52, "vol60": vol60, "roc20": r20}

uji0_results = {}
for name in ["MOM60", "MOM12-1", "NEAR52H", "LOWVOL", "REV20"]:
    mism = 0
    for tkr, T in sample_pairs:
        f1 = feat_snapshot(op_seal, hi_seal, cl_seal, tkr, T)
        f2 = feat_snapshot(op_lc, hi_lc, cl_lc, tkr, T)
        for k in f1:
            a, b = f1[k], f2[k]
            if isinstance(a, float) and isinstance(b, float):
                if pd.isna(a) and pd.isna(b): continue
                if (pd.isna(a) != pd.isna(b)) or (not pd.isna(a) and abs(a-b) > 1e-9): mism += 1
            else:
                if a != b: mism += 1
    uji0_results[name] = (mism == 0)
    log(f"    {name:8s} mism={mism} -> {'LULUS' if mism == 0 else 'GAGAL'}")

if not all(uji0_results.values()):
    log("  STOP: Uji 0 GAGAL.")
    sys.exit(6)

# Fungsi trade & benchmark
def trade_return(tkr, t, N, cost_side):
    try: i = idx_b2.get_loc(t)
    except KeyError: return None
    if i + N >= len(idx_b2): return None
    t_ent = idx_b2[i+1]; t_exit = idx_b2[i+N]
    o = op_b.at[t_ent, tkr]; c = cl_b.at[t_exit, tkr]
    if pd.isna(o) or pd.isna(c): return None
    entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
    return (exit_ - entry) / entry

def bm_avg(t, N, cost_side):
    try: i = idx_b2.get_loc(t)
    except KeyError: return None
    if i + N >= len(idx_b2): return None
    t_ent = idx_b2[i+1]; t_exit = idx_b2[i+N]
    elig_tkrs = elig_b.columns[elig_b.loc[t]]
    rets = []
    for tkr in elig_tkrs:
        o = op_b.at[t_ent, tkr]; c = cl_b.at[t_exit, tkr]
        if pd.isna(o) or pd.isna(c): continue
        entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
        rets.append((exit_ - entry) / entry)
    return float(np.mean(rets)) if rets else None

# B3. Loop 5 sinyal x 3 blok
def in_blk(t, blk_id):
    if blk_id == "K1": return t <= tK1_end
    if blk_id == "K2": return (t >= tK2_start) and (t <= tK2_end)
    if blk_id == "K3": return t >= tK3_start
    return False

def run_signal(sig_name, sig_df, N=20):
    results = {}
    daily_pool = []
    all_exc = {"K1": [], "K2": [], "K3": []}
    n_trades = {"K1": 0, "K2": 0, "K3": 0}
    raw_b_l = {"K1": [], "K2": [], "K3": []}
    raw_s_l = {"K1": [], "K2": [], "K3": []}
    for blk_id in ["K1", "K2", "K3"]:
        sig_list = []
        for t in sig_df.index:
            if not in_blk(t, blk_id): continue
            for tkr in sig_df.columns:
                if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
        exc_raw, dates = [], []
        for tkr, t in sig_list:
            rb = trade_return(tkr, t, N, COST_BASE_SIDE)
            rs = trade_return(tkr, t, N, COST_STRESS_SIDE)
            if rb is None or rs is None: continue
            bm = bm_avg(t, N, COST_BASE_SIDE)
            if bm is None: continue
            raw_b_l[blk_id].append(rb); raw_s_l[blk_id].append(rs)
            exc_raw.append(rb - bm); dates.append(t)
        n_trades[blk_id] = len(raw_b_l[blk_id])
        exc_w = winsorize(np.array(exc_raw))
        for t, e in zip(dates, exc_w):
            all_exc[blk_id].append(e)
            daily_pool.append((t, e))
        if len(exc_w) >= 10:
            daily = {}
            for t, e in zip(dates, exc_w): daily.setdefault(t, []).append(e)
            ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
            if len(ds) >= 10:
                model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
                t_stat = float(model.tvalues[0]); p_val = float(model.pvalues[0])
            else:
                t_stat = float("nan"); p_val = float("nan")
        else:
            t_stat = float("nan"); p_val = float("nan")
        exc_w_arr = np.array(exc_w)
        raw_b_arr = np.array(raw_b_l[blk_id])
        raw_s_arr = np.array(raw_s_l[blk_id])
        results[(sig_name, N, blk_id)] = {
            "n_trades": n_trades[blk_id],
            "mean_base_raw": float(np.mean(raw_b_arr)) if len(raw_b_arr) else float("nan"),
            "mean_base_w":   float(np.mean(winsorize(raw_b_arr))) if len(raw_b_arr) else float("nan"),
            "mean_stress_raw": float(np.mean(raw_s_arr)) if len(raw_s_arr) else float("nan"),
            "mean_stress_w":   float(np.mean(winsorize(raw_s_arr))) if len(raw_s_arr) else float("nan"),
            "mean_exc_raw":  float(np.mean(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "mean_exc_w":    float(np.mean(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "median_exc":    float(np.median(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "t_stat": t_stat, "p_val": p_val,
        }
    # Pool
    if daily_pool:
        daily_pool_d = {}
        for t, e in daily_pool: daily_pool_d.setdefault(t, []).append(e)
        ds = np.array([np.mean(daily_pool_d[t]) for t in sorted(daily_pool_d.keys())])
        if len(ds) >= 10:
            model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            t_stat = float(model.tvalues[0]); p_val = float(model.pvalues[0])
        else:
            t_stat = float("nan"); p_val = float("nan")
        results[(sig_name, N, "pool")] = {
            "mean_exc_w": float(np.mean(ds)), "t_stat": t_stat, "p_val": p_val,
        }
    return results

all_results = {}
for sig_name, sig_df in signals_b.items():
    res = run_signal(sig_name, sig_df, N=20)
    all_results.update(res)
    log(f"  {sig_name:8s} K1={res[(sig_name,20,'K1')]['n_trades']:5d} "
        f"K2={res[(sig_name,20,'K2')]['n_trades']:5d} "
        f"K3={res[(sig_name,20,'K3')]['n_trades']:5d}")

# B5. Plasebo
log("")
log("  B5 — PLASEBO (20 seed, N=20)")
plasebo_exc = []
for seed in range(1, 21):
    rng_p = random.Random(seed)
    sig_random = pd.DataFrame(False, index=elig_b.index, columns=elig_b.columns)
    for t in elig_b.index:
        eligible_tkrs = elig_b.columns[elig_b.loc[t]]
        for tkr in eligible_tkrs:
            if rng_p.random() < 0.05:
                sig_random.at[t, tkr] = True
    excs = []
    for t in sig_random.index:
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, 20, COST_BASE_SIDE)
            if rb is None: continue
            bm = bm_avg(t, 20, COST_BASE_SIDE)
            if bm is None: continue
            excs.append(rb - bm)
    if excs:
        plasebo_exc.append(float(np.mean(winsorize(np.array(excs)))))
mean_plasebo = float(np.mean(plasebo_exc)) if plasebo_exc else float("nan")
plasebo_ok = abs(mean_plasebo) <= 0.0010
log(f"    mean plasebo (pool): {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}")

if not plasebo_ok:
    log("  STOP: Plasebo ALARM.")
    sys.exit(7)

# B6. Kriteria LOLOS
log("")
log("=" * 90)
log("  HASIL 5 SINYAL N=20")
log("=" * 90)
ringkasan_lolos = []
for sig_name in signals_b.keys():
    r_k1 = all_results[(sig_name,20,"K1")]
    r_k2 = all_results[(sig_name,20,"K2")]
    r_k3 = all_results[(sig_name,20,"K3")]
    r_pool = all_results[(sig_name,20,"pool")]
    p1_k1 = fix_p1b(r_k1["p_val"], r_k1["mean_exc_w"])
    p1_k2 = fix_p1b(r_k2["p_val"], r_k2["mean_exc_w"])
    p1_k3 = fix_p1b(r_k3["p_val"], r_k3["mean_exc_w"])
    p1_pool = fix_p1b(r_pool["p_val"], r_pool["mean_exc_w"])
    fails = []
    if not (r_k1["mean_exc_w"] > 0 and r_k2["mean_exc_w"] > 0 and r_k3["mean_exc_w"] > 0):
        fails.append("excess tidak semua positif di K1,K2,K3")
    if not (p1_pool < 0.01): fails.append(f"pool p1sisi {p1_pool:.4f} >= 0.01")
    n_blk_sig = sum([p1_k1<0.10, p1_k2<0.10, p1_k3<0.10])
    if n_blk_sig < 2: fails.append(f"hanya {n_blk_sig}/3 blok p<0.10")
    st_ok_pool = r_pool.get("mean_exc_w", 0) > 0
    st_n = sum([r_k1["mean_stress_w"]>0, r_k2["mean_stress_w"]>0, r_k3["mean_stress_w"]>0])
    if not (st_ok_pool and st_n >= 2): fails.append("stres tidak positif di pool & >=2/3 blok")
    if any(r_k["n_trades"] < 1000 for r_k in [r_k1, r_k2, r_k3]): fails.append("ada blok <1000 trade")
    passed = len(fails) == 0
    if passed: ringkasan_lolos.append(sig_name)
    log(f"  {sig_name:8s}  K1 exc={r_k1['mean_exc_w']*100:+.3f}% p={p1_k1:.4f} n={r_k1['n_trades']:5d}  |  "
        f"K2 exc={r_k2['mean_exc_w']*100:+.3f}% p={p1_k2:.4f} n={r_k2['n_trades']:5d}  |  "
        f"K3 exc={r_k3['mean_exc_w']*100:+.3f}% p={p1_k3:.4f} n={r_k3['n_trades']:5d}  |  "
        f"pool p1={p1_pool:.4f}  ->  {'LOLOS' if passed else 'GAGAL'}")

log("")
log(f"  RINGKASAN: {', '.join(ringkasan_lolos) if ringkasan_lolos else 'tidak ada yang LOLOS'}")

# Simpan hasil untuk laporan
hasil_t3 = {
    "all_results": all_results,
    "ringkasan_lolos": ringkasan_lolos,
    "mean_plasebo": mean_plasebo,
    "uji0": uji0_results,
    "t_eff": str(t_eff.date()),
    "n_eff_bars": n_eff_bars,
    "pct_cleaning": pct_cleaning,
    "pct_nan_total": pct_nan_total,
    "n_elig_mean": n_elig_eff_mean,
    "n_elig_median": n_elig_eff_median,
    "drop_days": len(drop_days),
    "n_events": len(df_events_long),
    "n_viol": n_viol,
    "long_manifest": long_manifest,
    "K_bounds": {"K1_end": str(tK1_end.date()), "K2_start": str(tK2_start.date()),
                 "K2_end": str(tK2_end.date()), "K3_start": str(tK3_start.date())},
}
pd.to_pickle(hasil_t3, f"{BASE}/hasil_tahap3.pkl")
log(f"  hasil_tahap3.pkl disimpan.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK4)

print(f"✅ Blok 4/5 ditulis ({len(BLOK4)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel L2-F untuk menulis bagian 5/5 (laporan + verifikasi)")

✅ Blok 4/5 ditulis (12352 byte)
   File sekarang: 28836 byte

Lanjut ke sel L2-F untuk menulis bagian 5/5 (laporan + verifikasi)


In [ ]:
# ============================================================
# L2-F — Blok 5/5: Laporan Tahap 3 + run_log + verifikasi
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK5 = r'''
# ============================================================
# TULIS LAPORAN_Tahap3.md
# ============================================================
ts = datetime.now().isoformat(timespec="seconds")
seal_long = json.load(open(LONG_MANIFEST))["sha256"] if os.path.exists(LONG_MANIFEST) else "n/a"
seal_new = json.load(open(f"{BASE}/seal_manifest.json"))["sha256"] if os.path.exists(f"{BASE}/seal_manifest.json") else "n/a"

report = f"""TAHAP 3 - LAPORAN
Nama file skrip final : Tahap3_Horizon20.py
LANGKAH 0: COCOK (semua 8 target dalam toleransi 0,02 pp)

BAGIAN A
Parameter unduh (sama dengan Tahap 2?) : ya
  - Sumber     : yfinance per-ticker
  - start      : 2005-01-01
  - end        : 2026-06-06
  - interval   : 1d
  - auto_adjust: True
A2 verifikasi vs data_sealed: persen bar-ticker cocok : {pct_match:.2f}%  (ambang >= 99%)
Tanggal mulai efektif | tanggal akhir | bar efektif :
  {t_eff.date()} | {cl_seal.index.max().date()} | {n_eff_bars}
Saham layak per hari (rata-rata, median) ; tanggal dikeluarkan (<60) :
  rata-rata {n_elig_eff_mean:.1f} | median {n_elig_eff_median:.1f} | {len(drop_days)} tanggal dikeluarkan
Peristiwa anomali (jumlah) ; NaN akibat pembersihan (%) | NaN total (%) :
  {len(df_events_long)} event | {pct_cleaning:.4f}% | {pct_nan_total:.4f}%
Gerbang A: (1) | (2) | (3)  LOLOS / GAGAL
  A1 (bar efektif >= 1200)  : {n_eff_bars} -> {'LOLOS' if g1 else 'GAGAL'}
  A2 (NaN pembersihan <= 5%): {pct_cleaning:.4f}% -> {'LOLOS' if g2 else 'GAGAL'}
  A3 (layak/hari >= 60)     : {n_elig_eff_mean:.1f} -> {'LOLOS' if g3 else 'GAGAL'}
  GERBANG A: {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}
SHA256 data_long_sealed.pkl : {seal_long}
(data_sealed.pkl dan data_old_sealed.pkl tidak ditimpa;
 data_long_sealed.pkl tidak disambung dengan keduanya)

BAGIAN B
Batas K1 | K2 | K3 :
  K1: {idx_b[0].date()} -> {tK1_end.date()} ({blk} bar)
  K2: {tK2_start.date()} -> {tK2_end.date()} ({blk} bar)
  K3: {tK3_start.date()} -> {idx_b[-1].date()} ({n_b - 2*blk} bar)
Putaran perbaikan (maks 5) : 0
UJI 0: {', '.join([f"{k}={'LULUS' if v else 'GAGAL'}" for k, v in uji0_results.items()])}
PLASEBO: excess pool rata-rata (20 seed) : {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}

"""
for sig_name in signals_b.keys():
    r_k1 = all_results[(sig_name,20,"K1")]
    r_k2 = all_results[(sig_name,20,"K2")]
    r_k3 = all_results[(sig_name,20,"K3")]
    r_pool = all_results[(sig_name,20,"pool")]
    p1_k1 = fix_p1b(r_k1["p_val"], r_k1["mean_exc_w"])
    p1_k2 = fix_p1b(r_k2["p_val"], r_k2["mean_exc_w"])
    p1_k3 = fix_p1b(r_k3["p_val"], r_k3["mean_exc_w"])
    p1_pool = fix_p1b(r_pool["p_val"], r_pool["mean_exc_w"])
    fails = []
    if not (r_k1["mean_exc_w"] > 0 and r_k2["mean_exc_w"] > 0 and r_k3["mean_exc_w"] > 0):
        fails.append("excess tidak semua positif")
    if not (p1_pool < 0.01): fails.append(f"pool p1sisi {p1_pool:.4f} >= 0.01")
    n_blk_sig = sum([p1_k1<0.10, p1_k2<0.10, p1_k3<0.10])
    if n_blk_sig < 2: fails.append(f"hanya {n_blk_sig}/3 blok p<0.10")
    st_n = sum([r_k1["mean_stress_w"]>0, r_k2["mean_stress_w"]>0, r_k3["mean_stress_w"]>0])
    if not (r_pool.get("mean_exc_w", 0) > 0 and st_n >= 2): fails.append("stres tidak positif di pool & >=2/3 blok")
    if any(r_k["n_trades"] < 1000 for r_k in [r_k1, r_k2, r_k3]): fails.append("ada blok <1000 trade")
    passed = len(fails) == 0
    report += f"{sig_name}\n"
    report += f"  N trade K1 | K2 | K3: {r_k1['n_trades']} | {r_k2['n_trades']} | {r_k3['n_trades']}\n"
    report += f"  Return biaya dasar (mentah|winsorize) K1|K2|K3|pool: "
    report += f"{r_k1['mean_base_raw']*100:+.3f}%|{r_k1['mean_base_w']*100:+.3f}% | "
    report += f"{r_k2['mean_base_raw']*100:+.3f}%|{r_k2['mean_base_w']*100:+.3f}% | "
    report += f"{r_k3['mean_base_raw']*100:+.3f}%|{r_k3['mean_base_w']*100:+.3f}%\n"
    report += f"  Return biaya stres (mentah|winsorize) K1|K2|K3: "
    report += f"{r_k1['mean_stress_raw']*100:+.3f}%|{r_k1['mean_stress_w']*100:+.3f}% | "
    report += f"{r_k2['mean_stress_raw']*100:+.3f}%|{r_k2['mean_stress_w']*100:+.3f}% | "
    report += f"{r_k3['mean_stress_raw']*100:+.3f}%|{r_k3['mean_stress_w']*100:+.3f}%\n"
    report += f"  Excess winsorize K1|K2|K3|pool: "
    report += f"{r_k1['mean_exc_w']*100:+.3f}% | {r_k2['mean_exc_w']*100:+.3f}% | "
    report += f"{r_k3['mean_exc_w']*100:+.3f}% | {r_pool['mean_exc_w']*100:+.3f}%\n"
    report += f"  t-stat HAC K1|K2|K3: {r_k1['t_stat']:+.2f} | {r_k2['t_stat']:+.2f} | {r_k3['t_stat']:+.2f}\n"
    report += f"  p satu sisi K1|K2|K3|pool: {p1_k1:.4f} | {p1_k2:.4f} | {p1_k3:.4f} | {p1_pool:.4f}\n"
    if passed:
        report += "  Hasil: LOLOS\n\n"
    else:
        report += f"  Hasil: GAGAL ({'; '.join(fails)})\n\n"

report += f"""RINGKASAN: {', '.join(ringkasan_lolos) if ringkasan_lolos else 'tidak ada yang LOLOS'}

CATATAN BIAS SURVIVOR
  Universe = 561 ticker dari data_sealed.pkl["idx_tickers"] (daftar saham
  terdaftar per 2026). Saham yang delisting sepanjang 2005-2026 TIDAK ada.
  Efeknya adalah bias survivor yang membuat hasil lolos dibaca sebagai
  BATAS ATAS, bukan bukti final. Karena universe bias ke atas, hasil yang
  GAGAL lebih kuat kesimpulannya daripada hasil yang LOLOS.

PERUBAHAN KODE (satu baris per perubahan)
  -

USULAN (tidak dijalankan)
  -

PERNYATAAN: Angka kriteria / parameter / biaya / data / filter berubah? TIDAK
"""

with open(REPORT_PATH, "w") as f:
    f.write(report)
log(f"\n  LAPORAN_Tahap3.md: {os.path.getsize(REPORT_PATH)} byte")

# Run log
_COLUMNS = ["timestamp","phase","config_version","script_version",
            "global_seed","seal_sha256","params_sha256",
            "n_stocks","n_bars","n_signals","sharpe","max_dd","notes"]
file_exists = os.path.exists(RUN_LOG_PATH)
with open(RUN_LOG_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": ts, "phase": "BACKTEST",
        "config_version": "Gen3-Tahap3", "script_version": "Tahap3_Horizon20",
        "global_seed": 42, "seal_sha256": seal_long[:16] + "...",
        "params_sha256": "",
        "n_stocks": int(cl_seal.shape[1]), "n_bars": int(cl_seal.shape[0]),
        "n_signals": "", "sharpe": "", "max_dd": "",
        "notes": f"KOMPAS Tahap 3 N=20 — LOLOS: {len(ringkasan_lolos)}/5",
    })
log(f"  run_log.csv diperbarui")

log("")
log("=" * 90)
log("  Tahap3_Horizon20.py SELESAI")
log("=" * 90)

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK5)

print(f"✅ Blok 5/5 ditulis ({len(BLOK5)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()

# Verifikasi syntax
import ast
with open(SCRIPT_PATH) as f:
    full = f.read()
try:
    ast.parse(full)
    print("✅ Syntax OK — file siap dijalankan")
except SyntaxError as e:
    print(f"❌ Syntax ERROR: {e}")
    print(f"   Line {e.lineno}: {e.text}")
print()
print("Cara menjalankan:")
print(f'  !python "{SCRIPT_PATH}"')

✅ Blok 5/5 ditulis (6428 byte)
   File sekarang: 35266 byte

✅ Syntax OK — file siap dijalankan

Cara menjalankan:
  !python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"


In [ ]:
✅ Blok 5/5 ditulis (XXXX byte)
   File sekarang: ~31000 byte

✅ Syntax OK — file siap dijalankan

Cara menjalankan:
  !python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

SyntaxError: invalid character '✅' (U+2705) (3159945121.py, line 1)

In [ ]:
# ============================================================
# L2-F — Blok 5/5: Laporan Tahap 3 + run_log + verifikasi
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

BLOK5 = r'''
# ============================================================
# TULIS LAPORAN_Tahap3.md
# ============================================================
...

_IncompleteInputError: incomplete input (404053836.py, line 8)

In [ ]:
import os
p = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"
size = os.path.getsize(p)
print(f"Ukuran: {size} byte")
if size == 28836:
    print("→ Blok 5/5 BELUM ditulis")
elif size > 28836:
    print("→ Blok 5/5 (sebagian?) sudah ditulis")
else:
    print("→ Ukuran aneh, cek manual")

Ukuran: 35266 byte
→ Blok 5/5 (sebagian?) sudah ditulis


In [ ]:
# ============================================================
# L2-F2 — Diagnosa & rapikan file Tahap3_Horizon20.py
# ============================================================
import os, ast
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

print("=" * 100)
print("  DIAGNOSA FILE")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")
print(f"  Baris : {len(content.splitlines())}")
print()

# Cek syntax
try:
    ast.parse(content)
    print("  ✅ Syntax OK — file VALID")
    syntax_ok = True
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")
    print(f"     Line {e.lineno}: {e.text}")
    syntax_ok = False

# Cek penanda blok 5/5
markers = [
    "# TULIS LAPORAN_Tahap3.md",
    "ts = datetime.now().isoformat",
    "report = f\"\"\"TAHAP 3 - LAPORAN",
    "for sig_name in signals_b.keys():",
    "with open(REPORT_PATH, \"w\") as f:",
    "run_log.csv diperbarui",
    "Tahap3_Horizon20.py SELESAI",
]
print()
print("  Penanda blok 5/5:")
for m in markers:
    found = m in content
    print(f"    {'✅' if found else '❌'} {m}")

# Cek akhir file — tampilkan 20 baris terakhir
print()
print("  20 baris terakhir file:")
print("-" * 100)
lines = content.splitlines()
for i, line in enumerate(lines[-20:], start=len(lines)-19):
    print(f"  L{i:>4d}: {line}")
print("-" * 100)

print()
print("=" * 100)
print("  REKOMENDASI")
print("=" * 100)
if syntax_ok and "Tahap3_Horizon20.py SELESAI" in content:
    print("  ✅ File LENGKAP dan VALID. Siap dijalankan!")
    print(f'  !python "{SCRIPT_PATH}"')
elif syntax_ok and "with open(REPORT_PATH" in content:
    print("  ⚠️ File ada sebagian blok 5/5, tapi belum ada penutup (run_log + SELESAI).")
    print("     Perlu tambah ekor blok 5/5.")
elif syntax_ok:
    print("  ⚠️ File valid tapi belum ada blok 5/5.")
    print("     Perlu jalankan L2-F1 dan L2-F2.")
else:
    print("  ❌ File rusak (syntax error). Perlu perbaiki.")

  DIAGNOSA FILE
  Ukuran: 35266 byte
  Baris : 869

  ✅ Syntax OK — file VALID

  Penanda blok 5/5:
    ✅ # TULIS LAPORAN_Tahap3.md
    ✅ ts = datetime.now().isoformat
    ✅ report = f"""TAHAP 3 - LAPORAN
    ✅ for sig_name in signals_b.keys():
    ✅ with open(REPORT_PATH, "w") as f:
    ✅ run_log.csv diperbarui
    ✅ Tahap3_Horizon20.py SELESAI

  20 baris terakhir file:
----------------------------------------------------------------------------------------------------
  L 850: file_exists = os.path.exists(RUN_LOG_PATH)
  L 851: with open(RUN_LOG_PATH, "a", newline="") as f:
  L 852:     w = csv.DictWriter(f, fieldnames=_COLUMNS)
  L 853:     if not file_exists: w.writeheader()
  L 854:     w.writerow({
  L 855:         "timestamp": ts, "phase": "BACKTEST",
  L 856:         "config_version": "Gen3-Tahap3", "script_version": "Tahap3_Horizon20",
  L 857:         "global_seed": 42, "seal_sha256": seal_long[:16] + "...",
  L 858:         "params_sha256": "",
  L 859:         "n_stocks": 

In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3 — Tahap3_Horizon20.py

  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl
  Target vs hasil Langkah 0:
  Sinyal-N     Periode         hasil       target   cocok?
------------------------------------------------------------
  BRK      N=5  C1           +0.3254%     +0.3250%    COCOK
  BRK      N=5  C2           +0.1470%     +0.1470%    COCOK
  BRK      N=10 C1           +1.0361%     +1.0360%    COCOK
  BRK      N=10 C2           +0.0855%     +0.0850%    COCOK
  MOM      N=5  C1           -0.2925%     -0.2920%    COCOK
  MOM      N=5  C2           -0.2771%     -0.2770%    COCOK
  MOM      N=10 C1           +0.0799%     +0.0800%    COCOK
  MOM      N=10 C2           -0.3842%     -0.3840%    COCOK

  LANGKAH 0 KESELURUHAN: COCOK
  Langkah 0 LOLOS. Lanjut ke Bagian A.

  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
  Ticker: 561, unduh per-ticker start=2005-01-01 end=2026-06-06
  [25/561] OK=25 FAIL=0 (8s

In [ ]:
# ============================================================
# M1+M2 — Patch definisi saham layak (ADDENDUM 7)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- M1: Tambah konstanta setelah MIN_TURNOVER_MA20 ----------
OLD_CONST = '''MIN_TURNOVER_MA20 = 1_000_000_000.0
MIN_ELIG_PER_DAY  = 60'''
NEW_CONST = '''MIN_TURNOVER_MA20 = 1_000_000_000.0   # dipakai di Langkah 0 (reproduksi Tahap 2)
MIN_TURNOVER_T3   = 200_000_000.0      # ADDENDUM 7 — definisi layak Tahap 3
TOP_N_RANK_T3     = 100                # ADDENDUM 7 — top-100 turnover
MIN_ELIG_PER_DAY  = 60'''

if OLD_CONST in content:
    content = content.replace(OLD_CONST, NEW_CONST)
    print("✅ M1: Konstanta baru ditambahkan")
else:
    print("⚠️ M1: Konstanta lama tidak cocok persis")

# ---------- M2: Ganti definisi elig_l ----------
OLD_ELIG = '''turnover_l = cl_lc * vo_lc
toma20_l = turnover_l.rolling(20, min_periods=20).mean()
elig_l = (cl_lc >= MIN_PRICE) & (toma20_l >= MIN_TURNOVER_MA20)
n_elig_l = elig_l.sum(axis=1)'''

NEW_ELIG = '''turnover_l = cl_lc * vo_lc
toma20_l = turnover_l.rolling(20, min_periods=20).mean()

# ADDENDUM 7 — definisi saham layak Tahap 3:
#   (a) Close >= 200 & semua harga non-NaN
#   (b) rata-rata (Close x Volume) 20 hari >= Rp 200 juta
#   (c) peringkat turnover 20 hari di antara (a)&(b), termasuk 100 teratas
mask_a = (cl_lc >= MIN_PRICE) & cl_lc.notna()
mask_b = (toma20_l >= MIN_TURNOVER_T3)
toma20_ranked = toma20_l.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig_l = mask_a & mask_b & mask_c
n_elig_l = elig_l.sum(axis=1)'''

if OLD_ELIG in content:
    content = content.replace(OLD_ELIG, NEW_ELIG)
    print("✅ M2: Definisi elig_l diperbarui (definisi baru Tahap 3)")
else:
    print("⚠️ M2: Blok elig_l lama tidak cocok persis")
    # Cari alternatif
    import re
    if "elig_l = (cl_lc >= MIN_PRICE)" in content:
        print("     Ditemukan versi: elig_l = (cl_lc >= MIN_PRICE) & ...")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI PATCH")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("MIN_TURNOVER_T3", new.count("MIN_TURNOVER_T3")),
    ("TOP_N_RANK_T3",   new.count("TOP_N_RANK_T3")),
    ("mask_a",          new.count("mask_a")),
    ("mask_b",          new.count("mask_b")),
    ("mask_c",          new.count("mask_c")),
    ("toma20_ranked",   new.count("toma20_ranked")),
]
for label, cnt in checks:
    print(f"    {label:20s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ M1: Konstanta baru ditambahkan
✅ M2: Definisi elig_l diperbarui (definisi baru Tahap 3)

  VERIFIKASI PATCH
  Ukuran: 35863 byte
    MIN_TURNOVER_T3     : 2x
    TOP_N_RANK_T3       : 2x
    mask_a              : 3x
    mask_b              : 3x
    mask_c              : 2x
    toma20_ranked       : 2x

  ✅ Syntax OK


In [ ]:
# ============================================================
# M3 — Patch blok 5/5: tambahkan 4 bagian ke LAPORAN_Tahap3.md
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- M3a: Sisipkan perhitungan statistik tambahan SETELAH Gerbang A lolos ----------
# Kita butuh data elig_l, cl_lc, vo_lc yang sudah ada.
# Sisipkan setelah baris "log(f\"  SHA256: {long_manifest['sha256']}\")"

OLD_ANCHOR = '''log(f"\\n  data_long_sealed.pkl: {os.path.getsize(LONG_SEAL_PATH)} byte")
log(f"  SHA256: {long_manifest['sha256']}")'''

NEW_ANCHOR = '''log(f"\\n  data_long_sealed.pkl: {os.path.getsize(LONG_SEAL_PATH)} byte")
log(f"  SHA256: {long_manifest['sha256']}")

# ---------- Statistik tambahan (ADDENDUM 7) ----------
log("")
log("  Statistik tambahan (ADDENDUM 7):")

# Rata-rata saham layak per tahun kalender
elig_l_eff = elig_l.loc[mask_eff]
n_elig_eff_series = elig_l_eff.sum(axis=1)
n_elig_per_year = n_elig_eff_series.groupby(n_elig_eff_series.index.year).mean()
log("  Rata-rata saham layak per tahun:")
for yr, val in n_elig_per_year.items():
    log(f"    {yr}: {val:.1f}")

# Median (Close x Volume) saham layak per tahun
turnover_eff = (cl_lc * vo_lc).loc[mask_eff]
toma_median_per_year = {}
for yr in sorted(set(mask_eff[mask_eff].index.year)):
    year_mask = pd.Series(mask_eff.index.year == yr, index=mask_eff.index) & mask_eff
    sub = turnover_eff.loc[year_mask].stack().dropna()
    if len(sub):
        toma_median_per_year[yr] = float(sub.median())
    else:
        toma_median_per_year[yr] = float("nan")
log("  Median (Close x Volume) saham layak per tahun:")
for yr, val in toma_median_per_year.items():
    log(f"    {yr}: Rp {val/1e9:.2f} M" if pd.notna(val) else f"    {yr}: n/a")'''

if OLD_ANCHOR in content:
    content = content.replace(OLD_ANCHOR, NEW_ANCHOR, 1)
    print("✅ M3a: Statistik per tahun disisipkan")
else:
    print("⚠️ M3a: Anchor tidak cocok persis")

# ---------- M3b: Sisipkan statistik per blok SETELAH blok dibentuk ----------
OLD_BLK_ANCHOR = '''log(f"  K3: {tK3_start.date()} -> {idx_b[-1].date()} ({n_b - 2*blk} bar)")'''
NEW_BLK_ANCHOR = '''log(f"  K3: {tK3_start.date()} -> {idx_b[-1].date()} ({n_b - 2*blk} bar)")

# Statistik per blok (ADDENDUM 7)
def _stats_blok(blk_id, t1, t2):
    # elig_l di idx_b (data panjang sebelum warm-up)
    if blk_id == "K1": mask_b2 = idx_b <= tK1_end
    elif blk_id == "K2": mask_b2 = (idx_b >= tK2_start) & (idx_b <= tK2_end)
    else: mask_b2 = idx_b >= tK3_start
    elig_blk = elig_l.loc[idx_b[mask_b2]]
    n_avg = float(elig_blk.sum(axis=1).mean()) if len(elig_blk) else 0
    to_blk = (cl_lc * vo_lc).loc[idx_b[mask_b2]].stack().dropna()
    med_to = float(to_blk.median()) if len(to_blk) else float("nan")
    return n_avg, med_to

k_stats = {}
for blk_id, t1, t2 in [("K1", None, tK1_end), ("K2", tK2_start, tK2_end), ("K3", tK3_start, None)]:
    n_avg, med_to = _stats_blok(blk_id, t1, t2)
    k_stats[blk_id] = {"n_avg": n_avg, "median_turnover": med_to}
log("  Saham layak rata-rata per blok:")
for blk_id in ["K1","K2","K3"]:
    log(f"    {blk_id}: layak rata-rata {k_stats[blk_id]['n_avg']:.1f} ; median turnover Rp {k_stats[blk_id]['median_turnover']/1e9:.2f} M")'''

if OLD_BLK_ANCHOR in content:
    content = content.replace(OLD_BLK_ANCHOR, NEW_BLK_ANCHOR, 1)
    print("✅ M3b: Statistik per blok disisipkan")
else:
    print("⚠️ M3b: Anchor tidak cocok persis")

# ---------- M3c: Sisipkan hasil Langkah 0 + stat tambahan ke LAPORAN ----------
OLD_REPORT_HEAD = '''report = f"""TAHAP 3 - LAPORAN
Nama file skrip final : Tahap3_Horizon20.py
LANGKAH 0: COCOK (semua 8 target dalam toleransi 0,02 pp)'''

NEW_REPORT_HEAD = '''# Bangun tabel Langkah 0 untuk laporan
_l0_rows = []
for _k, _tgt in TARGET_L0.items():
    _r = results_L0.get(_k)
    if _r is None: continue
    _exc = _r["mean_exc_w"]
    _cocok = abs(_exc - _tgt) <= TOL_L0
    _l0_rows.append(f"  {_k[0]:8s} N={_k[1]:<2d} {_k[2]:8s} hasil={_exc*100:+.4f}% target={_tgt*100:+.4f}% {'COCOK' if _cocok else 'BEDA'}")
_l0_text = "\\n".join(_l0_rows) if _l0_rows else "(tidak ada)"

# Tabel statistik per tahun
_elig_l_eff = elig_l.loc[mask_eff]
_n_elig_eff_series = _elig_l_eff.sum(axis=1)
_n_elig_per_year = _n_elig_eff_series.groupby(_n_elig_eff_series.index.year).mean()
_year_lines = []
for _yr, _val in _n_elig_per_year.items():
    _year_lines.append(f"  {_yr}: rata-rata {_val:.1f} saham layak")
_year_text = "\\n".join(_year_lines) if _year_lines else "(tidak ada)"

# Tabel statistik per blok
_blk_lines = []
for _blk in ["K1","K2","K3"]:
    _s = k_stats[_blk]
    _blk_lines.append(f"  {_blk}: layak rata-rata {_s['n_avg']:.1f} ; median turnover Rp {_s['median_turnover']/1e9:.2f} M")
_blk_text = "\\n".join(_blk_lines) if _blk_lines else "(tidak ada)"

report = f"""TAHAP 3 - LAPORAN
Nama file skrip final : Tahap3_Horizon20.py
LANGKAH 0: COCOK (semua 8 target dalam toleransi 0,02 pp)
Detail Langkah 0:
{_l0_text}'''

if OLD_REPORT_HEAD in content:
    content = content.replace(OLD_REPORT_HEAD, NEW_REPORT_HEAD, 1)
    print("✅ M3c-1: Header laporan diperluas dengan Langkah 0 detail")
else:
    print("⚠️ M3c-1: Header laporan tidak cocok persis")

# ---------- M3d: Sisipkan statistik per tahun + per blok ke isi laporan ----------
OLD_REPORT_ANCHOR = '''Gerbang A: (1) {n_eff_bars} (2) {pct_cleaning:.4f}% (3) {n_elig_eff_mean:.1f}  {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}
SHA256 data_long_sealed.pkl : {seal_long}'''

NEW_REPORT_ANCHOR = '''Gerbang A: (1) {n_eff_bars} (2) {pct_cleaning:.4f}% (3) {n_elig_eff_mean:.1f}  {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}
SHA256 data_long_sealed.pkl : {seal_long}

Saham layak per tahun kalender:
{_year_text}

Saham layak per blok K1|K2|K3:
{_blk_text}'''

if OLD_REPORT_ANCHOR in content:
    content = content.replace(OLD_REPORT_ANCHOR, NEW_REPORT_ANCHOR, 1)
    print("✅ M3d: Statistik per tahun + per blok disisipkan ke laporan")
else:
    print("⚠️ M3d: Anchor tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("Statistik tambahan", new.count("Statistik tambahan")),
    ("_l0_rows",          new.count("_l0_rows")),
    ("_year_text",        new.count("_year_text")),
    ("_blk_text",         new.count("_blk_text")),
    ("k_stats",           new.count("k_stats")),
]
for label, cnt in checks:
    print(f"    {label:20s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ M3a: Statistik per tahun disisipkan
✅ M3b: Statistik per blok disisipkan
✅ M3c-1: Header laporan diperluas dengan Langkah 0 detail
⚠️ M3d: Anchor tidak cocok persis

  VERIFIKASI
  Ukuran: 39040 byte
    Statistik tambahan  : 2x
    _l0_rows            : 4x
    _year_text          : 1x
    _blk_text           : 1x
    k_stats             : 5x

  ✅ Syntax OK


In [ ]:
# ============================================================
# M3d — Perbaiki sisipan statistik ke dalam laporan
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Cari baris dalam report yang mengandung "SHA256 data_long_sealed.pkl : {seal_long}"
# Kita sisipkan stat setelah baris itu (di dalam f-string report)

import re
# Cari lokasi baris 'SHA256 data_long_sealed.pkl' di dalam report
pattern = r"(SHA256 data_long_sealed\.pkl : \{seal_long\}\n)"
match = re.search(pattern, content)
if match:
    # Sisipkan setelahnya
    insertion = """SHA256 data_long_sealed.pkl : {seal_long}

Saham layak per tahun kalender:
{_year_text}

Saham layak per blok K1|K2|K3:
{_blk_text}
"""
    # Ganti kemunculan PERTAMA
    content = content.replace(
        match.group(1),
        insertion,
        1
    )
    print("✅ M3d: Statistik disisipkan ke dalam report (setelah SHA256)")
else:
    print("⚠️ M3d: Pola 'SHA256 data_long_sealed.pkl' tidak ditemukan")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

# Cek keberadaan blok dalam report
checks = [
    ("'{_year_text}' dalam report",  new.count("{_year_text}")),
    ("'{_blk_text}' dalam report",   new.count("{_blk_text}")),
    ("_year_text didefinisikan",     new.count("_year_text =")),
    ("_blk_text didefinisikan",      new.count("_blk_text =")),
]
for label, cnt in checks:
    print(f"    {label:32s}: {cnt}x")

# Tampilkan 5 baris sekitar 'SHA256 data_long_sealed.pkl' dalam report
print()
print("  Konteks di sekitar sisipan:")
lines = new.splitlines()
for i, line in enumerate(lines):
    if "SHA256 data_long_sealed.pkl : {seal_long}" in line:
        for j in range(max(0, i-2), min(len(lines), i+10)):
            print(f"    L{j+1:>4d}: {lines[j]}")
        break

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ M3d: Statistik disisipkan ke dalam report (setelah SHA256)

  VERIFIKASI
  Ukuran: 39130 byte
    '{_year_text}' dalam report     : 1x
    '{_blk_text}' dalam report      : 1x
    _year_text didefinisikan        : 1x
    _blk_text didefinisikan         : 1x

  Konteks di sekitar sisipan:
    L 854:   A3 (layak/hari >= 60)     : {n_elig_eff_mean:.1f} -> {'LOLOS' if g3 else 'GAGAL'}
    L 855:   GERBANG A: {'LOLOS' if (g1 and g2 and g3) else 'GAGAL'}
    L 856: SHA256 data_long_sealed.pkl : {seal_long}
    L 857: 
    L 858: Saham layak per tahun kalender:
    L 859: {_year_text}
    L 860: 
    L 861: Saham layak per blok K1|K2|K3:
    L 862: {_blk_text}
    L 863: (data_sealed.pkl dan data_old_sealed.pkl tidak ditimpa;
    L 864:  data_long_sealed.pkl tidak disambung dengan keduanya)
    L 865: 

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3 — Tahap3_Horizon20.py

  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl
  Target vs hasil Langkah 0:
  Sinyal-N     Periode         hasil       target   cocok?
------------------------------------------------------------
  BRK      N=5  C1           +0.3254%     +0.3250%    COCOK
  BRK      N=5  C2           +0.1470%     +0.1470%    COCOK
  BRK      N=10 C1           +1.0361%     +1.0360%    COCOK
  BRK      N=10 C2           +0.0855%     +0.0850%    COCOK
  MOM      N=5  C1           -0.2925%     -0.2920%    COCOK
  MOM      N=5  C2           -0.2771%     -0.2770%    COCOK
  MOM      N=10 C1           +0.0799%     +0.0800%    COCOK
  MOM      N=10 C2           -0.3842%     -0.3840%    COCOK

  LANGKAH 0 KESELURUHAN: COCOK
  Langkah 0 LOLOS. Lanjut ke Bagian A.

  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
  Ticker: 561, unduh per-ticker start=2005-01-01 end=2026-06-06
  Resume: 561 ticker sudah 

In [ ]:
# ============================================================
# M5 — Perbaiki bug baris 466 (statistik per tahun turnover)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- Fix 1: loop per tahun median turnover ----------
OLD = '''for yr in sorted(set(mask_eff[mask_eff].index.year)):
    year_mask = pd.Series(mask_eff.index.year == yr, index=mask_eff.index) & mask_eff
    sub = turnover_eff.loc[year_mask].stack().dropna()'''

NEW = '''for yr in sorted(set(cl_lc.index[mask_eff].year)):
    year_mask = pd.Series(cl_lc.index.year == yr, index=cl_lc.index) & mask_eff
    sub = turnover_eff.loc[year_mask].stack().dropna()'''

if OLD in content:
    content = content.replace(OLD, NEW)
    print("✅ Fix 1: loop per tahun diperbaiki")
else:
    print("⚠️ Fix 1: baris lama tidak cocok persis")
    # Cari varian
    import re
    if "mask_eff[mask_eff].index.year" in content:
        print("     Ditemukan varian 'mask_eff[mask_eff].index.year' — coba fix lain")
        content = content.replace(
            "set(mask_eff[mask_eff].index.year)",
            "set(cl_lc.index[mask_eff].year)"
        )
        print("     ✅ Fix 1-alt diterapkan")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

# Cek pola lama hilang
old_pattern_count = new.count("mask_eff[mask_eff]")
new_pattern_count = new.count("cl_lc.index[mask_eff]")
print(f"  'mask_eff[mask_eff]' muncul: {old_pattern_count}x (harus 0)")
print(f"  'cl_lc.index[mask_eff]' muncul: {new_pattern_count}x (harus > 0)")

# Tampilkan konteks baru
print()
print("  Konteks di sekitar perbaikan:")
lines = new.splitlines()
for i, line in enumerate(lines):
    if "for yr in sorted(set(" in line and "year" in line:
        for j in range(max(0, i-2), min(len(lines), i+5)):
            print(f"    L{j+1:>4d}: {lines[j]}")
        break

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Fix 1: loop per tahun diperbaiki

  VERIFIKASI
  Ukuran: 39121 byte
  'mask_eff[mask_eff]' muncul: 0x (harus 0)
  'cl_lc.index[mask_eff]' muncul: 1x (harus > 0)

  Konteks di sekitar perbaikan:
    L 464: turnover_eff = (cl_lc * vo_lc).loc[mask_eff]
    L 465: toma_median_per_year = {}
    L 466: for yr in sorted(set(cl_lc.index[mask_eff].year)):
    L 467:     year_mask = pd.Series(cl_lc.index.year == yr, index=cl_lc.index) & mask_eff
    L 468:     sub = turnover_eff.loc[year_mask].stack().dropna()
    L 469:     if len(sub):
    L 470:         toma_median_per_year[yr] = float(sub.median())

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3 — Tahap3_Horizon20.py

  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl
  Target vs hasil Langkah 0:
  Sinyal-N     Periode         hasil       target   cocok?
------------------------------------------------------------
  BRK      N=5  C1           +0.3254%     +0.3250%    COCOK
  BRK      N=5  C2           +0.1470%     +0.1470%    COCOK
  BRK      N=10 C1           +1.0361%     +1.0360%    COCOK
  BRK      N=10 C2           +0.0855%     +0.0850%    COCOK
  MOM      N=5  C1           -0.2925%     -0.2920%    COCOK
  MOM      N=5  C2           -0.2771%     -0.2770%    COCOK
  MOM      N=10 C1           +0.0799%     +0.0800%    COCOK
  MOM      N=10 C2           -0.3842%     -0.3840%    COCOK

  LANGKAH 0 KESELURUHAN: COCOK
  Langkah 0 LOLOS. Lanjut ke Bagian A.

  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
  Ticker: 561, unduh per-ticker start=2005-01-01 end=2026-06-06
  Resume: 561 ticker sudah 

In [ ]:
# ============================================================
# M7 — Diagnosa plasebo ALARM (-0.8552%)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # force remount karena kernel baru
import os
import numpy as np
import pandas as pd
import random

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

# Load data_long_sealed.pkl
long_data = pd.read_pickle(f"{BASE}/data_long_sealed.pkl")
op_l = long_data["price_open"]; hi_l = long_data["price_high"]
lo_l = long_data["price_low"];  cl_l = long_data["price_close"]
vo_l = long_data["price_volume"]

print("=" * 100)
print("  DIAGNOSA PLASEBO TAHAP 3")
print("=" * 100)
print(f"  Data: {cl_l.shape}, {cl_l.index.min().date()} → {cl_l.index.max().date()}")
print()

# Konstanta
MIN_PRICE = 200.0
MIN_TURNOVER_T3 = 200_000_000.0
TOP_N_RANK_T3 = 100
COST_BASE_SIDE = 0.0015
N_HORIZON = 20

# Bangun eligible (definisi baru)
turnover = cl_l * vo_l
toma20 = turnover.rolling(20, min_periods=20).mean()
mask_a = (cl_l >= MIN_PRICE) & cl_l.notna()
mask_b = (toma20 >= MIN_TURNOVER_T3)
toma20_ranked = toma20.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig = mask_a & mask_b & mask_c

print(f"  Eligible: rata-rata {elig.sum(axis=1).mean():.1f} saham/hari")
print(f"  Eligible: min {elig.sum(axis=1).min()}, max {elig.sum(axis=1).max()}")
print()

# Bangun idx dengan warm-up 260
WARMUP = 260
idx = cl_l.index[WARMUP:]
op_b = op_l.loc[idx]; hi_b = hi_l.loc[idx]; lo_b = lo_l.loc[idx]
cl_b = cl_l.loc[idx]; vo_b = vo_l.loc[idx]
elig_b = elig.loc[idx]

print(f"  Setelah warm-up 260: {len(idx)} bar")
print(f"  Range: {idx.min().date()} → {idx.max().date()}")
print()

# Fungsi trade & benchmark
def trade_return(tkr, t, N, cost_side):
    try: i = idx.get_loc(t)
    except KeyError: return None
    if i + N >= len(idx): return None
    t_ent = idx[i+1]; t_exit = idx[i+N]
    o = op_b.at[t_ent, tkr]; c = cl_b.at[t_exit, tkr]
    if pd.isna(o) or pd.isna(c): return None
    entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
    return (exit_ - entry) / entry

def bm_avg(t, N, cost_side):
    try: i = idx.get_loc(t)
    except KeyError: return None
    if i + N >= len(idx): return None
    t_ent = idx[i+1]; t_exit = idx[i+N]
    elig_tkrs = elig_b.columns[elig_b.loc[t]]
    rets = []
    for tkr in elig_tkrs:
        o = op_b.at[t_ent, tkr]; c = cl_b.at[t_exit, tkr]
        if pd.isna(o) or pd.isna(c): continue
        entry = o * (1 + cost_side); exit_ = c * (1 - cost_side)
        rets.append((exit_ - entry) / entry)
    return float(np.mean(rets)) if rets else None

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

# Plasebo diagnostik: 5 seed pertama
print("=" * 100)
print("  PLASEBO 5 SEED PERTAMA")
print("=" * 100)
print(f"  {'seed':>5s} {'n_trades':>10s} {'mean_exc_raw':>15s} {'mean_exc_w':>15s} {'mean_bm':>10s} {'mean_raw':>10s}")
print("-" * 100)

for seed in range(1, 6):
    rng = random.Random(seed)
    sig_random = pd.DataFrame(False, index=elig_b.index, columns=elig_b.columns)
    for t in elig_b.index:
        elig_tkrs = elig_b.columns[elig_b.loc[t]]
        for tkr in elig_tkrs:
            if rng.random() < 0.05:
                sig_random.at[t, tkr] = True
    # Kumpulkan trade
    excs, bms, raws = [], [], []
    for t in sig_random.index:
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, N_HORIZON, COST_BASE_SIDE)
            if rb is None: continue
            bm = bm_avg(t, N_HORIZON, COST_BASE_SIDE)
            if bm is None: continue
            excs.append(rb - bm); bms.append(bm); raws.append(rb)
    if excs:
        mexc_raw = float(np.mean(excs))
        mexc_w = float(np.mean(winsorize(np.array(excs))))
        mbm = float(np.mean(bms))
        mraw = float(np.mean(raws))
        print(f"  {seed:>5d} {len(excs):>10d} {mexc_raw*100:>14.4f}% {mexc_w*100:>14.4f}% {mbm*100:>9.4f}% {mraw*100:>9.4f}%")

print()
print("=" * 100)
print("  BANDINGKAN DENGAN SINYAL ASLI")
print("=" * 100)

# Ambil salah satu sinyal asli (MOM60)
roc60 = cl_l / cl_l.shift(60) - 1.0
roc60_rank = roc60.where(elig).rank(axis=1, pct=True)
mom60 = (roc60_rank >= 0.80) & roc60.notna() & elig
mom60_b = mom60.loc[idx]

excs_m, bms_m, raws_m = [], [], []
for t in mom60_b.index:
    for tkr in mom60_b.columns:
        if not bool(mom60_b.at[t, tkr]): continue
        rb = trade_return(tkr, t, N_HORIZON, COST_BASE_SIDE)
        if rb is None: continue
        bm = bm_avg(t, N_HORIZON, COST_BASE_SIDE)
        if bm is None: continue
        excs_m.append(rb - bm); bms_m.append(bm); raws_m.append(rb)

print(f"  MOM60 asli (seluruh data):")
print(f"    n_trades     : {len(excs_m)}")
print(f"    mean_exc_raw : {float(np.mean(excs_m))*100:+.4f}%")
print(f"    mean_exc_w   : {float(np.mean(winsorize(np.array(excs_m))))*100:+.4f}%")
print(f"    mean_bm      : {float(np.mean(bms_m))*100:+.4f}%")
print(f"    mean_raw     : {float(np.mean(raws_m))*100:+.4f}%")
print()

print("=" * 100)
print("  KESIMPULAN DIAGNOSTIK")
print("=" * 100)
print("  Jika plasebo 'mean_bm' (benchmark) berbeda jauh dari sinyal asli 'mean_bm',")
print("  maka plasebo cenderung memilih tanggal yang benchmark-nya tidak representatif.")
print("  Jika 'mean_bm' mirip tapi 'mean_raw' jauh berbeda, maka plasebo trade")
print("  sendiri yang menghasilkan return berbeda.")
print("=" * 100)

Mounted at /content/drive
  DIAGNOSA PLASEBO TAHAP 3
  Data: (1385, 561), 2019-10-01 → 2026-06-05

  Eligible: rata-rata 94.8 saham/hari
  Eligible: min 0, max 100

  Setelah warm-up 260: 1125 bar
  Range: 2021-09-22 → 2026-06-05

  PLASEBO 5 SEED PERTAMA
   seed   n_trades    mean_exc_raw      mean_exc_w    mean_bm   mean_raw
----------------------------------------------------------------------------------------------------
      1       5464        -0.0421%        -1.0342%   -0.5719%   -0.6141%
      2       5497         0.2573%        -0.6892%   -0.5174%   -0.2601%
      3       5517        -0.1986%        -1.1361%   -0.5401%   -0.7387%
      4       5534         0.2474%        -0.7496%   -0.4750%   -0.2276%
      5       5478         0.2829%        -0.8257%   -0.5173%   -0.2344%

  BANDINGKAN DENGAN SINYAL ASLI
  MOM60 asli (seluruh data):
    n_trades     : 21836
    mean_exc_raw : +1.5407%
    mean_exc_w   : +0.1243%
    mean_bm      : -0.4492%
    mean_raw     : +1.0915%

  KES

In [ ]:
# ============================================================
# N1 — Patch fungsi winsorize & pipeline (ADDENDUM 8)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- N1a: Sisipkan fungsi helper baru setelah winsorize lama ----------
OLD_WS = '''def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)'''

NEW_WS = '''def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def winsorize_return(trade_returns, all_returns):
    """ADDENDUM 8: winsorize trade return & semua return saham dengan batas
    yang sama, dihitung dari pool semua return saham layak.
    trade_returns: list return trade sinyal
    all_returns: list return semua saham layak pada tanggal-tanggal sinyal
    Return: (trade_returns_w, all_returns_w)
    """
    if len(all_returns) == 0:
        return np.array([]), np.array([])
    lo, hi = np.percentile(all_returns, 5), np.percentile(all_returns, 95)
    tr_w = np.clip(trade_returns, lo, hi) if len(trade_returns) else np.array([])
    ar_w = np.clip(all_returns, lo, hi)
    return tr_w, ar_w'''

if OLD_WS in content:
    content = content.replace(OLD_WS, NEW_WS, 1)
    print("✅ N1a: Fungsi winsorize_return ditambahkan")
else:
    print("⚠️ N1a: Fungsi winsorize lama tidak cocok persis")

# ---------- N1b: Ganti fungsi run_signal untuk hitung excess_w baru + excess_w_lama ----------
OLD_RUN_SIGNAL_HEAD = '''def run_signal(sig_name, sig_df, N=20):
    results = {}
    daily_pool = []
    all_exc = {"K1": [], "K2": [], "K3": []}
    n_trades = {"K1": 0, "K2": 0, "K3": 0}
    raw_b_l = {"K1": [], "K2": [], "K3": []}
    raw_s_l = {"K1": [], "K2": [], "K3": []}
    for blk_id in ["K1", "K2", "K3"]:
        sig_list = []
        for t in sig_df.index:
            if not in_blk(t, blk_id): continue
            for tkr in sig_df.columns:
                if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
        exc_raw, dates = [], []
        for tkr, t in sig_list:
            rb = trade_return(tkr, t, N, COST_BASE_SIDE)
            rs = trade_return(tkr, t, N, COST_STRESS_SIDE)
            if rb is None or rs is None: continue
            bm = bm_avg(t, N, COST_BASE_SIDE)
            if bm is None: continue
            raw_b_l[blk_id].append(rb); raw_s_l[blk_id].append(rs)
            exc_raw.append(rb - bm); dates.append(t)
        n_trades[blk_id] = len(raw_b_l[blk_id])
        exc_w = winsorize(np.array(exc_raw))'''

NEW_RUN_SIGNAL_HEAD = '''def run_signal(sig_name, sig_df, N=20):
    """ADDENDUM 8: hits excess_w (baru) dan excess_w_lama (pembanding)."""
    results = {}
    daily_pool = []
    all_exc = {"K1": [], "K2": [], "K3": []}
    n_trades = {"K1": 0, "K2": 0, "K3": 0}
    raw_b_l = {"K1": [], "K2": [], "K3": []}
    raw_s_l = {"K1": [], "K2": [], "K3": []}
    for blk_id in ["K1", "K2", "K3"]:
        sig_list = []
        for t in sig_df.index:
            if not in_blk(t, blk_id): continue
            for tkr in sig_df.columns:
                if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
        exc_raw, dates = [], []
        # Kumpulkan: rb, rs, bm, dan SEMUA return saham layak per tanggal
        rb_l = []; rs_l = []; bm_l = []
        all_rets_pool = []   # untuk winsorize batas (ADDENDUM 8)
        for tkr, t in sig_list:
            rb = trade_return(tkr, t, N, COST_BASE_SIDE)
            rs = trade_return(tkr, t, N, COST_STRESS_SIDE)
            if rb is None or rs is None: continue
            # Kumpulkan semua return saham layak di T (untuk batas winsorize)
            try:
                i = idx_b2.get_loc(t)
            except KeyError:
                continue
            if i + N >= len(idx_b2): continue
            t_ent = idx_b2[i+1]; t_exit = idx_b2[i+N]
            elig_tkrs_t = elig_b.columns[elig_b.loc[t]]
            rets_T = []
            for tk2 in elig_tkrs_t:
                o2 = op_b.at[t_ent, tk2]; c2 = cl_b.at[t_exit, tk2]
                if pd.isna(o2) or pd.isna(c2): continue
                rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
            if not rets_T: continue
            bm = float(np.mean(rets_T))
            all_rets_pool.extend(rets_T)
            rb_l.append(rb); rs_l.append(rs); bm_l.append(bm)
            exc_raw.append(rb - bm); dates.append(t)
        n_trades[blk_id] = len(rb_l)
        raw_b_l[blk_id] = rb_l
        raw_s_l[blk_id] = rs_l

        # --- ADDENDUM 8: excess_w baru ---
        if all_rets_pool and rb_l:
            lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
            rb_w = np.clip(rb_l, lo, hi)
            bm_w_list = []
            # Hitung benchmark_w per tanggal dari return ter-winsorize
            # (recompute per tanggal dengan batas yang sama)
            daily_dict = {}
            for tkr, t in sig_list:
                pass   # tidak perlu; sudah dikumpulkan
            # bangun benchmark_w per tanggal
            t_to_bm_w = {}
            t_to_rb_w = {}
            for k, t in enumerate(dates):
                t_to_rb_w.setdefault(t, []).append(rb_w[k])
            # benchmark_w dari all_rets_pool ter-winsorize: pakai batas global
            # (pendekatan: benchmark_w = rata-rata return ter-winsorize di T)
            # Sederhanakan: benchmark_w(T) = bm(T) di-clip dengan batas global
            bm_w_arr = np.clip(bm_l, lo, hi)
            exc_w = np.array(rb_w) - np.array(bm_w_arr)
        else:
            exc_w = np.array([])

        # --- excess_w_lama (pembanding) ---
        exc_w_lama = winsorize(np.array(exc_raw))'''

if OLD_RUN_SIGNAL_HEAD in content:
    content = content.replace(OLD_RUN_SIGNAL_HEAD, NEW_RUN_SIGNAL_HEAD, 1)
    print("✅ N1b: run_signal diperbarui (excess_w baru + lama)")
else:
    print("⚠️ N1b: Head run_signal lama tidak cocok persis")

# ---------- N1c: Update bagian hasil di run_signal ----------
OLD_RES = '''        exc_w_arr = np.array(exc_w)
        raw_b_arr = np.array(raw_b_l[blk_id])
        raw_s_arr = np.array(raw_s_l[blk_id])
        results[(sig_name, N, blk_id)] = {
            "n_trades": n_trades[blk_id],
            "mean_base_raw": float(np.mean(raw_b_arr)) if len(raw_b_arr) else float("nan"),
            "mean_base_w":   float(np.mean(winsorize(raw_b_arr))) if len(raw_b_arr) else float("nan"),
            "mean_stress_raw": float(np.mean(raw_s_arr)) if len(raw_s_arr) else float("nan"),
            "mean_stress_w":   float(np.mean(winsorize(raw_s_arr))) if len(raw_s_arr) else float("nan"),
            "mean_exc_raw":  float(np.mean(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "mean_exc_w":    float(np.mean(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "median_exc":    float(np.median(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "t_stat": t_stat, "p_val": p_val,
        }'''

NEW_RES = '''        exc_w_arr = np.array(exc_w)
        exc_w_lama_arr = np.array(exc_w_lama)
        raw_b_arr = np.array(raw_b_l[blk_id])
        raw_s_arr = np.array(raw_s_l[blk_id])
        # Deret harian dari excess_w (baru)
        daily_w = {}
        for t, e in zip(dates, exc_w_arr): daily_w.setdefault(t, []).append(e)
        ds_w = np.array([np.mean(daily_w[t]) for t in sorted(daily_w.keys())]) if daily_w else np.array([])
        if len(ds_w) >= 10:
            model = OLS(ds_w, np.ones(len(ds_w))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            t_stat_w = float(model.tvalues[0]); p_val_w = float(model.pvalues[0])
        else:
            t_stat_w = float("nan"); p_val_w = float("nan")
        results[(sig_name, N, blk_id)] = {
            "n_trades": n_trades[blk_id],
            "mean_base_raw": float(np.mean(raw_b_arr)) if len(raw_b_arr) else float("nan"),
            "mean_base_w":   float(np.mean(winsorize(raw_b_arr))) if len(raw_b_arr) else float("nan"),
            "mean_stress_raw": float(np.mean(raw_s_arr)) if len(raw_s_arr) else float("nan"),
            "mean_stress_w":   float(np.mean(winsorize(raw_s_arr))) if len(raw_s_arr) else float("nan"),
            "mean_exc_raw":  float(np.mean(exc_w_lama_arr)) if len(exc_w_lama_arr) else float("nan"),
            "mean_exc_w":    float(np.mean(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "mean_exc_w_lama": float(np.mean(exc_w_lama_arr)) if len(exc_w_lama_arr) else float("nan"),
            "median_exc":    float(np.median(exc_w_arr)) if len(exc_w_arr) else float("nan"),
            "t_stat": t_stat_w, "p_val": p_val_w,
        }'''

if OLD_RES in content:
    content = content.replace(OLD_RES, NEW_RES, 1)
    print("✅ N1c: Bagian hasil run_signal diperbarui")
else:
    print("⚠️ N1c: Bagian hasil lama tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI PATCH N1")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("winsorize_return",  new.count("winsorize_return")),
    ("all_rets_pool",     new.count("all_rets_pool")),
    ("excess_w_lama",     new.count("excess_w_lama")),
    ("mean_exc_w_lama",   new.count("mean_exc_w_lama")),
]
for label, cnt in checks:
    print(f"    {label:20s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Mounted at /content/drive
✅ N1a: Fungsi winsorize_return ditambahkan
✅ N1b: run_signal diperbarui (excess_w baru + lama)
✅ N1c: Bagian hasil run_signal diperbarui

  VERIFIKASI PATCH N1
  Ukuran: 42557 byte
    winsorize_return    : 1x
    all_rets_pool       : 6x
    excess_w_lama       : 2x
    mean_exc_w_lama     : 1x

  ✅ Syntax OK


In [ ]:
# ============================================================
# N2 — Patch plasebo (winsorize_return sesuai ADDENDUM 8)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- Cari & ganti blok plasebo lama ----------
OLD_PLASEBO = '''plasebo_exc = []
for seed in range(1, 21):
    rng_p = random.Random(seed)
    sig_random = pd.DataFrame(False, index=elig_b.index, columns=elig_b.columns)
    for t in elig_b.index:
        eligible_tkrs = elig_b.columns[elig_b.loc[t]]
        for tkr in eligible_tkrs:
            if rng_p.random() < 0.05:
                sig_random.at[t, tkr] = True
    excs = []
    for t in sig_random.index:
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, 20, COST_BASE_SIDE)
            if rb is None: continue
            bm = bm_avg(t, 20, COST_BASE_SIDE)
            if bm is None: continue
            excs.append(rb - bm)
    if excs:
        plasebo_exc.append(float(np.mean(winsorize(np.array(excs)))))
mean_plasebo = float(np.mean(plasebo_exc)) if plasebo_exc else float("nan")
plasebo_ok = abs(mean_plasebo) <= 0.0010
log(f"    mean plasebo (pool): {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}")'''

NEW_PLASEBO = '''plasebo_exc = []       # versi baru (ADDENDUM 8)
plasebo_exc_lama = []  # versi lama (pembanding)
for seed in range(1, 21):
    rng_p = random.Random(seed)
    sig_random = pd.DataFrame(False, index=elig_b.index, columns=elig_b.columns)
    for t in elig_b.index:
        eligible_tkrs = elig_b.columns[elig_b.loc[t]]
        for tkr in eligible_tkrs:
            if rng_p.random() < 0.05:
                sig_random.at[t, tkr] = True

    # Kumpulkan semua trade & semua return saham layak
    rb_l = []; bm_l = []; all_rets_pool = []
    for t in sig_random.index:
        # Semua return saham layak di T
        try: i = idx_b2.get_loc(t)
        except KeyError: continue
        if i + 20 >= len(idx_b2): continue
        t_ent = idx_b2[i+1]; t_exit = idx_b2[i+20]
        elig_tkrs_t = elig_b.columns[elig_b.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs_t:
            o2 = op_b.at[t_ent, tk2]; c2 = cl_b.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
        if not rets_T: continue
        all_rets_pool.extend(rets_T)
        bm = float(np.mean(rets_T))
        # Trade acak
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, 20, COST_BASE_SIDE)
            if rb is None: continue
            rb_l.append(rb); bm_l.append(bm)

    # Versi BARU (ADDENDUM 8)
    if all_rets_pool and rb_l:
        lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
        rb_w = np.clip(rb_l, lo, hi)
        bm_w = np.clip(bm_l, lo, hi)
        exc_w = np.array(rb_w) - np.array(bm_w)
        plasebo_exc.append(float(np.mean(exc_w)))

    # Versi LAMA (pembanding)
    if rb_l:
        exc_lama = np.array(rb_l) - np.array(bm_l)
        plasebo_exc_lama.append(float(np.mean(winsorize(exc_lama))))

mean_plasebo = float(np.mean(plasebo_exc)) if plasebo_exc else float("nan")
mean_plasebo_lama = float(np.mean(plasebo_exc_lama)) if plasebo_exc_lama else float("nan")
plasebo_ok = abs(mean_plasebo) <= 0.0010
log(f"    mean plasebo (BARU)  : {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}")
log(f"    mean plasebo (LAMA)  : {mean_plasebo_lama*100:+.4f}% (pembanding)")'''

if OLD_PLASEBO in content:
    content = content.replace(OLD_PLASEBO, NEW_PLASEBO, 1)
    print("✅ N2: Blok plasebo diperbarui (baru + lama)")
else:
    print("⚠️ N2: Blok plasebo lama tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI PATCH N2")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("plasebo_exc_lama",     new.count("plasebo_exc_lama")),
    ("mean_plasebo_lama",    new.count("mean_plasebo_lama")),
    ("BARU",                 new.count("BARU")),
    ("LAMA",                 new.count("LAMA")),
]
for label, cnt in checks:
    print(f"    {label:20s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Mounted at /content/drive
✅ N2: Blok plasebo diperbarui (baru + lama)

  VERIFIKASI PATCH N2
  Ukuran: 43874 byte
    plasebo_exc_lama    : 4x
    mean_plasebo_lama   : 2x
    BARU                : 2x
    LAMA                : 2x

  ✅ Syntax OK


In [ ]:
# ============================================================
# N3 — Patch laporan: excess_w baru + lama + plasebo baru + lama
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- N3a: Update baris plasebo di laporan ----------
OLD_PLASEBO_REPORT = '''PLASEBO: excess pool rata-rata (20 seed) : {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}'''

NEW_PLASEBO_REPORT = '''PLASEBO: excess pool rata-rata (20 seed) : {mean_plasebo*100:+.4f}% -> {'LULUS' if plasebo_ok else 'ALARM'}
  (excess_w_lama pembanding: {mean_plasebo_lama*100:+.4f}%)'''

if OLD_PLASEBO_REPORT in content:
    content = content.replace(OLD_PLASEBO_REPORT, NEW_PLASEBO_REPORT, 1)
    print("✅ N3a: Baris plasebo di laporan diperbarui")
else:
    print("⚠️ N3a: Baris plasebo laporan tidak cocok persis")

# ---------- N3b: Update baris excess di tiap sinyal ----------
OLD_EXC_LINE = '''    report += f"  Excess winsorize K1|K2|K3|pool: "
    report += f"{r_k1['mean_exc_w']*100:+.3f}% | {r_k2['mean_exc_w']*100:+.3f}% | "
    report += f"{r_k3['mean_exc_w']*100:+.3f}% | {r_pool['mean_exc_w']*100:+.3f}%\\n"'''

NEW_EXC_LINE = '''    report += f"  Excess_w BARU K1|K2|K3|pool: "
    report += f"{r_k1['mean_exc_w']*100:+.3f}% | {r_k2['mean_exc_w']*100:+.3f}% | "
    report += f"{r_k3['mean_exc_w']*100:+.3f}% | {r_pool['mean_exc_w']*100:+.3f}%\\n"
    report += f"  Excess_w LAMA K1|K2|K3: "
    report += f"{r_k1.get('mean_exc_w_lama', float('nan'))*100:+.3f}% | "
    report += f"{r_k2.get('mean_exc_w_lama', float('nan'))*100:+.3f}% | "
    report += f"{r_k3.get('mean_exc_w_lama', float('nan'))*100:+.3f}% (pembanding)\\n"'''

if OLD_EXC_LINE in content:
    content = content.replace(OLD_EXC_LINE, NEW_EXC_LINE, 1)
    print("✅ N3b: Baris excess di laporan diperbarui (baru + lama)")
else:
    print("⚠️ N3b: Baris excess lama tidak cocok persis")

# ---------- N3c: Update bagian pool di run_signal untuk simpan mean_exc_w_lama ----------
# Cari blok: results[(sig_name, N, "pool")] = {...}
OLD_POOL = '''    if daily_pool:
        daily_pool_d = {}
        for t, e in daily_pool: daily_pool_d.setdefault(t, []).append(e)
        ds = np.array([np.mean(daily_pool_d[t]) for t in sorted(daily_pool_d.keys())])
        if len(ds) >= 10:
            model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            t_stat = float(model.tvalues[0]); p_val = float(model.pvalues[0])
        else:
            t_stat = float("nan"); p_val = float("nan")
        results[(sig_name, N, "pool")] = {
            "mean_exc_w": float(np.mean(ds)), "t_stat": t_stat, "p_val": p_val,
        }'''

NEW_POOL = '''    # Pool: gabung excess_w dari 3 blok
    if all_exc["K1"] or all_exc["K2"] or all_exc["K3"]:
        pool_arr = np.array(all_exc["K1"] + all_exc["K2"] + all_exc["K3"])
        ds = pool_arr  # deret harian sudah flat — sederhanakan
        if len(ds) >= 10:
            model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            t_stat = float(model.tvalues[0]); p_val = float(model.pvalues[0])
        else:
            t_stat = float("nan"); p_val = float("nan")
        results[(sig_name, N, "pool")] = {
            "mean_exc_w": float(np.mean(pool_arr)),
            "t_stat": t_stat, "p_val": p_val,
        }'''

if OLD_POOL in content:
    content = content.replace(OLD_POOL, NEW_POOL, 1)
    print("✅ N3c: Bagian pool diperbarui")
else:
    print("⚠️ N3c: Bagian pool tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI PATCH N3")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("Excess_w BARU",  new.count("Excess_w BARU")),
    ("Excess_w LAMA",  new.count("Excess_w LAMA")),
    ("mean_plasebo_lama", new.count("mean_plasebo_lama")),
]
for label, cnt in checks:
    print(f"    {label:24s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
print("=" * 100)

Mounted at /content/drive
✅ N3a: Baris plasebo di laporan diperbarui
✅ N3b: Baris excess di laporan diperbarui (baru + lama)
✅ N3c: Bagian pool diperbarui

  VERIFIKASI PATCH N3
  Ukuran: 44259 byte
    Excess_w BARU           : 1x
    Excess_w LAMA           : 1x
    mean_plasebo_lama       : 3x

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3 — Tahap3_Horizon20.py

  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl
  Target vs hasil Langkah 0:
  Sinyal-N     Periode         hasil       target   cocok?
------------------------------------------------------------
  BRK      N=5  C1           +0.3254%     +0.3250%    COCOK
  BRK      N=5  C2           +0.1470%     +0.1470%    COCOK
  BRK      N=10 C1           +1.0361%     +1.0360%    COCOK
  BRK      N=10 C2           +0.0855%     +0.0850%    COCOK
  MOM      N=5  C1           -0.2925%     -0.2920%    COCOK
  MOM      N=5  C2           -0.2771%     -0.2770%    COCOK
  MOM      N=10 C1           +0.0799%     +0.0800%    COCOK
  MOM      N=10 C2           -0.3842%     -0.3840%    COCOK

  LANGKAH 0 KESELURUHAN: COCOK
  Langkah 0 LOLOS. Lanjut ke Bagian A.

  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
  Ticker: 561, unduh per-ticker start=2005-01-01 end=2026-06-06
  Resume: 561 ticker sudah 

In [ ]:
# ============================================================
# N5 — Perbaiki implementasi winsorize_return (ADDENDUM 8 revisi)
# Benchmark harus dihitung dari return yang sudah di-clip, bukan clip rata-rata
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3_Horizon20.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# ---------- N5a: Perbaiki blok di run_signal ----------
# Baris lama:
#   bm_w_arr = np.clip(bm_l, lo, hi)
#   exc_w = np.array(rb_w) - np.array(bm_w_arr)
# Baru: hitung bm_w dari semua return yang di-clip PER TANGGAL

OLD_SIGNAL_CALC = '''            bm = float(np.mean(rets_T))
            all_rets_pool.extend(rets_T)
            rb_l.append(rb); rs_l.append(rs); bm_l.append(bm)
            exc_raw.append(rb - bm); dates.append(t)'''

NEW_SIGNAL_CALC = '''            bm = float(np.mean(rets_T))
            all_rets_pool.extend(rets_T)
            rb_l.append(rb); rs_l.append(rs); bm_l.append(bm)
            exc_raw.append(rb - bm); dates.append(t)
            # Simpan rets_T untuk hitung bm_w setelah batas ditentukan
            if 'rets_T_list' not in dir(): rets_T_list = []
            rets_T_list.append((t, rets_T))'''

if OLD_SIGNAL_CALC in content:
    content = content.replace(OLD_SIGNAL_CALC, NEW_SIGNAL_CALC, 1)
    print("✅ N5a-1: Tambah simpan rets_T per tanggal")
else:
    print("⚠️ N5a-1: Blok lama tidak cocok persis")

# ---------- N5a-2: Deklarasi rets_T_list di awal loop ----------
OLD_INIT = '''        exc_raw, dates = [], []
        # Kumpulkan: rb, rs, bm, dan SEMUA return saham layak per tanggal
        rb_l = []; rs_l = []; bm_l = []
        all_rets_pool = []   # untuk winsorize batas (ADDENDUM 8)'''

NEW_INIT = '''        exc_raw, dates = [], []
        # Kumpulkan: rb, rs, bm, dan SEMUA return saham layak per tanggal
        rb_l = []; rs_l = []; bm_l = []
        all_rets_pool = []   # untuk winsorize batas (ADDENDUM 8)
        rets_T_list = []     # (t, list return saham layak di T)'''

if OLD_INIT in content:
    content = content.replace(OLD_INIT, NEW_INIT, 1)
    print("✅ N5a-2: Deklarasi rets_T_list")
else:
    print("⚠️ N5a-2: Blok init tidak cocok persis")

# ---------- N5b: Perbaiki perhitungan excess_w ----------
OLD_EXC_CALC = '''        # --- ADDENDUM 8: excess_w baru ---
        if all_rets_pool and rb_l:
            lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
            rb_w = np.clip(rb_l, lo, hi)
            bm_w_list = []
            # Hitung benchmark_w per tanggal dari return ter-winsorize
            # (recompute per tanggal dengan batas yang sama)
            daily_dict = {}
            for tkr, t in sig_list:
                pass   # tidak perlu; sudah dikumpulkan
            # bangun benchmark_w per tanggal
            t_to_bm_w = {}
            t_to_rb_w = {}
            for k, t in enumerate(dates):
                t_to_rb_w.setdefault(t, []).append(rb_w[k])
            # benchmark_w dari all_rets_pool ter-winsorize: pakai batas global
            # (pendekatan: benchmark_w = rata-rata return ter-winsorize di T)
            # Sederhanakan: benchmark_w(T) = bm(T) di-clip dengan batas global
            bm_w_arr = np.clip(bm_l, lo, hi)
            exc_w = np.array(rb_w) - np.array(bm_w_arr)
        else:
            exc_w = np.array([])'''

NEW_EXC_CALC = '''        # --- ADDENDUM 8 (revisi): excess_w baru ---
        # Batas winsorize dari pool SEMUA return saham layak
        # bm_w(T) = mean(rets_T yang di-clip dengan batas global)
        if all_rets_pool and rb_l:
            lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
            rb_w_arr = np.clip(np.array(rb_l), lo, hi)
            # Hitung bm_w per tanggal dari return yang di-clip
            t_to_bm_w = {}
            for t, rets_T in rets_T_list:
                rets_T_w = np.clip(np.array(rets_T), lo, hi)
                t_to_bm_w[t] = float(np.mean(rets_T_w))
            # Hitung excess_w per trade
            bm_w_arr = np.array([t_to_bm_w[t] for t in dates])
            exc_w = rb_w_arr - bm_w_arr
        else:
            exc_w = np.array([])'''

if OLD_EXC_CALC in content:
    content = content.replace(OLD_EXC_CALC, NEW_EXC_CALC, 1)
    print("✅ N5b: Perhitungan excess_w diperbaiki (bm_w dari return yang di-clip)")
else:
    print("⚠️ N5b: Blok excess_w lama tidak cocok persis")

# ---------- N5c: Perbaiki blok plasebo juga ----------
OLD_PLASEBO_CALC = '''    # Versi BARU (ADDENDUM 8)
    if all_rets_pool and rb_l:
        lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
        rb_w = np.clip(rb_l, lo, hi)
        bm_w = np.clip(bm_l, lo, hi)
        exc_w = np.array(rb_w) - np.array(bm_w)
        plasebo_exc.append(float(np.mean(exc_w)))'''

NEW_PLASEBO_CALC = '''    # Versi BARU (ADDENDUM 8 revisi)
    if all_rets_pool and rb_l:
        lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
        rb_w = np.clip(rb_l, lo, hi)
        # bm_w dari return yang di-clip
        # Kita pakai rets_T_list juga
        pass
    # Karena rets_T_list belum disimpan di plasebo, kita simpan dulu'''

if OLD_PLASEBO_CALC in content:
    content = content.replace(OLD_PLASEBO_CALC, NEW_PLASEBO_CALC, 1)
    print("✅ N5c-1: Blok plasebo ditandai (perlu tambah rets_T_list)")
else:
    print("⚠️ N5c-1: Blok plasebo lama tidak cocok persis")

# ---------- N5d: Perbaiki plasebo — tambah simpan rets_T_list ----------
OLD_PLASEBO_LOOP = '''    # Kumpulkan semua trade & semua return saham layak
    rb_l = []; bm_l = []; all_rets_pool = []
    for t in sig_random.index:
        # Semua return saham layak di T
        try: i = idx_b2.get_loc(t)
        except KeyError: continue
        if i + 20 >= len(idx_b2): continue
        t_ent = idx_b2[i+1]; t_exit = idx_b2[i+20]
        elig_tkrs_t = elig_b.columns[elig_b.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs_t:
            o2 = op_b.at[t_ent, tk2]; c2 = cl_b.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
        if not rets_T: continue
        all_rets_pool.extend(rets_T)
        bm = float(np.mean(rets_T))
        # Trade acak
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, 20, COST_BASE_SIDE)
            if rb is None: continue
            rb_l.append(rb); bm_l.append(bm)

    # Versi BARU (ADDENDUM 8 revisi)
    if all_rets_pool and rb_l:
        lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
        rb_w = np.clip(rb_l, lo, hi)
        # bm_w dari return yang di-clip
        # Kita pakai rets_T_list juga
        pass
    # Karena rets_T_list belum disimpan di plasebo, kita simpan dulu'''

NEW_PLASEBO_LOOP = '''    # Kumpulkan semua trade & semua return saham layak
    rb_l = []; bm_l = []; all_rets_pool = []; t_to_bm_w_data = []
    for t in sig_random.index:
        # Semua return saham layak di T
        try: i = idx_b2.get_loc(t)
        except KeyError: continue
        if i + 20 >= len(idx_b2): continue
        t_ent = idx_b2[i+1]; t_exit = idx_b2[i+20]
        elig_tkrs_t = elig_b.columns[elig_b.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs_t:
            o2 = op_b.at[t_ent, tk2]; c2 = cl_b.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
        if not rets_T: continue
        all_rets_pool.extend(rets_T)
        bm = float(np.mean(rets_T))
        # Trade acak
        rb_t_trades = []
        for tkr in sig_random.columns:
            if not bool(sig_random.at[t, tkr]): continue
            rb = trade_return(tkr, t, 20, COST_BASE_SIDE)
            if rb is None: continue
            rb_l.append(rb); bm_l.append(bm)
            rb_t_trades.append(rb)
        if rb_t_trades:
            t_to_bm_w_data.append((t, rets_T, rb_t_trades))

    # Versi BARU (ADDENDUM 8 revisi): bm_w dari return yang di-clip
    if all_rets_pool and rb_l:
        lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
        # Hitung excess_w per trade
        excs_w = []
        for t, rets_T, rbs_t in t_to_bm_w_data:
            bm_w = float(np.mean(np.clip(rets_T, lo, hi)))
            for rb in rbs_t:
                rb_w = float(np.clip(rb, lo, hi))
                excs_w.append(rb_w - bm_w)
        if excs_w:
            plasebo_exc.append(float(np.mean(excs_w)))'''

if OLD_PLASEBO_LOOP in content:
    content = content.replace(OLD_PLASEBO_LOOP, NEW_PLASEBO_LOOP, 1)
    print("✅ N5d: Blok plasebo diperbaiki (rets_T_list + bm_w yang benar)")
else:
    print("⚠️ N5d: Blok plasebo tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print("=" * 100)
print("  VERIFIKASI PATCH N5")
print("=" * 100)
print(f"  Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()

checks = [
    ("rets_T_list",        new.count("rets_T_list")),
    ("t_to_bm_w",          new.count("t_to_bm_w")),
    ("t_to_bm_w_data",     new.count("t_to_bm_w_data")),
    ("np.clip(rets_T",     new.count("np.clip(rets_T")),
    ("np.clip(ret",        new.count("np.clip(ret")),
]
for label, cnt in checks:
    print(f"    {label:20s}: {cnt}x")

try:
    ast.parse(new)
    print("\n  ✅ Syntax OK")
except SyntaxError as e:
    print(f"\n  ❌ Syntax ERROR: {e}")
    print(f"   Line {e.lineno}: {e.text}")
print("=" * 100)

Mounted at /content/drive
✅ N5a-1: Tambah simpan rets_T per tanggal
✅ N5a-2: Deklarasi rets_T_list
✅ N5b: Perhitungan excess_w diperbaiki (bm_w dari return yang di-clip)
✅ N5c-1: Blok plasebo ditandai (perlu tambah rets_T_list)
✅ N5d: Blok plasebo diperbaiki (rets_T_list + bm_w yang benar)

  VERIFIKASI PATCH N5
  Ukuran: 44614 byte
    rets_T_list         : 5x
    t_to_bm_w           : 6x
    t_to_bm_w_data      : 3x
    np.clip(rets_T      : 1x
    np.clip(ret         : 1x

  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3 — Tahap3_Horizon20.py

  LANGKAH 0 — REPRODUKSI pada data_old_sealed.pkl
  Target vs hasil Langkah 0:
  Sinyal-N     Periode         hasil       target   cocok?
------------------------------------------------------------
  BRK      N=5  C1           +0.3254%     +0.3250%    COCOK
  BRK      N=5  C2           +0.1470%     +0.1470%    COCOK
  BRK      N=10 C1           +1.0361%     +1.0360%    COCOK
  BRK      N=10 C2           +0.0855%     +0.0850%    COCOK
  MOM      N=5  C1           -0.2925%     -0.2920%    COCOK
  MOM      N=5  C2           -0.2771%     -0.2770%    COCOK
  MOM      N=10 C1           +0.0799%     +0.0800%    COCOK
  MOM      N=10 C2           -0.3842%     -0.3840%    COCOK

  LANGKAH 0 KESELURUHAN: COCOK
  Langkah 0 LOLOS. Lanjut ke Bagian A.

  BAGIAN A — DATA PANJANG TUNGGAL (2005 → 2026-06-05)
  Ticker: 561, unduh per-ticker start=2005-01-01 end=2026-06-06
  Resume: 561 ticker sudah 

In [ ]:
# ============================================================
# Verifikasi LAPORAN_Tahap3.md + cek kriteria B6 NEAR52H
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import pandas as pd, os

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
REPORT = f"{BASE}/LAPORAN_Tahap3.md"
HASIL = f"{BASE}/hasil_tahap3.pkl"

print("=" * 100)
print("  ISI LAPORAN_Tahap3.md")
print("=" * 100)
with open(REPORT) as f:
    print(f.read())
print("=" * 100)

# Cek detail NEAR52H dari pkl
print()
print("=" * 100)
print("  DETAIL NEAR52H — cek kriteria B6")
print("=" * 100)
hasil = pd.read_pickle(HASIL)
all_results = hasil["all_results"]
for N in [20]:
    for blk in ["K1", "K2", "K3", "pool"]:
        r = all_results.get(("NEAR52H", N, blk))
        if r is None: continue
        print(f"  NEAR52H N={N} {blk}:")
        for k, v in r.items():
            if isinstance(v, float):
                print(f"    {k:20s}: {v*100:+.4f}%" if "mean" in k or "exc" in k or "stress" in k else f"    {k:20s}: {v:.4f}")
            else:
                print(f"    {k:20s}: {v}")
        print()

print("=" * 100)
print("  Cek kriteria B6 untuk NEAR52H:")
print("=" * 100)
r_k1 = all_results[("NEAR52H", 20, "K1")]
r_k2 = all_results[("NEAR52H", 20, "K2")]
r_k3 = all_results[("NEAR52H", 20, "K3")]
r_pool = all_results[("NEAR52H", 20, "pool")]

print(f"  1. Excess > 0 di K1,K2,K3: {r_k1['mean_exc_w'] > 0} {r_k2['mean_exc_w'] > 0} {r_k3['mean_exc_w'] > 0}")
print(f"  2. Pool p1sisi < 0.01     : {r_pool['p_val']/2:.4f} -> {r_pool['p_val']/2 < 0.01}")
print(f"  3. >=2/3 blok p < 0.10    : p1sisi K1={r_k1['p_val']/2 if r_k1['mean_exc_w']>0 else 1-r_k1['p_val']/2:.4f}, K2={r_k2['p_val']/2 if r_k2['mean_exc_w']>0 else 1-r_k2['p_val']/2:.4f}, K3={r_k3['p_val']/2 if r_k3['mean_exc_w']>0 else 1-r_k3['p_val']/2:.4f}")
print(f"  4. Stress_w > 0 pool & >=2/3 blok:")
print(f"     K1 stress_w={r_k1['mean_stress_w']*100:+.4f}%  ({r_k1['mean_stress_w']>0})")
print(f"     K2 stress_w={r_k2['mean_stress_w']*100:+.4f}%  ({r_k2['mean_stress_w']>0})")
print(f"     K3 stress_w={r_k3['mean_stress_w']*100:+.4f}%  ({r_k3['mean_stress_w']>0})")
print(f"  5. n_trades >= 1000 tiap blok: K1={r_k1['n_trades']}, K2={r_k2['n_trades']}, K3={r_k3['n_trades']}")

Mounted at /content/drive
  ISI LAPORAN_Tahap3.md
TAHAP 3 - LAPORAN
Nama file skrip final : Tahap3_Horizon20.py
LANGKAH 0: COCOK (semua 8 target dalam toleransi 0,02 pp)
Detail Langkah 0:
  BRK      N=5  C1       hasil=+0.3254% target=+0.3250% COCOK
  BRK      N=5  C2       hasil=+0.1470% target=+0.1470% COCOK
  BRK      N=10 C1       hasil=+1.0361% target=+1.0360% COCOK
  BRK      N=10 C2       hasil=+0.0855% target=+0.0850% COCOK
  MOM      N=5  C1       hasil=-0.2925% target=-0.2920% COCOK
  MOM      N=5  C2       hasil=-0.2771% target=-0.2770% COCOK
  MOM      N=10 C1       hasil=+0.0799% target=+0.0800% COCOK
  MOM      N=10 C2       hasil=-0.3842% target=-0.3840% COCOK

BAGIAN A
Parameter unduh (sama dengan Tahap 2?) : ya
  - Sumber     : yfinance per-ticker
  - start      : 2005-01-01
  - end        : 2026-06-06
  - interval   : 1d
  - auto_adjust: True
A2 verifikasi vs data_sealed: persen bar-ticker cocok : 100.00%  (ambang >= 99%)
Tanggal mulai efektif | tanggal akhir | bar ef

In [ ]:
# ============================================================
# O1 — Cek semua file Tahap 3 di folder
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

print("=" * 100)
print("  CEK FILE TAHAP 3 DI FOLDER")
print("=" * 100)
print(f"  Folder: {BASE}")
print()

files_t3 = [
    "Tahap3_Horizon20.py",
    "LAPORAN_Tahap3.md",
    "hasil_tahap3.pkl",
    "data_long_sealed.pkl",
    "data_long_manifest.json",
    "data_events_long.csv",
    "_download_long_progress.pkl",
    "data_old_sealed.pkl",
    "Tahap2_Replikasi.py",
    "LAPORAN_Tahap2.md",
    "run_log.csv",
]

total_size = 0
for name in files_t3:
    p = f"{BASE}/{name}"
    if os.path.exists(p):
        size = os.path.getsize(p)
        total_size += size
        print(f"  ✅ {name:35s} {size:>12,} byte")
    else:
        print(f"  ❌ {name:35s} (tidak ada)")
print("-" * 100)
print(f"  Total: {total_size:,} byte ({total_size/1024/1024:.2f} MB)")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  CEK FILE TAHAP 3 DI FOLDER
  Folder: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3

  ✅ Tahap3_Horizon20.py                       44,614 byte
  ✅ LAPORAN_Tahap3.md                          6,063 byte
  ✅ hasil_tahap3.pkl                           2,922 byte
  ✅ data_long_sealed.pkl                  31,161,317 byte
  ✅ data_long_manifest.json                      245 byte
  ✅ data_events_long.csv                      13,932 byte
  ✅ _download_long_progress.pkl          109,755,280 byte
  ✅ data_old_sealed.pkl                   10,425,567 byte
  ✅ Tahap2_Replikasi.py                       27,234 byte
  ✅ LAPORAN_Tahap2.md                         10,739 byte
  ✅ run_log.csv                                1,664 byte
----------------------------------------------------------------------------------------------------
  Total: 151,449,577 byte (144.43 MB)


In [ ]:
# ============================================================
# O2 — Buat folder arsip Tahap 3 + copy file + SHA256 manifest
# ============================================================
import os, hashlib, json, shutil
from datetime import datetime

BASE   = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = f"{BASE}/_archive_tahap3_{datetime.now().strftime('%Y%m%d_%H%M')}"

os.makedirs(ARCHIVE, exist_ok=True)

files_to_archive = [
    "Tahap3_Horizon20.py",
    "LAPORAN_Tahap3.md",
    "hasil_tahap3.pkl",
    "data_long_sealed.pkl",
    "data_long_manifest.json",
    "data_events_long.csv",
    "Tahap2_Replikasi.py",
    "LAPORAN_Tahap2.md",
    "data_old_sealed.pkl",
    "run_log.csv",
]

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

manifest = {
    "archived_at": datetime.now().isoformat(timespec="seconds"),
    "folder": ARCHIVE,
    "files": {},
}

print("=" * 100)
print("  ARSIP TAHAP 3")
print("=" * 100)
print(f"  Folder arsip: {ARCHIVE}")
print()

for name in files_to_archive:
    src = f"{BASE}/{name}"
    if not os.path.exists(src):
        print(f"  ⚠️  {name}: tidak ada, skip")
        continue
    dst = f"{ARCHIVE}/{name}"
    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    sha = sha256_file(dst)
    manifest["files"][name] = {"size": size, "sha256": sha}
    print(f"  ✅ {name:35s} {size:>12,} byte  SHA256 {sha[:16]}…")

# Tulis manifest arsip
manifest_path = f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print()
print("=" * 100)
print(f"  ✅ Arsip selesai: {len(manifest['files'])} file")
print(f"  Manifest: {manifest_path}")
print("=" * 100)

  ARSIP TAHAP 3
  Folder arsip: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_tahap3_20261013_0538

  ✅ Tahap3_Horizon20.py                       44,614 byte  SHA256 5f6c9b92e2af3990…
  ✅ LAPORAN_Tahap3.md                          6,063 byte  SHA256 76bab8a948508585…
  ✅ hasil_tahap3.pkl                           2,922 byte  SHA256 be86b76687e24e0a…
  ✅ data_long_sealed.pkl                  31,161,317 byte  SHA256 956221434b81744b…
  ✅ data_long_manifest.json                      245 byte  SHA256 c927d49e83bf051b…
  ✅ data_events_long.csv                      13,932 byte  SHA256 28db91017be673f2…
  ✅ Tahap2_Replikasi.py                       27,234 byte  SHA256 51275c2041956230…
  ✅ LAPORAN_Tahap2.md                         10,739 byte  SHA256 37c99045ec93142f…
  ✅ data_old_sealed.pkl                   10,425,567 byte  SHA256 6179ae1e695a0d50…
  ✅ run_log.csv                                1,664 byte  SHA256 8a98abed4cec28e5…

  ✅ Arsip selesai: 10 file
  Manifest: /co

In [ ]:
# ============================================================
# O3 — Verifikasi arsip (cek SHA256 arsip vs sumber)
# ============================================================
import os, hashlib, json

BASE    = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = [d for d in os.listdir(BASE) if d.startswith("_archive_tahap3_")][0]
ARCHIVE = f"{BASE}/{ARCHIVE}"

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

manifest = json.load(open(f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"))

print("=" * 100)
print("  VERIFIKASI ARSIP")
print("=" * 100)
print(f"  Folder arsip: {ARCHIVE}")
print()

all_ok = True
for name, meta in manifest["files"].items():
    p = f"{ARCHIVE}/{name}"
    if not os.path.exists(p):
        print(f"  ❌ {name}: tidak ada")
        all_ok = False
        continue
    sha = sha256_file(p)
    ok = (sha == meta["sha256"])
    if not ok: all_ok = False
    print(f"  {'✅' if ok else '❌'} {name:35s} SHA256 {sha[:16]}… {'OK' if ok else 'BEDA!'}")

print()
print(f"  KESELURUHAN: {'✅ SEMUA FILE UTUH' if all_ok else '❌ ADA YANG BEDA'}")
print("=" * 100)

# Info: cara restore
print()
print("  CARA RESTORE jika file asli rusak/hilang:")
print(f"    cp -r {ARCHIVE}/* {BASE}/")
print("=" * 100)

  VERIFIKASI ARSIP
  Folder arsip: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_tahap3_20261013_0538

  ✅ Tahap3_Horizon20.py                 SHA256 5f6c9b92e2af3990… OK
  ✅ LAPORAN_Tahap3.md                   SHA256 76bab8a948508585… OK
  ✅ hasil_tahap3.pkl                    SHA256 be86b76687e24e0a… OK
  ✅ data_long_sealed.pkl                SHA256 956221434b81744b… OK
  ✅ data_long_manifest.json             SHA256 c927d49e83bf051b… OK
  ✅ data_events_long.csv                SHA256 28db91017be673f2… OK
  ✅ Tahap2_Replikasi.py                 SHA256 51275c2041956230… OK
  ✅ LAPORAN_Tahap2.md                   SHA256 37c99045ec93142f… OK
  ✅ data_old_sealed.pkl                 SHA256 6179ae1e695a0d50… OK
  ✅ run_log.csv                         SHA256 8a98abed4cec28e5… OK

  KESELURUHAN: ✅ SEMUA FILE UTUH

  CARA RESTORE jika file asli rusak/hilang:
    cp -r /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_tahap3_20261013_0538/* /content/drive/MyDrive/3 

In [ ]:
# Cek file Tahap 0/1/1b yang bisa diarsipkan
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
for name in [
    "LAPORAN_Tahap0.md", "LAPORAN_Tahap1.md", "LAPORAN_Tahap1b.md",
    "data_sealed.pkl", "data_events.csv", "seal_manifest.json",
    "params_frozen.json", "4. Anthropi_AF_G_Evo4.py",
    "Tahap0_Prep.py", "fundamental_soft.py",
]:
    p = f"{BASE}/{name}"
    print(f"  {'✅' if os.path.exists(p) else '❌'} {name}")

  ✅ LAPORAN_Tahap0.md
  ✅ LAPORAN_Tahap1.md
  ✅ LAPORAN_Tahap1b.md
  ✅ data_sealed.pkl
  ✅ data_events.csv
  ✅ seal_manifest.json
  ✅ params_frozen.json
  ✅ 4. Anthropi_AF_G_Evo4.py
  ✅ Tahap0_Prep.py
  ✅ fundamental_soft.py


In [ ]:
# ============================================================
# O4 — Arsip lengkap semua tahap (0, 1, 1b, 2, 3)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, hashlib, json, shutil
from datetime import datetime

BASE    = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = f"{BASE}/_archive_KOMPAS_full_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(ARCHIVE, exist_ok=True)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

# Daftar file per tahap
FILES = {
    "Tahap_0": [
        "LAPORAN_Tahap0.md",
        "Tahap0_Prep.py",
        "seal_manifest.json",
        "params_frozen.json",
        "data_sealed.pkl",
        "data_events.csv",
    ],
    "Tahap_1": [
        "LAPORAN_Tahap1.md",
        "4. Anthropi_AF_G_Evo4.py",
    ],
    "Tahap_1b": [
        "LAPORAN_Tahap1b.md",
    ],
    "Tahap_2": [
        "Tahap2_Replikasi.py",
        "LAPORAN_Tahap2.md",
        "data_old_sealed.pkl",
        "data_events_old.csv",
    ],
    "Tahap_3": [
        "Tahap3_Horizon20.py",
        "LAPORAN_Tahap3.md",
        "hasil_tahap3.pkl",
        "data_long_sealed.pkl",
        "data_long_manifest.json",
        "data_events_long.csv",
    ],
    "Umum": [
        "run_log.csv",
        "fundamental_soft.py",
    ],
}

manifest = {
    "archived_at": datetime.now().isoformat(timespec="seconds"),
    "folder": ARCHIVE,
    "tahap": {},
    "total_files": 0,
    "total_size": 0,
}

print("=" * 100)
print("  ARSIP KOMPAS LENGKAP (Tahap 0-3)")
print("=" * 100)
print(f"  Folder arsip: {ARCHIVE}")
print()

for tahap, files in FILES.items():
    print(f"  ── {tahap} ──")
    manifest["tahap"][tahap] = {}
    for name in files:
        src = f"{BASE}/{name}"
        if not os.path.exists(src):
            print(f"    ⚠️  {name:35s} tidak ada, skip")
            continue
        dst = f"{ARCHIVE}/{name}"
        shutil.copy2(src, dst)
        size = os.path.getsize(dst)
        sha = sha256_file(dst)
        manifest["tahap"][tahap][name] = {"size": size, "sha256": sha}
        manifest["total_files"] += 1
        manifest["total_size"] += size
        print(f"    ✅ {name:35s} {size:>12,} byte  {sha[:16]}…")
    print()

manifest_path = f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 100)
print(f"  ✅ ARSIP SELESAI")
print(f"     Total file: {manifest['total_files']}")
print(f"     Total size: {manifest['total_size']:,} byte "
      f"({manifest['total_size']/1024/1024:.2f} MB)")
print(f"     Manifest  : {manifest_path}")
print("=" * 100)

# Buat file CATATAN_ARSIP.md
note = f"""# Arsip KOMPAS — Tahap 0-3

**Tanggal arsip:** {manifest['archived_at']}
**Folder:** `{ARCHIVE}`

## Isi Arsip

| Tahap | File | Status |
|-------|------|--------|
"""

for tahap, files in manifest["tahap"].items():
    for name, meta in files.items():
        note += f"| {tahap} | `{name}` | {meta['size']:,} byte |\n"

note += f"""
## Total

- **{manifest['total_files']} file**
- **{manifest['total_size']:,} byte** ({manifest['total_size']/1024/1024:.2f} MB)

## Hasil Akhir Proyek

| Tahap | Verdict |
|-------|---------|
| Tahap 0 | ✅ LOLOS (mesin jujur) |
| Tahap 1 | ❌ 0/8 (N=2,3) |
| Tahap 1b | ❌ 0/8 (N=5,10) |
| Tahap 2 | ❌ BRK dicoret (tidak tereplikasi) |
| **Tahap 3** | ✅ **NEAR52H LOLOS** (N=20) |

## Cara Restore

```bash
cp -r {ARCHIVE}/* {BASE}/

_IncompleteInputError: incomplete input (4228008554.py, line 117)

In [ ]:
# ============================================================
# O4 — Arsip lengkap semua tahap (0, 1, 1b, 2, 3)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, hashlib, json, shutil
from datetime import datetime

BASE    = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = f"{BASE}/_archive_KOMPAS_full_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(ARCHIVE, exist_ok=True)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

# Daftar file per tahap
FILES = {
    "Tahap_0": [
        "LAPORAN_Tahap0.md",
        "Tahap0_Prep.py",
        "seal_manifest.json",
        "params_frozen.json",
        "data_sealed.pkl",
        "data_events.csv",
    ],
    "Tahap_1": [
        "LAPORAN_Tahap1.md",
        "4. Anthropi_AF_G_Evo4.py",
    ],
    "Tahap_1b": [
        "LAPORAN_Tahap1b.md",
    ],
    "Tahap_2": [
        "Tahap2_Replikasi.py",
        "LAPORAN_Tahap2.md",
        "data_old_sealed.pkl",
        "data_events_old.csv",
    ],
    "Tahap_3": [
        "Tahap3_Horizon20.py",
        "LAPORAN_Tahap3.md",
        "hasil_tahap3.pkl",
        "data_long_sealed.pkl",
        "data_long_manifest.json",
        "data_events_long.csv",
    ],
    "Umum": [
        "run_log.csv",
        "fundamental_soft.py",
    ],
}

manifest = {
    "archived_at": datetime.now().isoformat(timespec="seconds"),
    "folder": ARCHIVE,
    "tahap": {},
    "total_files": 0,
    "total_size": 0,
}

print("=" * 100)
print("  ARSIP KOMPAS LENGKAP (Tahap 0-3)")
print("=" * 100)
print(f"  Folder arsip: {ARCHIVE}")
print()

for tahap, files in FILES.items():
    print(f"  ── {tahap} ──")
    manifest["tahap"][tahap] = {}
    for name in files:
        src = f"{BASE}/{name}"
        if not os.path.exists(src):
            print(f"    ⚠️  {name:35s} tidak ada, skip")
            continue
        dst = f"{ARCHIVE}/{name}"
        shutil.copy2(src, dst)
        size = os.path.getsize(dst)
        sha = sha256_file(dst)
        manifest["tahap"][tahap][name] = {"size": size, "sha256": sha}
        manifest["total_files"] += 1
        manifest["total_size"] += size
        print(f"    ✅ {name:35s} {size:>12,} byte  {sha[:16]}…")
    print()

manifest_path = f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 100)
print(f"  ✅ ARSIP SELESAI")
print(f"     Total file: {manifest['total_files']}")
print(f"     Total size: {manifest['total_size']:,} byte "
      f"({manifest['total_size']/1024/1024:.2f} MB)")
print(f"     Manifest  : {manifest_path}")
print("=" * 100)

# Buat file CATATAN_ARSIP.md
note = f"""# Arsip KOMPAS — Tahap 0-3

**Tanggal arsip:** {manifest['archived_at']}
**Folder:** `{ARCHIVE}`

## Isi Arsip

| Tahap | File | Status |
|-------|------|--------|
"""

for tahap, files in manifest["tahap"].items():
    for name, meta in files.items():
        note += f"| {tahap} | `{name}` | {meta['size']:,} byte |\n"

note += f"""
## Total

- **{manifest['total_files']} file**
- **{manifest['total_size']:,} byte** ({manifest['total_size']/1024/1024:.2f} MB)

## Hasil Akhir Proyek

| Tahap | Verdict |
|-------|---------|
| Tahap 0 | ✅ LOLOS (mesin jujur) |
| Tahap 1 | ❌ 0/8 (N=2,3) |
| Tahap 1b | ❌ 0/8 (N=5,10) |
| Tahap 2 | ❌ BRK dicoret (tidak tereplikasi) |
| **Tahap 3** | ✅ **NEAR52H LOLOS** (N=20) |

## Cara Restore

```bash
cp -r {ARCHIVE}/* {BASE}/

_IncompleteInputError: incomplete input (4228008554.py, line 117)

In [ ]:
# ============================================================
# O4-A — Arsip file Tahap 0-3 + manifest SHA256
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, hashlib, json, shutil
from datetime import datetime

BASE    = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = f"{BASE}/_archive_KOMPAS_full_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(ARCHIVE, exist_ok=True)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

FILES = {
    "Tahap_0": ["LAPORAN_Tahap0.md", "Tahap0_Prep.py", "seal_manifest.json",
                "params_frozen.json", "data_sealed.pkl", "data_events.csv"],
    "Tahap_1": ["LAPORAN_Tahap1.md", "4. Anthropi_AF_G_Evo4.py"],
    "Tahap_1b": ["LAPORAN_Tahap1b.md"],
    "Tahap_2": ["Tahap2_Replikasi.py", "LAPORAN_Tahap2.md",
                "data_old_sealed.pkl", "data_events_old.csv"],
    "Tahap_3": ["Tahap3_Horizon20.py", "LAPORAN_Tahap3.md",
                "hasil_tahap3.pkl", "data_long_sealed.pkl",
                "data_long_manifest.json", "data_events_long.csv"],
    "Umum": ["run_log.csv", "fundamental_soft.py"],
}

manifest = {
    "archived_at": datetime.now().isoformat(timespec="seconds"),
    "folder": ARCHIVE,
    "tahap": {},
    "total_files": 0,
    "total_size": 0,
}

print("=" * 100)
print("  ARSIP KOMPAS LENGKAP (Tahap 0-3)")
print("=" * 100)
print(f"  Folder arsip: {ARCHIVE}")
print()

for tahap, files in FILES.items():
    print(f"  ── {tahap} ──")
    manifest["tahap"][tahap] = {}
    for name in files:
        src = f"{BASE}/{name}"
        if not os.path.exists(src):
            print(f"    ⚠️  {name:35s} tidak ada, skip")
            continue
        dst = f"{ARCHIVE}/{name}"
        shutil.copy2(src, dst)
        size = os.path.getsize(dst)
        sha = sha256_file(dst)
        manifest["tahap"][tahap][name] = {"size": size, "sha256": sha}
        manifest["total_files"] += 1
        manifest["total_size"] += size
        print(f"    ✅ {name:35s} {size:>12,} byte  {sha[:16]}…")
    print()

manifest_path = f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 100)
print(f"  ✅ ARSIP SELESAI")
print(f"     Total file: {manifest['total_files']}")
print(f"     Total size: {manifest['total_size']:,} byte "
      f"({manifest['total_size']/1024/1024:.2f} MB)")
print(f"     Manifest  : {manifest_path}")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ARSIP KOMPAS LENGKAP (Tahap 0-3)
  Folder arsip: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_KOMPAS_full_20261013_0547

  ── Tahap_0 ──
    ✅ LAPORAN_Tahap0.md                          2,669 byte  6052aaf2fc09bd59…
    ✅ Tahap0_Prep.py                             2,636 byte  fe092e82ea459f5b…
    ✅ seal_manifest.json                           309 byte  221f415300629438…
    ✅ params_frozen.json                           739 byte  66d4c2a9ed27352f…
    ✅ data_sealed.pkl                       14,161,504 byte  860e5faa948cfcb1…
    ✅ data_events.csv                            1,975 byte  c69ea25a85887caa…

  ── Tahap_1 ──
    ✅ LAPORAN_Tahap1.md                          4,030 byte  7a37c500b3b81aad…
    ✅ 4. Anthropi_AF_G_Evo4.py                  19,411 byte  4606c6c8bbc8a988…

  ── Tahap_1b ──
    ✅ LAPORAN_Tahap1b.md                       

In [ ]:
# ============================================================
# O4-B — Buat CATATAN_ARSIP.md (ringkasan proyek)
# ============================================================
import os, json

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
# Cari arsip terbaru
candidates = [d for d in os.listdir(BASE) if d.startswith("_archive_KOMPAS_full_")]
ARCHIVE = f"{BASE}/{sorted(candidates)[-1]}"

manifest = json.load(open(f"{ARCHIVE}/_ARCHIVE_MANIFEST.json"))

lines = []
lines.append("# Arsip KOMPAS — Tahap 0-3")
lines.append("")
lines.append(f"**Tanggal arsip:** {manifest['archived_at']}")
lines.append(f"**Folder:** `{ARCHIVE}`")
lines.append("")
lines.append("## Isi Arsip")
lines.append("")
lines.append("| Tahap | File | Ukuran |")
lines.append("|-------|------|--------|")
for tahap, files in manifest["tahap"].items():
    for name, meta in files.items():
        lines.append(f"| {tahap} | `{name}` | {meta['size']:,} byte |")
lines.append("")
lines.append("## Total")
lines.append("")
lines.append(f"- **{manifest['total_files']} file**")
lines.append(f"- **{manifest['total_size']:,} byte** ({manifest['total_size']/1024/1024:.2f} MB)")
lines.append("")
lines.append("## Hasil Akhir Proyek")
lines.append("")
lines.append("| Tahap | Verdict |")
lines.append("|-------|---------|")
lines.append("| Tahap 0 | ✅ LOLOS (mesin jujur) |")
lines.append("| Tahap 1 | ❌ 0/8 (N=2,3) |")
lines.append("| Tahap 1b | ❌ 0/8 (N=5,10) |")
lines.append("| Tahap 2 | ❌ BRK dicoret (tidak tereplikasi) |")
lines.append("| **Tahap 3** | ✅ **NEAR52H LOLOS** (N=20) |")
lines.append("")
lines.append("## Cara Restore")
lines.append("")
lines.append("```bash")
lines.append(f"cp -r {ARCHIVE}/* {BASE}/")
lines.append("```")
lines.append("")
lines.append("## Catatan Penting")
lines.append("")
lines.append("- Tahap 3: **NEAR52H LOLOS** 5/5 kriteria B6")
lines.append("- Koreksi ADDENDUM 8 (winsorize return) berhasil: plasebo −0,92% → +0,07%")
lines.append("- **Bias survivor** dicatat sebagai batas atas — bukan bukti final")
lines.append("- **Belum boleh uang nyata** — butuh paper trading 60 hari")
lines.append("")
lines.append("## File Kunci")
lines.append("")
lines.append("- **Script Tahap 3:** `Tahap3_Horizon20.py`")
lines.append("- **Data panjang:** `data_long_sealed.pkl` (2005-2026)")
lines.append("- **Laporan resmi:** `LAPORAN_Tahap3.md`")
lines.append("- **Hasil mentah:** `hasil_tahap3.pkl`")

note = "\n".join(lines)
note_path = f"{ARCHIVE}/CATATAN_ARSIP.md"
with open(note_path, "w") as f:
    f.write(note)

print("=" * 100)
print(f"  ✅ CATATAN_ARSIP.md dibuat")
print(f"     Path: {note_path}")
print(f"     Ukuran: {os.path.getsize(note_path)} byte")
print("=" * 100)

# Tampilkan isi folder arsip
print()
print("  Isi folder arsip:")
for f in sorted(os.listdir(ARCHIVE)):
    p = f"{ARCHIVE}/{f}"
    size = os.path.getsize(p) if os.path.isfile(p) else 0
    print(f"    {f:40s} {size:>12,} byte")
print("=" * 100)

  ✅ CATATAN_ARSIP.md dibuat
     Path: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_KOMPAS_full_20261013_0547/CATATAN_ARSIP.md
     Ukuran: 2223 byte

  Isi folder arsip:
    4. Anthropi_AF_G_Evo4.py                       19,411 byte
    CATATAN_ARSIP.md                                2,223 byte
    LAPORAN_Tahap0.md                               2,669 byte
    LAPORAN_Tahap1.md                               4,030 byte
    LAPORAN_Tahap1b.md                              5,277 byte
    LAPORAN_Tahap2.md                              10,739 byte
    LAPORAN_Tahap3.md                               6,063 byte
    Tahap0_Prep.py                                  2,636 byte
    Tahap2_Replikasi.py                            27,234 byte
    Tahap3_Horizon20.py                            44,614 byte
    _ARCHIVE_MANIFEST.json                          3,424 byte
    data_events.csv                                 1,975 byte
    data_events_long.csv                           13,9

In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"

python3: can't open file '/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py': [Errno 2] No such file or directory


In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
print(f"Folder ada: {os.path.isdir(BASE)}")
if os.path.isdir(BASE):
    files = sorted(os.listdir(BASE))
    print(f"Total isi: {len(files)}")
    for f in files:
        if "Produksi" in f or f.endswith(".py"):
            p = f"{BASE}/{f}"
            print(f"  {f:40s} {os.path.getsize(p):>10} byte")

Folder ada: True
Total isi: 29
  1. AladdinFoundry_Gen_v1_Step25.py           292829 byte
  4. Anthropi_AF_G_Evo4.py                      19411 byte
  Tahap0_Prep.py                                 2636 byte
  Tahap2_Replikasi.py                           27234 byte
  Tahap3_Horizon20.py                           44614 byte
  fundamental_soft.py                            4326 byte


In [ ]:
%%writefile "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"
import os, sys, json, csv
from datetime import datetime
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

BASE   = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL   = f"{BASE}/data_long_sealed.pkl"
OUT    = f"{BASE}/produksi_signals.csv"
LOG    = f"{BASE}/produksi_log.csv"

MIN_PRICE         = 200.0
MIN_TURNOVER_T3   = 200_000_000.0
TOP_N_RANK_T3     = 100
NEAR52H_THRESHOLD = 0.95
LOOKBACK_52W      = 252
COST_BASE_SIDE    = 0.0015
N_HOLD            = 20

print("=" * 80)
print("  PRODUKSI NEAR52H")
print("=" * 80)
print(f"  Waktu: {datetime.now().isoformat(timespec='seconds')}")

if not os.path.exists(SEAL):
    print(f"STOP: {SEAL} tidak ada."); sys.exit(1)

data = pd.read_pickle(SEAL)
op = data["price_open"]; hi = data["price_high"]
cl = data["price_close"]; vo = data["price_volume"]

print(f"  Data: {cl.shape}, range {cl.index.min().date()} -> {cl.index.max().date()}")

turnover = cl * vo
toma20 = turnover.rolling(20, min_periods=20).mean()
mask_a = (cl >= MIN_PRICE) & cl.notna()
mask_b = (toma20 >= MIN_TURNOVER_T3)
toma20_ranked = toma20.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig = mask_a & mask_b & mask_c

high_52w = hi.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
near52h = (cl >= NEAR52H_THRESHOLD * high_52w) & elig

t_last = cl.index[-1]
t_prev = cl.index[-2]
print()
print(f"  Sinyal di T = {t_prev.date()}, entry di T+1 = {t_last.date()}")
print()
print("  KANDIDAT BELI:")
print(f"  {'ticker':10s} {'Close(T)':>10s} {'High52w':>10s} {'%52w':>7s} {'Open(T+1)':>10s}")
print("-" * 60)

kandidat = []
sig_prev = near52h.loc[t_prev]
for tkr in cl.columns:
    if not bool(sig_prev.get(tkr, False)): continue
    o_entry = op.at[t_last, tkr]
    c_sig   = cl.at[t_prev, tkr]
    h_52w   = high_52w.at[t_prev, tkr]
    if pd.isna(o_entry) or pd.isna(c_sig) or pd.isna(h_52w): continue
    kandidat.append({
        "ticker": tkr, "close_signal": c_sig, "high_52w": h_52w,
        "pct_of_52w": c_sig / h_52w, "open_entry": o_entry,
    })
    print(f"  {tkr:10s} {c_sig:>10.2f} {h_52w:>10.2f} {c_sig/h_52w*100:>6.2f}% {o_entry:>10.2f}")

print()
print(f"  Total kandidat: {len(kandidat)}")

df_out = pd.DataFrame(kandidat)
df_out.to_csv(OUT, index=False)
print(f"  Disimpan: {OUT}")

_COLUMNS = ["timestamp", "t_signal", "t_entry", "n_kandidat"]
file_exists = os.path.exists(LOG)
with open(LOG, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "t_signal": str(t_prev.date()),
        "t_entry": str(t_last.date()),
        "n_kandidat": len(kandidat),
    })
print(f"  Log: {LOG}")
print("=" * 80)

Overwriting /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py


In [ ]:
import os
p = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"
print(f"Ada: {os.path.exists(p)}")
if os.path.exists(p):
    print(f"Ukuran: {os.path.getsize(p)} byte")

Ada: True
Ukuran: 2866 byte


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"

  PRODUKSI NEAR52H
  Waktu: 2026-10-13T06:04:53
  Data: (1385, 561), range 2019-10-01 -> 2026-06-05

  Sinyal di T = 2026-06-04, entry di T+1 = 2026-06-05

  KANDIDAT BELI:
  ticker       Close(T)    High52w    %52w  Open(T+1)
------------------------------------------------------------
  KING.JK        565.00     565.00 100.00%     565.00
  MMIX.JK        655.00     655.00 100.00%     700.00

  Total kandidat: 2
  Disimpan: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/produksi_signals.csv
  Log: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/produksi_log.csv


In [ ]:
import pandas as pd, os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

print("=" * 80)
print("  VERIFIKASI FILE OUTPUT PRODUKSI")
print("=" * 80)

for name in ["produksi_signals.csv", "produksi_log.csv"]:
    p = f"{BASE}/{name}"
    if os.path.exists(p):
        print(f"\n  ✅ {name} ({os.path.getsize(p)} byte)")
        df = pd.read_csv(p)
        print(df.to_string(index=False))
    else:
        print(f"  ❌ {name} tidak ada")
print("=" * 80)

  VERIFIKASI FILE OUTPUT PRODUKSI

  ✅ produksi_signals.csv (111 byte)
 ticker  close_signal  high_52w  pct_of_52w  open_entry
KING.JK         565.0     565.0         1.0       565.0
MMIX.JK         655.0     655.0         1.0       700.0

  ✅ produksi_log.csv (84 byte)
          timestamp   t_signal    t_entry  n_kandidat
2026-10-13T06:04:53 2026-06-04 2026-06-05           2


In [ ]:
%%writefile "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"
import os, sys, json, csv
from datetime import datetime
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

BASE   = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SEAL   = f"{BASE}/data_long_sealed.pkl"
OUT    = f"{BASE}/produksi_signals.csv"
LOG    = f"{BASE}/produksi_log.csv"

# Fee Stockbit riil (per 2026)
FEE_BELI_STOCKBIT = 0.0015   # 0.15%
FEE_JUAL_STOCKBIT = 0.0025   # 0.25%
FEE_RT_STOCKBIT   = FEE_BELI_STOCKBIT + FEE_JUAL_STOCKBIT  # 0.40%

MIN_PRICE         = 200.0
MIN_TURNOVER_T3   = 200_000_000.0
TOP_N_RANK_T3     = 100
NEAR52H_THRESHOLD = 0.95
LOOKBACK_52W      = 252
N_HOLD            = 20

print("=" * 80)
print("  PRODUKSI NEAR52H — Fee Stockbit")
print("=" * 80)
print(f"  Waktu       : {datetime.now().isoformat(timespec='seconds')}")
print(f"  Fee beli    : {FEE_BELI_STOCKBIT*100:.2f}% (Stockbit)")
print(f"  Fee jual    : {FEE_JUAL_STOCKBIT*100:.2f}% (Stockbit, termasuk PPh 0.1%)")
print(f"  Total RT    : {FEE_RT_STOCKBIT*100:.2f}%")
print()

if not os.path.exists(SEAL):
    print(f"STOP: {SEAL} tidak ada."); sys.exit(1)

data = pd.read_pickle(SEAL)
op = data["price_open"]; hi = data["price_high"]
cl = data["price_close"]; vo = data["price_volume"]

print(f"  Data: {cl.shape}, range {cl.index.min().date()} -> {cl.index.max().date()}")

turnover = cl * vo
toma20 = turnover.rolling(20, min_periods=20).mean()
mask_a = (cl >= MIN_PRICE) & cl.notna()
mask_b = (toma20 >= MIN_TURNOVER_T3)
toma20_ranked = toma20.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig = mask_a & mask_b & mask_c

high_52w = hi.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
near52h = (cl >= NEAR52H_THRESHOLD * high_52w) & elig

t_last = cl.index[-1]
t_prev = cl.index[-2]
print()
print(f"  Sinyal di T = {t_prev.date()}, entry di T+1 = {t_last.date()}")
print()
print("  KANDIDAT BELI (dengan fee Stockbit):")
print(f"  {'ticker':10s} {'Close(T)':>10s} {'High52w':>10s} {'%52w':>7s} {'Open(T+1)':>10s} {'Entry+fee':>11s}")
print("-" * 72)

kandidat = []
sig_prev = near52h.loc[t_prev]
for tkr in cl.columns:
    if not bool(sig_prev.get(tkr, False)): continue
    o_entry = op.at[t_last, tkr]
    c_sig   = cl.at[t_prev, tkr]
    h_52w   = high_52w.at[t_prev, tkr]
    if pd.isna(o_entry) or pd.isna(c_sig) or pd.isna(h_52w): continue
    entry_w_fee = o_entry * (1 + FEE_BELI_STOCKBIT)
    kandidat.append({
        "ticker": tkr, "close_signal": c_sig, "high_52w": h_52w,
        "pct_of_52w": c_sig / h_52w, "open_entry": o_entry,
        "entry_after_fee": entry_w_fee,
    })
    print(f"  {tkr:10s} {c_sig:>10.2f} {h_52w:>10.2f} {c_sig/h_52w*100:>6.2f}% "
          f"{o_entry:>10.2f} {entry_w_fee:>11.2f}")

print()
print(f"  Total kandidat: {len(kandidat)}")

df_out = pd.DataFrame(kandidat)
df_out.to_csv(OUT, index=False)
print(f"  Disimpan: {OUT}")

_COLUMNS = ["timestamp", "t_signal", "t_entry", "n_kandidat", "fee_beli", "fee_jual"]
file_exists = os.path.exists(LOG)
with open(LOG, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "t_signal": str(t_prev.date()),
        "t_entry": str(t_last.date()),
        "n_kandidat": len(kandidat),
        "fee_beli": FEE_BELI_STOCKBIT,
        "fee_jual": FEE_JUAL_STOCKBIT,
    })
print(f"  Log: {LOG}")
print("=" * 80)
print("  Exit target: Close(T+20) = 20 hari bursa setelah entry")
print("  Fee jual 0.25% akan dikenakan saat exit")
print("=" * 80)

Overwriting /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Produksi_NEAR52H.py"

  PRODUKSI NEAR52H — Fee Stockbit
  Waktu       : 2026-10-13T06:11:30
  Fee beli    : 0.15% (Stockbit)
  Fee jual    : 0.25% (Stockbit, termasuk PPh 0.1%)
  Total RT    : 0.40%

  Data: (1385, 561), range 2019-10-01 -> 2026-06-05

  Sinyal di T = 2026-06-04, entry di T+1 = 2026-06-05

  KANDIDAT BELI (dengan fee Stockbit):
  ticker       Close(T)    High52w    %52w  Open(T+1)   Entry+fee
------------------------------------------------------------------------
  KING.JK        565.00     565.00 100.00%     565.00      565.85
  MMIX.JK        655.00     655.00 100.00%     700.00      701.05

  Total kandidat: 2
  Disimpan: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/produksi_signals.csv
  Log: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/produksi_log.csv
  Exit target: Close(T+20) = 20 hari bursa setelah entry
  Fee jual 0.25% akan dikenakan saat exit


In [ ]:
# ============================================================
# P0-A — Mount + Cek File + Buat Tahap3b_Ketahanan.py kosong
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

print("=" * 100)
print("  CEK FILE YANG DITUJU")
print("=" * 100)
print(f"  Folder : {BASE}  →  {'ADA' if os.path.isdir(BASE) else '❌ TIDAK ADA'}")
for name, path in [
    ("data_long_sealed.pkl",     f"{BASE}/data_long_sealed.pkl"),
    ("data_sealed.pkl",          f"{BASE}/data_sealed.pkl"),
    ("data_old_sealed.pkl",      f"{BASE}/data_old_sealed.pkl"),
    ("Tahap3_Horizon20.py",      f"{BASE}/Tahap3_Horizon20.py"),
    ("LAPORAN_Tahap3.md",        f"{BASE}/LAPORAN_Tahap3.md"),
    ("hasil_tahap3.pkl",         f"{BASE}/hasil_tahap3.pkl"),
    ("data_long_manifest.json",  f"{BASE}/data_long_manifest.json"),
]:
    ada = os.path.exists(path)
    size = os.path.getsize(path) if ada else 0
    print(f"  {'✅' if ada else '❌'} {name:28s} {size:>12,} byte")
print("=" * 100)

# Buat file kosong
with open(SCRIPT_PATH, "w") as f:
    f.write("")
print(f"\n✅ File dibuat kosong: {SCRIPT_PATH}")
print(f"   Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-B untuk menulis bagian 1/7")

Mounted at /content/drive
  CEK FILE YANG DITUJU
  Folder : /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3  →  ADA
  ✅ data_long_sealed.pkl           31,161,317 byte
  ✅ data_sealed.pkl                14,161,504 byte
  ✅ data_old_sealed.pkl            10,425,567 byte
  ✅ Tahap3_Horizon20.py                44,614 byte
  ✅ LAPORAN_Tahap3.md                   6,063 byte
  ✅ hasil_tahap3.pkl                    2,922 byte
  ✅ data_long_manifest.json               245 byte

✅ File dibuat kosong: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py
   Ukuran: 0 byte

Lanjut ke sel P0-B untuk menulis bagian 1/7


In [ ]:
# ============================================================
# P0-B — Blok 1/7: Header, konstanta, fungsi util
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK1 = r'''# ============================================================
# Tahap3b_Ketahanan.py
# KOMPAS TAHAP 3b — Uji Ketahanan NEAR52H
# ============================================================
# 6 uji ketahanan (R1-R6) + R7 (VALIDATE sekali)
# Statistik: Addendum 8 (winsorize return, benchmark_w per T)
# Sinyal: HANYA NEAR52H, N=20
# Data: data_long_sealed.pkl (kecuali R7)
# ============================================================

import os, sys, csv, json, random, hashlib, time
from datetime import datetime
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"[WARN] Mount Drive: {e}")

BASE              = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
LONG_SEAL_PATH    = f"{BASE}/data_long_sealed.pkl"
LONG_SEALED_V2    = f"{BASE}/data_long_sealed_v2.pkl"   # untuk R7
LONG_MANIFEST_V2  = f"{BASE}/data_long_manifest_v2.json"
RUN_LOG_PATH      = f"{BASE}/run_log.csv"
REPORT_PATH       = f"{BASE}/LAPORAN_Tahap3b.md"
PROGRESS_V2_PATH  = f"{BASE}/_download_long_v2_progress.pkl"

CUTOFF_SEALED     = pd.Timestamp("2026-06-05")
CUTOFF_NEW        = pd.Timestamp("2026-06-07")

# Filter layak (Addendum 7)
MIN_PRICE         = 200.0
MIN_TURNOVER_T3   = 200_000_000.0
TOP_N_RANK_T3     = 100
TOP_N_RANK_R4     = 50       # untuk R4

# Sinyal NEAR52H
NEAR52H_THRESHOLD = 0.95
LOOKBACK_52W      = 252

WARMUP_BARS       = 260
WINDOW_BACK       = 3
WINDOW_FWD        = 25

# Biaya
COST_BASE_SIDE    = 0.0015   # 0.3% RT
COST_STRESS_SIDE  = 0.0030   # 0.6% RT
COST_R3_SIDE      = 0.0050   # R3: 1.0% RT
N_HORIZON         = 20
HAC_MAXLAGS       = 20

# Target reproduksi Langkah 0
TARGET_L0 = {
    "K1": {"exc_w": +0.05367, "stress_w": +0.03535, "n": 1099},
    "K2": {"exc_w": +0.01635, "stress_w": +0.00200, "n": 2426},
    "K3": {"exc_w": +0.02174, "stress_w": +0.02485, "n": 3750},
}
TOL_L0_N  = 0.01     # 1%
TOL_L0_PP = 0.0002   # 0.02 pp

random.seed(42); np.random.seed(42)

def log(msg=""):
    print(msg, flush=True)

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def fix_p1b(p, exc):
    if pd.isna(p): return float("nan")
    if exc > 0:   return p / 2
    if exc < 0:   return 1 - p / 2
    return 0.5

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

log("=" * 90)
log("  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py")
log("=" * 90)
log(f"  Waktu: {datetime.now().isoformat(timespec='seconds')}")
log("")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK1)

print(f"✅ Blok 1/7 ditulis ({len(BLOK1)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-C untuk menulis bagian 2/7 (load + Langkah 0)")

✅ Blok 1/7 ditulis (2742 byte)
   File sekarang: 2746 byte

Lanjut ke sel P0-C untuk menulis bagian 2/7 (load + Langkah 0)


In [ ]:
# ============================================================
# P0-C — Blok 2/7: Load data + Langkah 0 (reproduksi)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK2 = r'''
# ============================================================
# LOAD DATA + PREPROCESSING
# ============================================================
log("=" * 90)
log("  LOAD DATA PANJANG + PREPROCESSING")
log("=" * 90)

if not os.path.exists(LONG_SEAL_PATH):
    log(f"STOP: {LONG_SEAL_PATH} tidak ada."); sys.exit(1)

data_long = pd.read_pickle(LONG_SEAL_PATH)
op = data_long["price_open"].copy()
hi = data_long["price_high"].copy()
lo = data_long["price_low"].copy()
cl = data_long["price_close"].copy()
vo = data_long["price_volume"].copy()

log(f"  Data: {cl.shape}, range {cl.index.min().date()} → {cl.index.max().date()}")

# Filter saham layak (Addendum 7)
turnover = cl * vo
toma20 = turnover.rolling(20, min_periods=20).mean()
mask_a = (cl >= MIN_PRICE) & cl.notna()
mask_b = (toma20 >= MIN_TURNOVER_T3)
toma20_ranked = toma20.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig = mask_a & mask_b & mask_c

# Sinyal NEAR52H
high_52w = hi.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
near52h = (cl >= NEAR52H_THRESHOLD * high_52w) & elig

# Buang warm-up 260
idx_full = cl.index
idx = idx_full[WARMUP_BARS:]
op_b = op.loc[idx]; hi_b = hi.loc[idx]; lo_b = lo.loc[idx]
cl_b = cl.loc[idx]; vo_b = vo.loc[idx]
elig_b = elig.loc[idx]
near52h_b = near52h.loc[idx]

n_b = len(idx)
blk = n_b // 3
tK1_end = idx[blk - 1]
tK2_start = idx[blk]; tK2_end = idx[2*blk - 1]
tK3_start = idx[2*blk]

log(f"  Setelah warm-up 260: {n_b} bar")
log(f"  K1: {idx[0].date()} → {tK1_end.date()} ({blk} bar)")
log(f"  K2: {tK2_start.date()} → {tK2_end.date()} ({blk} bar)")
log(f"  K3: {tK3_start.date()} → {idx[-1].date()} ({n_b - 2*blk} bar)")
log("")

# ============================================================
# FUNGSI INTI — trade return, benchmark, excess (Addendum 8)
# ============================================================
def build_trades_and_excess(idx_arr, op_arr, cl_arr, elig_arr, sig_df,
                            N=20, cost_side=COST_BASE_SIDE,
                            delay=1, drop_gap=False):
    """Bangun trade dan excess_w (Addendum 8).
    delay: 1 = entry Open(T+1); 2 = entry Open(T+2)
    drop_gap: True = buang trade dengan Open(T+1)/Close(T) - 1 >= 0.10
    """
    trade_records = []   # list of dict: t, tkr, rb, rb_w, bm_w, exc_w, rs, rs_w, bm_stress_w
    all_rets_pool = []   # untuk batas winsorize
    dates = []
    for t in sig_df.index:
        try: i = idx_arr.get_loc(t)
        except KeyError: continue
        if i + delay + N - 1 >= len(idx_arr): continue
        t_ent = idx_arr[i + delay]
        t_exit = idx_arr[i + delay + N - 1]
        elig_tkrs = elig_arr.columns[elig_arr.loc[t]]
        # Kumpulkan semua return saham layak di T (untuk batas winsorize)
        rets_T_base = []
        rets_T_stress = []
        for tk2 in elig_tkrs:
            o2 = op_arr.at[t_ent, tk2]; c2 = cl_arr.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rb2 = (c2*(1-cost_side) - o2*(1+cost_side)) / (o2*(1+cost_side))
            rets_T_base.append(rb2)
        if not rets_T_base: continue
        all_rets_pool.extend(rets_T_base)
        # Trade dari sinyal
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_arr.at[t_ent, tkr]; c_tr = cl_arr.at[t_exit, tkr]
            c_sig = cl_arr.at[t, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            # R2: drop gap besar
            if drop_gap and pd.notna(c_sig) and c_sig > 0:
                gap = o_tr / c_sig - 1
                if gap >= 0.10: continue
            rb_tr = (c_tr*(1-cost_side) - o_tr*(1+cost_side)) / (o_tr*(1+cost_side))
            trade_records.append({
                "t": t, "tkr": tkr, "rb": rb_tr,
                "t_ent": t_ent, "t_exit": t_exit,
                "rets_T_base": rets_T_base,
            })
            dates.append(t)
    return trade_records, all_rets_pool, dates


def compute_excess_w(trade_records, all_rets_pool, cost_side=COST_BASE_SIDE):
    """Hitung excess_w (Addendum 8) + excess_w_lama (pembanding)."""
    if not all_rets_pool or not trade_records:
        return {"n_trades": 0, "mean_exc_w": float("nan"),
                "mean_exc_w_lama": float("nan"),
                "mean_stress_w": float("nan"),
                "mean_stress_raw": float("nan"),
                "dates": [], "exc_w_list": []}
    lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
    exc_w_list = []
    exc_w_lama_list = []
    rb_list = []
    for rec in trade_records:
        rb = rec["rb"]
        rets_T = rec["rets_T_base"]
        bm_w = float(np.mean(np.clip(rets_T, lo, hi)))
        rb_w = float(np.clip(rb, lo, hi))
        exc_w = rb_w - bm_w
        exc_w_list.append(exc_w)
        # lama: excess raw, winsorize terpisah
        bm_raw = float(np.mean(rets_T))
        exc_w_lama_list.append(rb - bm_raw)
        rb_list.append(rb)
    exc_w_arr = np.array(exc_w_list)
    exc_w_lama_arr = np.array(exc_w_lama_list)
    return {
        "n_trades": len(trade_records),
        "mean_exc_w": float(np.mean(exc_w_arr)),
        "mean_exc_w_lama": float(np.mean(winsorize(exc_w_lama_arr))),
        "median_exc": float(np.median(exc_w_arr)),
        "dates": [r["t"] for r in trade_records],
        "exc_w_list": exc_w_list,
    }


def hac_p_value(dates_list, exc_w_list, N=20):
    """HAC p-value + t-stat dari deret harian."""
    if len(dates_list) < 10:
        return {"t_stat": float("nan"), "p_val": float("nan"), "n_dates": 0}
    daily = {}
    for t, e in zip(dates_list, exc_w_list):
        daily.setdefault(t, []).append(e)
    ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
    if len(ds) < 10:
        return {"t_stat": float("nan"), "p_val": float("nan"), "n_dates": len(ds)}
    model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
    return {"t_stat": float(model.tvalues[0]),
            "p_val":  float(model.pvalues[0]),
            "n_dates": len(ds)}


def aggregate_block(idx_arr, t_start, t_end, sig_df, eligible_df,
                    op_arr, cl_arr, N=20, cost_side=COST_BASE_SIDE,
                    delay=1, drop_gap=False):
    """Jalankan trade+excess untuk satu blok (t_start..t_end)."""
    # Filter sinyal di blok
    mask_t = pd.Series((sig_df.index >= t_start) & (sig_df.index <= t_end),
                       index=sig_df.index)
    sig_block = sig_df.copy()
    for t in sig_block.index:
        if not mask_t.loc[t]: sig_block.loc[t] = False
    trades, all_rets, dates = build_trades_and_excess(
        idx_arr, op_arr, cl_arr, eligible_df, sig_block,
        N=N, cost_side=cost_side, delay=delay, drop_gap=drop_gap)
    res = compute_excess_w(trades, all_rets, cost_side=cost_side)
    res["n_trades"] = len(trades)
    return res, trades, all_rets


# ============================================================
# LANGKAH 0 — REPRODUKSI
# ============================================================
log("=" * 90)
log("  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)")
log("=" * 90)
log(f"  {'Blok':6s} {'hasil exc_w':>14s} {'target':>14s} {'cocok?':>8s}  "
    f"{'hasil str_w':>14s} {'target':>14s} {'cocok?':>8s}  "
    f"{'n':>6s} {'target':>6s} {'cocok?':>8s}")
log("-" * 110)

l0_results = {}
l0_pass = True
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    res, trades, all_rets = aggregate_block(
        idx, t1, t2, near52h_b, elig_b, op_b, cl_b, N=20,
        cost_side=COST_BASE_SIDE, delay=1, drop_gap=False)
    # stress
    trades_s, _, _ = build_trades_and_excess(
        idx, op_b, cl_b, elig_b, near52h_b.loc[(near52h_b.index >= t1) & (near52h_b.index <= t2)],
        N=20, cost_side=COST_STRESS_SIDE, delay=1, drop_gap=False)
    # Batas winsorize stress dari pool semua return stress
    # (untuk reproduksi, kita pakai cara yang sama: hitung excess stress dengan winsorize)
    res_stress, _, _ = aggregate_block(
        idx, t1, t2, near52h_b, elig_b, op_b, cl_b, N=20,
        cost_side=COST_STRESS_SIDE, delay=1, drop_gap=False)
    exc_w = res["mean_exc_w"]
    stress_w = res_stress["mean_exc_w"]
    n_tr = res["n_trades"]
    tgt = TARGET_L0[blk_id]
    ok_exc = abs(exc_w - tgt["exc_w"]) <= TOL_L0_PP
    ok_str = abs(stress_w - tgt["stress_w"]) <= TOL_L0_PP
    ok_n = abs(n_tr - tgt["n"]) / max(1, tgt["n"]) <= TOL_L0_N
    if not (ok_exc and ok_str and ok_n): l0_pass = False
    log(f"  {blk_id:6s} {exc_w*100:>+13.4f}% {tgt['exc_w']*100:>+13.4f}% "
        f"{'COCOK' if ok_exc else 'BEDA':>8s}  "
        f"{stress_w*100:>+13.4f}% {tgt['stress_w']*100:>+13.4f}% "
        f"{'COCOK' if ok_str else 'BEDA':>8s}  "
        f"{n_tr:>6d} {tgt['n']:>6d} {'COCOK' if ok_n else 'BEDA':>8s}")
    l0_results[blk_id] = res

log("-" * 110)
log(f"  LANGKAH 0: {'COCOK' if l0_pass else 'TIDAK COCOK'}")

if not l0_pass:
    log("  STOP: Langkah 0 tidak cocok. Tidak lanjut ke uji ketahanan.")
    sys.exit(2)

log("  Langkah 0 LOLOS. Lanjut ke uji ketahanan R1-R6.")
'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK2)

print(f"✅ Blok 2/7 ditulis ({len(BLOK2)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-D untuk menulis bagian 3/7 (uji R1-R4)")

✅ Blok 2/7 ditulis (9115 byte)
   File sekarang: 11875 byte

Lanjut ke sel P0-D untuk menulis bagian 3/7 (uji R1-R4)


In [ ]:
# ============================================================
# P0-D — Blok 3/7: uji ketahanan R1-R4
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK3 = r'''
# ============================================================
# UJI KETAHANAN R1-R4
# ============================================================
log("")
log("=" * 90)
log("  UJI KETAHANAN R1-R4")
log("=" * 90)

# ---------- Fungsi bantu untuk agregasi pool dari 3 blok ----------
def jalankan_3_blok(idx_arr, op_arr, cl_arr, elig_arr, sig_df,
                    N=20, cost_side=COST_BASE_SIDE, delay=1, drop_gap=False,
                    top_n_override=None):
    """Jalankan 3 blok. Return dict hasil per blok + pool."""
    if top_n_override is not None:
        # R4: filter elig tambahan — 50 teratas turnover 20 hari
        to20 = (cl_arr * vo_b)  # vo_b dari scope global
        to20_rank = to20.where(elig_arr).rank(axis=1, ascending=False)
        elig_use = elig_arr & (to20_rank <= top_n_override)
    else:
        elig_use = elig_arr

    out = {}
    all_dates_pool = []
    all_exc_pool = []
    all_stress_pool = []

    for blk_id, t1, t2 in [("K1", idx_arr[0], tK1_end),
                            ("K2", tK2_start, tK2_end),
                            ("K3", tK3_start, idx_arr[-1])]:
        # Filter sinyal di blok
        mask_t = pd.Series((sig_df.index >= t1) & (sig_df.index <= t2),
                           index=sig_df.index)
        sig_blk = sig_df.where(mask_t, False)
        trades, all_rets, dates = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=cost_side, delay=delay, drop_gap=drop_gap)
        res = compute_excess_w(trades, all_rets, cost_side=cost_side)
        hac = hac_p_value(res["dates"], res["exc_w_list"], N=N)
        res.update(hac)
        res["n_trades"] = len(trades)
        out[blk_id] = res
        all_dates_pool.extend(res["dates"])
        all_exc_pool.extend(res["exc_w_list"])

        # Stress untuk blok ini
        trades_s, all_rets_s, _ = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=COST_STRESS_SIDE, delay=delay, drop_gap=drop_gap)
        res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
        out[blk_id]["stress_w"] = res_s["mean_exc_w"]
        all_stress_pool.extend(res_s["exc_w_list"])

    # Pool
    if all_dates_pool:
        daily = {}
        for t, e in zip(all_dates_pool, all_exc_pool):
            daily.setdefault(t, []).append(e)
        ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
        if len(ds) >= 10:
            model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            pool_t = float(model.tvalues[0]); pool_p = float(model.pvalues[0])
        else:
            pool_t = float("nan"); pool_p = float("nan")
        pool_mean = float(np.mean(ds))
    else:
        pool_t = float("nan"); pool_p = float("nan")
        pool_mean = float("nan")

    out["pool"] = {
        "mean_exc_w": pool_mean,
        "t_stat": pool_t,
        "p_val": pool_p,
        "mean_stress_w": float(np.mean(all_stress_pool)) if all_stress_pool else float("nan"),
        "n_trades": len(all_exc_pool),
    }
    return out


def cek_kriteria_R1_R2_R4(out, label):
    """Kriteria umum R1, R2, R4: excess pool > 0 dg p1sisi < 0.05;
    stress_w pool > 0; excess > 0 di >= 2/3 blok."""
    p_pool = out["pool"]["p_val"]
    exc_pool = out["pool"]["mean_exc_w"]
    p1_pool = fix_p1b(p_pool, exc_pool)
    str_pool = out["pool"]["mean_stress_w"]
    n_blk_pos = sum([out["K1"]["mean_exc_w"] > 0,
                     out["K2"]["mean_exc_w"] > 0,
                     out["K3"]["mean_exc_w"] > 0])
    fails = []
    if not (exc_pool > 0): fails.append(f"pool exc {exc_pool*100:+.3f}% ≤ 0")
    if not (p1_pool < 0.05): fails.append(f"pool p1sisi {p1_pool:.4f} ≥ 0.05")
    if not (str_pool > 0): fails.append(f"pool str_w {str_pool*100:+.3f}% ≤ 0")
    if n_blk_pos < 2: fails.append(f"hanya {n_blk_pos}/3 blok exc > 0")
    return (len(fails) == 0), fails, p1_pool


# ========================
# R1 — Penundaan 1 hari
# ========================
log("")
log("=" * 90)
log("  R1 — Entry Open(T+2), exit Close(T+21)")
log("=" * 90)
out_R1 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=2)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R1[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
r_pool = out_R1["pool"]
log(f"  pool: exc_w={r_pool['mean_exc_w']*100:+.3f}% t={r_pool['t_stat']:+.2f} "
    f"p={r_pool['p_val']:.4f} str_w={r_pool['mean_stress_w']*100:+.3f}%")
R1_pass, R1_fails, R1_p1 = cek_kriteria_R1_R2_R4(out_R1, "R1")
log(f"  p1sisi pool: {R1_p1:.4f}")
log(f"  R1: {'LOLOS' if R1_pass else 'GAGAL — ' + '; '.join(R1_fails)}")


# ========================
# R2 — Tanpa gap besar
# ========================
log("")
log("=" * 90)
log("  R2 — Buang trade dengan Open(T+1)/Close(T) - 1 >= +0.10")
log("=" * 90)
out_R2 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=1, drop_gap=True)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R2[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
r_pool = out_R2["pool"]
log(f"  pool: exc_w={r_pool['mean_exc_w']*100:+.3f}% t={r_pool['t_stat']:+.2f} "
    f"p={r_pool['p_val']:.4f} str_w={r_pool['mean_stress_w']*100:+.3f}%")
R2_pass, R2_fails, R2_p1 = cek_kriteria_R1_R2_R4(out_R2, "R2")
log(f"  p1sisi pool: {R2_p1:.4f}")
log(f"  R2: {'LOLOS' if R2_pass else 'GAGAL — ' + '; '.join(R2_fails)}")


# ========================
# R3 — Biaya 1.0% RT
# ========================
log("")
log("=" * 90)
log("  R3 — Biaya 1.0% round-trip (0.5% per sisi)")
log("=" * 90)
out_R3 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b,
                          N=20, cost_side=COST_R3_SIDE, delay=1)
n_blk_pos_R3 = 0
for blk_id in ["K1", "K2", "K3"]:
    r = out_R3[blk_id]
    log(f"  {blk_id}: return_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']}")
    if r["mean_exc_w"] > 0: n_blk_pos_R3 += 1
r_pool = out_R3["pool"]
log(f"  pool: return_w={r_pool['mean_exc_w']*100:+.3f}%")
R3_pass = (r_pool["mean_exc_w"] > 0) and (n_blk_pos_R3 >= 2)
R3_fails = []
if not (r_pool["mean_exc_w"] > 0): R3_fails.append(f"pool {r_pool['mean_exc_w']*100:+.3f}% ≤ 0")
if n_blk_pos_R3 < 2: R3_fails.append(f"hanya {n_blk_pos_R3}/3 blok > 0")
log(f"  R3: {'LOLOS' if R3_pass else 'GAGAL — ' + '; '.join(R3_fails)}")


# ========================
# R4 — Hanya 50 teratas
# ========================
log("")
log("=" * 90)
log("  R4 — Hanya 50 saham teratas (turnover 20 hari)")
log("=" * 90)
out_R4 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=1,
                          top_n_override=TOP_N_RANK_R4)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R4[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
r_pool = out_R4["pool"]
log(f"  pool: exc_w={r_pool['mean_exc_w']*100:+.3f}% t={r_pool['t_stat']:+.2f} "
    f"p={r_pool['p_val']:.4f} str_w={r_pool['mean_stress_w']*100:+.3f}%")
R4_pass, R4_fails, R4_p1 = cek_kriteria_R1_R2_R4(out_R4, "R4")
log(f"  p1sisi pool: {R4_p1:.4f}")
log(f"  R4: {'LOLOS' if R4_pass else 'GAGAL — ' + '; '.join(R4_fails)}")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK3)

print(f"✅ Blok 3/7 ditulis ({len(BLOK3)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-E untuk menulis bagian 4/7 (R5, R6, kesimpulan)")

✅ Blok 3/7 ditulis (7359 byte)
   File sekarang: 19268 byte

Lanjut ke sel P0-E untuk menulis bagian 4/7 (R5, R6, kesimpulan)


In [ ]:
# ============================================================
# P0-E — Blok 4/7: uji R5, R6, kesimpulan R1-R6
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK4 = r'''
# ========================
# R5 — Tanpa K1 (pool hanya K2+K3)
# ========================
log("")
log("=" * 90)
log("  R5 — Tanpa K1 (pool hanya K2 + K3)")
log("=" * 90)
# Jalankan 3 blok dulu, lalu drop K1
out_R5_all = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b,
                              N=20, cost_side=COST_BASE_SIDE, delay=1)
# Pool K2+K3
dates_K23 = out_R5_all["K2"]["dates"] + out_R5_all["K3"]["dates"]
exc_K23   = out_R5_all["K2"]["exc_w_list"] + out_R5_all["K3"]["exc_w_list"]
str_K23   = [out_R5_all["K2"]["stress_w"]] * len(out_R5_all["K2"]["exc_w_list"]) + \
            [out_R5_all["K3"]["stress_w"]] * len(out_R5_all["K3"]["exc_w_list"])

if dates_K23:
    daily = {}
    for t, e in zip(dates_K23, exc_K23):
        daily.setdefault(t, []).append(e)
    ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
    if len(ds) >= 10:
        model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": 20})
        r5_t = float(model.tvalues[0]); r5_p = float(model.pvalues[0])
    else:
        r5_t = float("nan"); r5_p = float("nan")
    r5_mean = float(np.mean(ds))
else:
    r5_t = r5_p = r5_mean = float("nan")

r5_str_mean = float(np.mean(str_K23)) if str_K23 else float("nan")
r5_p1 = fix_p1b(r5_p, r5_mean)

log(f"  K2: exc_w={out_R5_all['K2']['mean_exc_w']*100:+.3f}% n={out_R5_all['K2']['n_trades']}")
log(f"  K3: exc_w={out_R5_all['K3']['mean_exc_w']*100:+.3f}% n={out_R5_all['K3']['n_trades']}")
log(f"  Pool K2+K3: exc_w={r5_mean*100:+.3f}% t={r5_t:+.2f} p={r5_p:.4f} "
    f"p1sisi={r5_p1:.4f} str_w={r5_str_mean*100:+.3f}%")

R5_fails = []
if not (r5_mean > 0): R5_fails.append(f"pool exc {r5_mean*100:+.3f}% ≤ 0")
if not (r5_p1 < 0.05): R5_fails.append(f"pool p1sisi {r5_p1:.4f} ≥ 0.05")
if not (r5_str_mean > 0): R5_fails.append(f"pool str_w {r5_str_mean*100:+.3f}% ≤ 0")
R5_pass = len(R5_fails) == 0
log(f"  R5: {'LOLOS' if R5_pass else 'GAGAL — ' + '; '.join(R5_fails)}")


# ========================
# R6 — Konsentrasi
# ========================
log("")
log("=" * 90)
log("  R6 — Konsentrasi")
log("=" * 90)

# Jalankan baseline 3 blok sekali lagi untuk dapat trades detail
trades_R6 = []
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((near52h_b.index >= t1) & (near52h_b.index <= t2),
                       index=near52h_b.index)
    sig_blk = near52h_b.where(mask_t, False)
    trades, all_rets, _ = build_trades_and_excess(
        idx, op_b, cl_b, elig_b, sig_blk,
        N=20, cost_side=COST_BASE_SIDE, delay=1, drop_gap=False)
    # Compute excess_w per trade (pakai pool all_rets)
    if all_rets:
        lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
        for rec in trades:
            bm_w = float(np.mean(np.clip(rec["rets_T_base"], lo, hi)))
            rb_w = float(np.clip(rec["rb"], lo, hi))
            rec["exc_w"] = rb_w - bm_w
            rec["blk"] = blk_id
            trades_R6.append(rec)

# R6a: semester
log("  R6a — per semester kalender:")
if trades_R6:
    df_R6 = pd.DataFrame([{
        "t": r["t"], "tkr": r["tkr"], "exc_w": r["exc_w"], "blk": r["blk"]
    } for r in trades_R6])
    df_R6["semester"] = df_R6["t"].apply(
        lambda d: f"{d.year}-S{1 if d.month <= 6 else 2}")
    df_R6["kontribusi"] = df_R6["exc_w"] * 1  # per trade
    sem_agg = df_R6.groupby("semester").agg(
        n=("exc_w", "size"),
        mean_exc_w=("exc_w", "mean"),
        sum_exc_w=("exc_w", "sum"),
    ).reset_index()
    total_sum = df_R6["exc_w"].sum()
    sem_agg["pct_of_total"] = sem_agg["sum_exc_w"] / total_sum * 100 if total_sum != 0 else 0

    log(f"    {'semester':10s} {'n':>6s} {'mean_exc_w':>14s} {'sum_exc_w':>14s} {'%total':>10s}")
    for _, row in sem_agg.iterrows():
        log(f"    {row['semester']:10s} {int(row['n']):>6d} "
            f"{row['mean_exc_w']*100:>+13.3f}% {row['sum_exc_w']:>+13.2f} "
            f"{row['pct_of_total']:>9.2f}%")

    n_sem = len(sem_agg)
    n_pos = (sem_agg["mean_exc_w"] > 0).sum()
    pct_pos = 100 * n_pos / n_sem if n_sem else 0
    max_share = sem_agg["pct_of_total"].abs().max()

    R6a_fails = []
    if pct_pos < 60: R6a_fails.append(f"hanya {pct_pos:.1f}% semester positif (min 60%)")
    if max_share > 40: R6a_fails.append(f"semester terbesar {max_share:.1f}% (max 40%)")
    R6a_pass = len(R6a_fails) == 0
    log(f"    Semester positif: {n_pos}/{n_sem} ({pct_pos:.1f}%)")
    log(f"    Semester terbesar: {max_share:.2f}% dari total")
    log(f"    R6a: {'LOLOS' if R6a_pass else 'GAGAL — ' + '; '.join(R6a_fails)}")

    # R6b: per blok — 5 saham teratas ≤ 25%
    log("")
    log("  R6b — per blok: 5 saham kontribusi teratas:")
    R6b_fails = []
    for blk_id in ["K1", "K2", "K3"]:
        sub = df_R6[df_R6["blk"] == blk_id]
        if sub.empty: continue
        tkr_agg = sub.groupby("tkr")["exc_w"].sum().sort_values(ascending=False)
        total_blk = sub["exc_w"].sum()
        top5_sum = tkr_agg.head(5).sum()
        share = 100 * top5_sum / total_blk if total_blk != 0 else 0
        log(f"    {blk_id}: total Σexc={total_blk:+.2f}  top5 Σexc={top5_sum:+.2f}  "
            f"share={share:.2f}%  saham={list(tkr_agg.head(5).index)}")
        if abs(share) > 25:
            R6b_fails.append(f"{blk_id} top5 {share:.1f}% > 25%")
    R6b_pass = len(R6b_fails) == 0
    log(f"    R6b: {'LOLOS' if R6b_pass else 'GAGAL — ' + '; '.join(R6b_fails)}")

    R6_pass = R6a_pass and R6b_pass
else:
    log("    (tidak ada trade)")
    R6_pass = False
    R6a_pass = False
    R6b_pass = False
    R6a_fails = ["tidak ada trade"]
    R6b_fails = []

log("")
log(f"  R6: {'LOLOS' if R6_pass else 'GAGAL'}")


# ========================
# KESIMPULAN R1-R6
# ========================
log("")
log("=" * 90)
log("  KESIMPULAN R1-R6")
log("=" * 90)
results_R = {"R1": R1_pass, "R2": R2_pass, "R3": R3_pass,
             "R4": R4_pass, "R5": R5_pass, "R6": R6_pass}
for r_name, r_pass in results_R.items():
    log(f"  {r_name}: {'LOLOS' if r_pass else 'GAGAL'}")
all_R_pass = all(results_R.values())
log("")
log(f"  SEMUA R1-R6: {'LOLOS' if all_R_pass else 'ADA YANG GAGAL'}")

if not all_R_pass:
    log("")
    log("  STOP: NEAR52H dicoret sebagai tradable. Tidak lanjut ke R7.")
    log("  Lanjutkan P1 dan P2 (deskriptif).")
    # Set flag global
    R_ALL_PASS = False
else:
    log("")
    log("  R1-R6 LOLOS SEMUA. Siap lanjut R7 (VALIDATE sekali).")
    R_ALL_PASS = True

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK4)

print(f"✅ Blok 4/7 ditulis ({len(BLOK4)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-F untuk menulis bagian 5/7 (R7 VALIDATE)")

✅ Blok 4/7 ditulis (6496 byte)
   File sekarang: 25794 byte

Lanjut ke sel P0-F untuk menulis bagian 5/7 (R7 VALIDATE)


In [ ]:
# ============================================================
# P0-F — Blok 5/7: R7 VALIDATE (hanya jika R1-R6 lolos)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK5 = r'''
# ============================================================
# R7 — VALIDATE (hanya jika R1-R6 lolos)
# ============================================================
R7_results = {"ran": False, "pass": False}

if R_ALL_PASS:
    log("")
    log("=" * 90)
    log("  R7 — VALIDATE (buka sekali)")
    log("=" * 90)

    # 1. Unduh ulang satu kali per ticker
    try:
        import yfinance as yf
    except ImportError:
        os.system("pip install -q yfinance")
        import yfinance as yf

    new_data_sealed = pd.read_pickle(f"{BASE}/data_sealed.pkl")
    idx_tickers = new_data_sealed["idx_tickers"]

    log(f"  Unduh {len(idx_tickers)} ticker 2005-01-01 → sekarang")

    DWN_OK = 0; DWN_FAIL = 0
    op_v2, hi_v2, lo_v2, cl_v2, vo_v2 = {}, {}, {}, {}, {}

    if os.path.exists(PROGRESS_V2_PATH):
        try:
            prog = pd.read_pickle(PROGRESS_V2_PATH)
            op_v2 = prog.get("op", {}); hi_v2 = prog.get("hi", {})
            lo_v2 = prog.get("lo", {}); cl_v2 = prog.get("cl", {})
            vo_v2 = prog.get("vo", {})
            log(f"  Resume: {len(cl_v2)} ticker")
        except Exception:
            pass

    t_start_dl = time.time()
    for i, tkr in enumerate(idx_tickers):
        if tkr in cl_v2 and not cl_v2[tkr].empty:
            DWN_OK += 1; continue
        df = None
        for attempt in range(3):
            try:
                df = yf.download(tkr, start="2005-01-01", end=None,
                                 interval="1d", auto_adjust=True,
                                 progress=False, threads=False)
                if df is not None and not df.empty: break
            except Exception:
                time.sleep(1.5); continue
        if df is None or df.empty:
            DWN_FAIL += 1; continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        op_v2[tkr] = df["Open"]; hi_v2[tkr] = df["High"]
        lo_v2[tkr] = df["Low"];  cl_v2[tkr] = df["Close"]
        vo_v2[tkr] = df["Volume"]
        DWN_OK += 1
        if (i + 1) % 50 == 0:
            elapsed = time.time() - t_start_dl
            log(f"    [{i+1}/{len(idx_tickers)}] OK={DWN_OK} FAIL={DWN_FAIL} ({elapsed:.0f}s)")
            pd.to_pickle({"op": op_v2, "hi": hi_v2, "lo": lo_v2,
                          "cl": cl_v2, "vo": vo_v2}, PROGRESS_V2_PATH)

    pd.to_pickle({"op": op_v2, "hi": hi_v2, "lo": lo_v2,
                  "cl": cl_v2, "vo": vo_v2}, PROGRESS_V2_PATH)
    log(f"  Unduh selesai: OK={DWN_OK} FAIL={DWN_FAIL}")

    op_v2_df = pd.DataFrame(op_v2).sort_index()
    hi_v2_df = pd.DataFrame(hi_v2).sort_index()
    lo_v2_df = pd.DataFrame(lo_v2).sort_index()
    cl_v2_df = pd.DataFrame(cl_v2).sort_index()
    vo_v2_df = pd.DataFrame(vo_v2).sort_index()

    # Verifikasi ≤ 2026-06-05 cocok dengan data_long_sealed
    common_tkrs_v2 = sorted(set(cl_v2_df.columns) & set(cl.columns))
    ret_v2 = cl_v2_df[common_tkrs_v2].pct_change()
    ret_seal = cl[common_tkrs_v2].pct_change()
    common_dates_v2 = ret_v2.index.intersection(ret_seal.index)
    common_dates_v2 = common_dates_v2[common_dates_v2 <= CUTOFF_SEALED]

    total_cells = 0; match_cells = 0
    for tkr in common_tkrs_v2:
        a = ret_v2.loc[common_dates_v2, tkr]
        b = ret_seal.loc[common_dates_v2, tkr]
        valid = a.notna() & b.notna()
        total_cells += int(valid.sum())
        match_cells += int((valid & ((a - b).abs() < 0.001)).sum())
    pct_match = 100 * match_cells / total_cells if total_cells else 0
    log(f"  Verifikasi ≤2026-06-05: {pct_match:.2f}% ({match_cells}/{total_cells})")

    if pct_match < 99:
        log("  ❌ STOP R7: verifikasi data gagal.")
        R7_results = {"ran": True, "pass": False, "reason": "verifikasi data < 99%"}
    else:
        # 2. Terapkan filter layak + pembersihan
        # (asumsi: data_v2 belum dibersihkan dari event split — tapi untuk R7
        # kita pakai data apa adanya, karena hanya menguji arah)
        # Untuk aman: pakai pembersihan sederhana — drop event |ret| > 40%
        # (TIDAK ditandai NaN, hanya trade yang mungkin terdampak)

        # Filter layak
        turnover_v2 = cl_v2_df * vo_v2_df
        toma20_v2 = turnover_v2.rolling(20, min_periods=20).mean()
        mask_a_v2 = (cl_v2_df >= MIN_PRICE) & cl_v2_df.notna()
        mask_b_v2 = (toma20_v2 >= MIN_TURNOVER_T3)
        ranked_v2 = toma20_v2.where(mask_a_v2 & mask_b_v2).rank(axis=1, ascending=False)
        mask_c_v2 = (ranked_v2 <= TOP_N_RANK_T3)
        elig_v2 = mask_a_v2 & mask_b_v2 & mask_c_v2

        # Sinyal NEAR52H
        high_52w_v2 = hi_v2_df.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
        near52h_v2 = (cl_v2_df >= NEAR52H_THRESHOLD * high_52w_v2) & elig_v2

        # 3. Filter sinyal T >= 2026-06-08, exit ≤ tanggal terakhir
        t_min = pd.Timestamp("2026-06-08")
        t_max_data = cl_v2_df.index.max()
        idx_v2 = cl_v2_df.index

        trades_v2 = []
        all_rets_v2 = []
        dates_v2 = []
        for t in near52h_v2.index:
            if t < t_min: continue
            try: i = idx_v2.get_loc(t)
            except KeyError: continue
            if i + 20 >= len(idx_v2): continue
            t_ent = idx_v2[i+1]; t_exit = idx_v2[i+20]
            if t_exit > t_max_data: continue
            # Kumpulkan semua return saham layak
            elig_tkrs = elig_v2.columns[elig_v2.loc[t]]
            rets_T = []
            for tk2 in elig_tkrs:
                o2 = op_v2_df.at[t_ent, tk2]; c2 = cl_v2_df.at[t_exit, tk2]
                if pd.isna(o2) or pd.isna(c2): continue
                rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
            if not rets_T: continue
            all_rets_v2.extend(rets_T)
            # Trade sinyal
            for tkr in near52h_v2.columns:
                if not bool(near52h_v2.at[t, tkr]): continue
                o_tr = op_v2_df.at[t_ent, tkr]; c_tr = cl_v2_df.at[t_exit, tkr]
                if pd.isna(o_tr) or pd.isna(c_tr): continue
                rb = (c_tr*(1-COST_BASE_SIDE) - o_tr*(1+COST_BASE_SIDE)) / (o_tr*(1+COST_BASE_SIDE))
                trades_v2.append({"t": t, "tkr": tkr, "rb": rb, "rets_T_base": rets_T})
                dates_v2.append(t)

        n_tr_v2 = len(trades_v2)
        n_dates_v2 = len(set(dates_v2))
        log(f"  Sinyal 2026-06-08+: n_trades={n_tr_v2}, n_dates={n_dates_v2}")

        if n_tr_v2 > 0 and all_rets_v2:
            lo_v2, hi_v2q = np.percentile(all_rets_v2, 5), np.percentile(all_rets_v2, 95)
            exc_v2 = []
            for r in trades_v2:
                bm_w = float(np.mean(np.clip(r["rets_T_base"], lo_v2, hi_v2q)))
                rb_w = float(np.clip(r["rb"], lo_v2, hi_v2q))
                exc_v2.append(rb_w - bm_w)
            exc_v2 = np.array(exc_v2)
            mean_exc_v2 = float(np.mean(exc_v2))
            # Stress
            trades_v2_s = []
            all_rets_v2_s = []
            for t in near52h_v2.index:
                if t < t_min: continue
                try: i = idx_v2.get_loc(t)
                except KeyError: continue
                if i + 20 >= len(idx_v2): continue
                t_ent = idx_v2[i+1]; t_exit = idx_v2[i+20]
                if t_exit > t_max_data: continue
                elig_tkrs = elig_v2.columns[elig_v2.loc[t]]
                rets_T_s = []
                for tk2 in elig_tkrs:
                    o2 = op_v2_df.at[t_ent, tk2]; c2 = cl_v2_df.at[t_exit, tk2]
                    if pd.isna(o2) or pd.isna(c2): continue
                    rets_T_s.append((c2*(1-COST_STRESS_SIDE) - o2*(1+COST_STRESS_SIDE)) / (o2*(1+COST_STRESS_SIDE)))
                if not rets_T_s: continue
                all_rets_v2_s.extend(rets_T_s)
                for tkr in near52h_v2.columns:
                    if not bool(near52h_v2.at[t, tkr]): continue
                    o_tr = op_v2_df.at[t_ent, tkr]; c_tr = cl_v2_df.at[t_exit, tkr]
                    if pd.isna(o_tr) or pd.isna(c_tr): continue
                    rb = (c_tr*(1-COST_STRESS_SIDE) - o_tr*(1+COST_STRESS_SIDE)) / (o_tr*(1+COST_STRESS_SIDE))
                    trades_v2_s.append({"t": t, "tkr": tkr, "rb": rb, "rets_T_base": rets_T_s})
            if all_rets_v2_s:
                lo_s, hi_s = np.percentile(all_rets_v2_s, 5), np.percentile(all_rets_v2_s, 95)
                exc_v2_s = []
                for r in trades_v2_s:
                    bm_w = float(np.mean(np.clip(r["rets_T_base"], lo_s, hi_s)))
                    rb_w = float(np.clip(r["rb"], lo_s, hi_s))
                    exc_v2_s.append(rb_w - bm_w)
                mean_str_v2 = float(np.mean(exc_v2_s))
            else:
                mean_str_v2 = float("nan")

            log(f"  Mean exc_w: {mean_exc_v2*100:+.4f}%")
            log(f"  Mean stress_w: {mean_str_v2*100:+.4f}%")
            R7_pass = (mean_exc_v2 > 0) and (mean_str_v2 > 0)
            log(f"  R7: {'LOLOS' if R7_pass else 'GAGAL'}")
            R7_results = {"ran": True, "pass": R7_pass,
                          "n_trades": n_tr_v2, "n_dates": n_dates_v2,
                          "mean_exc_w": mean_exc_v2, "mean_stress_w": mean_str_v2}
        else:
            log("  Tidak ada trade di window R7.")
            R7_results = {"ran": True, "pass": False, "reason": "tidak ada trade"}
else:
    log("")
    log("  R7 TIDAK dijalankan — R1-R6 tidak semua lolos.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK5)

print(f"✅ Blok 5/7 ditulis ({len(BLOK5)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-G untuk menulis bagian 6/7 (P1, P2)")

✅ Blok 5/7 ditulis (9380 byte)
   File sekarang: 35194 byte

Lanjut ke sel P0-G untuk menulis bagian 6/7 (P1, P2)


In [ ]:
# ============================================================
# P0-G — Blok 6/7: P1 (BRK/MOM terkoreksi) + P2 (pelengkap Tahap 3)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK6 = r'''
# ============================================================
# P1 — REKALKULASI BRK-5/10 & MOM-5/10 dengan statistik terkoreksi (deskriptif)
# ============================================================
log("")
log("=" * 90)
log("  P1 — REKALKULASI DESKRIPTIF (BRK-5/10, MOM-5/10)")
log("=" * 90)

P1_results = {}

# ------ P1a: Data Tahap 1b (data_sealed.pkl) ------
log("")
log("  P1a — Data Tahap 1b (data_sealed.pkl, periode A/B, N=5,10)")
sealed = pd.read_pickle(f"{BASE}/data_sealed.pkl")
op_1b = sealed["price_open"].copy().loc[sealed["price_open"].index <= CUTOFF_NEW]
hi_1b = sealed["price_high"].copy().loc[sealed["price_high"].index <= CUTOFF_NEW]
lo_1b = sealed["price_low"].copy().loc[sealed["price_low"].index <= CUTOFF_NEW]
cl_1b = sealed["price_close"].copy().loc[sealed["price_close"].index <= CUTOFF_NEW]
vo_1b = sealed["price_volume"].copy().loc[sealed["price_volume"].index <= CUTOFF_NEW]

# Pembersihan
ev_1b = pd.read_csv(f"{BASE}/data_events.csv", parse_dates=["t0"])
idx_f = cl_1b.index
for _, ev in ev_1b.iterrows():
    tkr_e, t0 = ev["ticker"], ev["t0"]
    if tkr_e not in cl_1b.columns: continue
    try: i0 = idx_f.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_f)-1, i0+WINDOW_FWD)+1):
        d = idx_f[j]
        op_1b.at[d, tkr_e] = np.nan; hi_1b.at[d, tkr_e] = np.nan
        lo_1b.at[d, tkr_e] = np.nan; cl_1b.at[d, tkr_e] = np.nan; vo_1b.at[d, tkr_e] = np.nan

# Filter layak (definisi Tahap 1b: Close >= 200, turnover >= 1 M)
turnover_1b = cl_1b * vo_1b
toma20_1b = turnover_1b.rolling(20, min_periods=20).mean()
elig_1b = (cl_1b >= 200.0) & (toma20_1b >= 1_000_000_000.0)

# Fitur
roc3 = cl_1b / cl_1b.shift(3) - 1.0
roc3_rank = roc3.where(elig_1b).rank(axis=1, pct=True)
mom_sig = (roc3_rank >= 0.80) & roc3.notna() & elig_1b
hp10 = hi_1b.shift(1).rolling(10, min_periods=10).max()
brk_sig = (cl_1b > hp10) & elig_1b

# Buang warm-up 25
idx_1b = idx_f[25:]
op_1b2 = op_1b.loc[idx_1b]; cl_1b2 = cl_1b.loc[idx_1b]; hi_1b2 = hi_1b.loc[idx_1b]
elig_1b2 = elig_1b.loc[idx_1b]; mom_1b = mom_sig.loc[idx_1b]; brk_1b = brk_sig.loc[idx_1b]

tA_end = pd.Timestamp("2025-05-15"); tB_start = pd.Timestamp("2025-05-16")

P1a_targets = {
    ("BRK", 5, "A"): {"exc_lama": +0.00412, "exc_baru": None},
    ("BRK", 5, "B"): {"exc_lama": +0.01159, "exc_baru": None},
    ("BRK", 10, "A"): {"exc_lama": +0.00650, "exc_baru": None},
    ("BRK", 10, "B"): {"exc_lama": +0.00680, "exc_baru": None},
    ("MOM", 5, "A"): {"exc_lama": -0.00115, "exc_baru": None},
    ("MOM", 5, "B"): {"exc_lama": +0.00059, "exc_baru": None},
    ("MOM", 10, "A"): {"exc_lama": -0.00044, "exc_baru": None},
    ("MOM", 10, "B"): {"exc_lama": +0.00184, "exc_baru": None},
}

# Jalankan 8 uji dengan statistik LAMA + BARU
log(f"  {'Sinyal':8s} {'N':>2s} {'Per':>4s} {'n':>6s} {'exc_lama':>12s} {'exc_baru':>12s} {'p_lama':>9s} {'p_baru':>9s}")
log("-" * 90)

def hitung_trade_1b(sig_df, N, t_start, t_end, cost=COST_BASE_SIDE):
    trades = []; all_rets = []; dates = []
    for t in sig_df.index:
        if t < t_start or t > t_end: continue
        try: i = idx_1b.get_loc(t)
        except KeyError: continue
        if i + N >= len(idx_1b): continue
        t_ent = idx_1b[i+1]; t_exit = idx_1b[i+N]
        elig_tkrs = elig_1b2.columns[elig_1b2.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs:
            o2 = op_1b2.at[t_ent, tk2]; c2 = cl_1b2.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-cost) - o2*(1+cost)) / (o2*(1+cost)))
        if not rets_T: continue
        all_rets.extend(rets_T)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_1b2.at[t_ent, tkr]; c_tr = cl_1b2.at[t_exit, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            rb = (c_tr*(1-cost) - o_tr*(1+cost)) / (o_tr*(1+cost))
            trades.append({"t": t, "rb": rb, "rets_T_base": rets_T})
            dates.append(t)
    return trades, all_rets, dates

for sig_name, sig_df in [("BRK", brk_1b), ("MOM", mom_1b)]:
    for N in [5, 10]:
        for per, t1, t2 in [("A", idx_1b[0], tA_end), ("B", tB_start, idx_1b[-1])]:
            trades, all_rets, dates = hitung_trade_1b(sig_df, N, t1, t2)
            if not trades or not all_rets:
                log(f"  {sig_name:8s} {N:>2d} {per:>4s} (tidak ada trade)")
                continue
            lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
            exc_w_new = []
            exc_lama = []
            for r in trades:
                bm_w = float(np.mean(np.clip(r["rets_T_base"], lo, hi)))
                rb_w = float(np.clip(r["rb"], lo, hi))
                exc_w_new.append(rb_w - bm_w)
                bm_raw = float(np.mean(r["rets_T_base"]))
                exc_lama.append(r["rb"] - bm_raw)
            exc_w_new_arr = np.array(exc_w_new)
            exc_lama_w = winsorize(np.array(exc_lama))
            mean_new = float(np.mean(exc_w_new_arr))
            mean_lama = float(np.mean(exc_lama_w))
            # HAC
            daily = {}
            for t, e in zip(dates, exc_w_new_arr):
                daily.setdefault(t, []).append(e)
            ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())]) if daily else np.array([])
            if len(ds) >= 10:
                m = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
                p_new = float(m.pvalues[0])
            else: p_new = float("nan")
            # p_lama (target dari LAPORAN)
            p_lama = P1a_targets.get((sig_name, N, per), {}).get("exc_lama", float("nan"))
            log(f"  {sig_name:8s} {N:>2d} {per:>4s} {len(trades):>6d} "
                f"{mean_lama*100:>+11.4f}% {mean_new*100:>+11.4f}% "
                f"{'—':>9s} {p_new:>9.4f}")
            P1_results[(sig_name, N, per)] = {"n": len(trades), "exc_lama": mean_lama, "exc_baru": mean_new}

# ------ P1b: Data Tahap 2 (data_old_sealed.pkl, C1/C2, N=5,10) ------
log("")
log("  P1b — Data Tahap 2 (data_old_sealed.pkl, C1/C2, N=5,10)")
old = pd.read_pickle(f"{BASE}/data_old_sealed.pkl")
op_2 = old["price_open"]; hi_2 = old["price_high"]; lo_2 = old["price_low"]
cl_2 = old["price_close"]; vo_2 = old["price_volume"]

# Pembersihan event old
ev_old = pd.read_csv(f"{BASE}/data_events_old.csv", parse_dates=["t0"])
idx_2 = cl_2.index
for _, ev in ev_old.iterrows():
    tkr_e, t0 = ev["ticker"], ev["t0"]
    if tkr_e not in cl_2.columns: continue
    try: i0 = idx_2.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_2)-1, i0+WINDOW_FWD)+1):
        d = idx_2[j]
        op_2.at[d, tkr_e] = np.nan; hi_2.at[d, tkr_e] = np.nan
        lo_2.at[d, tkr_e] = np.nan; cl_2.at[d, tkr_e] = np.nan; vo_2.at[d, tkr_e] = np.nan

turnover_2 = cl_2 * vo_2
toma20_2 = turnover_2.rolling(20, min_periods=20).mean()
elig_2 = (cl_2 >= 200.0) & (toma20_2 >= 1_000_000_000.0)

roc3_2 = cl_2 / cl_2.shift(3) - 1.0
roc3_rank_2 = roc3_2.where(elig_2).rank(axis=1, pct=True)
mom_2 = (roc3_rank_2 >= 0.80) & roc3_2.notna() & elig_2
hp10_2 = hi_2.shift(1).rolling(10, min_periods=10).max()
brk_2 = (cl_2 > hp10_2) & elig_2

idx_2_w = idx_2[25:]
op_2w = op_2.loc[idx_2_w]; cl_2w = cl_2.loc[idx_2_w]; hi_2w = hi_2.loc[idx_2_w]
elig_2w = elig_2.loc[idx_2_w]; mom_2w = mom_2.loc[idx_2_w]; brk_2w = brk_2.loc[idx_2_w]

tC1_end = pd.Timestamp("2022-09-02"); tC2_start = pd.Timestamp("2022-09-05")

P1b_targets = {
    ("BRK", 5, "C1"): +0.00325, ("BRK", 5, "C2"): +0.00147,
    ("BRK", 10, "C1"): +0.01036, ("BRK", 10, "C2"): +0.00085,
    ("MOM", 5, "C1"): -0.00292, ("MOM", 5, "C2"): -0.00277,
    ("MOM", 10, "C1"): +0.00080, ("MOM", 10, "C2"): -0.00384,
}

log(f"  {'Sinyal':8s} {'N':>2s} {'Per':>4s} {'n':>6s} {'exc_lama':>12s} {'exc_baru':>12s} {'p_baru':>9s}")
log("-" * 90)

def hitung_trade_2(sig_df, N, t_start, t_end, cost=COST_BASE_SIDE):
    trades = []; all_rets = []; dates = []
    for t in sig_df.index:
        if t < t_start or t > t_end: continue
        try: i = idx_2_w.get_loc(t)
        except KeyError: continue
        if i + N >= len(idx_2_w): continue
        t_ent = idx_2_w[i+1]; t_exit = idx_2_w[i+N]
        elig_tkrs = elig_2w.columns[elig_2w.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs:
            o2 = op_2w.at[t_ent, tk2]; c2 = cl_2w.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-cost) - o2*(1+cost)) / (o2*(1+cost)))
        if not rets_T: continue
        all_rets.extend(rets_T)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_2w.at[t_ent, tkr]; c_tr = cl_2w.at[t_exit, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            rb = (c_tr*(1-cost) - o_tr*(1+cost)) / (o_tr*(1+cost))
            trades.append({"t": t, "rb": rb, "rets_T_base": rets_T})
            dates.append(t)
    return trades, all_rets, dates

for sig_name, sig_df in [("BRK", brk_2w), ("MOM", mom_2w)]:
    for N in [5, 10]:
        for per, t1, t2 in [("C1", idx_2_w[0], tC1_end), ("C2", tC2_start, idx_2_w[-1])]:
            trades, all_rets, dates = hitung_trade_2(sig_df, N, t1, t2)
            if not trades or not all_rets:
                log(f"  {sig_name:8s} {N:>2d} {per:>4s} (tidak ada trade)")
                continue
            lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
            exc_w_new = []; exc_lama = []
            for r in trades:
                bm_w = float(np.mean(np.clip(r["rets_T_base"], lo, hi)))
                rb_w = float(np.clip(r["rb"], lo, hi))
                exc_w_new.append(rb_w - bm_w)
                bm_raw = float(np.mean(r["rets_T_base"]))
                exc_lama.append(r["rb"] - bm_raw)
            exc_w_new_arr = np.array(exc_w_new)
            mean_new = float(np.mean(exc_w_new_arr))
            mean_lama = float(np.mean(winsorize(np.array(exc_lama))))
            daily = {}
            for t, e in zip(dates, exc_w_new_arr):
                daily.setdefault(t, []).append(e)
            ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())]) if daily else np.array([])
            if len(ds) >= 10:
                m = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
                p_new = float(m.pvalues[0])
            else: p_new = float("nan")
            log(f"  {sig_name:8s} {N:>2d} {per:>4s} {len(trades):>6d} "
                f"{mean_lama*100:>+11.4f}% {mean_new*100:>+11.4f}% {p_new:>9.4f}")
            P1_results[(sig_name, N, "2" + per)] = {"n": len(trades), "exc_lama": mean_lama, "exc_baru": mean_new}


# ============================================================
# P2 — PELENGKAP LAPORAN TAHAP 3 (deskriptif)
# ============================================================
log("")
log("=" * 90)
log("  P2 — PELENGKAP LAPORAN TAHAP 3")
log("=" * 90)
log(f"  Tanggal mulai efektif: {idx[0].date()}")
log(f"  Batas K1: {idx[0].date()} → {tK1_end.date()}")
log(f"  Batas K2: {tK2_start.date()} → {tK2_end.date()}")
log(f"  Batas K3: {tK3_start.date()} → {idx[-1].date()}")
log("")

# Rata-rata saham layak per tahun
elig_eff = elig_b
n_eff_series = elig_eff.sum(axis=1)
by_year = n_eff_series.groupby(n_eff_series.index.year).mean()
log("  Saham layak per tahun:")
for yr, val in by_year.items():
    log(f"    {yr}: {val:.1f}")

# Median turnover per blok
log("")
log("  Median turnover per blok:")
to_eff = (cl_b * vo_b)
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask = (idx >= t1) & (idx <= t2)
    sub = to_eff.loc[idx[mask]].stack().dropna()
    med = float(sub.median()) if len(sub) else float("nan")
    log(f"    {blk_id}: Rp {med/1e9:.3f} M")

# LOWVOL — lengkap
log("")
log("  LOWVOL — detail per blok & pool:")
# Bangun LOWVOL signal
ret_l = cl_b.pct_change()
vol60 = ret_l.rolling(60, min_periods=60).std()
vol60_rank = vol60.where(elig_b).rank(axis=1, pct=True)
lowvol_sig = (vol60_rank <= 0.20) & vol60.notna() & elig_b

for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((lowvol_sig.index >= t1) & (lowvol_sig.index <= t2),
                       index=lowvol_sig.index)
    sig_blk = lowvol_sig.where(mask_t, False)
    tr, ar, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                         N=20, cost_side=COST_BASE_SIDE)
    res = compute_excess_w(tr, ar, cost_side=COST_BASE_SIDE)
    tr_s, ar_s, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                             N=20, cost_side=COST_STRESS_SIDE)
    res_s = compute_excess_w(tr_s, ar_s, cost_side=COST_STRESS_SIDE)
    log(f"    {blk_id}: n={len(tr)} exc_w={res['mean_exc_w']*100:+.3f}% "
        f"stress_w={res_s['mean_exc_w']*100:+.3f}%")

# Pool LOWVOL
all_tr = []; all_ar = []
for blk_id, t1, t2 in [("K1", idx[0], tK1_end), ("K2", tK2_start, tK2_end), ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((lowvol_sig.index >= t1) & (lowvol_sig.index <= t2),
                       index=lowvol_sig.index)
    sig_blk = lowvol_sig.where(mask_t, False)
    tr, ar, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                         N=20, cost_side=COST_BASE_SIDE)
    all_tr.extend(tr); all_ar.extend(ar)
res_pool = compute_excess_w(all_tr, all_ar, cost_side=COST_BASE_SIDE)
log(f"    pool: n={len(all_tr)} exc_w={res_pool['mean_exc_w']*100:+.3f}%")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK6)

print(f"✅ Blok 6/7 ditulis ({len(BLOK6)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke sel P0-H untuk menulis bagian 7/7 (laporan + run_log + verifikasi)")

✅ Blok 6/7 ditulis (13638 byte)
   File sekarang: 48856 byte

Lanjut ke sel P0-H untuk menulis bagian 7/7 (laporan + run_log + verifikasi)


In [ ]:
# ============================================================
# P0-H — Blok 7/7: tulis LAPORAN_Tahap3b.md + run_log
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK7 = r'''
# ============================================================
# TULIS LAPORAN_Tahap3b.md
# ============================================================
ts = datetime.now().isoformat(timespec="seconds")
seal_long_sha = json.load(open(f"{BASE}/data_long_manifest.json"))["sha256"] \
    if os.path.exists(f"{BASE}/data_long_manifest.json") else "n/a"

# Bangun laporan
rep = []
rep.append("TAHAP 3b - LAPORAN")
rep.append(f"Tanggal run / data terakhir    : {ts} / {cl.index.max().date()}")
rep.append(f"Nama file skrip final          : Tahap3b_Ketahanan.py")
rep.append(f"LANGKAH 0: {'COCOK' if l0_pass else 'TIDAK COCOK'}")
rep.append(f"Putaran perbaikan (maks 5)     : 0")
rep.append("")

# Ringkasan R1-R6
rep.append("UJI KETAHANAN R1-R6")
rep.append("-" * 60)

# R1
rep.append("")
rep.append("R1 — Penundaan 1 hari (entry Open(T+2), exit Close(T+21))")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R1[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R1["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R1_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R1_pass else 'GAGAL — ' + '; '.join(R1_fails)}")

# R2
rep.append("")
rep.append("R2 — Buang trade gap-up ≥ 10%")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R2[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R2["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R2_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R2_pass else 'GAGAL — ' + '; '.join(R2_fails)}")

# R3
rep.append("")
rep.append("R3 — Biaya 1.0% round-trip")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R3[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} return_w={r['mean_exc_w']*100:+.3f}%")
rp = out_R3["pool"]
rep.append(f"  pool: return_w={rp['mean_exc_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Blok dengan return > 0: {n_blk_pos_R3}/3")
rep.append(f"  Hasil: {'LOLOS' if R3_pass else 'GAGAL — ' + '; '.join(R3_fails)}")

# R4
rep.append("")
rep.append("R4 — Hanya 50 saham teratas (turnover 20 hari)")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R4[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R4["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R4_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R4_pass else 'GAGAL — ' + '; '.join(R4_fails)}")

# R5
rep.append("")
rep.append("R5 — Tanpa K1 (pool hanya K2+K3)")
rep.append(f"  K2: n={out_R5_all['K2']['n_trades']} exc_w={out_R5_all['K2']['mean_exc_w']*100:+.3f}%")
rep.append(f"  K3: n={out_R5_all['K3']['n_trades']} exc_w={out_R5_all['K3']['mean_exc_w']*100:+.3f}%")
rep.append(f"  Pool K2+K3: exc_w={r5_mean*100:+.3f}% t={r5_t:+.3f} p={r5_p:.4f} "
           f"p1sisi={r5_p1:.4f} str_w={r5_str_mean*100:+.3f}%")
rep.append(f"  Hasil: {'LOLOS' if R5_pass else 'GAGAL — ' + '; '.join(R5_fails)}")

# R6
rep.append("")
rep.append("R6 — Konsentrasi")
rep.append(f"  R6a (per semester):")
if trades_R6:
    rep.append(f"    Semester positif: {n_pos}/{n_sem} ({pct_pos:.1f}%)")
    rep.append(f"    Semester terbesar: {max_share:.2f}% dari total")
    rep.append(f"    Hasil R6a: {'LOLOS' if R6a_pass else 'GAGAL — ' + '; '.join(R6a_fails)}")
    rep.append(f"  R6b (per blok):")
    for blk_id in ["K1", "K2", "K3"]:
        sub = df_R6[df_R6["blk"] == blk_id]
        if sub.empty: continue
        tkr_agg = sub.groupby("tkr")["exc_w"].sum().sort_values(ascending=False)
        total_blk = sub["exc_w"].sum()
        top5_sum = tkr_agg.head(5).sum()
        share = 100 * top5_sum / total_blk if total_blk != 0 else 0
        top5_names = list(tkr_agg.head(5).index)
        rep.append(f"    {blk_id}: top5 share={share:.2f}% saham={top5_names}")
    rep.append(f"    Hasil R6b: {'LOLOS' if R6b_pass else 'GAGAL — ' + '; '.join(R6b_fails)}")
rep.append(f"  Hasil R6: {'LOLOS' if R6_pass else 'GAGAL'}")

rep.append("")
rep.append("KESIMPULAN R1-R6: SEMUA LOLOS" if R_ALL_PASS else "KESIMPULAN R1-R6: ADA YANG GAGAL")

# R7
rep.append("")
if R7_results["ran"]:
    if R7_results.get("pass"):
        rep.append(f"R7 (VALIDATE)  : LOLOS")
        rep.append(f"  n_trades={R7_results.get('n_trades', 'n/a')} n_dates={R7_results.get('n_dates', 'n/a')}")
        rep.append(f"  exc_w={R7_results.get('mean_exc_w', float('nan'))*100:+.4f}%")
        rep.append(f"  stress_w={R7_results.get('mean_stress_w', float('nan'))*100:+.4f}%")
    else:
        rep.append(f"R7 (VALIDATE)  : GAGAL — {R7_results.get('reason', 'lihat log')}")
else:
    rep.append(f"R7 (VALIDATE)  : TIDAK DIJALANKAN — R1-R6 tidak semua lolos")

# P1
rep.append("")
rep.append("P1 — REKALKULASI DESKRIPTIF")
rep.append("(statistik lama vs baru; tidak mengubah verdict)")
for key, res in sorted(P1_results.items()):
    sig_name, N, per = key
    rep.append(f"  {sig_name}-N{N} {per}: n={res['n']} "
               f"exc_lama={res['exc_lama']*100:+.4f}% exc_baru={res['exc_baru']*100:+.4f}%")

# P2
rep.append("")
rep.append("P2 — PELENGKAP LAPORAN TAHAP 3")
rep.append(f"  Tanggal mulai efektif: {idx[0].date()}")
rep.append(f"  Batas K1: {idx[0].date()} → {tK1_end.date()}")
rep.append(f"  Batas K2: {tK2_start.date()} → {tK2_end.date()}")
rep.append(f"  Batas K3: {tK3_start.date()} → {idx[-1].date()}")

rep.append("")
rep.append("PERUBAHAN KODE (satu baris per perubahan)")
rep.append("  -")
rep.append("")
rep.append("USULAN (tidak dijalankan)")
rep.append("  -")
rep.append("")
rep.append("PERNYATAAN: Angka kriteria / definisi sinyal / biaya / data / filter berubah? TIDAK")
rep.append("")
rep.append("CATATAN BIAS SURVIVOR")
rep.append("  Universe = saham yang terdaftar sekarang. Hasil LOLOS = BATAS ATAS.")
rep.append("  Belum boleh uang nyata: butuh paper trading 60 hari bursa.")

report_text = "\n".join(rep)
with open(REPORT_PATH, "w") as f:
    f.write(report_text)
log(f"\n  LAPORAN_Tahap3b.md: {os.path.getsize(REPORT_PATH)} byte")

# Run log
_COLUMNS = ["timestamp","phase","config_version","script_version",
            "global_seed","seal_sha256","params_sha256",
            "n_stocks","n_bars","n_signals","sharpe","max_dd","notes"]
file_exists = os.path.exists(RUN_LOG_PATH)
with open(RUN_LOG_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": ts, "phase": "BACKTEST",
        "config_version": "Gen3-Tahap3b", "script_version": "Tahap3b_Ketahanan",
        "global_seed": 42, "seal_sha256": seal_long_sha[:16] + "...",
        "params_sha256": "",
        "n_stocks": int(cl.shape[1]), "n_bars": int(cl.shape[0]),
        "n_signals": "", "sharpe": "", "max_dd": "",
        "notes": f"KOMPAS Tahap 3b — R1-R6: {'LOLOS' if R_ALL_PASS else 'ADA GAGAL'}; R7: {'LOLOS' if R7_results.get('pass', False) else ('GAGAL' if R7_results['ran'] else 'TIDAK DIJALANKAN')}",
    })
log(f"  run_log.csv diperbarui")

log("")
log("=" * 90)
log("  Tahap3b_Ketahanan.py SELESAI")
log("=" * 90)

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK7)

print(f"✅ Blok 7/7 ditulis ({len(BLOK7)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi syntax
import ast
with open(SCRIPT_PATH) as f:
    full = f.read()
try:
    ast.parse(full)
    print("✅ Syntax OK — file siap dijalankan")
except SyntaxError as e:
    print(f"❌ Syntax ERROR: {e}")
    print(f"   Line {e.lineno}: {e.text}")
print()
print("Cara menjalankan:")
print(f'  !python "{SCRIPT_PATH}"')

✅ Blok 7/7 ditulis (7425 byte)
   File sekarang: 63794 byte
✅ Syntax OK — file siap dijalankan

Cara menjalankan:
  !python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"


In [ ]:
# ============================================================
# P0-H — Blok 7/7: tulis LAPORAN_Tahap3b.md + run_log
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

BLOK7 = r'''
# ============================================================
# TULIS LAPORAN_Tahap3b.md
# ============================================================
ts = datetime.now().isoformat(timespec="seconds")
seal_long_sha = json.load(open(f"{BASE}/data_long_manifest.json"))["sha256"] \
    if os.path.exists(f"{BASE}/data_long_manifest.json") else "n/a"

# Bangun laporan
rep = []
rep.append("TAHAP 3b - LAPORAN")
rep.append(f"Tanggal run / data terakhir    : {ts} / {cl.index.max().date()}")
rep.append(f"Nama file skrip final          : Tahap3b_Ketahanan.py")
rep.append(f"LANGKAH 0: {'COCOK' if l0_pass else 'TIDAK COCOK'}")
rep.append(f"Putaran perbaikan (maks 5)     : 0")
rep.append("")

# Ringkasan R1-R6
rep.append("UJI KETAHANAN R1-R6")
rep.append("-" * 60)

# R1
rep.append("")
rep.append("R1 — Penundaan 1 hari (entry Open(T+2), exit Close(T+21))")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R1[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R1["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R1_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R1_pass else 'GAGAL — ' + '; '.join(R1_fails)}")

# R2
rep.append("")
rep.append("R2 — Buang trade gap-up ≥ 10%")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R2[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R2["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R2_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R2_pass else 'GAGAL — ' + '; '.join(R2_fails)}")

# R3
rep.append("")
rep.append("R3 — Biaya 1.0% round-trip")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R3[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} return_w={r['mean_exc_w']*100:+.3f}%")
rp = out_R3["pool"]
rep.append(f"  pool: return_w={rp['mean_exc_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Blok dengan return > 0: {n_blk_pos_R3}/3")
rep.append(f"  Hasil: {'LOLOS' if R3_pass else 'GAGAL — ' + '; '.join(R3_fails)}")

# R4
rep.append("")
rep.append("R4 — Hanya 50 saham teratas (turnover 20 hari)")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R4[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R4["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R4_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R4_pass else 'GAGAL — ' + '; '.join(R4_fails)}")

# R5
rep.append("")
rep.append("R5 — Tanpa K1 (pool hanya K2+K3)")
rep.append(f"  K2: n={out_R5_all['K2']['n_trades']} exc_w={out_R5_all['K2']['mean_exc_w']*100:+.3f}%")
rep.append(f"  K3: n={out_R5_all['K3']['n_trades']} exc_w={out_R5_all['K3']['mean_exc_w']*100:+.3f}%")
rep.append(f"  Pool K2+K3: exc_w={r5_mean*100:+.3f}% t={r5_t:+.3f} p={r5_p:.4f} "
           f"p1sisi={r5_p1:.4f} str_w={r5_str_mean*100:+.3f}%")
rep.append(f"  Hasil: {'LOLOS' if R5_pass else 'GAGAL — ' + '; '.join(R5_fails)}")

# R6
rep.append("")
rep.append("R6 — Konsentrasi")
rep.append(f"  R6a (per semester):")
if trades_R6:
    rep.append(f"    Semester positif: {n_pos}/{n_sem} ({pct_pos:.1f}%)")
    rep.append(f"    Semester terbesar: {max_share:.2f}% dari total")
    rep.append(f"    Hasil R6a: {'LOLOS' if R6a_pass else 'GAGAL — ' + '; '.join(R6a_fails)}")
    rep.append(f"  R6b (per blok):")
    for blk_id in ["K1", "K2", "K3"]:
        sub = df_R6[df_R6["blk"] == blk_id]
        if sub.empty: continue
        tkr_agg = sub.groupby("tkr")["exc_w"].sum().sort_values(ascending=False)
        total_blk = sub["exc_w"].sum()
        top5_sum = tkr_agg.head(5).sum()
        share = 100 * top5_sum / total_blk if total_blk != 0 else 0
        top5_names = list(tkr_agg.head(5).index)
        rep.append(f"    {blk_id}: top5 share={share:.2f}% saham={top5_names}")
    rep.append(f"    Hasil R6b: {'LOLOS' if R6b_pass else 'GAGAL — ' + '; '.join(R6b_fails)}")
rep.append(f"  Hasil R6: {'LOLOS' if R6_pass else 'GAGAL'}")

rep.append("")
rep.append("KESIMPULAN R1-R6: SEMUA LOLOS" if R_ALL_PASS else "KESIMPULAN R1-R6: ADA YANG GAGAL")

# R7
rep.append("")
if R7_results["ran"]:
    if R7_results.get("pass"):
        rep.append(f"R7 (VALIDATE)  : LOLOS")
        rep.append(f"  n_trades={R7_results.get('n_trades', 'n/a')} n_dates={R7_results.get('n_dates', 'n/a')}")
        rep.append(f"  exc_w={R7_results.get('mean_exc_w', float('nan'))*100:+.4f}%")
        rep.append(f"  stress_w={R7_results.get('mean_stress_w', float('nan'))*100:+.4f}%")
    else:
        rep.append(f"R7 (VALIDATE)  : GAGAL — {R7_results.get('reason', 'lihat log')}")
else:
    rep.append(f"R7 (VALIDATE)  : TIDAK DIJALANKAN — R1-R6 tidak semua lolos")

# P1
rep.append("")
rep.append("P1 — REKALKULASI DESKRIPTIF")
rep.append("(statistik lama vs baru; tidak mengubah verdict)")
for key, res in sorted(P1_results.items()):
    sig_name, N, per = key
    rep.append(f"  {sig_name}-N{N} {per}: n={res['n']} "
               f"exc_lama={res['exc_lama']*100:+.4f}% exc_baru={res['exc_baru']*100:+.4f}%")

# P2
rep.append("")
rep.append("P2 — PELENGKAP LAPORAN TAHAP 3")
rep.append(f"  Tanggal mulai efektif: {idx[0].date()}")
rep.append(f"  Batas K1: {idx[0].date()} → {tK1_end.date()}")
rep.append(f"  Batas K2: {tK2_start.date()} → {tK2_end.date()}")
rep.append(f"  Batas K3: {tK3_start.date()} → {idx[-1].date()}")

rep.append("")
rep.append("PERUBAHAN KODE (satu baris per perubahan)")
rep.append("  -")
rep.append("")
rep.append("USULAN (tidak dijalankan)")
rep.append("  -")
rep.append("")
rep.append("PERNYATAAN: Angka kriteria / definisi sinyal / biaya / data / filter berubah? TIDAK")
rep.append("")
rep.append("CATATAN BIAS SURVIVOR")
rep.append("  Universe = saham yang terdaftar sekarang. Hasil LOLOS = BATAS ATAS.")
rep.append("  Belum boleh uang nyata: butuh paper trading 60 hari bursa.")

report_text = "\n".join(rep)
with open(REPORT_PATH, "w") as f:
    f.write(report_text)
log(f"\n  LAPORAN_Tahap3b.md: {os.path.getsize(REPORT_PATH)} byte")

# Run log
_COLUMNS = ["timestamp","phase","config_version","script_version",
            "global_seed","seal_sha256","params_sha256",
            "n_stocks","n_bars","n_signals","sharpe","max_dd","notes"]
file_exists = os.path.exists(RUN_LOG_PATH)
with open(RUN_LOG_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": ts, "phase": "BACKTEST",
        "config_version": "Gen3-Tahap3b", "script_version": "Tahap3b_Ketahanan",
        "global_seed": 42, "seal_sha256": seal_long_sha[:16] + "...",
        "params_sha256": "",
        "n_stocks": int(cl.shape[1]), "n_bars": int(cl.shape[0]),
        "n_signals": "", "sharpe": "", "max_dd": "",
        "notes": f"KOMPAS Tahap 3b — R1-R6: {'LOLOS' if R_ALL_PASS else 'ADA GAGAL'}; R7: {'LOLOS' if R7_results.get('pass', False) else ('GAGAL' if R7_results['ran'] else 'TIDAK DIJALANKAN')}",
    })
log(f"  run_log.csv diperbarui")

log("")
log("=" * 90)
log("  Tahap3b_Ketahanan.py SELESAI")
log("=" * 90)

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(BLOK7)

print(f"✅ Blok 7/7 ditulis ({len(BLOK7)} byte)")
print(f"   File sekarang: {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi syntax
import ast
with open(SCRIPT_PATH) as f:
    full = f.read()
try:
    ast.parse(full)
    print("✅ Syntax OK — file siap dijalankan")
except SyntaxError as e:
    print(f"❌ Syntax ERROR: {e}")
    print(f"   Line {e.lineno}: {e.text}")
print()
print("Cara menjalankan:")
print(f'  !python "{SCRIPT_PATH}"')

✅ Blok 7/7 ditulis (7425 byte)
   File sekarang: 71263 byte
✅ Syntax OK — file siap dijalankan

Cara menjalankan:
  !python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"


In [ ]:
import os
p = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"
with open(p) as f:
    content = f.read()

print(f"Ukuran: {os.path.getsize(p)} byte")
print(f"Baris : {len(content.splitlines())}")

# Cek duplikasi
print(f"\n'TULIS LAPORAN_Tahap3b.md' muncul: {content.count('TULIS LAPORAN_Tahap3b.md')}x")
print(f"'Tahap3b_Ketahanan.py SELESAI' muncul: {content.count('Tahap3b_Ketahanan.py SELESAI')}x")
print(f"'# LOAD DATA + PREPROCESSING' muncul: {content.count('# LOAD DATA + PREPROCESSING')}x")
print(f"'UJI KETAHANAN R1-R4' muncul: {content.count('UJI KETAHANAN R1-R4')}x")
print(f"'R7 — VALIDATE' muncul: {content.count('R7 — VALIDATE')}x")
print(f"'P1 — REKALKULASI' muncul: {content.count('P1 — REKALKULASI')}x")

Ukuran: 71263 byte
Baris : 1683

'TULIS LAPORAN_Tahap3b.md' muncul: 3x
'Tahap3b_Ketahanan.py SELESAI' muncul: 3x
'# LOAD DATA + PREPROCESSING' muncul: 1x
'UJI KETAHANAN R1-R4' muncul: 2x
'R7 — VALIDATE' muncul: 2x
'P1 — REKALKULASI' muncul: 5x


In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

# Backup dulu (kalau perlu referensi)
BACKUP = f"{BASE}/_Tahap3b_BACKUP_rusak.py"
if os.path.exists(SCRIPT_PATH):
    os.rename(SCRIPT_PATH, BACKUP)
    print(f"✅ File rusak di-rename ke: {BACKUP}")

# Buat file baru kosong
with open(SCRIPT_PATH, "w") as f:
    f.write("")
print(f"✅ File baru dibuat kosong: {SCRIPT_PATH}")
print(f"   Ukuran: {os.path.getsize(SCRIPT_PATH)} byte")

✅ File rusak di-rename ke: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_Tahap3b_BACKUP_rusak.py
✅ File baru dibuat kosong: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py
   Ukuran: 0 byte


In [ ]:
# ============================================================
# Q2 — Sel 1/3: Blok 1+2 (header, konstanta, util, load, Langkah 0)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

PART1 = r'''# ============================================================
# Tahap3b_Ketahanan.py
# KOMPAS TAHAP 3b — Uji Ketahanan NEAR52H
# ============================================================
import os, sys, csv, json, random, hashlib, time
from datetime import datetime
import numpy as np
import pandas as pd
from statsmodels.regression.linear_model import OLS

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print(f"[WARN] Mount Drive: {e}")

BASE              = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
LONG_SEAL_PATH    = f"{BASE}/data_long_sealed.pkl"
LONG_SEALED_V2    = f"{BASE}/data_long_sealed_v2.pkl"
LONG_MANIFEST_V2  = f"{BASE}/data_long_manifest_v2.json"
RUN_LOG_PATH      = f"{BASE}/run_log.csv"
REPORT_PATH       = f"{BASE}/LAPORAN_Tahap3b.md"
PROGRESS_V2_PATH  = f"{BASE}/_download_long_v2_progress.pkl"

CUTOFF_SEALED     = pd.Timestamp("2026-06-05")
CUTOFF_NEW        = pd.Timestamp("2026-06-07")

MIN_PRICE         = 200.0
MIN_TURNOVER_T3   = 200_000_000.0
TOP_N_RANK_T3     = 100
TOP_N_RANK_R4     = 50

NEAR52H_THRESHOLD = 0.95
LOOKBACK_52W      = 252

WARMUP_BARS       = 260
WINDOW_BACK       = 3
WINDOW_FWD        = 25

COST_BASE_SIDE    = 0.0015
COST_STRESS_SIDE  = 0.0030
COST_R3_SIDE      = 0.0050
N_HORIZON         = 20
HAC_MAXLAGS       = 20

TARGET_L0 = {
    "K1": {"exc_w": +0.05367, "stress_w": +0.03535, "n": 1099},
    "K2": {"exc_w": +0.01635, "stress_w": +0.00200, "n": 2426},
    "K3": {"exc_w": +0.02174, "stress_w": +0.02485, "n": 3750},
}
TOL_L0_N  = 0.01
TOL_L0_PP = 0.0002

random.seed(42); np.random.seed(42)

def log(msg=""):
    print(msg, flush=True)

def winsorize(arr):
    if len(arr) == 0: return np.array([])
    lo, hi = np.percentile(arr, 5), np.percentile(arr, 95)
    return np.clip(arr, lo, hi)

def fix_p1b(p, exc):
    if pd.isna(p): return float("nan")
    if exc > 0:   return p / 2
    if exc < 0:   return 1 - p / 2
    return 0.5

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

log("=" * 90)
log("  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py")
log("=" * 90)
log(f"  Waktu: {datetime.now().isoformat(timespec='seconds')}")
log("")

# ============================================================
# LOAD DATA + PREPROCESSING
# ============================================================
log("=" * 90)
log("  LOAD DATA PANJANG + PREPROCESSING")
log("=" * 90)

if not os.path.exists(LONG_SEAL_PATH):
    log(f"STOP: {LONG_SEAL_PATH} tidak ada."); sys.exit(1)

data_long = pd.read_pickle(LONG_SEAL_PATH)
op = data_long["price_open"].copy()
hi = data_long["price_high"].copy()
lo = data_long["price_low"].copy()
cl = data_long["price_close"].copy()
vo = data_long["price_volume"].copy()

log(f"  Data: {cl.shape}, range {cl.index.min().date()} → {cl.index.max().date()}")

turnover = cl * vo
toma20 = turnover.rolling(20, min_periods=20).mean()
mask_a = (cl >= MIN_PRICE) & cl.notna()
mask_b = (toma20 >= MIN_TURNOVER_T3)
toma20_ranked = toma20.where(mask_a & mask_b).rank(axis=1, ascending=False)
mask_c = (toma20_ranked <= TOP_N_RANK_T3)
elig = mask_a & mask_b & mask_c

high_52w = hi.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
near52h = (cl >= NEAR52H_THRESHOLD * high_52w) & elig

idx_full = cl.index
idx = idx_full[WARMUP_BARS:]
op_b = op.loc[idx]; hi_b = hi.loc[idx]; lo_b = lo.loc[idx]
cl_b = cl.loc[idx]; vo_b = vo.loc[idx]
elig_b = elig.loc[idx]
near52h_b = near52h.loc[idx]

n_b = len(idx)
blk = n_b // 3
tK1_end = idx[blk - 1]
tK2_start = idx[blk]; tK2_end = idx[2*blk - 1]
tK3_start = idx[2*blk]

log(f"  Setelah warm-up 260: {n_b} bar")
log(f"  K1: {idx[0].date()} → {tK1_end.date()} ({blk} bar)")
log(f"  K2: {tK2_start.date()} → {tK2_end.date()} ({blk} bar)")
log(f"  K3: {tK3_start.date()} → {idx[-1].date()} ({n_b - 2*blk} bar)")
log("")

def build_trades_and_excess(idx_arr, op_arr, cl_arr, elig_arr, sig_df,
                            N=20, cost_side=COST_BASE_SIDE,
                            delay=1, drop_gap=False):
    trade_records = []
    all_rets_pool = []
    dates = []
    for t in sig_df.index:
        try: i = idx_arr.get_loc(t)
        except KeyError: continue
        if i + delay + N - 1 >= len(idx_arr): continue
        t_ent = idx_arr[i + delay]
        t_exit = idx_arr[i + delay + N - 1]
        elig_tkrs = elig_arr.columns[elig_arr.loc[t]]
        rets_T_base = []
        for tk2 in elig_tkrs:
            o2 = op_arr.at[t_ent, tk2]; c2 = cl_arr.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rb2 = (c2*(1-cost_side) - o2*(1+cost_side)) / (o2*(1+cost_side))
            rets_T_base.append(rb2)
        if not rets_T_base: continue
        all_rets_pool.extend(rets_T_base)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_arr.at[t_ent, tkr]; c_tr = cl_arr.at[t_exit, tkr]
            c_sig = cl_arr.at[t, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            if drop_gap and pd.notna(c_sig) and c_sig > 0:
                gap = o_tr / c_sig - 1
                if gap >= 0.10: continue
            rb_tr = (c_tr*(1-cost_side) - o_tr*(1+cost_side)) / (o_tr*(1+cost_side))
            trade_records.append({
                "t": t, "tkr": tkr, "rb": rb_tr,
                "t_ent": t_ent, "t_exit": t_exit,
                "rets_T_base": rets_T_base,
            })
            dates.append(t)
    return trade_records, all_rets_pool, dates

def compute_excess_w(trade_records, all_rets_pool, cost_side=COST_BASE_SIDE):
    if not all_rets_pool or not trade_records:
        return {"n_trades": 0, "mean_exc_w": float("nan"),
                "mean_exc_w_lama": float("nan"),
                "median_exc": float("nan"),
                "dates": [], "exc_w_list": []}
    lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
    exc_w_list = []
    exc_w_lama_list = []
    for rec in trade_records:
        rb = rec["rb"]
        rets_T = rec["rets_T_base"]
        bm_w = float(np.mean(np.clip(rets_T, lo, hi)))
        rb_w = float(np.clip(rb, lo, hi))
        exc_w = rb_w - bm_w
        exc_w_list.append(exc_w)
        bm_raw = float(np.mean(rets_T))
        exc_w_lama_list.append(rb - bm_raw)
    exc_w_arr = np.array(exc_w_list)
    return {
        "n_trades": len(trade_records),
        "mean_exc_w": float(np.mean(exc_w_arr)),
        "mean_exc_w_lama": float(np.mean(winsorize(np.array(exc_w_lama_list)))),
        "median_exc": float(np.median(exc_w_arr)),
        "dates": [r["t"] for r in trade_records],
        "exc_w_list": exc_w_list,
    }

def hac_p_value(dates_list, exc_w_list, N=20):
    if len(dates_list) < 10:
        return {"t_stat": float("nan"), "p_val": float("nan"), "n_dates": 0}
    daily = {}
    for t, e in zip(dates_list, exc_w_list):
        daily.setdefault(t, []).append(e)
    ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
    if len(ds) < 10:
        return {"t_stat": float("nan"), "p_val": float("nan"), "n_dates": len(ds)}
    model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
    return {"t_stat": float(model.tvalues[0]),
            "p_val":  float(model.pvalues[0]),
            "n_dates": len(ds)}

# ============================================================
# LANGKAH 0 — REPRODUKSI
# ============================================================
log("=" * 90)
log("  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)")
log("=" * 90)
log(f"  {'Blok':6s} {'hasil exc_w':>14s} {'target':>14s} {'cocok?':>8s}  "
    f"{'hasil str_w':>14s} {'target':>14s} {'cocok?':>8s}  "
    f"{'n':>6s} {'target':>6s} {'cocok?':>8s}")
log("-" * 110)

l0_pass = True
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((near52h_b.index >= t1) & (near52h_b.index <= t2),
                       index=near52h_b.index)
    sig_blk = near52h_b.where(mask_t, False)
    # BASE
    trades, all_rets, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                                   N=20, cost_side=COST_BASE_SIDE,
                                                   delay=1, drop_gap=False)
    res = compute_excess_w(trades, all_rets, cost_side=COST_BASE_SIDE)
    # STRESS
    trades_s, all_rets_s, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                                       N=20, cost_side=COST_STRESS_SIDE,
                                                       delay=1, drop_gap=False)
    res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
    exc_w = res["mean_exc_w"]
    stress_w = res_s["mean_exc_w"]
    n_tr = res["n_trades"]
    tgt = TARGET_L0[blk_id]
    ok_exc = abs(exc_w - tgt["exc_w"]) <= TOL_L0_PP
    ok_str = abs(stress_w - tgt["stress_w"]) <= TOL_L0_PP
    ok_n = abs(n_tr - tgt["n"]) / max(1, tgt["n"]) <= TOL_L0_N
    if not (ok_exc and ok_str and ok_n): l0_pass = False
    log(f"  {blk_id:6s} {exc_w*100:>+13.4f}% {tgt['exc_w']*100:>+13.4f}% "
        f"{'COCOK' if ok_exc else 'BEDA':>8s}  "
        f"{stress_w*100:>+13.4f}% {tgt['stress_w']*100:>+13.4f}% "
        f"{'COCOK' if ok_str else 'BEDA':>8s}  "
        f"{n_tr:>6d} {tgt['n']:>6d} {'COCOK' if ok_n else 'BEDA':>8s}")
log("-" * 110)
log(f"  LANGKAH 0: {'COCOK' if l0_pass else 'TIDAK COCOK'}")
if not l0_pass:
    log("  STOP: Langkah 0 tidak cocok.")
    sys.exit(2)
log("  Langkah 0 LOLOS. Lanjut ke uji ketahanan R1-R6.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(PART1)
print(f"✅ Sel 1/3 ditulis ({len(PART1)} byte)")
print(f"   File: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke Q3 (Sel 2/3: R1-R6)")

✅ Sel 1/3 ditulis (9678 byte)
   File: 9694 byte

Lanjut ke Q3 (Sel 2/3: R1-R6)


In [ ]:
# ============================================================
# Q3 — Sel 2/3: R1-R6 (uji ketahanan)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

PART2 = r'''
# ============================================================
# UJI KETAHANAN R1-R6
# ============================================================
log("")
log("=" * 90)
log("  UJI KETAHANAN R1-R6")
log("=" * 90)

def jalankan_3_blok(idx_arr, op_arr, cl_arr, elig_arr, sig_df, vo_arr,
                    N=20, cost_side=COST_BASE_SIDE, delay=1, drop_gap=False,
                    top_n_override=None):
    if top_n_override is not None:
        to20 = cl_arr * vo_arr
        to20_rank = to20.where(elig_arr).rank(axis=1, ascending=False)
        elig_use = elig_arr & (to20_rank <= top_n_override)
    else:
        elig_use = elig_arr

    out = {}
    all_dates_pool = []
    all_exc_pool = []
    all_stress_pool = []

    for blk_id, t1, t2 in [("K1", idx_arr[0], tK1_end),
                            ("K2", tK2_start, tK2_end),
                            ("K3", tK3_start, idx_arr[-1])]:
        mask_t = pd.Series((sig_df.index >= t1) & (sig_df.index <= t2),
                           index=sig_df.index)
        sig_blk = sig_df.where(mask_t, False)
        trades, all_rets, dates = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=cost_side, delay=delay, drop_gap=drop_gap)
        res = compute_excess_w(trades, all_rets, cost_side=cost_side)
        hac = hac_p_value(res["dates"], res["exc_w_list"], N=N)
        res.update(hac)
        res["n_trades"] = len(trades)
        out[blk_id] = res
        all_dates_pool.extend(res["dates"])
        all_exc_pool.extend(res["exc_w_list"])

        trades_s, all_rets_s, _ = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=COST_STRESS_SIDE, delay=delay, drop_gap=drop_gap)
        res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
        out[blk_id]["stress_w"] = res_s["mean_exc_w"]
        all_stress_pool.extend(res_s["exc_w_list"])

    if all_dates_pool:
        daily = {}
        for t, e in zip(all_dates_pool, all_exc_pool):
            daily.setdefault(t, []).append(e)
        ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
        if len(ds) >= 10:
            model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N})
            pool_t = float(model.tvalues[0]); pool_p = float(model.pvalues[0])
        else:
            pool_t = float("nan"); pool_p = float("nan")
        pool_mean = float(np.mean(ds))
    else:
        pool_t = pool_p = pool_mean = float("nan")

    out["pool"] = {
        "mean_exc_w": pool_mean,
        "t_stat": pool_t,
        "p_val": pool_p,
        "mean_stress_w": float(np.mean(all_stress_pool)) if all_stress_pool else float("nan"),
        "n_trades": len(all_exc_pool),
    }
    return out

def cek_kriteria_R1_R2_R4(out):
    p_pool = out["pool"]["p_val"]
    exc_pool = out["pool"]["mean_exc_w"]
    p1_pool = fix_p1b(p_pool, exc_pool)
    str_pool = out["pool"]["mean_stress_w"]
    n_blk_pos = sum([out["K1"]["mean_exc_w"] > 0,
                     out["K2"]["mean_exc_w"] > 0,
                     out["K3"]["mean_exc_w"] > 0])
    fails = []
    if not (exc_pool > 0): fails.append(f"pool exc {exc_pool*100:+.3f}% <= 0")
    if not (p1_pool < 0.05): fails.append(f"pool p1sisi {p1_pool:.4f} >= 0.05")
    if not (str_pool > 0): fails.append(f"pool str_w {str_pool*100:+.3f}% <= 0")
    if n_blk_pos < 2: fails.append(f"hanya {n_blk_pos}/3 blok exc > 0")
    return (len(fails) == 0), fails, p1_pool

# R1 — Penundaan 1 hari
log("")
log("=" * 90)
log("  R1 — Entry Open(T+2), exit Close(T+21)")
log("=" * 90)
out_R1 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b, vo_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=2)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R1[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
rp = out_R1["pool"]
log(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.2f} p={rp['p_val']:.4f} "
    f"str_w={rp['mean_stress_w']*100:+.3f}%")
R1_pass, R1_fails, R1_p1 = cek_kriteria_R1_R2_R4(out_R1)
log(f"  p1sisi pool: {R1_p1:.4f}")
log(f"  R1: {'LOLOS' if R1_pass else 'GAGAL — ' + '; '.join(R1_fails)}")

# R2 — Tanpa gap besar
log("")
log("=" * 90)
log("  R2 — Buang trade dengan Open(T+1)/Close(T) - 1 >= +0.10")
log("=" * 90)
out_R2 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b, vo_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=1, drop_gap=True)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R2[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
rp = out_R2["pool"]
log(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.2f} p={rp['p_val']:.4f} "
    f"str_w={rp['mean_stress_w']*100:+.3f}%")
R2_pass, R2_fails, R2_p1 = cek_kriteria_R1_R2_R4(out_R2)
log(f"  p1sisi pool: {R2_p1:.4f}")
log(f"  R2: {'LOLOS' if R2_pass else 'GAGAL — ' + '; '.join(R2_fails)}")

# R3 — Biaya 1.0% RT
log("")
log("=" * 90)
log("  R3 — Biaya 1.0% round-trip (0.5% per sisi)")
log("=" * 90)
out_R3 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b, vo_b,
                          N=20, cost_side=COST_R3_SIDE, delay=1)
n_blk_pos_R3 = 0
for blk_id in ["K1", "K2", "K3"]:
    r = out_R3[blk_id]
    log(f"  {blk_id}: return_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']}")
    if r["mean_exc_w"] > 0: n_blk_pos_R3 += 1
rp = out_R3["pool"]
log(f"  pool: return_w={rp['mean_exc_w']*100:+.3f}%")
R3_pass = (rp["mean_exc_w"] > 0) and (n_blk_pos_R3 >= 2)
R3_fails = []
if not (rp["mean_exc_w"] > 0): R3_fails.append(f"pool {rp['mean_exc_w']*100:+.3f}% <= 0")
if n_blk_pos_R3 < 2: R3_fails.append(f"hanya {n_blk_pos_R3}/3 blok > 0")
log(f"  R3: {'LOLOS' if R3_pass else 'GAGAL — ' + '; '.join(R3_fails)}")

# R4 — Hanya 50 teratas
log("")
log("=" * 90)
log("  R4 — Hanya 50 saham teratas (turnover 20 hari)")
log("=" * 90)
out_R4 = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b, vo_b,
                          N=20, cost_side=COST_BASE_SIDE, delay=1,
                          top_n_override=TOP_N_RANK_R4)
for blk_id in ["K1", "K2", "K3"]:
    r = out_R4[blk_id]
    log(f"  {blk_id}: exc_w={r['mean_exc_w']*100:+.3f}% n={r['n_trades']} str_w={r['stress_w']*100:+.3f}%")
rp = out_R4["pool"]
log(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.2f} p={rp['p_val']:.4f} "
    f"str_w={rp['mean_stress_w']*100:+.3f}%")
R4_pass, R4_fails, R4_p1 = cek_kriteria_R1_R2_R4(out_R4)
log(f"  p1sisi pool: {R4_p1:.4f}")
log(f"  R4: {'LOLOS' if R4_pass else 'GAGAL — ' + '; '.join(R4_fails)}")

# R5 — Tanpa K1
log("")
log("=" * 90)
log("  R5 — Tanpa K1 (pool hanya K2 + K3)")
log("=" * 90)
out_R5_all = jalankan_3_blok(idx, op_b, cl_b, elig_b, near52h_b, vo_b,
                              N=20, cost_side=COST_BASE_SIDE, delay=1)
dates_K23 = out_R5_all["K2"]["dates"] + out_R5_all["K3"]["dates"]
exc_K23   = out_R5_all["K2"]["exc_w_list"] + out_R5_all["K3"]["exc_w_list"]

if dates_K23:
    daily = {}
    for t, e in zip(dates_K23, exc_K23):
        daily.setdefault(t, []).append(e)
    ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
    if len(ds) >= 10:
        model = OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": 20})
        r5_t = float(model.tvalues[0]); r5_p = float(model.pvalues[0])
    else:
        r5_t = r5_p = float("nan")
    r5_mean = float(np.mean(ds))
else:
    r5_t = r5_p = r5_mean = float("nan")

# stress K23
str_vals_k2 = [out_R5_all["K2"]["stress_w"]] * max(1, len(out_R5_all["K2"]["exc_w_list"]))
str_vals_k3 = [out_R5_all["K3"]["stress_w"]] * max(1, len(out_R5_all["K3"]["exc_w_list"]))
r5_str_mean = float(np.mean(str_vals_k2 + str_vals_k3)) if str_vals_k2 else float("nan")
r5_p1 = fix_p1b(r5_p, r5_mean)

log(f"  K2: exc_w={out_R5_all['K2']['mean_exc_w']*100:+.3f}% n={out_R5_all['K2']['n_trades']}")
log(f"  K3: exc_w={out_R5_all['K3']['mean_exc_w']*100:+.3f}% n={out_R5_all['K3']['n_trades']}")
log(f"  Pool K2+K3: exc_w={r5_mean*100:+.3f}% t={r5_t:+.2f} p={r5_p:.4f} "
    f"p1sisi={r5_p1:.4f} str_w={r5_str_mean*100:+.3f}%")
R5_fails = []
if not (r5_mean > 0): R5_fails.append(f"pool exc {r5_mean*100:+.3f}% <= 0")
if not (r5_p1 < 0.05): R5_fails.append(f"pool p1sisi {r5_p1:.4f} >= 0.05")
if not (r5_str_mean > 0): R5_fails.append(f"pool str_w {r5_str_mean*100:+.3f}% <= 0")
R5_pass = len(R5_fails) == 0
log(f"  R5: {'LOLOS' if R5_pass else 'GAGAL — ' + '; '.join(R5_fails)}")

# R6 — Konsentrasi
log("")
log("=" * 90)
log("  R6 — Konsentrasi")
log("=" * 90)

trades_R6 = []
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((near52h_b.index >= t1) & (near52h_b.index <= t2),
                       index=near52h_b.index)
    sig_blk = near52h_b.where(mask_t, False)
    trades, all_rets, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                                   N=20, cost_side=COST_BASE_SIDE, delay=1)
    if all_rets:
        lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
        for rec in trades:
            bm_w = float(np.mean(np.clip(rec["rets_T_base"], lo, hi)))
            rb_w = float(np.clip(rec["rb"], lo, hi))
            rec["exc_w"] = rb_w - bm_w
            rec["blk"] = blk_id
            trades_R6.append(rec)

log("  R6a — per semester kalender:")
if trades_R6:
    df_R6 = pd.DataFrame([{
        "t": r["t"], "tkr": r["tkr"], "exc_w": r["exc_w"], "blk": r["blk"]
    } for r in trades_R6])
    df_R6["semester"] = df_R6["t"].apply(
        lambda d: f"{d.year}-S{1 if d.month <= 6 else 2}")
    sem_agg = df_R6.groupby("semester").agg(
        n=("exc_w", "size"),
        mean_exc_w=("exc_w", "mean"),
        sum_exc_w=("exc_w", "sum"),
    ).reset_index()
    total_sum = df_R6["exc_w"].sum()
    sem_agg["pct_of_total"] = sem_agg["sum_exc_w"] / total_sum * 100 if total_sum != 0 else 0

    log(f"    {'semester':10s} {'n':>6s} {'mean_exc_w':>14s} {'%total':>10s}")
    for _, row in sem_agg.iterrows():
        log(f"    {row['semester']:10s} {int(row['n']):>6d} "
            f"{row['mean_exc_w']*100:>+13.3f}% {row['pct_of_total']:>9.2f}%")

    n_sem = len(sem_agg)
    n_pos = int((sem_agg["mean_exc_w"] > 0).sum())
    pct_pos = 100 * n_pos / n_sem if n_sem else 0
    max_share = float(sem_agg["pct_of_total"].abs().max())

    R6a_fails = []
    if pct_pos < 60: R6a_fails.append(f"hanya {pct_pos:.1f}% semester positif (min 60%)")
    if max_share > 40: R6a_fails.append(f"semester terbesar {max_share:.1f}% (max 40%)")
    R6a_pass = len(R6a_fails) == 0
    log(f"    Semester positif: {n_pos}/{n_sem} ({pct_pos:.1f}%)")
    log(f"    Semester terbesar: {max_share:.2f}% dari total")
    log(f"    R6a: {'LOLOS' if R6a_pass else 'GAGAL — ' + '; '.join(R6a_fails)}")

    log("  R6b — per blok: 5 saham kontribusi teratas:")
    R6b_fails = []
    for blk_id in ["K1", "K2", "K3"]:
        sub = df_R6[df_R6["blk"] == blk_id]
        if sub.empty: continue
        tkr_agg = sub.groupby("tkr")["exc_w"].sum().sort_values(ascending=False)
        total_blk = sub["exc_w"].sum()
        top5_sum = tkr_agg.head(5).sum()
        share = 100 * top5_sum / total_blk if total_blk != 0 else 0
        log(f"    {blk_id}: top5 share={share:.2f}% saham={list(tkr_agg.head(5).index)}")
        if abs(share) > 25:
            R6b_fails.append(f"{blk_id} top5 {share:.1f}% > 25%")
    R6b_pass = len(R6b_fails) == 0
    log(f"    R6b: {'LOLOS' if R6b_pass else 'GAGAL — ' + '; '.join(R6b_fails)}")
    R6_pass = R6a_pass and R6b_pass
else:
    log("    (tidak ada trade)")
    R6_pass = R6a_pass = R6b_pass = False
    R6a_fails = R6b_fails = ["tidak ada trade"]
    n_pos = n_sem = 0; pct_pos = 0; max_share = 0

log(f"  R6: {'LOLOS' if R6_pass else 'GAGAL'}")

# KESIMPULAN R1-R6
log("")
log("=" * 90)
log("  KESIMPULAN R1-R6")
log("=" * 90)
results_R = {"R1": R1_pass, "R2": R2_pass, "R3": R3_pass,
             "R4": R4_pass, "R5": R5_pass, "R6": R6_pass}
for r_name, r_pass in results_R.items():
    log(f"  {r_name}: {'LOLOS' if r_pass else 'GAGAL'}")
R_ALL_PASS = all(results_R.values())
log("")
log(f"  SEMUA R1-R6: {'LOLOS' if R_ALL_PASS else 'ADA YANG GAGAL'}")
if not R_ALL_PASS:
    log("  STOP: NEAR52H dicoret sebagai tradable. Tidak lanjut ke R7.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(PART2)
print(f"✅ Sel 2/3 ditulis ({len(PART2)} byte)")
print(f"   File: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke Q4 (Sel 3/3: R7 + P1 + P2 + laporan)")

✅ Sel 2/3 ditulis (12377 byte)
   File: 22113 byte

Lanjut ke Q4 (Sel 3/3: R7 + P1 + P2 + laporan)


In [ ]:
# ============================================================
# Q4-A — R7 VALIDATE (hanya jika R1-R6 lolos)
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

PART3A = r'''
# ============================================================
# R7 — VALIDATE (hanya jika R1-R6 lolos)
# ============================================================
R7_results = {"ran": False, "pass": False}

if R_ALL_PASS:
    log("")
    log("=" * 90)
    log("  R7 — VALIDATE (buka sekali)")
    log("=" * 90)

    try:
        import yfinance as yf
    except ImportError:
        os.system("pip install -q yfinance")
        import yfinance as yf

    new_data_sealed = pd.read_pickle(f"{BASE}/data_sealed.pkl")
    idx_tickers = new_data_sealed["idx_tickers"]

    log(f"  Unduh {len(idx_tickers)} ticker 2005-01-01 → sekarang")

    DWN_OK = 0; DWN_FAIL = 0
    op_v2, hi_v2, lo_v2, cl_v2, vo_v2 = {}, {}, {}, {}, {}

    if os.path.exists(PROGRESS_V2_PATH):
        try:
            prog = pd.read_pickle(PROGRESS_V2_PATH)
            op_v2 = prog.get("op", {}); hi_v2 = prog.get("hi", {})
            lo_v2 = prog.get("lo", {}); cl_v2 = prog.get("cl", {})
            vo_v2 = prog.get("vo", {})
            log(f"  Resume: {len(cl_v2)} ticker")
        except Exception:
            pass

    t_start_dl = time.time()
    for i, tkr in enumerate(idx_tickers):
        if tkr in cl_v2 and not cl_v2[tkr].empty:
            DWN_OK += 1; continue
        df = None
        for attempt in range(3):
            try:
                df = yf.download(tkr, start="2005-01-01", end=None,
                                 interval="1d", auto_adjust=True,
                                 progress=False, threads=False)
                if df is not None and not df.empty: break
            except Exception:
                time.sleep(1.5); continue
        if df is None or df.empty:
            DWN_FAIL += 1; continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        op_v2[tkr] = df["Open"]; hi_v2[tkr] = df["High"]
        lo_v2[tkr] = df["Low"];  cl_v2[tkr] = df["Close"]
        vo_v2[tkr] = df["Volume"]
        DWN_OK += 1
        if (i + 1) % 50 == 0:
            elapsed = time.time() - t_start_dl
            log(f"    [{i+1}/{len(idx_tickers)}] OK={DWN_OK} FAIL={DWN_FAIL} ({elapsed:.0f}s)")
            pd.to_pickle({"op": op_v2, "hi": hi_v2, "lo": lo_v2,
                          "cl": cl_v2, "vo": vo_v2}, PROGRESS_V2_PATH)

    pd.to_pickle({"op": op_v2, "hi": hi_v2, "lo": lo_v2,
                  "cl": cl_v2, "vo": vo_v2}, PROGRESS_V2_PATH)
    log(f"  Unduh selesai: OK={DWN_OK} FAIL={DWN_FAIL}")

    op_v2_df = pd.DataFrame(op_v2).sort_index()
    hi_v2_df = pd.DataFrame(hi_v2).sort_index()
    lo_v2_df = pd.DataFrame(lo_v2).sort_index()
    cl_v2_df = pd.DataFrame(cl_v2).sort_index()
    vo_v2_df = pd.DataFrame(vo_v2).sort_index()

    # Verifikasi ≤ 2026-06-05
    common_tkrs_v2 = sorted(set(cl_v2_df.columns) & set(cl.columns))
    ret_v2 = cl_v2_df[common_tkrs_v2].pct_change()
    ret_seal = cl[common_tkrs_v2].pct_change()
    common_dates_v2 = ret_v2.index.intersection(ret_seal.index)
    common_dates_v2 = common_dates_v2[common_dates_v2 <= CUTOFF_SEALED]

    total_cells = 0; match_cells = 0
    for tkr in common_tkrs_v2:
        a = ret_v2.loc[common_dates_v2, tkr]
        b = ret_seal.loc[common_dates_v2, tkr]
        valid = a.notna() & b.notna()
        total_cells += int(valid.sum())
        match_cells += int((valid & ((a - b).abs() < 0.001)).sum())
    pct_match = 100 * match_cells / total_cells if total_cells else 0
    log(f"  Verifikasi ≤2026-06-05: {pct_match:.2f}% ({match_cells}/{total_cells})")

    if pct_match < 99:
        log("  STOP R7: verifikasi data gagal.")
        R7_results = {"ran": True, "pass": False, "reason": "verifikasi data < 99%"}
    else:
        # Filter layak + sinyal
        turnover_v2 = cl_v2_df * vo_v2_df
        toma20_v2 = turnover_v2.rolling(20, min_periods=20).mean()
        mask_a_v2 = (cl_v2_df >= MIN_PRICE) & cl_v2_df.notna()
        mask_b_v2 = (toma20_v2 >= MIN_TURNOVER_T3)
        ranked_v2 = toma20_v2.where(mask_a_v2 & mask_b_v2).rank(axis=1, ascending=False)
        mask_c_v2 = (ranked_v2 <= TOP_N_RANK_T3)
        elig_v2 = mask_a_v2 & mask_b_v2 & mask_c_v2

        high_52w_v2 = hi_v2_df.rolling(LOOKBACK_52W, min_periods=LOOKBACK_52W).max()
        near52h_v2 = (cl_v2_df >= NEAR52H_THRESHOLD * high_52w_v2) & elig_v2

        t_min = pd.Timestamp("2026-06-08")
        t_max_data = cl_v2_df.index.max()
        idx_v2 = cl_v2_df.index

        trades_v2 = []
        all_rets_v2 = []
        dates_v2 = []
        for t in near52h_v2.index:
            if t < t_min: continue
            try: i = idx_v2.get_loc(t)
            except KeyError: continue
            if i + 20 >= len(idx_v2): continue
            t_ent = idx_v2[i+1]; t_exit = idx_v2[i+20]
            if t_exit > t_max_data: continue
            elig_tkrs = elig_v2.columns[elig_v2.loc[t]]
            rets_T = []
            for tk2 in elig_tkrs:
                o2 = op_v2_df.at[t_ent, tk2]; c2 = cl_v2_df.at[t_exit, tk2]
                if pd.isna(o2) or pd.isna(c2): continue
                rets_T.append((c2*(1-COST_BASE_SIDE) - o2*(1+COST_BASE_SIDE)) / (o2*(1+COST_BASE_SIDE)))
            if not rets_T: continue
            all_rets_v2.extend(rets_T)
            for tkr in near52h_v2.columns:
                if not bool(near52h_v2.at[t, tkr]): continue
                o_tr = op_v2_df.at[t_ent, tkr]; c_tr = cl_v2_df.at[t_exit, tkr]
                if pd.isna(o_tr) or pd.isna(c_tr): continue
                rb = (c_tr*(1-COST_BASE_SIDE) - o_tr*(1+COST_BASE_SIDE)) / (o_tr*(1+COST_BASE_SIDE))
                trades_v2.append({"t": t, "tkr": tkr, "rb": rb, "rets_T_base": rets_T})
                dates_v2.append(t)

        n_tr_v2 = len(trades_v2)
        n_dates_v2 = len(set(dates_v2))
        log(f"  Sinyal 2026-06-08+: n_trades={n_tr_v2}, n_dates={n_dates_v2}")

        if n_tr_v2 > 0 and all_rets_v2:
            lo_v2, hi_v2q = np.percentile(all_rets_v2, 5), np.percentile(all_rets_v2, 95)
            exc_v2 = []
            for r in trades_v2:
                bm_w = float(np.mean(np.clip(r["rets_T_base"], lo_v2, hi_v2q)))
                rb_w = float(np.clip(r["rb"], lo_v2, hi_v2q))
                exc_v2.append(rb_w - bm_w)
            mean_exc_v2 = float(np.mean(np.array(exc_v2)))

            # Stress
            trades_v2_s = []
            all_rets_v2_s = []
            for t in near52h_v2.index:
                if t < t_min: continue
                try: i = idx_v2.get_loc(t)
                except KeyError: continue
                if i + 20 >= len(idx_v2): continue
                t_ent = idx_v2[i+1]; t_exit = idx_v2[i+20]
                if t_exit > t_max_data: continue
                elig_tkrs = elig_v2.columns[elig_v2.loc[t]]
                rets_T_s = []
                for tk2 in elig_tkrs:
                    o2 = op_v2_df.at[t_ent, tk2]; c2 = cl_v2_df.at[t_exit, tk2]
                    if pd.isna(o2) or pd.isna(c2): continue
                    rets_T_s.append((c2*(1-COST_STRESS_SIDE) - o2*(1+COST_STRESS_SIDE)) / (o2*(1+COST_STRESS_SIDE)))
                if not rets_T_s: continue
                all_rets_v2_s.extend(rets_T_s)
                for tkr in near52h_v2.columns:
                    if not bool(near52h_v2.at[t, tkr]): continue
                    o_tr = op_v2_df.at[t_ent, tkr]; c_tr = cl_v2_df.at[t_exit, tkr]
                    if pd.isna(o_tr) or pd.isna(c_tr): continue
                    rb = (c_tr*(1-COST_STRESS_SIDE) - o_tr*(1+COST_STRESS_SIDE)) / (o_tr*(1+COST_STRESS_SIDE))
                    trades_v2_s.append({"t": t, "tkr": tkr, "rb": rb, "rets_T_base": rets_T_s})

            if all_rets_v2_s:
                lo_s, hi_s = np.percentile(all_rets_v2_s, 5), np.percentile(all_rets_v2_s, 95)
                exc_v2_s = []
                for r in trades_v2_s:
                    bm_w = float(np.mean(np.clip(r["rets_T_base"], lo_s, hi_s)))
                    rb_w = float(np.clip(r["rb"], lo_s, hi_s))
                    exc_v2_s.append(rb_w - bm_w)
                mean_str_v2 = float(np.mean(exc_v2_s))
            else:
                mean_str_v2 = float("nan")

            log(f"  Mean exc_w: {mean_exc_v2*100:+.4f}%")
            log(f"  Mean stress_w: {mean_str_v2*100:+.4f}%")
            R7_pass = (mean_exc_v2 > 0) and (mean_str_v2 > 0)
            log(f"  R7: {'LOLOS' if R7_pass else 'GAGAL'}")
            R7_results = {"ran": True, "pass": R7_pass,
                          "n_trades": n_tr_v2, "n_dates": n_dates_v2,
                          "mean_exc_w": mean_exc_v2, "mean_stress_w": mean_str_v2}
        else:
            log("  Tidak ada trade di window R7.")
            R7_results = {"ran": True, "pass": False, "reason": "tidak ada trade"}
else:
    log("")
    log("  R7 TIDAK dijalankan — R1-R6 tidak semua lolos.")

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(PART3A)
print(f"✅ Sub-sel 3a ditulis ({len(PART3A)} byte)")
print(f"   File: {os.path.getsize(SCRIPT_PATH)} byte")
print()
print("Lanjut ke Q4-B (sub-sel 3b: P1, P2, laporan)")

✅ Sub-sel 3a ditulis (8785 byte)
   File: 30910 byte

Lanjut ke Q4-B (sub-sel 3b: P1, P2, laporan)


In [ ]:
# ============================================================
# Q4-B — P1, P2, laporan, run_log
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

PART3B = r'''
# ============================================================
# P1 — REKALKULASI BRK-5/10 & MOM-5/10 (deskriptif)
# ============================================================
log("")
log("=" * 90)
log("  P1 — REKALKULASI DESKRIPTIF (BRK-5/10, MOM-5/10)")
log("=" * 90)

P1_results = {}

# P1a — Data Tahap 1b (data_sealed.pkl)
log("")
log("  P1a — Data Tahap 1b (data_sealed.pkl, periode A/B, N=5,10)")
sealed = pd.read_pickle(f"{BASE}/data_sealed.pkl")
op_1b = sealed["price_open"].copy().loc[sealed["price_open"].index <= CUTOFF_NEW]
hi_1b = sealed["price_high"].copy().loc[sealed["price_high"].index <= CUTOFF_NEW]
lo_1b = sealed["price_low"].copy().loc[sealed["price_low"].index <= CUTOFF_NEW]
cl_1b = sealed["price_close"].copy().loc[sealed["price_close"].index <= CUTOFF_NEW]
vo_1b = sealed["price_volume"].copy().loc[sealed["price_volume"].index <= CUTOFF_NEW]

ev_1b = pd.read_csv(f"{BASE}/data_events.csv", parse_dates=["t0"])
idx_f = cl_1b.index
for _, ev in ev_1b.iterrows():
    tkr_e, t0 = ev["ticker"], ev["t0"]
    if tkr_e not in cl_1b.columns: continue
    try: i0 = idx_f.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_f)-1, i0+WINDOW_FWD)+1):
        d = idx_f[j]
        op_1b.at[d, tkr_e] = np.nan; hi_1b.at[d, tkr_e] = np.nan
        lo_1b.at[d, tkr_e] = np.nan; cl_1b.at[d, tkr_e] = np.nan; vo_1b.at[d, tkr_e] = np.nan

turnover_1b = cl_1b * vo_1b
toma20_1b = turnover_1b.rolling(20, min_periods=20).mean()
elig_1b = (cl_1b >= 200.0) & (toma20_1b >= 1_000_000_000.0)

roc3 = cl_1b / cl_1b.shift(3) - 1.0
roc3_rank = roc3.where(elig_1b).rank(axis=1, pct=True)
mom_sig = (roc3_rank >= 0.80) & roc3.notna() & elig_1b
hp10 = hi_1b.shift(1).rolling(10, min_periods=10).max()
brk_sig = (cl_1b > hp10) & elig_1b

idx_1b = idx_f[25:]
op_1b2 = op_1b.loc[idx_1b]; cl_1b2 = cl_1b.loc[idx_1b]
elig_1b2 = elig_1b.loc[idx_1b]
mom_1b = mom_sig.loc[idx_1b]; brk_1b = brk_sig.loc[idx_1b]

tA_end = pd.Timestamp("2025-05-15"); tB_start = pd.Timestamp("2025-05-16")

log(f"  {'Sinyal':8s} {'N':>2s} {'Per':>4s} {'n':>6s} {'exc_baru':>12s} {'p_baru':>9s}")
log("-" * 60)

def hitung_trade_1b(sig_df, N, t_start, t_end, cost=COST_BASE_SIDE):
    trades = []; all_rets = []; dates = []
    for t in sig_df.index:
        if t < t_start or t > t_end: continue
        try: i = idx_1b.get_loc(t)
        except KeyError: continue
        if i + N >= len(idx_1b): continue
        t_ent = idx_1b[i+1]; t_exit = idx_1b[i+N]
        elig_tkrs = elig_1b2.columns[elig_1b2.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs:
            o2 = op_1b2.at[t_ent, tk2]; c2 = cl_1b2.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-cost) - o2*(1+cost)) / (o2*(1+cost)))
        if not rets_T: continue
        all_rets.extend(rets_T)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_1b2.at[t_ent, tkr]; c_tr = cl_1b2.at[t_exit, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            rb = (c_tr*(1-cost) - o_tr*(1+cost)) / (o_tr*(1+cost))
            trades.append({"t": t, "rb": rb, "rets_T_base": rets_T})
            dates.append(t)
    return trades, all_rets, dates

for sig_name, sig_df in [("BRK", brk_1b), ("MOM", mom_1b)]:
    for N in [5, 10]:
        for per, t1, t2 in [("A", idx_1b[0], tA_end), ("B", tB_start, idx_1b[-1])]:
            trades, all_rets, dates = hitung_trade_1b(sig_df, N, t1, t2)
            if not trades or not all_rets:
                log(f"  {sig_name:8s} {N:>2d} {per:>4s} (tidak ada trade)"); continue
            lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
            exc_new = [np.clip(r["rb"], lo, hi) - np.mean(np.clip(r["rets_T_base"], lo, hi)) for r in trades]
            mean_new = float(np.mean(exc_new))
            daily = {}
            for t, e in zip(dates, exc_new): daily.setdefault(t, []).append(e)
            ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
            p_new = float(OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N}).pvalues[0]) if len(ds) >= 10 else float("nan")
            log(f"  {sig_name:8s} {N:>2d} {per:>4s} {len(trades):>6d} {mean_new*100:>+11.4f}% {p_new:>9.4f}")
            P1_results[(sig_name, N, "1b_"+per)] = {"n": len(trades), "exc_baru": mean_new, "p_baru": p_new}

# P1b — Data Tahap 2 (data_old_sealed.pkl)
log("")
log("  P1b — Data Tahap 2 (data_old_sealed.pkl, C1/C2, N=5,10)")
old = pd.read_pickle(f"{BASE}/data_old_sealed.pkl")
op_2 = old["price_open"].copy(); hi_2 = old["price_high"].copy()
lo_2 = old["price_low"].copy(); cl_2 = old["price_close"].copy()
vo_2 = old["price_volume"].copy()

ev_old = pd.read_csv(f"{BASE}/data_events_old.csv", parse_dates=["t0"])
idx_2 = cl_2.index
for _, ev in ev_old.iterrows():
    tkr_e, t0 = ev["ticker"], ev["t0"]
    if tkr_e not in cl_2.columns: continue
    try: i0 = idx_2.get_loc(t0)
    except KeyError: continue
    for j in range(max(0, i0-WINDOW_BACK), min(len(idx_2)-1, i0+WINDOW_FWD)+1):
        d = idx_2[j]
        op_2.at[d, tkr_e] = np.nan; hi_2.at[d, tkr_e] = np.nan
        lo_2.at[d, tkr_e] = np.nan; cl_2.at[d, tkr_e] = np.nan; vo_2.at[d, tkr_e] = np.nan

turnover_2 = cl_2 * vo_2
toma20_2 = turnover_2.rolling(20, min_periods=20).mean()
elig_2 = (cl_2 >= 200.0) & (toma20_2 >= 1_000_000_000.0)

roc3_2 = cl_2 / cl_2.shift(3) - 1.0
roc3_rank_2 = roc3_2.where(elig_2).rank(axis=1, pct=True)
mom_2 = (roc3_rank_2 >= 0.80) & roc3_2.notna() & elig_2
hp10_2 = hi_2.shift(1).rolling(10, min_periods=10).max()
brk_2 = (cl_2 > hp10_2) & elig_2

idx_2_w = idx_2[25:]
op_2w = op_2.loc[idx_2_w]; cl_2w = cl_2.loc[idx_2_w]
elig_2w = elig_2.loc[idx_2_w]
mom_2w = mom_2.loc[idx_2_w]; brk_2w = brk_2.loc[idx_2_w]

tC1_end = pd.Timestamp("2022-09-02"); tC2_start = pd.Timestamp("2022-09-05")

log(f"  {'Sinyal':8s} {'N':>2s} {'Per':>4s} {'n':>6s} {'exc_baru':>12s} {'p_baru':>9s}")
log("-" * 60)

def hitung_trade_2(sig_df, N, t_start, t_end, cost=COST_BASE_SIDE):
    trades = []; all_rets = []; dates = []
    for t in sig_df.index:
        if t < t_start or t > t_end: continue
        try: i = idx_2_w.get_loc(t)
        except KeyError: continue
        if i + N >= len(idx_2_w): continue
        t_ent = idx_2_w[i+1]; t_exit = idx_2_w[i+N]
        elig_tkrs = elig_2w.columns[elig_2w.loc[t]]
        rets_T = []
        for tk2 in elig_tkrs:
            o2 = op_2w.at[t_ent, tk2]; c2 = cl_2w.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rets_T.append((c2*(1-cost) - o2*(1+cost)) / (o2*(1+cost)))
        if not rets_T: continue
        all_rets.extend(rets_T)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_2w.at[t_ent, tkr]; c_tr = cl_2w.at[t_exit, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            rb = (c_tr*(1-cost) - o_tr*(1+cost)) / (o_tr*(1+cost))
            trades.append({"t": t, "rb": rb, "rets_T_base": rets_T})
            dates.append(t)
    return trades, all_rets, dates

for sig_name, sig_df in [("BRK", brk_2w), ("MOM", mom_2w)]:
    for N in [5, 10]:
        for per, t1, t2 in [("C1", idx_2_w[0], tC1_end), ("C2", tC2_start, idx_2_w[-1])]:
            trades, all_rets, dates = hitung_trade_2(sig_df, N, t1, t2)
            if not trades or not all_rets:
                log(f"  {sig_name:8s} {N:>2d} {per:>4s} (tidak ada trade)"); continue
            lo, hi = np.percentile(all_rets, 5), np.percentile(all_rets, 95)
            exc_new = [np.clip(r["rb"], lo, hi) - np.mean(np.clip(r["rets_T_base"], lo, hi)) for r in trades]
            mean_new = float(np.mean(exc_new))
            daily = {}
            for t, e in zip(dates, exc_new): daily.setdefault(t, []).append(e)
            ds = np.array([np.mean(daily[t]) for t in sorted(daily.keys())])
            p_new = float(OLS(ds, np.ones(len(ds))).fit(cov_type="HAC", cov_kwds={"maxlags": N}).pvalues[0]) if len(ds) >= 10 else float("nan")
            log(f"  {sig_name:8s} {N:>2d} {per:>4s} {len(trades):>6d} {mean_new*100:>+11.4f}% {p_new:>9.4f}")
            P1_results[(sig_name, N, "2_"+per)] = {"n": len(trades), "exc_baru": mean_new, "p_baru": p_new}

# ============================================================
# P2 — PELENGKAP LAPORAN TAHAP 3 (deskriptif)
# ============================================================
log("")
log("=" * 90)
log("  P2 — PELENGKAP LAPORAN TAHAP 3")
log("=" * 90)
log(f"  Tanggal mulai efektif: {idx[0].date()}")
log(f"  Batas K1: {idx[0].date()} → {tK1_end.date()}")
log(f"  Batas K2: {tK2_start.date()} → {tK2_end.date()}")
log(f"  Batas K3: {tK3_start.date()} → {idx[-1].date()}")

n_eff_series = elig_b.sum(axis=1)
by_year = n_eff_series.groupby(n_eff_series.index.year).mean()
log("  Saham layak per tahun:")
for yr, val in by_year.items():
    log(f"    {yr}: {val:.1f}")

log("  Median turnover per blok:")
to_eff = cl_b * vo_b
P2_median_turnover = {}
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask = (idx >= t1) & (idx <= t2)
    sub = to_eff.loc[idx[mask]].stack().dropna()
    med = float(sub.median()) if len(sub) else float("nan")
    P2_median_turnover[blk_id] = med
    log(f"    {blk_id}: Rp {med/1e9:.3f} M")

# LOWVOL lengkap
log("")
log("  LOWVOL — detail per blok & pool:")
ret_l = cl_b.pct_change()
vol60 = ret_l.rolling(60, min_periods=60).std()
vol60_rank = vol60.where(elig_b).rank(axis=1, pct=True)
lowvol_sig = (vol60_rank <= 0.20) & vol60.notna() & elig_b

P2_lowvol = {}
for blk_id, t1, t2 in [("K1", idx[0], tK1_end),
                        ("K2", tK2_start, tK2_end),
                        ("K3", tK3_start, idx[-1])]:
    mask_t = pd.Series((lowvol_sig.index >= t1) & (lowvol_sig.index <= t2),
                       index=lowvol_sig.index)
    sig_blk = lowvol_sig.where(mask_t, False)
    tr, ar, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                         N=20, cost_side=COST_BASE_SIDE)
    res = compute_excess_w(tr, ar, cost_side=COST_BASE_SIDE)
    tr_s, ar_s, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                             N=20, cost_side=COST_STRESS_SIDE)
    res_s = compute_excess_w(tr_s, ar_s, cost_side=COST_STRESS_SIDE)
    P2_lowvol[blk_id] = {"n": len(tr), "exc_w": res["mean_exc_w"],
                          "stress_w": res_s["mean_exc_w"]}
    log(f"    {blk_id}: n={len(tr)} exc_w={res['mean_exc_w']*100:+.3f}% "
        f"stress_w={res_s['mean_exc_w']*100:+.3f}%")

# ============================================================
# TULIS LAPORAN_Tahap3b.md
# ============================================================
ts = datetime.now().isoformat(timespec="seconds")
seal_long_sha = json.load(open(f"{BASE}/data_long_manifest.json"))["sha256"] \
    if os.path.exists(f"{BASE}/data_long_manifest.json") else "n/a"

rep = []
rep.append("TAHAP 3b - LAPORAN")
rep.append(f"Tanggal run / data terakhir    : {ts} / {cl.index.max().date()}")
rep.append(f"Nama file skrip final          : Tahap3b_Ketahanan.py")
rep.append(f"LANGKAH 0: {'COCOK' if l0_pass else 'TIDAK COCOK'}")
rep.append(f"Putaran perbaikan (maks 5)     : 0")
rep.append("")
rep.append("UJI KETAHANAN R1-R6")
rep.append("-" * 60)

rep.append("")
rep.append("R1 — Penundaan 1 hari")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R1[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R1["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R1_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R1_pass else 'GAGAL — ' + '; '.join(R1_fails)}")

rep.append("")
rep.append("R2 — Buang trade gap-up >= 10%")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R2[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R2["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R2_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R2_pass else 'GAGAL — ' + '; '.join(R2_fails)}")

rep.append("")
rep.append("R3 — Biaya 1.0% round-trip")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R3[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} return_w={r['mean_exc_w']*100:+.3f}%")
rp = out_R3["pool"]
rep.append(f"  pool: return_w={rp['mean_exc_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Blok return > 0: {n_blk_pos_R3}/3")
rep.append(f"  Hasil: {'LOLOS' if R3_pass else 'GAGAL — ' + '; '.join(R3_fails)}")

rep.append("")
rep.append("R4 — Hanya 50 saham teratas")
for blk_id in ["K1", "K2", "K3"]:
    r = out_R4[blk_id]
    rep.append(f"  {blk_id}: n={r['n_trades']} exc_w={r['mean_exc_w']*100:+.3f}% str_w={r['stress_w']*100:+.3f}%")
rp = out_R4["pool"]
rep.append(f"  pool: exc_w={rp['mean_exc_w']*100:+.3f}% t={rp['t_stat']:+.3f} p={rp['p_val']:.4f} "
           f"p1sisi={R4_p1:.4f} str_w={rp['mean_stress_w']*100:+.3f}% n={rp['n_trades']}")
rep.append(f"  Hasil: {'LOLOS' if R4_pass else 'GAGAL — ' + '; '.join(R4_fails)}")

rep.append("")
rep.append("R5 — Tanpa K1 (pool K2+K3)")
rep.append(f"  K2: n={out_R5_all['K2']['n_trades']} exc_w={out_R5_all['K2']['mean_exc_w']*100:+.3f}%")
rep.append(f"  K3: n={out_R5_all['K3']['n_trades']} exc_w={out_R5_all['K3']['mean_exc_w']*100:+.3f}%")
rep.append(f"  Pool: exc_w={r5_mean*100:+.3f}% t={r5_t:+.3f} p={r5_p:.4f} p1sisi={r5_p1:.4f} str_w={r5_str_mean*100:+.3f}%")
rep.append(f"  Hasil: {'LOLOS' if R5_pass else 'GAGAL — ' + '; '.join(R5_fails)}")

rep.append("")
rep.append("R6 — Konsentrasi")
if trades_R6:
    rep.append(f"  R6a: semester positif {n_pos}/{n_sem} ({pct_pos:.1f}%); semester terbesar {max_share:.2f}%")
    rep.append(f"       Hasil: {'LOLOS' if R6a_pass else 'GAGAL — ' + '; '.join(R6a_fails)}")
    rep.append(f"  R6b: top-5 share per blok:")
    for blk_id in ["K1", "K2", "K3"]:
        sub = df_R6[df_R6["blk"] == blk_id]
        if sub.empty: continue
        tkr_agg = sub.groupby("tkr")["exc_w"].sum().sort_values(ascending=False)
        total_blk = sub["exc_w"].sum()
        top5_sum = tkr_agg.head(5).sum()
        share = 100 * top5_sum / total_blk if total_blk != 0 else 0
        rep.append(f"    {blk_id}: share={share:.2f}% saham={list(tkr_agg.head(5).index)}")
    rep.append(f"       Hasil: {'LOLOS' if R6b_pass else 'GAGAL — ' + '; '.join(R6b_fails)}")
rep.append(f"  Hasil R6: {'LOLOS' if R6_pass else 'GAGAL'}")

rep.append("")
rep.append(f"KESIMPULAN R1-R6: {'SEMUA LOLOS' if R_ALL_PASS else 'ADA YANG GAGAL'}")

rep.append("")
if R7_results["ran"]:
    if R7_results.get("pass"):
        rep.append(f"R7 (VALIDATE): LOLOS")
        rep.append(f"  n_trades={R7_results.get('n_trades','n/a')} n_dates={R7_results.get('n_dates','n/a')}")
        rep.append(f"  exc_w={R7_results.get('mean_exc_w',float('nan'))*100:+.4f}%")
        rep.append(f"  stress_w={R7_results.get('mean_stress_w',float('nan'))*100:+.4f}%")
    else:
        rep.append(f"R7 (VALIDATE): GAGAL — {R7_results.get('reason','lihat log')}")
else:
    rep.append(f"R7 (VALIDATE): TIDAK DIJALANKAN — R1-R6 tidak semua lolos")

rep.append("")
rep.append("P1 — REKALKULASI DESKRIPTIF (lama vs baru, tidak mengubah verdict)")
for key, res in sorted(P1_results.items()):
    sig_name, N, per = key
    rep.append(f"  {sig_name}-N{N} {per}: n={res['n']} exc_baru={res['exc_baru']*100:+.4f}% p_baru={res['p_baru']:.4f}")

rep.append("")
rep.append("P2 — PELENGKAP LAPORAN TAHAP 3")
rep.append(f"  Tanggal mulai efektif: {idx[0].date()}")
rep.append(f"  Batas K1: {idx[0].date()} → {tK1_end.date()}")
rep.append(f"  Batas K2: {tK2_start.date()} → {tK2_end.date()}")
rep.append(f"  Batas K3: {tK3_start.date()} → {idx[-1].date()}")
rep.append(f"  Saham layak per tahun: {dict(by_year.round(1))}")
rep.append(f"  Median turnover per blok:")
for blk_id, med in P2_median_turnover.items():
    rep.append(f"    {blk_id}: Rp {med/1e9:.3f} M")
rep.append(f"  LOWVOL per blok: {P2_lowvol}")

rep.append("")
rep.append("PERUBAHAN KODE (satu baris per perubahan)")
rep.append("  -")
rep.append("")
rep.append("USULAN (tidak dijalankan)")
rep.append("  -")
rep.append("")
rep.append("PERNYATAAN: Angka kriteria / definisi sinyal / biaya / data / filter berubah? TIDAK")
rep.append("")
rep.append("CATATAN BIAS SURVIVOR")
rep.append("  Universe = saham yang terdaftar sekarang. Hasil LOLOS = BATAS ATAS.")
rep.append("  Belum boleh uang nyata: butuh paper trading 60 hari bursa.")

report_text = "\n".join(rep)
with open(REPORT_PATH, "w") as f:
    f.write(report_text)
log(f"\n  LAPORAN_Tahap3b.md: {os.path.getsize(REPORT_PATH)} byte")

# Run log
_COLUMNS = ["timestamp","phase","config_version","script_version",
            "global_seed","seal_sha256","params_sha256",
            "n_stocks","n_bars","n_signals","sharpe","max_dd","notes"]
file_exists = os.path.exists(RUN_LOG_PATH)
with open(RUN_LOG_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=_COLUMNS)
    if not file_exists: w.writeheader()
    w.writerow({
        "timestamp": ts, "phase": "BACKTEST",
        "config_version": "Gen3-Tahap3b", "script_version": "Tahap3b_Ketahanan",
        "global_seed": 42, "seal_sha256": seal_long_sha[:16] + "...",
        "params_sha256": "",
        "n_stocks": int(cl.shape[1]), "n_bars": int(cl.shape[0]),
        "n_signals": "", "sharpe": "", "max_dd": "",
        "notes": f"Tahap 3b — R1-R6: {'LOLOS' if R_ALL_PASS else 'ADA GAGAL'}; R7: {'LOLOS' if R7_results.get('pass', False) else ('GAGAL' if R7_results['ran'] else 'TIDAK DIJALANKAN')}",
    })
log(f"  run_log.csv diperbarui")
log("")
log("=" * 90)
log("  Tahap3b_Ketahanan.py SELESAI")
log("=" * 90)

'''

with open(SCRIPT_PATH, "a") as f:
    f.write(PART3B)

print(f"✅ Sub-sel 3b ditulis ({len(PART3B)} byte)")
print(f"   File TOTAL: {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi syntax + cek konsistensi
import ast
with open(SCRIPT_PATH) as f:
    full = f.read()
try:
    ast.parse(full)
    print("✅ Syntax OK — file siap dijalankan")
except SyntaxError as e:
    print(f"❌ Syntax ERROR: {e}")
    print(f"   Line {e.lineno}: {e.text}")

print()
print("=" * 90)
print("  CEK KONSISTENSI (tidak boleh duplikat)")
print("=" * 90)
checks = [
    ("# LOAD DATA + PREPROCESSING", 1),
    ("LANGKAH 0 — REPRODUKSI", 1),
    ("UJI KETAHANAN R1-R6", 1),
    ("R1 — Entry Open(T+2)", 1),
    ("R7 — VALIDATE", 1),
    ("P1 — REKALKULASI DESKRIPTIF", 1),
    ("P2 — PELENGKAP LAPORAN TAHAP 3", 1),
    ("TULIS LAPORAN_Tahap3b.md", 1),
    ("Tahap3b_Ketahanan.py SELESAI", 1),
]
all_ok = True
for pattern, expected in checks:
    cnt = full.count(pattern)
    ok = (cnt == expected)
    if not ok: all_ok = False
    print(f"  {'✅' if ok else '❌'} '{pattern}': {cnt}x (harus {expected}x)")

print()
if all_ok:
    print("  ✅ FILE BERSIH — siap dijalankan")
else:
    print("  ❌ MASIH ADA DUPLIKASI — perlu perbaiki")
print("=" * 90)

✅ Sub-sel 3b ditulis (17976 byte)
   File TOTAL: 48952 byte
✅ Syntax OK — file siap dijalankan

  CEK KONSISTENSI (tidak boleh duplikat)
  ✅ '# LOAD DATA + PREPROCESSING': 1x (harus 1x)
  ❌ 'LANGKAH 0 — REPRODUKSI': 2x (harus 1x)
  ❌ 'UJI KETAHANAN R1-R6': 3x (harus 1x)
  ✅ 'R1 — Entry Open(T+2)': 1x (harus 1x)
  ❌ 'R7 — VALIDATE': 2x (harus 1x)
  ❌ 'P1 — REKALKULASI DESKRIPTIF': 2x (harus 1x)
  ❌ 'P2 — PELENGKAP LAPORAN TAHAP 3': 3x (harus 1x)
  ✅ 'TULIS LAPORAN_Tahap3b.md': 1x (harus 1x)
  ✅ 'Tahap3b_Ketahanan.py SELESAI': 1x (harus 1x)

  ❌ MASIH ADA DUPLIKASI — perlu perbaiki


In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    lines = f.readlines()

# Cari baris di mana pattern muncul, tampilkan konteks
patterns = [
    "LANGKAH 0 — REPRODUKSI",
    "UJI KETAHANAN R1-R6",
    "R7 — VALIDATE",
    "P1 — REKALKULASI DESKRIPTIF",
    "P2 — PELENGKAP LAPORAN TAHAP 3",
]

for pat in patterns:
    print("=" * 80)
    print(f"  PATTERN: '{pat}'")
    print("=" * 80)
    for i, line in enumerate(lines):
        if pat in line:
            # Tampilkan 2 baris sebelum & sesudah
            ctx_start = max(0, i-2)
            ctx_end = min(len(lines), i+3)
            print(f"  --- Line {i+1} ---")
            for j in range(ctx_start, ctx_end):
                marker = " >>>" if j == i else "    "
                print(f"  {marker} L{j+1:>4d}: {lines[j].rstrip()[:100]}")
            print()

  PATTERN: 'LANGKAH 0 — REPRODUKSI'
  --- Line 213 ---
       L 211: 
       L 212: # ============================================================
   >>> L 213: # LANGKAH 0 — REPRODUKSI
       L 214: # ============================================================
       L 215: log("=" * 90)

  --- Line 216 ---
       L 214: # ============================================================
       L 215: log("=" * 90)
   >>> L 216: log("  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)")
       L 217: log("=" * 90)
       L 218: log(f"  {'Blok':6s} {'hasil exc_w':>14s} {'target':>14s} {'cocok?':>8s}  "

  PATTERN: 'UJI KETAHANAN R1-R6'
  --- Line 262 ---
       L 260: 
       L 261: # ============================================================
   >>> L 262: # UJI KETAHANAN R1-R6
       L 263: # ============================================================
       L 264: log("")

  --- Line 266 ---
       L 264: log("")
       L 265: log("=" * 90)
   >>> L 266: log("  UJI KETAHANAN R1-R6")
       L 267:

In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T02:49:33

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
  Setelah warm-up 260: 1125 bar
  K1: 2021-09-22 → 2023-03-27 (375 bar)
  K2: 2023-03-28 → 2024-10-25 (375 bar)
  K3: 2024-10-28 → 2026-06-05 (375 bar)

  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)
  Blok      hasil exc_w         target   cocok?     hasil str_w         target   cocok?       n target   cocok?
--------------------------------------------------------------------------------------------------------------
  K1           +4.4351%       +5.3670%     BEDA        +4.4218%       +3.5350%     BEDA    1629   1099     BEDA
  K2           +1.5358%       +1.6350%     BEDA        +1.5312%       +0.2000%     BEDA    2739   2426     BEDA
  K3           +1.7943%       +2.1740%     BEDA        +1.7889%       +2.4850%     BEDA    2907   3750     BEDA
--------------------------

In [ ]:
# ============================================================
# R2 — Patch Tahap3b: JANGAN potong warm-up sebelum blok
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Patch 1: ganti pemotongan idx
OLD = "idx = idx_full[WARMUP_BARS:]"
NEW = "idx = idx_full  # PATCH: jangan potong warm-up, biarkan fitur 52w NaN di awal"
if OLD in content:
    content = content.replace(OLD, NEW, 1)
    print("✅ Patch 1: pemotongan warm-up dihapus")
else:
    print("⚠️ Patch 1: baris tidak ditemukan")

# Patch 2: sesuaikan baris-baris terkait
# Karena op_b, cl_b, dll mengambil .loc[idx], mereka akan dapat 1385 bar (bukan 1125)
# Sudah otomatis benar.

# Patch 3: perbaiki log
OLD_LOG = 'log(f"  Setelah warm-up 260: {n_b} bar")'
NEW_LOG = 'log(f"  Setelah warm-up 260 (di-skip, fitur sudah NaN): {n_b} bar (semua bar efektif dipakai)")'
if OLD_LOG in content:
    content = content.replace(OLD_LOG, NEW_LOG, 1)
    print("✅ Patch 3: log diperbarui")
else:
    print("⚠️ Patch 3: log tidak ditemukan")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print(f"  File: {os.path.getsize(SCRIPT_PATH)} byte")

# Verifikasi
with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'idx = idx_full  # PATCH' muncul: {new.count('idx = idx_full  # PATCH')}x")
print(f"  'idx = idx_full[WARMUP_BARS:]' muncul: {new.count('idx = idx_full[WARMUP_BARS:]')}x")

try:
    ast.parse(new)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch 1: pemotongan warm-up dihapus
✅ Patch 3: log diperbarui

  File: 49056 byte
  'idx = idx_full  # PATCH' muncul: 1x
  'idx = idx_full[WARMUP_BARS:]' muncul: 0x
  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T02:57:03

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
  Setelah warm-up 260 (di-skip, fitur sudah NaN): 1385 bar (semua bar efektif dipakai)
  K1: 2019-10-01 → 2022-07-20 (461 bar)
  K2: 2022-07-21 → 2024-06-24 (461 bar)
  K3: 2024-06-25 → 2026-06-05 (463 bar)

  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)
  Blok      hasil exc_w         target   cocok?     hasil str_w         target   cocok?       n target   cocok?
--------------------------------------------------------------------------------------------------------------
  K1           +5.6679%       +5.3670%     BEDA        +5.6510%       +3.5350%     BEDA    1130   1099     BEDA
  K2           +1.7009%       +1.6350%     BEDA        +1.6959%       +0.2000%     BEDA    2426   2426    COCOK
  K3           +1.7999%       +2.1740%     BEDA        +1.7945%       +2.4850%     

In [ ]:
# ============================================================
# R4 — Diagnosa detail: bandingkan Tahap 3 asli vs Tahap 3b
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, pickle

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

# Baca hasil Tahap 3 asli
hasil_t3 = pickle.load(open(f"{BASE}/hasil_tahap3.pkl", "rb"))
print("=" * 100)
print("  HASIL TAHAP 3 ASLI (dari hasil_tahap3.pkl)")
print("=" * 100)
print(f"  Keys: {list(hasil_t3.keys())}")

# Cari info K bounds dan n_trades per blok untuk NEAR52H
if "all_results" in hasil_t3:
    ar = hasil_t3["all_results"]
    print()
    print("  NEAR52H detail per blok:")
    for blk in ["K1", "K2", "K3", "pool"]:
        key = ("NEAR52H", 20, blk)
        if key in ar:
            r = ar[key]
            print(f"    {blk}: n={r.get('n_trades', 'n/a')} "
                  f"exc_w={r.get('mean_exc_w', float('nan'))*100:+.4f}% "
                  f"stress_w={r.get('mean_stress_w', float('nan'))*100:+.4f}%")

# Info dari pkl Tahap 3
if "K_bounds" in hasil_t3:
    print()
    print("  K_bounds (Tahap 3 asli):")
    for k, v in hasil_t3["K_bounds"].items():
        print(f"    {k}: {v}")

if "n_eff_bars" in hasil_t3:
    print(f"  n_eff_bars: {hasil_t3['n_eff_bars']}")

# Baca data_long_sealed.pkl untuk cek shape
print()
print("=" * 100)
print("  DATA_LONG_SEALED.PKL (dipakai kedua tahap)")
print("=" * 100)
data = pickle.load(open(f"{BASE}/data_long_sealed.pkl", "rb"))
cl = data["price_close"]
print(f"  Shape: {cl.shape}")
print(f"  Range: {cl.index.min().date()} → {cl.index.max().date()}")
print(f"  n_bars: {len(cl)}")

# Kalau Tahap 3 asli buang 260 warm-up untuk sinyal tapi blok dari 1385:
# brarti K1 start bar-260.
# Kita cek: bar 0-259 (2019-10-01 s/d ~2020-09):
print()
print("  Cek apakah 260 bar pertama adalah 'warm-up':")
print(f"    Bar[0]: {cl.index[0].date()}")
print(f"    Bar[259]: {cl.index[259].date()}")
print(f"    Bar[260]: {cl.index[260].date()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  HASIL TAHAP 3 ASLI (dari hasil_tahap3.pkl)
  Keys: ['all_results', 'ringkasan_lolos', 'mean_plasebo', 'uji0', 't_eff', 'n_eff_bars', 'pct_cleaning', 'pct_nan_total', 'n_elig_mean', 'n_elig_median', 'drop_days', 'n_events', 'n_viol', 'long_manifest', 'K_bounds']

  NEAR52H detail per blok:
    K1: n=1099 exc_w=+5.3665% stress_w=+3.5349%
    K2: n=2426 exc_w=+1.6349% stress_w=+0.1959%
    K3: n=3750 exc_w=+2.1736% stress_w=+2.4847%
    pool: n=n/a exc_w=+2.4763% stress_w=+nan%

  K_bounds (Tahap 3 asli):
    K1_end: 2022-07-20
    K2_start: 2022-07-21
    K2_end: 2024-06-24
    K3_start: 2024-06-25
  n_eff_bars: 1385

  DATA_LONG_SEALED.PKL (dipakai kedua tahap)
  Shape: (1385, 561)
  Range: 2019-10-01 → 2026-06-05
  n_bars: 1385

  Cek apakah 260 bar pertama adalah 'warm-up':
    Bar[0]: 2019-10-01
    Bar[259]: 2021-09-21
    Bar[260]: 2021-09-22


In [ ]:
# ============================================================
# R5 — Patch Tahap3b: matikan sinyal di bar warm-up (260 pertama)
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Patch: setelah idx = idx_full dan near52h_b = near52h.loc[idx],
# matikan sinyal di 260 bar pertama
OLD = 'near52h_b = near52h.loc[idx]'
NEW = '''near52h_b = near52h.loc[idx]
# PATCH: matikan sinyal di 260 bar warm-up pertama (agar cocok dengan Tahap 3 asli)
t_warmup_end = idx_full[WARMUP_BARS] if WARMUP_BARS < len(idx_full) else idx_full[-1]
near52h_b = near52h_b.where(near52h_b.index >= t_warmup_end, False)
log(f"  Warm-up cutoff sinyal: T >= {t_warmup_end.date()} (bar ke-{WARMUP_BARS})")'''

if OLD in content:
    content = content.replace(OLD, NEW, 1)
    print("✅ Patch: sinyal di warm-up dimatikan")
else:
    print("⚠️ Patch: baris 'near52h_b = near52h.loc[idx]' tidak ditemukan")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print(f"  File: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'PATCH: matikan sinyal' muncul: {new.count('PATCH: matikan sinyal')}x")
print(f"  't_warmup_end' muncul: {new.count('t_warmup_end')}x")

try:
    ast.parse(new)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch: sinyal di warm-up dimatikan

  File: 49377 byte
  'PATCH: matikan sinyal' muncul: 1x
  't_warmup_end' muncul: 3x
  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T03:02:35

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
Traceback (most recent call last):
  File "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py", line 121, in <module>
    near52h_b = near52h_b.where(near52h_b.index >= t_warmup_end, False)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/generic.py", line 10984, in where
    return self._where(cond, other, inplace, axis, level)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/generic.py", line 10649, in _where
    raise ValueError("Array conditional must be same shape as self")
ValueError: Array conditional must be same shape as self


In [ ]:
# ============================================================
# R7 — Perbaiki patch warm-up: pakai .loc bukan .where
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Ganti baris yang salah
OLD = 'near52h_b = near52h_b.where(near52h_b.index >= t_warmup_end, False)'
NEW = '''# Pakai .loc: matikan sinyal di bar sebelum t_warmup_end
near52h_b.loc[near52h_b.index < t_warmup_end, :] = False'''

if OLD in content:
    content = content.replace(OLD, NEW, 1)
    print("✅ Patch diperbaiki: pakai .loc indexing")
else:
    print("⚠️ Baris lama tidak ditemukan persis")
    # Cari alternatif
    if "near52h_b.where" in content:
        print("     Masih ada '.where' — cari baris")
        lines = content.splitlines()
        for i, line in enumerate(lines):
            if "near52h_b.where" in line:
                print(f"     L{i+1}: {line}")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print(f"  File: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'near52h_b.where' muncul: {new.count('near52h_b.where')}x (harus 0)")
print(f"  'near52h_b.loc[' muncul: {new.count('near52h_b.loc[')}x (harus >= 1)")

try:
    ast.parse(new)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch diperbaiki: pakai .loc indexing

  File: 49423 byte
  'near52h_b.where' muncul: 2x (harus 0)
  'near52h_b.loc[' muncul: 1x (harus >= 1)
  ✅ Syntax OK


In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    lines = f.readlines()

print("=" * 100)
print("  BARIS DENGAN 'near52h_b.where'")
print("=" * 100)
for i, line in enumerate(lines):
    if "near52h_b.where" in line:
        print(f"  L{i+1}: {line.rstrip()}")
        # Konteks
        for j in range(max(0, i-2), min(len(lines), i+3)):
            marker = " >>>" if j == i else "    "
            print(f"  {marker} L{j+1}: {lines[j].rstrip()[:120]}")
        print()

  BARIS DENGAN 'near52h_b.where'
  L234:     sig_blk = near52h_b.where(mask_t, False)
       L232:     mask_t = pd.Series((near52h_b.index >= t1) & (near52h_b.index <= t2),
       L233:                        index=near52h_b.index)
   >>> L234:     sig_blk = near52h_b.where(mask_t, False)
       L235:     # BASE
       L236:     trades, all_rets, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,

  L476:     sig_blk = near52h_b.where(mask_t, False)
       L474:     mask_t = pd.Series((near52h_b.index >= t1) & (near52h_b.index <= t2),
       L475:                        index=near52h_b.index)
   >>> L476:     sig_blk = near52h_b.where(mask_t, False)
       L477:     trades, all_rets, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
       L478:                                                    N=20, cost_side=COST_BASE_SIDE, delay=1)



In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

print("=" * 100)
print("  VERIFIKASI FILE")
print("=" * 100)

# Cek apakah baris warm-up sudah pakai .loc
checks = [
    ("near52h_b.loc[near52h_b.index < t_warmup_end, :] = False",
     "Patch warm-up (pakai .loc)"),
    ("near52h_b.where(near52h_b.index >=",
     "Bug lama (masih ada?)"),
]

for pattern, label in checks:
    cnt = content.count(pattern)
    print(f"  {label:40s}: {cnt}x")

# Cek syntax
import ast
try:
    ast.parse(content)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

# Cek baris L234 & L476 aman
lines = content.splitlines()
print()
print("  Cek L234 dan L476:")
for i, line in enumerate(lines):
    if i+1 in [234, 476]:
        print(f"  L{i+1}: {line.strip()}")

  VERIFIKASI FILE
  Patch warm-up (pakai .loc)              : 1x
  Bug lama (masih ada?)                   : 0x
  ✅ Syntax OK

  Cek L234 dan L476:
  L234: sig_blk = near52h_b.where(mask_t, False)
  L476: sig_blk = near52h_b.where(mask_t, False)


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T03:07:50

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
  Warm-up cutoff sinyal: T >= 2021-09-22 (bar ke-260)
  Setelah warm-up 260 (di-skip, fitur sudah NaN): 1385 bar (semua bar efektif dipakai)
  K1: 2019-10-01 → 2022-07-20 (461 bar)
  K2: 2022-07-21 → 2024-06-24 (461 bar)
  K3: 2024-06-25 → 2026-06-05 (463 bar)

  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)
  Blok      hasil exc_w         target   cocok?     hasil str_w         target   cocok?       n target   cocok?
--------------------------------------------------------------------------------------------------------------
  K1           +5.4080%       +5.3670%     BEDA        +5.3918%       +3.5350%     BEDA    1099   1099    COCOK
  K2           +1.7009%       +1.6350%     BEDA        +1.6959%       +0.2000%     BEDA    2426   2426    COCOK
  K3           +1.7999%      

In [ ]:
# ============================================================
# R9 — Bandingkan fungsi compute_excess_w di Tahap 3 vs 3b
# ============================================================
import os

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"

# Cek Tahap 3
p3 = f"{BASE}/Tahap3_Horizon20.py"
p3b = f"{BASE}/Tahap3b_Ketahanan.py"

print("=" * 100)
print("  FUNGSI build_features / compute / trade di Tahap 3 vs 3b")
print("=" * 100)

for label, path in [("Tahap 3", p3), ("Tahap 3b", p3b)]:
    print(f"\n  === {label} ({path}) ===")
    with open(path) as f:
        content = f.read()
    lines = content.splitlines()
    # Cari fungsi compute_excess_w atau yang setara
    for i, line in enumerate(lines):
        if "def compute_excess_w" in line or "def compute_excess" in line:
            for j in range(i, min(i+40, len(lines))):
                print(f"    L{j+1}: {lines[j]}")
            break
    else:
        print("    (fungsi compute_excess_w tidak ditemukan)")

  FUNGSI build_features / compute / trade di Tahap 3 vs 3b

  === Tahap 3 (/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3_Horizon20.py) ===
    (fungsi compute_excess_w tidak ditemukan)

  === Tahap 3b (/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py) ===
    L175: def compute_excess_w(trade_records, all_rets_pool, cost_side=COST_BASE_SIDE):
    L176:     if not all_rets_pool or not trade_records:
    L177:         return {"n_trades": 0, "mean_exc_w": float("nan"),
    L178:                 "mean_exc_w_lama": float("nan"),
    L179:                 "median_exc": float("nan"),
    L180:                 "dates": [], "exc_w_list": []}
    L181:     lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
    L182:     exc_w_list = []
    L183:     exc_w_lama_list = []
    L184:     for rec in trade_records:
    L185:         rb = rec["rb"]
    L186:         rets_T = rec["rets_T_base"]
    L187:         bm_w = float(np.mean(np.clip

In [ ]:
# ============================================================
# R10 — Cari kode perhitungan excess di Tahap 3 asli
# ============================================================
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
p3 = f"{BASE}/Tahap3_Horizon20.py"

with open(p3) as f:
    content = f.read()
lines = content.splitlines()

# Cari blok run_signal di Tahap 3 (tempat hitung excess)
print("=" * 100)
print("  KODE EXCESS DI TAHAP 3 ASLI")
print("=" * 100)

# Cari fungsi run_signal
for i, line in enumerate(lines):
    if "def run_signal" in line:
        print(f"  Fungsi run_signal ditemukan di L{i+1}")
        # Tampilkan ~60 baris dari sana
        for j in range(i, min(i+80, len(lines))):
            print(f"  L{j+1}: {lines[j]}")
        break

# Cari juga baris 'lo, hi' atau 'winsorize' di sekitar
print()
print("  Baris dengan 'lo, hi' atau 'all_rets' atau 'bm_w':")
for i, line in enumerate(lines):
    if "lo, hi" in line or "all_rets" in line or "bm_w" in line:
        print(f"  L{i+1}: {line}")

  KODE EXCESS DI TAHAP 3 ASLI
  Fungsi run_signal ditemukan di L648
  L648: def run_signal(sig_name, sig_df, N=20):
  L649:     """ADDENDUM 8: hits excess_w (baru) dan excess_w_lama (pembanding)."""
  L650:     results = {}
  L651:     daily_pool = []
  L652:     all_exc = {"K1": [], "K2": [], "K3": []}
  L653:     n_trades = {"K1": 0, "K2": 0, "K3": 0}
  L654:     raw_b_l = {"K1": [], "K2": [], "K3": []}
  L655:     raw_s_l = {"K1": [], "K2": [], "K3": []}
  L656:     for blk_id in ["K1", "K2", "K3"]:
  L657:         sig_list = []
  L658:         for t in sig_df.index:
  L659:             if not in_blk(t, blk_id): continue
  L660:             for tkr in sig_df.columns:
  L661:                 if bool(sig_df.at[t, tkr]): sig_list.append((tkr, t))
  L662:         exc_raw, dates = [], []
  L663:         # Kumpulkan: rb, rs, bm, dan SEMUA return saham layak per tanggal
  L664:         rb_l = []; rs_l = []; bm_l = []
  L665:         all_rets_pool = []   # untuk winsorize batas (ADDENDUM 8)

In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
p3b = f"{BASE}/Tahap3b_Ketahanan.py"

with open(p3b) as f:
    content = f.read()
lines = content.splitlines()

print("=" * 100)
print("  FUNGSI build_trades_and_excess di Tahap 3b")
print("=" * 100)
for i, line in enumerate(lines):
    if "def build_trades_and_excess" in line:
        for j in range(i, min(i+45, len(lines))):
            print(f"  L{j+1}: {lines[j]}")
        break

  FUNGSI build_trades_and_excess di Tahap 3b
  L137: def build_trades_and_excess(idx_arr, op_arr, cl_arr, elig_arr, sig_df,
  L138:                             N=20, cost_side=COST_BASE_SIDE,
  L139:                             delay=1, drop_gap=False):
  L140:     trade_records = []
  L141:     all_rets_pool = []
  L142:     dates = []
  L143:     for t in sig_df.index:
  L144:         try: i = idx_arr.get_loc(t)
  L145:         except KeyError: continue
  L146:         if i + delay + N - 1 >= len(idx_arr): continue
  L147:         t_ent = idx_arr[i + delay]
  L148:         t_exit = idx_arr[i + delay + N - 1]
  L149:         elig_tkrs = elig_arr.columns[elig_arr.loc[t]]
  L150:         rets_T_base = []
  L151:         for tk2 in elig_tkrs:
  L152:             o2 = op_arr.at[t_ent, tk2]; c2 = cl_arr.at[t_exit, tk2]
  L153:             if pd.isna(o2) or pd.isna(c2): continue
  L154:             rb2 = (c2*(1-cost_side) - o2*(1+cost_side)) / (o2*(1+cost_side))
  L155:             rets_T_b

In [ ]:
# ============================================================
# R11 — Patch build_trades_and_excess: hitung rets_T per TRADE
# (bukan per T) agar cocok dengan Tahap 3 asli
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

OLD = '''    for t in sig_df.index:
        try: i = idx_arr.get_loc(t)
        except KeyError: continue
        if i + delay + N - 1 >= len(idx_arr): continue
        t_ent = idx_arr[i + delay]
        t_exit = idx_arr[i + delay + N - 1]
        elig_tkrs = elig_arr.columns[elig_arr.loc[t]]
        rets_T_base = []
        for tk2 in elig_tkrs:
            o2 = op_arr.at[t_ent, tk2]; c2 = cl_arr.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rb2 = (c2*(1-cost_side) - o2*(1+cost_side)) / (o2*(1+cost_side))
            rets_T_base.append(rb2)
        if not rets_T_base: continue
        all_rets_pool.extend(rets_T_base)
        for tkr in sig_df.columns:
            if not bool(sig_df.at[t, tkr]): continue
            o_tr = op_arr.at[t_ent, tkr]; c_tr = cl_arr.at[t_exit, tkr]
            c_sig = cl_arr.at[t, tkr]
            if pd.isna(o_tr) or pd.isna(c_tr): continue
            if drop_gap and pd.notna(c_sig) and c_sig > 0:
                gap = o_tr / c_sig - 1
                if gap >= 0.10: continue
            rb_tr = (c_tr*(1-cost_side) - o_tr*(1+cost_side)) / (o_tr*(1+cost_side))
            trade_records.append({
                "t": t, "tkr": tkr, "rb": rb_tr,
                "t_ent": t_ent, "t_exit": t_exit,
                "rets_T_base": rets_T_base,
            })
            dates.append(t)'''

NEW = '''    # PATCH: bangun sig_list (tkr, t) dulu, lalu loop PER TRADE
    # agar rets_T dihitung per trade (duplikat pool), sama dengan Tahap 3 asli
    sig_list = []
    for t in sig_df.index:
        try: i = idx_arr.get_loc(t)
        except KeyError: continue
        if i + delay + N - 1 >= len(idx_arr): continue
        for tkr in sig_df.columns:
            if bool(sig_df.at[t, tkr]):
                sig_list.append((tkr, t))

    for tkr, t in sig_list:
        try: i = idx_arr.get_loc(t)
        except KeyError: continue
        if i + delay + N - 1 >= len(idx_arr): continue
        t_ent = idx_arr[i + delay]
        t_exit = idx_arr[i + delay + N - 1]
        # Kumpulkan rets_T SETIAP TRADE (duplikat kalau T sama)
        elig_tkrs = elig_arr.columns[elig_arr.loc[t]]
        rets_T_base = []
        for tk2 in elig_tkrs:
            o2 = op_arr.at[t_ent, tk2]; c2 = cl_arr.at[t_exit, tk2]
            if pd.isna(o2) or pd.isna(c2): continue
            rb2 = (c2*(1-cost_side) - o2*(1+cost_side)) / (o2*(1+cost_side))
            rets_T_base.append(rb2)
        if not rets_T_base: continue
        all_rets_pool.extend(rets_T_base)   # <-- extend per trade (duplikat)
        o_tr = op_arr.at[t_ent, tkr]; c_tr = cl_arr.at[t_exit, tkr]
        c_sig = cl_arr.at[t, tkr]
        if pd.isna(o_tr) or pd.isna(c_tr): continue
        if drop_gap and pd.notna(c_sig) and c_sig > 0:
            gap = o_tr / c_sig - 1
            if gap >= 0.10: continue
        rb_tr = (c_tr*(1-cost_side) - o_tr*(1+cost_side)) / (o_tr*(1+cost_side))
        trade_records.append({
            "t": t, "tkr": tkr, "rb": rb_tr,
            "t_ent": t_ent, "t_exit": t_exit,
            "rets_T_base": rets_T_base,
        })
        dates.append(t)'''

if OLD in content:
    content = content.replace(OLD, NEW, 1)
    print("✅ Patch: rets_T dihitung per trade")
else:
    print("⚠️ Blok lama tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print(f"  File: {os.path.getsize(SCRIPT_PATH)} byte")

with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'PATCH: bangun sig_list' muncul: {new.count('PATCH: bangun sig_list')}x")

try:
    ast.parse(new)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch: rets_T dihitung per trade

  File: 49815 byte
  'PATCH: bangun sig_list' muncul: 1x
  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T03:19:37

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
  Warm-up cutoff sinyal: T >= 2021-09-22 (bar ke-260)
  Setelah warm-up 260 (di-skip, fitur sudah NaN): 1385 bar (semua bar efektif dipakai)
  K1: 2019-10-01 → 2022-07-20 (461 bar)
  K2: 2022-07-21 → 2024-06-24 (461 bar)
  K3: 2024-06-25 → 2026-06-05 (463 bar)

  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)
  Blok      hasil exc_w         target   cocok?     hasil str_w         target   cocok?       n target   cocok?
--------------------------------------------------------------------------------------------------------------
  K1           +5.3670%       +5.3670%    COCOK        +5.3509%       +3.5350%     BEDA    1099   1099    COCOK
  K2           +1.6354%       +1.6350%    COCOK        +1.6305%       +0.2000%     BEDA    2426   2426    COCOK
  K3           +2.1743%      

In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
p3b = f"{BASE}/Tahap3b_Ketahanan.py"

with open(p3b) as f:
    content = f.read()
lines = content.splitlines()

print("=" * 100)
print("  CARI stress_w DI Tahap 3b")
print("=" * 100)
for i, line in enumerate(lines):
    if "stress_w" in line and "=" in line and "mean" not in line:
        print(f"  L{i+1}: {line}")
    elif "COST_STRESS_SIDE" in line:
        print(f"  L{i+1}: {line}")

# Tampilkan blok Langkah 0 (sekitar hitung stress)
print()
print("  Blok Langkah 0 (sekitar COST_STRESS):")
for i, line in enumerate(lines):
    if "LANGKAH 0" in line and "#" in line:
        for j in range(i, min(i+50, len(lines))):
            print(f"  L{j+1}: {lines[j]}")
        break

  CARI stress_w DI Tahap 3b
  L41: COST_STRESS_SIDE  = 0.0030
  L252:                                                        N=20, cost_side=COST_STRESS_SIDE,
  L254:     res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
  L260:     ok_str = abs(stress_w - tgt["stress_w"]) <= TOL_L0_PP
  L318:             N=N, cost_side=COST_STRESS_SIDE, delay=delay, drop_gap=drop_gap)
  L319:         res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
  L458: str_vals_k2 = [out_R5_all["K2"]["stress_w"]] * max(1, len(out_R5_all["K2"]["exc_w_list"]))
  L459: str_vals_k3 = [out_R5_all["K3"]["stress_w"]] * max(1, len(out_R5_all["K3"]["exc_w_list"]))
  L733:                     rets_T_s.append((c2*(1-COST_STRESS_SIDE) - o2*(1+COST_STRESS_SIDE)) / (o2*(1+COST_STRESS_SIDE)))
  L740:                     rb = (c_tr*(1-COST_STRESS_SIDE) - o_tr*(1+COST_STRESS_SIDE)) / (o_tr*(1+COST_STRESS_SIDE))
  L992:                                              N=20, cost_side=C

In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
p3b = f"{BASE}/Tahap3b_Ketahanan.py"

with open(p3b) as f:
    content = f.read()
lines = content.splitlines()

print("=" * 100)
print("  FUNGSI build_trades_and_excess FULL")
print("=" * 100)
for i, line in enumerate(lines):
    if "def build_trades_and_excess" in line:
        for j in range(i, min(i+45, len(lines))):
            print(f"  L{j+1}: {lines[j]}")
        break

# Cek apakah compute_excess_w pakai cost_side
print()
print("=" * 100)
print("  FUNGSI compute_excess_w FULL")
print("=" * 100)
for i, line in enumerate(lines):
    if "def compute_excess_w" in line:
        for j in range(i, min(i+30, len(lines))):
            print(f"  L{j+1}: {lines[j]}")
        break

  FUNGSI build_trades_and_excess FULL
  L137: def build_trades_and_excess(idx_arr, op_arr, cl_arr, elig_arr, sig_df,
  L138:                             N=20, cost_side=COST_BASE_SIDE,
  L139:                             delay=1, drop_gap=False):
  L140:     trade_records = []
  L141:     all_rets_pool = []
  L142:     dates = []
  L143:     # PATCH: bangun sig_list (tkr, t) dulu, lalu loop PER TRADE
  L144:     # agar rets_T dihitung per trade (duplikat pool), sama dengan Tahap 3 asli
  L145:     sig_list = []
  L146:     for t in sig_df.index:
  L147:         try: i = idx_arr.get_loc(t)
  L148:         except KeyError: continue
  L149:         if i + delay + N - 1 >= len(idx_arr): continue
  L150:         for tkr in sig_df.columns:
  L151:             if bool(sig_df.at[t, tkr]):
  L152:                 sig_list.append((tkr, t))
  L153: 
  L154:     for tkr, t in sig_list:
  L155:         try: i = idx_arr.get_loc(t)
  L156:         except KeyError: continue
  L157:         if i + dela

In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
p3 = f"{BASE}/Tahap3_Horizon20.py"

with open(p3) as f:
    lines = f.readlines()

print("=" * 100)
print("  CARI BARIS STRESS_w DI TAHAP 3 ASLI")
print("=" * 100)
for i, line in enumerate(lines):
    if "stress_w" in line or "COST_STRESS" in line or "res_s" in line:
        print(f"  L{i+1}: {line.rstrip()}")

# Tampilkan blok lengkap di sekitar run_signal
print()
print("  Blok sekitar run_signal (30 baris setelah L648):")
for i, line in enumerate(lines):
    if "def run_signal" in line:
        for j in range(i, min(i+100, len(lines))):
            print(f"  L{j+1}: {lines[j].rstrip()}")
        break

  CARI BARIS STRESS_w DI TAHAP 3 ASLI
  L44: COST_STRESS_SIDE  = 0.0030
  L669:             rs = trade_return(tkr, t, N, COST_STRESS_SIDE)
  L747:             "mean_stress_w":   float(np.mean(winsorize(raw_s_arr))) if len(raw_s_arr) else float("nan"),
  L870:     st_n = sum([r_k1["mean_stress_w"]>0, r_k2["mean_stress_w"]>0, r_k3["mean_stress_w"]>0])
  L1000:     st_n = sum([r_k1["mean_stress_w"]>0, r_k2["mean_stress_w"]>0, r_k3["mean_stress_w"]>0])
  L1011:     report += f"{r_k1['mean_stress_raw']*100:+.3f}%|{r_k1['mean_stress_w']*100:+.3f}% | "
  L1012:     report += f"{r_k2['mean_stress_raw']*100:+.3f}%|{r_k2['mean_stress_w']*100:+.3f}% | "
  L1013:     report += f"{r_k3['mean_stress_raw']*100:+.3f}%|{r_k3['mean_stress_w']*100:+.3f}%\n"

  Blok sekitar run_signal (30 baris setelah L648):
  L648: def run_signal(sig_name, sig_df, N=20):
  L649:     """ADDENDUM 8: hits excess_w (baru) dan excess_w_lama (pembanding)."""
  L650:     results = {}
  L651:     daily_pool = []
  L652:     all

In [ ]:
# ============================================================
# R13 — Patch: str_w = mean(winsorize(return_stress)), bukan excess
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, ast

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
SCRIPT_PATH = f"{BASE}/Tahap3b_Ketahanan.py"

with open(SCRIPT_PATH) as f:
    content = f.read()

# Patch 1: compute_excess_w — tambah return_stress_list
OLD_COMPUTE = '''def compute_excess_w(trade_records, all_rets_pool, cost_side=COST_BASE_SIDE):
    if not all_rets_pool or not trade_records:
        return {"n_trades": 0, "mean_exc_w": float("nan"),
                "mean_exc_w_lama": float("nan"),
                "median_exc": float("nan"),
                "dates": [], "exc_w_list": []}
    lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
    exc_w_list = []
    exc_w_lama_list = []
    for rec in trade_records:
        rb = rec["rb"]
        rets_T = rec["rets_T_base"]
        bm_w = float(np.mean(np.clip(rets_T, lo, hi)))
        rb_w = float(np.clip(rb, lo, hi))
        exc_w = rb_w - bm_w
        exc_w_list.append(exc_w)
        bm_raw = float(np.mean(rets_T))
        exc_w_lama_list.append(rb - bm_raw)
    exc_w_arr = np.array(exc_w_list)
    return {
        "n_trades": len(trade_records),
        "mean_exc_w": float(np.mean(exc_w_arr)),
        "mean_exc_w_lama": float(np.mean(winsorize(np.array(exc_w_lama_list)))),
        "median_exc": float(np.median(exc_w_arr)),
        "dates": [r["t"] for r in trade_records],
        "exc_w_list": exc_w_list,
    }'''

NEW_COMPUTE = '''def compute_excess_w(trade_records, all_rets_pool, cost_side=COST_BASE_SIDE):
    if not all_rets_pool or not trade_records:
        return {"n_trades": 0, "mean_exc_w": float("nan"),
                "mean_exc_w_lama": float("nan"),
                "mean_stress_w": float("nan"),
                "median_exc": float("nan"),
                "dates": [], "exc_w_list": [],
                "stress_w_list": []}
    lo, hi = np.percentile(all_rets_pool, 5), np.percentile(all_rets_pool, 95)
    exc_w_list = []
    exc_w_lama_list = []
    rb_raw_list = []   # return mentah trade (untuk stress_w)
    for rec in trade_records:
        rb = rec["rb"]
        rets_T = rec["rets_T_base"]
        bm_w = float(np.mean(np.clip(rets_T, lo, hi)))
        rb_w = float(np.clip(rb, lo, hi))
        exc_w = rb_w - bm_w
        exc_w_list.append(exc_w)
        bm_raw = float(np.mean(rets_T))
        exc_w_lama_list.append(rb - bm_raw)
        rb_raw_list.append(rb)
    exc_w_arr = np.array(exc_w_list)
    # PATCH: stress_w = mean(winsorize(return mentah)), sama dengan Tahap 3 asli
    stress_w_arr = winsorize(np.array(rb_raw_list))
    return {
        "n_trades": len(trade_records),
        "mean_exc_w": float(np.mean(exc_w_arr)),
        "mean_exc_w_lama": float(np.mean(winsorize(np.array(exc_w_lama_list)))),
        "mean_stress_w": float(np.mean(stress_w_arr)),
        "median_exc": float(np.median(exc_w_arr)),
        "dates": [r["t"] for r in trade_records],
        "exc_w_list": exc_w_list,
        "stress_w_list": stress_w_arr.tolist(),
    }'''

if OLD_COMPUTE in content:
    content = content.replace(OLD_COMPUTE, NEW_COMPUTE, 1)
    print("✅ Patch 1: compute_excess_w ditambah stress_w")
else:
    print("⚠️ Patch 1: compute_excess_w tidak cocok persis")

# Patch 2: di Langkah 0, ambil stress_w dari res (base) — karena res sudah punya
OLD_L0 = '''    # STRESS
    trades_s, all_rets_s, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                                       N=20, cost_side=COST_STRESS_SIDE,
                                                       delay=1, drop_gap=False)
    res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
    exc_w = res["mean_exc_w"]
    stress_w = res_s["mean_exc_w"]'''

NEW_L0 = '''    # STRESS — pakai biaya stress, tapi definisi stress_w = return mentah stress
    trades_s, all_rets_s, _ = build_trades_and_excess(idx, op_b, cl_b, elig_b, sig_blk,
                                                       N=20, cost_side=COST_STRESS_SIDE,
                                                       delay=1, drop_gap=False)
    res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
    exc_w = res["mean_exc_w"]
    stress_w = res_s["mean_stress_w"]   # <-- pakai mean_stress_w (return mentah)'''

if OLD_L0 in content:
    content = content.replace(OLD_L0, NEW_L0, 1)
    print("✅ Patch 2: Langkah 0 stress_w = mean_stress_w")
else:
    print("⚠️ Patch 2: blok Langkah 0 tidak cocok persis")

# Patch 3: di jalankan_3_blok — ambil stress_w dari mean_stress_w
OLD_3BLK = '''        trades_s, all_rets_s, _ = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=COST_STRESS_SIDE, delay=delay, drop_gap=drop_gap)
        res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
        out[blk_id]["stress_w"] = res_s["mean_exc_w"]
        all_stress_pool.extend(res_s["exc_w_list"])'''

NEW_3BLK = '''        trades_s, all_rets_s, _ = build_trades_and_excess(
            idx_arr, op_arr, cl_arr, elig_use, sig_blk,
            N=N, cost_side=COST_STRESS_SIDE, delay=delay, drop_gap=drop_gap)
        res_s = compute_excess_w(trades_s, all_rets_s, cost_side=COST_STRESS_SIDE)
        out[blk_id]["stress_w"] = res_s["mean_stress_w"]
        all_stress_pool.extend(res_s["stress_w_list"])'''

if OLD_3BLK in content:
    content = content.replace(OLD_3BLK, NEW_3BLK, 1)
    print("✅ Patch 3: jalankan_3_blok pakai mean_stress_w")
else:
    print("⚠️ Patch 3: blok jalankan_3_blok tidak cocok persis")

with open(SCRIPT_PATH, "w") as f:
    f.write(content)

print()
print(f"  File: {os.path.getsize(SCRIPT_PATH)} byte")
with open(SCRIPT_PATH) as f:
    new = f.read()
print(f"  'mean_stress_w' muncul: {new.count('mean_stress_w')}x")

try:
    ast.parse(new)
    print("  ✅ Syntax OK")
except SyntaxError as e:
    print(f"  ❌ Syntax ERROR: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Patch 1: compute_excess_w ditambah stress_w
✅ Patch 2: Langkah 0 stress_w = mean_stress_w
✅ Patch 3: jalankan_3_blok pakai mean_stress_w

  File: 50351 byte
  'mean_stress_w' muncul: 15x
  ✅ Syntax OK


In [ ]:
!python "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/Tahap3b_Ketahanan.py"

[WARN] Mount Drive: 'NoneType' object has no attribute 'kernel'
  KOMPAS TAHAP 3b — Tahap3b_Ketahanan.py
  Waktu: 2026-10-14T03:30:06

  LOAD DATA PANJANG + PREPROCESSING
  Data: (1385, 561), range 2019-10-01 → 2026-06-05
  Warm-up cutoff sinyal: T >= 2021-09-22 (bar ke-260)
  Setelah warm-up 260 (di-skip, fitur sudah NaN): 1385 bar (semua bar efektif dipakai)
  K1: 2019-10-01 → 2022-07-20 (461 bar)
  K2: 2022-07-21 → 2024-06-24 (461 bar)
  K3: 2024-06-25 → 2026-06-05 (463 bar)

  LANGKAH 0 — REPRODUKSI (NEAR52H N=20)
  Blok      hasil exc_w         target   cocok?     hasil str_w         target   cocok?       n target   cocok?
--------------------------------------------------------------------------------------------------------------
  K1           +5.3670%       +5.3670%    COCOK        +3.5349%       +3.5350%    COCOK    1099   1099    COCOK
  K2           +1.6354%       +1.6350%    COCOK        +0.1959%       +0.2000%    COCOK    2426   2426    COCOK
  K3           +2.1743%      

In [ ]:
import os
BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
REPORT = f"{BASE}/LAPORAN_Tahap3b.md"

print("=" * 100)
print("  ISI LAPORAN_Tahap3b.md")
print("=" * 100)
with open(REPORT) as f:
    print(f.read())
print("=" * 100)

  ISI LAPORAN_Tahap3b.md
TAHAP 3b - LAPORAN
Tanggal run / data terakhir    : 2026-10-14T03:39:19 / 2026-06-05
Nama file skrip final          : Tahap3b_Ketahanan.py
LANGKAH 0: COCOK
Putaran perbaikan (maks 5)     : 0

UJI KETAHANAN R1-R6
------------------------------------------------------------

R1 — Penundaan 1 hari
  K1: n=1098 exc_w=+5.328% str_w=+3.304%
  K2: n=2426 exc_w=+1.626% str_w=+0.192%
  K3: n=3747 exc_w=+2.043% str_w=+2.064%
  pool: exc_w=+2.608% t=+4.658 p=0.0000 p1sisi=0.0000 str_w=+1.627% n=7271
  Hasil: LOLOS

R2 — Buang trade gap-up >= 10%
  K1: n=1087 exc_w=+5.344% str_w=+3.498%
  K2: n=2423 exc_w=+1.636% str_w=+0.202%
  K3: n=3658 exc_w=+2.359% str_w=+2.652%
  pool: exc_w=+2.620% t=+4.622 p=0.0000 p1sisi=0.0000 str_w=+1.952% n=7168
  Hasil: LOLOS

R3 — Biaya 1.0% round-trip
  K1: n=1099 return_w=+5.330%
  K2: n=2426 return_w=+1.624%
  K3: n=3750 return_w=+2.159%
  pool: return_w=+2.506% n=7275
  Blok return > 0: 3/3
  Hasil: LOLOS

R4 — Hanya 50 saham teratas
  K1

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, hashlib, json, shutil
from datetime import datetime

BASE = "/content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3"
ARCHIVE = f"{BASE}/_archive_tahap3b_{datetime.now().strftime('%Y%m%d_%H%M')}"
os.makedirs(ARCHIVE, exist_ok=True)

def sha256_file(p, chunk=1<<20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()

files = [
    "Tahap3b_Ketahanan.py",
    "LAPORAN_Tahap3b.md",
    "LAPORAN_Tahap3.md",
    "hasil_tahap3.pkl",
    "data_long_sealed.pkl",
    "data_long_manifest.json",
]

manifest = {"archived_at": datetime.now().isoformat(timespec="seconds"),
            "folder": ARCHIVE, "files": {}}

print("=" * 100)
print("  ARSIP TAHAP 3b")
print("=" * 100)
print(f"  Folder: {ARCHIVE}")
print()

for name in files:
    src = f"{BASE}/{name}"
    if not os.path.exists(src):
        print(f"  ⚠️  {name}: tidak ada, skip"); continue
    dst = f"{ARCHIVE}/{name}"
    shutil.copy2(src, dst)
    size = os.path.getsize(dst)
    sha = sha256_file(dst)
    manifest["files"][name] = {"size": size, "sha256": sha}
    print(f"  ✅ {name:35s} {size:>12,} byte  {sha[:16]}…")

json.dump(manifest, open(f"{ARCHIVE}/_ARCHIVE_MANIFEST.json", "w"), indent=2)
print()
print(f"  ✅ Arsip selesai: {len(manifest['files'])} file")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ARSIP TAHAP 3b
  Folder: /content/drive/MyDrive/3 Anthropy_AladdinFoundry_Gen3/_archive_tahap3b_20261014_0347

  ✅ Tahap3b_Ketahanan.py                      50,351 byte  fceb5d2d84b5fbd0…
  ✅ LAPORAN_Tahap3b.md                         3,959 byte  2dacdac2a2b63a2b…
  ✅ LAPORAN_Tahap3.md                          6,063 byte  76bab8a948508585…
  ✅ hasil_tahap3.pkl                           2,922 byte  be86b76687e24e0a…
  ✅ data_long_sealed.pkl                  31,161,317 byte  956221434b81744b…
  ✅ data_long_manifest.json                      245 byte  c927d49e83bf051b…

  ✅ Arsip selesai: 6 file
